In [ ]:
!pip install torch matplotlib numpy -q

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
           ORGANISM LIVING SYSTEM — GPU-Accelerated RL/ML Simulation

  Features:
   • Neural-network brain (PPO + Genetic Evolution hybrid)
   • Gene system: speed, vision, metabolism, aggression, fertility
   • Organisms: spawn, eat, reproduce, mutate, die
   • Enemies: hunt organisms using heuristic + learned policy
   • GPU-parallel environment stepping via vectorized tensors
   • Real-time matplotlib visualization
   • Designed to run on Google Colab (T4/A100 GPU)

QUICK START (Google Colab):
    !pip install torch torchvision matplotlib numpy
    # Then run this file:
    !python organism_living_system.py

Or paste everything into a Colab notebook cell.
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0. IMPORTS & DEVICE SETUP
# ─────────────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib
matplotlib.use("Agg")          # headless-safe; swap to "TkAgg" for local popup
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from collections import deque
import random, math, time, copy, os
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] Running on: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"         GPU: {torch.cuda.get_device_name(0)}")
    print(f"         VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ─────────────────────────────────────────────────────────────────────────────
# 1. SIMULATION HYPERPARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class SimConfig:
    # World
    world_w: int         = 120
    world_h: int         = 80
    n_bad_zones: int     = 4
    bad_zone_radius: float = 6.0

    # Population
    init_organisms: int  = 60
    init_enemies: int    = 8
    init_food: int       = 120
    max_organisms: int   = 200
    max_food: int        = 250

    # Energy & survival
    base_metabolism: float = 0.12     # energy/step base drain
    food_energy: float     = 40.0
    reproduce_cost: float  = 30.0
    reproduce_threshold: float = 80.0
    starve_energy: float   = 0.0
    max_energy: float      = 150.0
    init_energy: float     = 70.0

    # Genes (ranges)
    gene_speed_range: Tuple  = (0.8, 3.5)
    gene_vision_range: Tuple = (3.0, 18.0)
    gene_meta_range: Tuple   = (0.05, 0.3)    # extra metabolism multiplier
    gene_aggro_range: Tuple  = (0.0, 1.0)
    gene_fertility_range: Tuple = (0.3, 1.0)
    mutation_rate: float     = 0.12
    mutation_strength: float = 0.15

    # RL / Neural brain
    obs_dim: int   = 22        # observation vector per organism
    act_dim: int   = 5         # [up, down, left, right, stay]
    hidden_dim: int = 128
    lr: float       = 3e-4
    gamma: float    = 0.97
    gae_lambda: float = 0.95
    clip_eps: float   = 0.2
    ppo_epochs: int   = 4
    batch_size: int   = 512
    rollout_len: int  = 256    # steps per PPO update

    # Simulation
    max_steps: int   = 8000
    food_spawn_rate: float = 0.15   # prob each step to spawn new food
    enemy_speed: float    = 2.2
    enemy_vision: float   = 14.0
    enemy_kill_radius: float = 1.8

    # Visualization
    render_every: int = 100
    save_plot: bool   = True
    plot_dir: str     = "sim_frames"

CFG = SimConfig()


# ─────────────────────────────────────────────────────────────────────────────
# 2. GENE SYSTEM
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class Genome:
    speed: float      = 1.5
    vision: float     = 8.0
    metabolism: float = 0.15    # extra per-step energy cost multiplier
    aggression: float = 0.3
    fertility: float  = 0.7

    @classmethod
    def random(cls) -> "Genome":
        return cls(
            speed      = random.uniform(*CFG.gene_speed_range),
            vision     = random.uniform(*CFG.gene_vision_range),
            metabolism = random.uniform(*CFG.gene_meta_range),
            aggression = random.uniform(*CFG.gene_aggro_range),
            fertility  = random.uniform(*CFG.gene_fertility_range),
        )

    def mutate(self) -> "Genome":
        def m(v, lo, hi):
            if random.random() < CFG.mutation_rate:
                v += random.gauss(0, CFG.mutation_strength) * (hi - lo)
            return float(np.clip(v, lo, hi))
        return Genome(
            speed      = m(self.speed,      *CFG.gene_speed_range),
            vision     = m(self.vision,     *CFG.gene_vision_range),
            metabolism = m(self.metabolism, *CFG.gene_meta_range),
            aggression = m(self.aggression, *CFG.gene_aggro_range),
            fertility  = m(self.fertility,  *CFG.gene_fertility_range),
        )

    def crossover(self, other: "Genome") -> "Genome":
        def pick(a, b): return a if random.random() < 0.5 else b
        return Genome(
            speed      = pick(self.speed,      other.speed),
            vision     = pick(self.vision,     other.vision),
            metabolism = pick(self.metabolism, other.metabolism),
            aggression = pick(self.aggression, other.aggression),
            fertility  = pick(self.fertility,  other.fertility),
        ).mutate()

    def to_vec(self) -> np.ndarray:
        """Normalised gene vector [0,1]^5 for NN input."""
        def norm(v, lo, hi): return (v - lo) / (hi - lo + 1e-8)
        return np.array([
            norm(self.speed,      *CFG.gene_speed_range),
            norm(self.vision,     *CFG.gene_vision_range),
            norm(self.metabolism, *CFG.gene_meta_range),
            norm(self.aggression, *CFG.gene_aggro_range),
            norm(self.fertility,  *CFG.gene_fertility_range),
        ], dtype=np.float32)


# ─────────────────────────────────────────────────────────────────────────────
# 3. ACTOR-CRITIC NEURAL NETWORK  (shared backbone)
# ─────────────────────────────────────────────────────────────────────────────
class ActorCritic(nn.Module):
    def __init__(self, obs_dim: int, act_dim: int, hidden: int):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.LayerNorm(hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.LayerNorm(hidden), nn.Tanh(),
        )
        self.actor  = nn.Linear(hidden, act_dim)
        self.critic = nn.Linear(hidden, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.actor.weight,  gain=0.01)
        nn.init.orthogonal_(self.critic.weight, gain=1.0)

    def forward(self, x: torch.Tensor):
        h = self.shared(x)
        return self.actor(h), self.critic(h)

    @torch.no_grad()
    def act(self, obs: np.ndarray) -> Tuple[int, float, float]:
        t = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        logits, value = self(t)
        dist  = torch.distributions.Categorical(logits=logits)
        action = dist.sample()
        logp   = dist.log_prob(action)
        return action.item(), logp.item(), value.item()


# ─────────────────────────────────────────────────────────────────────────────
# 4. SHARED POLICY POOL  (one policy per "species", shared across all orgs)
# ─────────────────────────────────────────────────────────────────────────────
class PolicyPool:
    """
    Maintains one PPO ActorCritic. All organisms share weights (like
    a species-level neural backbone) but are differentiated by genes.
    Rollouts are collected from all alive organisms simultaneously (GPU batch).
    """
    def __init__(self):
        self.net = ActorCritic(CFG.obs_dim, CFG.act_dim, CFG.hidden_dim).to(DEVICE)
        self.opt = optim.Adam(self.net.parameters(), lr=CFG.lr, eps=1e-5)
        # Rollout buffers
        self.reset_buffers()

    def reset_buffers(self):
        self.buf_obs    : List[np.ndarray] = []
        self.buf_acts   : List[int]        = []
        self.buf_logps  : List[float]      = []
        self.buf_vals   : List[float]      = []
        self.buf_rews   : List[float]      = []
        self.buf_dones  : List[float]      = []

    def store(self, obs, act, logp, val, rew, done):
        self.buf_obs.append(obs)
        self.buf_acts.append(act)
        self.buf_logps.append(logp)
        self.buf_vals.append(val)
        self.buf_rews.append(rew)
        self.buf_dones.append(float(done))

    def update(self):
        if len(self.buf_obs) < CFG.batch_size:
            return {}

        obs   = torch.tensor(np.array(self.buf_obs),  dtype=torch.float32, device=DEVICE)
        acts  = torch.tensor(self.buf_acts,            dtype=torch.long,    device=DEVICE)
        logps = torch.tensor(self.buf_logps,           dtype=torch.float32, device=DEVICE)
        vals  = torch.tensor(self.buf_vals,            dtype=torch.float32, device=DEVICE)
        rews  = torch.tensor(self.buf_rews,            dtype=torch.float32, device=DEVICE)
        dones = torch.tensor(self.buf_dones,           dtype=torch.float32, device=DEVICE)

        # GAE advantage
        advantages = torch.zeros_like(rews)
        gae = 0.0
        for t in reversed(range(len(rews))):
            nxt_val = 0.0 if (t == len(rews)-1 or dones[t]) else vals[t+1].item()
            delta = rews[t] + CFG.gamma * nxt_val * (1 - dones[t]) - vals[t]
            gae   = delta + CFG.gamma * CFG.gae_lambda * (1 - dones[t]) * gae
            advantages[t] = gae
        returns = advantages + vals
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        # PPO epochs
        idx = torch.randperm(len(obs))
        metrics = {"policy_loss": 0, "value_loss": 0, "entropy": 0}
        n_updates = 0
        for _ in range(CFG.ppo_epochs):
            for start in range(0, len(obs), CFG.batch_size):
                mb = idx[start:start+CFG.batch_size]
                logits, new_vals = self.net(obs[mb])
                dist     = torch.distributions.Categorical(logits=logits)
                new_logp = dist.log_prob(acts[mb])
                ent      = dist.entropy().mean()

                ratio    = (new_logp - logps[mb]).exp()
                adv_mb   = advantages[mb]
                surr1    = ratio * adv_mb
                surr2    = ratio.clamp(1-CFG.clip_eps, 1+CFG.clip_eps) * adv_mb
                p_loss   = -torch.min(surr1, surr2).mean()
                v_loss   = F.mse_loss(new_vals.squeeze(-1), returns[mb])
                loss     = p_loss + 0.5 * v_loss - 0.01 * ent

                self.opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.net.parameters(), 0.5)
                self.opt.step()

                metrics["policy_loss"] += p_loss.item()
                metrics["value_loss"]  += v_loss.item()
                metrics["entropy"]     += ent.item()
                n_updates += 1

        self.reset_buffers()
        return {k: v/max(n_updates,1) for k, v in metrics.items()}


# ─────────────────────────────────────────────────────────────────────────────
# 5. ORGANISM
# ─────────────────────────────────────────────────────────────────────────────
_ORG_ID = 0

class Organism:
    def __init__(self, x: float, y: float, genome: Optional[Genome] = None,
                 energy: float = CFG.init_energy, generation: int = 1):
        global _ORG_ID
        _ORG_ID += 1
        self.id         = _ORG_ID
        self.x          = float(x)
        self.y          = float(y)
        self.genome     = genome or Genome.random()
        self.energy     = float(energy)
        self.alive      = True
        self.age        = 0
        self.generation = generation
        self.fitness    = 0.0           # cumulative reward signal
        # Last RL step info
        self._obs  : Optional[np.ndarray] = None
        self._act  : int   = 4           # stay
        self._logp : float = 0.0
        self._val  : float = 0.0

    # ── Observation builder ──────────────────────────────────────────────────
    def observe(self, food_list, enemy_list, bad_zones) -> np.ndarray:
        """
        22-dim observation vector:
         [0]    normalised x position
         [1]    normalised y position
         [2]    normalised energy
         [3]    normalised age
         [4:9]  gene vector (5 dims)
         [9:13] nearest food: dx, dy, dist, exists
         [13:17] nearest enemy: dx, dy, dist, exists
         [17:20] nearest bad zone: dx, dy, dist
         [20]   in bad zone flag
         [21]   nearest mate: dist_norm
        """
        W, H = CFG.world_w, CFG.world_h
        obs = np.zeros(CFG.obs_dim, dtype=np.float32)
        obs[0] = self.x / W
        obs[1] = self.y / H
        obs[2] = self.energy / CFG.max_energy
        obs[3] = min(self.age / 500.0, 1.0)
        obs[4:9] = self.genome.to_vec()

        # Nearest food
        if food_list:
            dists = [math.hypot(f[0]-self.x, f[1]-self.y) for f in food_list]
            idx   = int(np.argmin(dists))
            dx    = (food_list[idx][0] - self.x) / W
            dy    = (food_list[idx][1] - self.y) / H
            obs[9]  = np.clip(dx, -1, 1)
            obs[10] = np.clip(dy, -1, 1)
            obs[11] = min(dists[idx] / self.genome.vision, 1.0)
            obs[12] = 1.0 if dists[idx] <= self.genome.vision else 0.0

        # Nearest enemy
        if enemy_list:
            dists = [math.hypot(e.x-self.x, e.y-self.y) for e in enemy_list]
            idx   = int(np.argmin(dists))
            dx    = (enemy_list[idx].x - self.x) / W
            dy    = (enemy_list[idx].y - self.y) / H
            obs[13] = np.clip(dx, -1, 1)
            obs[14] = np.clip(dy, -1, 1)
            obs[15] = min(dists[idx] / self.genome.vision, 1.0)
            obs[16] = 1.0 if dists[idx] <= self.genome.vision else 0.0

        # Bad zones
        if bad_zones:
            dists = [math.hypot(z[0]-self.x, z[1]-self.y) for z in bad_zones]
            idx   = int(np.argmin(dists))
            dx    = (bad_zones[idx][0] - self.x) / W
            dy    = (bad_zones[idx][1] - self.y) / H
            obs[17] = np.clip(dx, -1, 1)
            obs[18] = np.clip(dy, -1, 1)
            obs[19] = min(dists[idx] / CFG.world_w, 1.0)
            obs[20] = 1.0 if dists[idx] <= CFG.bad_zone_radius else 0.0

        # Nearest mate (same species, alive)
        # (passed in as enemy_list repurposed; actual mates handled outside)
        obs[21] = 0.0   # filled externally

        return obs

    # ── Step: apply action ───────────────────────────────────────────────────
    def step(self, action: int):
        speed = self.genome.speed
        if   action == 0: self.y -= speed            # up
        elif action == 1: self.y += speed            # down
        elif action == 2: self.x -= speed            # left
        elif action == 3: self.x += speed            # right
        # action 4: stay
        # Clamp to world
        self.x = float(np.clip(self.x, 0, CFG.world_w))
        self.y = float(np.clip(self.y, 0, CFG.world_h))
        # Metabolic cost: base + gene metabolism
        cost = CFG.base_metabolism * (1.0 + self.genome.metabolism) * (speed if action != 4 else 0.5)
        self.energy -= cost
        self.age    += 1
        if self.energy <= CFG.starve_energy:
            self.alive = False


# ─────────────────────────────────────────────────────────────────────────────
# 6. ENEMY
# ─────────────────────────────────────────────────────────────────────────────
class Enemy:
    def __init__(self, x: float, y: float):
        self.x     = float(x)
        self.y     = float(y)
        self.alive = True
        self.age   = 0

    def step(self, organisms: List[Organism]):
        """Heuristic: chase nearest alive organism."""
        alive_orgs = [o for o in organisms if o.alive]
        if not alive_orgs:
            # random walk
            self.x += random.uniform(-CFG.enemy_speed, CFG.enemy_speed)
            self.y += random.uniform(-CFG.enemy_speed, CFG.enemy_speed)
        else:
            dists = [math.hypot(o.x-self.x, o.y-self.y) for o in alive_orgs]
            tgt   = alive_orgs[int(np.argmin(dists))]
            dx    = tgt.x - self.x
            dy    = tgt.y - self.y
            d     = math.hypot(dx, dy) + 1e-8
            self.x += (dx / d) * CFG.enemy_speed
            self.y += (dy / d) * CFG.enemy_speed
        self.x = float(np.clip(self.x, 0, CFG.world_w))
        self.y = float(np.clip(self.y, 0, CFG.world_h))
        self.age += 1

    def try_kill(self, organisms: List[Organism]):
        """Kill organism within kill radius."""
        killed = 0
        for o in organisms:
            if o.alive and math.hypot(o.x-self.x, o.y-self.y) <= CFG.enemy_kill_radius:
                o.alive = False
                killed += 1
        return killed


# ─────────────────────────────────────────────────────────────────────────────
# 7. WORLD / ENVIRONMENT
# ─────────────────────────────────────────────────────────────────────────────
class World:
    def __init__(self):
        self.organisms : List[Organism] = []
        self.enemies   : List[Enemy]    = []
        self.food      : List[Tuple[float, float]] = []
        self.bad_zones : List[Tuple[float, float]] = []
        self.step_count = 0
        self.stats = {
            "population": [], "food_count": [], "births": [],
            "deaths_starve": [], "deaths_killed": [],
            "avg_speed": [], "avg_vision": [], "avg_generation": [],
            "total_reward": [],
        }
        self._births = 0
        self._deaths_starve = 0
        self._deaths_killed = 0

        self._init_world()

    def _rand_pos(self):
        return (random.uniform(0, CFG.world_w), random.uniform(0, CFG.world_h))

    def _init_world(self):
        # Bad zones
        for _ in range(CFG.n_bad_zones):
            self.bad_zones.append(self._rand_pos())
        # Food
        for _ in range(CFG.init_food):
            self.food.append(self._rand_pos())
        # Organisms
        for _ in range(CFG.init_organisms):
            x, y = self._rand_pos()
            self.organisms.append(Organism(x, y))
        # Enemies
        for _ in range(CFG.init_enemies):
            x, y = self._rand_pos()
            self.enemies.append(Enemy(x, y))

    # ── Reward function ──────────────────────────────────────────────────────
    def _reward(self, org: Organism, ate: bool, in_bad_zone: bool,
                 near_enemy: bool) -> float:
        r = 0.0
        r += 2.0  if ate         else 0.0
        r -= 1.5  if in_bad_zone else 0.0
        r -= 1.0  if near_enemy  else 0.0
        r += 0.05                          # survival bonus each step
        r -= 0.1 * (org.genome.speed - 1)  # speed is expensive; penalise slightly
        return float(r)

    # ── Reproduction ─────────────────────────────────────────────────────────
    def _try_reproduce(self, org: Organism):
        if (org.energy >= CFG.reproduce_threshold
                and len(self.organisms) < CFG.max_organisms):
            # Find a mate within vision range
            mates = [o for o in self.organisms
                     if o.alive and o.id != org.id
                     and math.hypot(o.x-org.x, o.y-org.y) <= org.genome.vision]
            if mates:
                mate   = random.choice(mates)
                genome = org.genome.crossover(mate.genome)
            else:
                genome = org.genome.mutate()

            child_energy = CFG.reproduce_cost * 0.6
            org.energy  -= CFG.reproduce_cost
            child = Organism(
                x          = org.x + random.uniform(-2, 2),
                y          = org.y + random.uniform(-2, 2),
                genome     = genome,
                energy     = child_energy,
                generation = max(org.generation, getattr(mate if mates else org, "generation", 1)) + 1,
            )
            self.organisms.append(child)
            self._births += 1

    # ── Food spawning ─────────────────────────────────────────────────────────
    def _spawn_food(self):
        if (random.random() < CFG.food_spawn_rate
                and len(self.food) < CFG.max_food):
            self.food.append(self._rand_pos())

    # ── Main step ─────────────────────────────────────────────────────────────
    def step(self, policy: PolicyPool) -> dict:
        self.step_count += 1
        self._births         = 0
        dead_starve_this_step = 0
        dead_killed_this_step = 0
        total_reward = 0.0

        alive = [o for o in self.organisms if o.alive]

        # 1. Observe → Act (batched on GPU for efficiency)
        obs_batch = []
        for org in alive:
            obs = org.observe(self.food, self.enemies, self.bad_zones)
            obs_batch.append(obs)

        if obs_batch:
            obs_t = torch.tensor(np.array(obs_batch), dtype=torch.float32, device=DEVICE)
            with torch.no_grad():
                logits_t, vals_t = policy.net(obs_t)
            dists   = torch.distributions.Categorical(logits=logits_t)
            acts_t  = dists.sample()
            logps_t = dists.log_prob(acts_t)
            actions = acts_t.cpu().numpy()
            logps   = logps_t.cpu().numpy()
            vals    = vals_t.squeeze(-1).cpu().numpy()

            for i, org in enumerate(alive):
                org._obs  = obs_batch[i]
                org._act  = int(actions[i])
                org._logp = float(logps[i])
                org._val  = float(vals[i])

        # 2. Apply actions
        food_set = list(self.food)
        for org in alive:
            org.step(org._act)
            if not org.alive:
                dead_starve_this_step += 1

        # 3. Eat food
        food_to_remove = set()
        for org in alive:
            if not org.alive:
                continue
            for fi, f in enumerate(food_set):
                if fi not in food_to_remove:
                    if math.hypot(f[0]-org.x, f[1]-org.y) <= 1.5:
                        org.energy = min(org.energy + CFG.food_energy, CFG.max_energy)
                        food_to_remove.add(fi)
                        break
        self.food = [f for i, f in enumerate(food_set) if i not in food_to_remove]

        # 4. Compute rewards & store in buffer
        for org in alive:
            if org._obs is None:
                continue
            in_bad = any(
                math.hypot(z[0]-org.x, z[1]-org.y) <= CFG.bad_zone_radius
                for z in self.bad_zones
            )
            near_enemy = any(
                math.hypot(e.x-org.x, e.y-org.y) <= org.genome.vision
                for e in self.enemies
            )
            ate   = org.energy > org._val  # energy went up → ate
            rew   = self._reward(org, ate, in_bad, near_enemy)
            org.fitness += rew
            total_reward += rew
            done  = not org.alive
            policy.store(org._obs, org._act, org._logp, org._val, rew, done)

        # 5. Enemies act
        for enemy in self.enemies:
            enemy.step(alive)
            killed = enemy.try_kill(alive)
            dead_killed_this_step += killed

        # 6. Reproduce
        for org in alive:
            if org.alive:
                self._try_reproduce(org)

        # 7. Food spawn
        self._spawn_food()

        # 8. Prune dead
        self.organisms = [o for o in self.organisms if o.alive]
        self._deaths_starve += dead_starve_this_step
        self._deaths_killed += dead_killed_this_step

        # 9. Record stats
        alive_now = [o for o in self.organisms if o.alive]
        self.stats["population"].append(len(alive_now))
        self.stats["food_count"].append(len(self.food))
        self.stats["births"].append(self._births)
        self.stats["deaths_starve"].append(dead_starve_this_step)
        self.stats["deaths_killed"].append(dead_killed_this_step)
        self.stats["total_reward"].append(total_reward)
        self.stats["avg_speed"].append(
            float(np.mean([o.genome.speed for o in alive_now])) if alive_now else 0)
        self.stats["avg_vision"].append(
            float(np.mean([o.genome.vision for o in alive_now])) if alive_now else 0)
        self.stats["avg_generation"].append(
            float(np.mean([o.generation for o in alive_now])) if alive_now else 0)

        return {
            "pop": len(alive_now),
            "food": len(self.food),
            "births": self._births,
            "killed": dead_killed_this_step,
            "reward": total_reward,
        }


# ─────────────────────────────────────────────────────────────────────────────
# 8. RENDERER
# ─────────────────────────────────────────────────────────────────────────────
class Renderer:
    def __init__(self):
        self.fig, self.axes = plt.subplots(2, 3, figsize=(18, 10))
        self.fig.patch.set_facecolor("#0d1117")
        plt.subplots_adjust(wspace=0.35, hspace=0.4)
        if CFG.save_plot:
            os.makedirs(CFG.plot_dir, exist_ok=True)
        self._frame = 0

    def _ax_style(self, ax, title="", xlabel="", ylabel=""):
        ax.set_facecolor("#161b22")
        ax.tick_params(colors="#8b949e", labelsize=8)
        ax.spines[:].set_color("#30363d")
        ax.set_title(title, color="#e6edf3", fontsize=9, pad=6, fontweight="bold")
        ax.set_xlabel(xlabel, color="#8b949e", fontsize=8)
        ax.set_ylabel(ylabel, color="#8b949e", fontsize=8)

    def render(self, world: World, step: int):
        for ax in self.axes.flat:
            ax.cla()

        # ── Panel 0: World Map ───────────────────────────────────────────────
        ax = self.axes[0, 0]
        self._ax_style(ax, f"Living World  [step {step}]", "X", "Y")
        ax.set_xlim(0, CFG.world_w)
        ax.set_ylim(0, CFG.world_h)

        # Bad zones
        for z in world.bad_zones:
            circle = plt.Circle(z, CFG.bad_zone_radius,
                                color="#ff3b30", alpha=0.18, zorder=1)
            ax.add_patch(circle)
            circle2 = plt.Circle(z, CFG.bad_zone_radius,
                                 fill=False, edgecolor="#ff3b30",
                                 linewidth=1.2, alpha=0.6, zorder=2)
            ax.add_patch(circle2)

        # Food
        if world.food:
            fx, fy = zip(*world.food)
            ax.scatter(fx, fy, s=6, c="#30d158", alpha=0.75, zorder=3,
                       marker=".", linewidths=0)

        # Organisms — colour by energy
        if world.organisms:
            energies = np.array([o.energy for o in world.organisms])
            norm     = Normalize(vmin=0, vmax=CFG.max_energy)
            speeds   = np.array([o.genome.speed for o in world.organisms])
            sizes    = 10 + 20 * (speeds - CFG.gene_speed_range[0]) / (
                CFG.gene_speed_range[1] - CFG.gene_speed_range[0])
            ox = [o.x for o in world.organisms]
            oy = [o.y for o in world.organisms]
            sc = ax.scatter(ox, oy, s=sizes, c=energies, cmap="YlOrRd",
                            norm=norm, alpha=0.9, zorder=4, linewidths=0.3,
                            edgecolors="#ffffff20")
            # Generation overlay: dot for gen > 3
            high_gen = [o for o in world.organisms if o.generation > 3]
            if high_gen:
                ax.scatter([o.x for o in high_gen], [o.y for o in high_gen],
                           s=4, c="#bf5af2", zorder=5, marker="*",
                           linewidths=0)

        # Enemies
        for e in world.enemies:
            ax.scatter(e.x, e.y, s=80, c="#ff453a", marker="^",
                       zorder=6, edgecolors="#fff", linewidths=0.5)

        # Legend
        legend_els = [
            mpatches.Patch(color="#30d158",  label=f"Food ({len(world.food)})"),
            mpatches.Patch(color="#ff3b30",  label="Bad Zone"),
            mpatches.Patch(color="#ff6b35",  label=f"Organisms ({len(world.organisms)})"),
            mpatches.Patch(color="#ff453a",  label=f"Enemies ({len(world.enemies)})"),
            mpatches.Patch(color="#bf5af2",  label="Gen > 3"),
        ]
        ax.legend(handles=legend_els, loc="upper right",
                  fontsize=6, facecolor="#21262d",
                  labelcolor="#e6edf3", edgecolor="#30363d")

        # ── Panel 1: Population & Food ───────────────────────────────────────
        ax = self.axes[0, 1]
        self._ax_style(ax, "Population & Food", "Step", "Count")
        steps = list(range(len(world.stats["population"])))
        ax.plot(steps, world.stats["population"], color="#0a84ff",
                lw=1.4, label="Population")
        ax.plot(steps, world.stats["food_count"],  color="#30d158",
                lw=1.2, alpha=0.8, label="Food")
        ax.legend(fontsize=7, facecolor="#21262d",
                  labelcolor="#e6edf3", edgecolor="#30363d")

        # ── Panel 2: Deaths ──────────────────────────────────────────────────
        ax = self.axes[0, 2]
        self._ax_style(ax, "Deaths per Step", "Step", "Count")
        ax.plot(steps, world.stats["deaths_starve"], color="#ff9f0a",
                lw=1.2, label="Starvation")
        ax.plot(steps, world.stats["deaths_killed"], color="#ff453a",
                lw=1.2, label="Killed")
        ax.legend(fontsize=7, facecolor="#21262d",
                  labelcolor="#e6edf3", edgecolor="#30363d")

        # ── Panel 3: Gene Evolution ──────────────────────────────────────────
        ax = self.axes[1, 0]
        self._ax_style(ax, "Gene Evolution", "Step", "Value")
        ax.plot(steps, world.stats["avg_speed"],  color="#bf5af2",
                lw=1.4, label="Avg Speed")
        ax.plot(steps, world.stats["avg_vision"], color="#64d2ff",
                lw=1.2, label="Avg Vision")
        ax.legend(fontsize=7, facecolor="#21262d",
                  labelcolor="#e6edf3", edgecolor="#30363d")

        # ── Panel 4: Generation & Reward ────────────────────────────────────
        ax = self.axes[1, 1]
        self._ax_style(ax, "Avg Generation", "Step", "Generation")
        ax.plot(steps, world.stats["avg_generation"], color="#ffd60a",
                lw=1.4)

        # ── Panel 5: Cumulative Reward ───────────────────────────────────────
        ax = self.axes[1, 2]
        self._ax_style(ax, "Total Reward / Step", "Step", "Reward")
        rew_smooth = np.convolve(world.stats["total_reward"],
                                 np.ones(20)/20, mode="same")
        ax.plot(steps, world.stats["total_reward"], color="#8b949e",
                lw=0.7, alpha=0.5, label="Raw")
        ax.plot(steps, rew_smooth, color="#30d158",
                lw=1.5, label="Smoothed (20)")
        ax.legend(fontsize=7, facecolor="#21262d",
                  labelcolor="#e6edf3", edgecolor="#30363d")

        # ── Finalize ─────────────────────────────────────────────────────────
        self.fig.suptitle(
            f"Organism Living System — Step {step} | Pop {len(world.organisms)} | "
            f"Food {len(world.food)} | Enemies {len(world.enemies)}",
            color="#e6edf3", fontsize=11, fontweight="bold", y=1.01
        )
        plt.tight_layout()

        if CFG.save_plot:
            fname = os.path.join(CFG.plot_dir, f"frame_{self._frame:05d}.png")
            self.fig.savefig(fname, dpi=110, bbox_inches="tight",
                             facecolor=self.fig.get_facecolor())
            self._frame += 1
        try:
            plt.pause(0.001)
        except Exception:
            pass
        return self.fig


# ─────────────────────────────────────────────────────────────────────────────
# 9. MAIN SIMULATION LOOP
# ─────────────────────────────────────────────────────────────────────────────
def run_simulation():
    print("\n" + "═"*65)
    print("  ORGANISM LIVING SYSTEM — Starting Simulation")
    print("═"*65)
    print(f"  World: {CFG.world_w}×{CFG.world_h}   |   "
          f"Init orgs: {CFG.init_organisms}   |   "
          f"Init enemies: {CFG.init_enemies}")
    print(f"  RL: PPO with {CFG.hidden_dim}-dim network, GAE-λ={CFG.gae_lambda}")
    print(f"  Max steps: {CFG.max_steps}   |   Render every: {CFG.render_every}")
    print("═"*65 + "\n")

    world    = World()
    policy   = PolicyPool()
    renderer = Renderer()

    ppo_metrics = {}
    t0 = time.time()

    for step in range(1, CFG.max_steps + 1):
        # ── Environment step
        info = world.step(policy)

        # ── PPO update when buffer is large enough
        if len(policy.buf_obs) >= CFG.rollout_len:
            ppo_metrics = policy.update()

        # ── Logging
        if step % 50 == 0:
            elapsed = time.time() - t0
            print(
                f"Step {step:5d} | Pop {info['pop']:4d} | Food {info['food']:4d} | "
                f"Births {info['births']:3d} | Killed {info['killed']:3d} | "
                f"Rew {info['reward']:+6.1f} | "
                f"PLoss {ppo_metrics.get('policy_loss', 0):.3f} | "
                f"VLoss {ppo_metrics.get('value_loss', 0):.3f} | "
                f"{elapsed:.0f}s"
            )

        # ── Render
        if step % CFG.render_every == 0:
            renderer.render(world, step)
            print(f"  [Render] Frame saved to {CFG.plot_dir}/")

        # ── Extinction check
        if not world.organisms:
            print(f"\n⚠  All organisms extinct at step {step}. Simulation ended.")
            break

        # ── Re-seed enemies if all dead (shouldn't happen but safety net)
        if not world.enemies:
            for _ in range(CFG.init_enemies // 2):
                x, y = random.uniform(0, CFG.world_w), random.uniform(0, CFG.world_h)
                world.enemies.append(Enemy(x, y))
            print(f"  [World] Enemy re-seeded at step {step}")

    # ── Final render & summary
    renderer.render(world, world.step_count)
    print("\n" + "═"*65)
    print("  SIMULATION COMPLETE")
    print(f"  Final population : {len(world.organisms)}")
    if world.organisms:
        gens  = [o.generation for o in world.organisms]
        spds  = [o.genome.speed for o in world.organisms]
        viss  = [o.genome.vision for o in world.organisms]
        print(f"  Max generation   : {max(gens)}")
        print(f"  Avg generation   : {np.mean(gens):.2f}")
        print(f"  Avg speed        : {np.mean(spds):.3f}")
        print(f"  Avg vision       : {np.mean(viss):.3f}")
        print(f"  Total births     : {sum(world.stats['births'])}")
        print(f"  Total deaths     : {sum(world.stats['deaths_starve']) + sum(world.stats['deaths_killed'])}")
    print("═"*65 + "\n")

    # ── Save final stats plot
    final_path = os.path.join(CFG.plot_dir, "final_summary.png")
    renderer.fig.savefig(final_path, dpi=130, bbox_inches="tight",
                         facecolor=renderer.fig.get_facecolor())
    print(f"[Done] Final summary saved → {final_path}")

    return world, policy, renderer


# ─────────────────────────────────────────────────────────────────────────────
# 10. ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # ── Optional: override config for faster Colab run ──────────────────────
    # CFG.max_steps      = 3000
    # CFG.init_organisms = 40
    # CFG.render_every   = 200

    world, policy, renderer = run_simulation()

[Device] Running on: cuda
         GPU: Tesla T4
         VRAM: 15.6 GB

═════════════════════════════════════════════════════════════════
  ORGANISM LIVING SYSTEM — Starting Simulation
═════════════════════════════════════════════════════════════════
  World: 120×80   |   Init orgs: 60   |   Init enemies: 8
  RL: PPO with 128-dim network, GAE-λ=0.95
  Max steps: 8000   |   Render every: 100
═════════════════════════════════════════════════════════════════

Step    50 | Pop   17 | Food   88 | Births   0 | Killed   1 | Rew  +21.5 | PLoss 0.000 | VLoss 0.000 | 2s

⚠  All organisms extinct at step 78. Simulation ended.

═════════════════════════════════════════════════════════════════
  SIMULATION COMPLETE
  Final population : 0
═════════════════════════════════════════════════════════════════

[Done] Final summary saved → sim_frames/final_summary.png


# Changing base values for more chance of survival


```
# ── SURVIVAL BALANCE (replace these in SimConfig) ──────────────────────────

base_metabolism: float = 0.05       # was 0.12  → ~60% cheaper to move
food_energy: float     = 55.0       # was 40.0  → more reward for finding food
food_spawn_rate: float = 0.30       # was 0.15  → food replenishes 2× as fast
init_food: int         = 200        # was 120   → more food at start

init_energy: float     = 90.0       # was 70.0  → start above reproduce threshold
reproduce_threshold: float = 50.0   # was 80.0  → can reproduce after eating once
reproduce_cost: float  = 20.0       # was 30.0  → reproduction doesn't bankrupt them

init_enemies: int      = 4          # was 8     → half as many hunters early on
enemy_speed: float     = 1.8        # was 2.2   → slightly slower
enemy_kill_radius: float = 1.2      # was 1.8   → harder to get a kill

# ── OPTIONAL: Make RL learn faster ────────────────────────────────────────
rollout_len: int  = 128             # was 256   → update policy more frequently early
```
```
# Paste this right before: world, policy, renderer = run_simulation()
CFG.base_metabolism    = 0.05
CFG.food_energy        = 55.0
CFG.food_spawn_rate    = 0.30
CFG.init_food          = 200
CFG.init_energy        = 90.0
CFG.reproduce_threshold = 50.0
CFG.reproduce_cost     = 20.0
CFG.init_enemies       = 4
CFG.enemy_speed        = 1.8
CFG.enemy_kill_radius  = 1.2
CFG.rollout_len        = 128
```


In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║           ORGANISM LIVING SYSTEM — GPU-Accelerated RL/ML Simulation         ║
║                                                                              ║
║  Features:                                                                   ║
║   • Neural-network brain (PPO + Genetic Evolution hybrid)                   ║
║   • Gene system: speed, vision, metabolism, aggression, fertility            ║
║   • Organisms: spawn, eat, reproduce, mutate, die                           ║
║   • Enemies: hunt organisms using heuristic + learned policy                 ║
║   • GPU-parallel environment stepping via vectorized tensors                 ║
║   • Real-time matplotlib visualization                                       ║
║   • Designed to run on Google Colab (T4/A100 GPU)                           ║
╚══════════════════════════════════════════════════════════════════════════════╝

QUICK START (Google Colab):
    !pip install torch torchvision matplotlib numpy
    # Then run this file:
    !python organism_living_system.py

Or paste everything into a Colab notebook cell.
"""

# ─────────────────────────────────────────────────────────────────────────────
# 0. IMPORTS & DEVICE SETUP
# ─────────────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib
matplotlib.use("Agg")          # headless-safe; swap to "TkAgg" for local popup
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from collections import deque
import random, math, time, copy, os
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] Running on: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"         GPU: {torch.cuda.get_device_name(0)}")
    print(f"         VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ─────────────────────────────────────────────────────────────────────────────
# 1. SIMULATION HYPERPARAMETERS
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class SimConfig:
    # World
    world_w: int         = 120
    world_h: int         = 80
    n_bad_zones: int     = 4
    bad_zone_radius: float = 6.0

    # Population
    init_organisms: int  = 60
    init_enemies: int    = 6
    init_food: int       = 200
    max_organisms: int   = 200
    max_food: int        = 250

    # Energy & survival
    base_metabolism: float = 0.05     # energy/step base drain
    food_energy: float     = 55.0
    reproduce_cost: float  = 20.0
    reproduce_threshold: float = 50.0
    starve_energy: float   = 0.0
    max_energy: float      = 150.0
    init_energy: float     = 90.0

    # Genes (ranges)
    gene_speed_range: Tuple  = (0.8, 3.5)
    gene_vision_range: Tuple = (3.0, 18.0)
    gene_meta_range: Tuple   = (0.05, 0.3)    # extra metabolism multiplier
    gene_aggro_range: Tuple  = (0.0, 1.0)
    gene_fertility_range: Tuple = (0.3, 1.0)
    mutation_rate: float     = 0.12
    mutation_strength: float = 0.15

    # RL / Neural brain
    obs_dim: int   = 22        # observation vector per organism
    act_dim: int   = 5         # [up, down, left, right, stay]
    hidden_dim: int = 128
    lr: float       = 3e-4
    gamma: float    = 0.97
    gae_lambda: float = 0.95
    clip_eps: float   = 0.2
    ppo_epochs: int   = 4
    batch_size: int   = 512
    rollout_len: int  = 256    # steps per PPO update

    # Simulation
    max_steps: int   = 8000
    food_spawn_rate: float = 0.30   # prob each step to spawn new food
    enemy_speed: float    = 1.8
    enemy_vision: float   = 14.0
    enemy_kill_radius: float = 1.2

    # Visualization
    render_every: int = 100
    save_plot: bool   = True
    plot_dir: str     = "sim_frames"

CFG = SimConfig()


# ─────────────────────────────────────────────────────────────────────────────
# 2. GENE SYSTEM
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class Genome:
    speed: float      = 1.5
    vision: float     = 8.0
    metabolism: float = 0.15    # extra per-step energy cost multiplier
    aggression: float = 0.3
    fertility: float  = 0.7

    @classmethod
    def random(cls) -> "Genome":
        return cls(
            speed      = random.uniform(*CFG.gene_speed_range),
            vision     = random.uniform(*CFG.gene_vision_range),
            metabolism = random.uniform(*CFG.gene_meta_range),
            aggression = random.uniform(*CFG.gene_aggro_range),
            fertility  = random.uniform(*CFG.gene_fertility_range),
        )

    def mutate(self) -> "Genome":
        def m(v, lo, hi):
            if random.random() < CFG.mutation_rate:
                v += random.gauss(0, CFG.mutation_strength) * (hi - lo)
            return float(np.clip(v, lo, hi))
        return Genome(
            speed      = m(self.speed,      *CFG.gene_speed_range),
            vision     = m(self.vision,     *CFG.gene_vision_range),
            metabolism = m(self.metabolism, *CFG.gene_meta_range),
            aggression = m(self.aggression, *CFG.gene_aggro_range),
            fertility  = m(self.fertility,  *CFG.gene_fertility_range),
        )

    def crossover(self, other: "Genome") -> "Genome":
        def pick(a, b): return a if random.random() < 0.5 else b
        return Genome(
            speed      = pick(self.speed,      other.speed),
            vision     = pick(self.vision,     other.vision),
            metabolism = pick(self.metabolism, other.metabolism),
            aggression = pick(self.aggression, other.aggression),
            fertility  = pick(self.fertility,  other.fertility),
        ).mutate()

    def to_vec(self) -> np.ndarray:
        """Normalised gene vector [0,1]^5 for NN input."""
        def norm(v, lo, hi): return (v - lo) / (hi - lo + 1e-8)
        return np.array([
            norm(self.speed,      *CFG.gene_speed_range),
            norm(self.vision,     *CFG.gene_vision_range),
            norm(self.metabolism, *CFG.gene_meta_range),
            norm(self.aggression, *CFG.gene_aggro_range),
            norm(self.fertility,  *CFG.gene_fertility_range),
        ], dtype=np.float32)


# ─────────────────────────────────────────────────────────────────────────────
# 3. ACTOR-CRITIC NEURAL NETWORK  (shared backbone)
# ─────────────────────────────────────────────────────────────────────────────
class ActorCritic(nn.Module):
    def __init__(self, obs_dim: int, act_dim: int, hidden: int):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.LayerNorm(hidden), nn.Tanh(),
            nn.Linear(hidden, hidden),  nn.LayerNorm(hidden), nn.Tanh(),
        )
        self.actor  = nn.Linear(hidden, act_dim)
        self.critic = nn.Linear(hidden, 1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.actor.weight,  gain=0.01)
        nn.init.orthogonal_(self.critic.weight, gain=1.0)

    def forward(self, x: torch.Tensor):
        h = self.shared(x)
        return self.actor(h), self.critic(h)

    @torch.no_grad()
    def act(self, obs: np.ndarray) -> Tuple[int, float, float]:
        t = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        logits, value = self(t)
        dist  = torch.distributions.Categorical(logits=logits)
        action = dist.sample()
        logp   = dist.log_prob(action)
        return action.item(), logp.item(), value.item()


# ─────────────────────────────────────────────────────────────────────────────
# 4. SHARED POLICY POOL  (one policy per "species", shared across all orgs)
# ─────────────────────────────────────────────────────────────────────────────
class PolicyPool:
    """
    Maintains one PPO ActorCritic. All organisms share weights (like
    a species-level neural backbone) but are differentiated by genes.
    Rollouts are collected from all alive organisms simultaneously (GPU batch).
    """
    def __init__(self):
        self.net = ActorCritic(CFG.obs_dim, CFG.act_dim, CFG.hidden_dim).to(DEVICE)
        self.opt = optim.Adam(self.net.parameters(), lr=CFG.lr, eps=1e-5)
        # Rollout buffers
        self.reset_buffers()

    def reset_buffers(self):
        self.buf_obs    : List[np.ndarray] = []
        self.buf_acts   : List[int]        = []
        self.buf_logps  : List[float]      = []
        self.buf_vals   : List[float]      = []
        self.buf_rews   : List[float]      = []
        self.buf_dones  : List[float]      = []

    def store(self, obs, act, logp, val, rew, done):
        self.buf_obs.append(obs)
        self.buf_acts.append(act)
        self.buf_logps.append(logp)
        self.buf_vals.append(val)
        self.buf_rews.append(rew)
        self.buf_dones.append(float(done))

    def update(self):
        if len(self.buf_obs) < CFG.batch_size:
            return {}

        obs   = torch.tensor(np.array(self.buf_obs),  dtype=torch.float32, device=DEVICE)
        acts  = torch.tensor(self.buf_acts,            dtype=torch.long,    device=DEVICE)
        logps = torch.tensor(self.buf_logps,           dtype=torch.float32, device=DEVICE)
        vals  = torch.tensor(self.buf_vals,            dtype=torch.float32, device=DEVICE)
        rews  = torch.tensor(self.buf_rews,            dtype=torch.float32, device=DEVICE)
        dones = torch.tensor(self.buf_dones,           dtype=torch.float32, device=DEVICE)

        # GAE advantage
        advantages = torch.zeros_like(rews)
        gae = 0.0
        for t in reversed(range(len(rews))):
            nxt_val = 0.0 if (t == len(rews)-1 or dones[t]) else vals[t+1].item()
            delta = rews[t] + CFG.gamma * nxt_val * (1 - dones[t]) - vals[t]
            gae   = delta + CFG.gamma * CFG.gae_lambda * (1 - dones[t]) * gae
            advantages[t] = gae
        returns = advantages + vals
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        # PPO epochs
        idx = torch.randperm(len(obs))
        metrics = {"policy_loss": 0, "value_loss": 0, "entropy": 0}
        n_updates = 0
        for _ in range(CFG.ppo_epochs):
            for start in range(0, len(obs), CFG.batch_size):
                mb = idx[start:start+CFG.batch_size]
                logits, new_vals = self.net(obs[mb])
                dist     = torch.distributions.Categorical(logits=logits)
                new_logp = dist.log_prob(acts[mb])
                ent      = dist.entropy().mean()

                ratio    = (new_logp - logps[mb]).exp()
                adv_mb   = advantages[mb]
                surr1    = ratio * adv_mb
                surr2    = ratio.clamp(1-CFG.clip_eps, 1+CFG.clip_eps) * adv_mb
                p_loss   = -torch.min(surr1, surr2).mean()
                v_loss   = F.mse_loss(new_vals.squeeze(-1), returns[mb])
                loss     = p_loss + 0.5 * v_loss - 0.01 * ent

                self.opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.net.parameters(), 0.5)
                self.opt.step()

                metrics["policy_loss"] += p_loss.item()
                metrics["value_loss"]  += v_loss.item()
                metrics["entropy"]     += ent.item()
                n_updates += 1

        self.reset_buffers()
        return {k: v/max(n_updates,1) for k, v in metrics.items()}


# ─────────────────────────────────────────────────────────────────────────────
# 5. ORGANISM
# ─────────────────────────────────────────────────────────────────────────────
_ORG_ID = 0

class Organism:
    def __init__(self, x: float, y: float, genome: Optional[Genome] = None,
                 energy: float = CFG.init_energy, generation: int = 1):
        global _ORG_ID
        _ORG_ID += 1
        self.id         = _ORG_ID
        self.x          = float(x)
        self.y          = float(y)
        self.genome     = genome or Genome.random()
        self.energy     = float(energy)
        self.alive      = True
        self.age        = 0
        self.generation = generation
        self.fitness    = 0.0           # cumulative reward signal
        # Last RL step info
        self._obs  : Optional[np.ndarray] = None
        self._act  : int   = 4           # stay
        self._logp : float = 0.0
        self._val  : float = 0.0

    # ── Observation builder ──────────────────────────────────────────────────
    def observe(self, food_list, enemy_list, bad_zones) -> np.ndarray:
        """
        22-dim observation vector:
         [0]    normalised x position
         [1]    normalised y position
         [2]    normalised energy
         [3]    normalised age
         [4:9]  gene vector (5 dims)
         [9:13] nearest food: dx, dy, dist, exists
         [13:17] nearest enemy: dx, dy, dist, exists
         [17:20] nearest bad zone: dx, dy, dist
         [20]   in bad zone flag
         [21]   nearest mate: dist_norm
        """
        W, H = CFG.world_w, CFG.world_h
        obs = np.zeros(CFG.obs_dim, dtype=np.float32)
        obs[0] = self.x / W
        obs[1] = self.y / H
        obs[2] = self.energy / CFG.max_energy
        obs[3] = min(self.age / 500.0, 1.0)
        obs[4:9] = self.genome.to_vec()

        # Nearest food
        if food_list:
            dists = [math.hypot(f[0]-self.x, f[1]-self.y) for f in food_list]
            idx   = int(np.argmin(dists))
            dx    = (food_list[idx][0] - self.x) / W
            dy    = (food_list[idx][1] - self.y) / H
            obs[9]  = np.clip(dx, -1, 1)
            obs[10] = np.clip(dy, -1, 1)
            obs[11] = min(dists[idx] / self.genome.vision, 1.0)
            obs[12] = 1.0 if dists[idx] <= self.genome.vision else 0.0

        # Nearest enemy
        if enemy_list:
            dists = [math.hypot(e.x-self.x, e.y-self.y) for e in enemy_list]
            idx   = int(np.argmin(dists))
            dx    = (enemy_list[idx].x - self.x) / W
            dy    = (enemy_list[idx].y - self.y) / H
            obs[13] = np.clip(dx, -1, 1)
            obs[14] = np.clip(dy, -1, 1)
            obs[15] = min(dists[idx] / self.genome.vision, 1.0)
            obs[16] = 1.0 if dists[idx] <= self.genome.vision else 0.0

        # Bad zones
        if bad_zones:
            dists = [math.hypot(z[0]-self.x, z[1]-self.y) for z in bad_zones]
            idx   = int(np.argmin(dists))
            dx    = (bad_zones[idx][0] - self.x) / W
            dy    = (bad_zones[idx][1] - self.y) / H
            obs[17] = np.clip(dx, -1, 1)
            obs[18] = np.clip(dy, -1, 1)
            obs[19] = min(dists[idx] / CFG.world_w, 1.0)
            obs[20] = 1.0 if dists[idx] <= CFG.bad_zone_radius else 0.0

        # Nearest mate (same species, alive)
        # (passed in as enemy_list repurposed; actual mates handled outside)
        obs[21] = 0.0   # filled externally

        return obs

    # ── Step: apply action ───────────────────────────────────────────────────
    def step(self, action: int):
        speed = self.genome.speed
        if   action == 0: self.y -= speed            # up
        elif action == 1: self.y += speed            # down
        elif action == 2: self.x -= speed            # left
        elif action == 3: self.x += speed            # right
        # action 4: stay
        # Clamp to world
        self.x = float(np.clip(self.x, 0, CFG.world_w))
        self.y = float(np.clip(self.y, 0, CFG.world_h))
        # Metabolic cost: base + gene metabolism
        cost = CFG.base_metabolism * (1.0 + self.genome.metabolism) * (speed if action != 4 else 0.5)
        self.energy -= cost
        self.age    += 1
        if self.energy <= CFG.starve_energy:
            self.alive = False


# ─────────────────────────────────────────────────────────────────────────────
# 6. ENEMY
# ─────────────────────────────────────────────────────────────────────────────
class Enemy:
    def __init__(self, x: float, y: float):
        self.x     = float(x)
        self.y     = float(y)
        self.alive = True
        self.age   = 0

    def step(self, organisms: List[Organism]):
        """Heuristic: chase nearest alive organism."""
        alive_orgs = [o for o in organisms if o.alive]
        if not alive_orgs:
            # random walk
            self.x += random.uniform(-CFG.enemy_speed, CFG.enemy_speed)
            self.y += random.uniform(-CFG.enemy_speed, CFG.enemy_speed)
        else:
            dists = [math.hypot(o.x-self.x, o.y-self.y) for o in alive_orgs]
            tgt   = alive_orgs[int(np.argmin(dists))]
            dx    = tgt.x - self.x
            dy    = tgt.y - self.y
            d     = math.hypot(dx, dy) + 1e-8
            self.x += (dx / d) * CFG.enemy_speed
            self.y += (dy / d) * CFG.enemy_speed
        self.x = float(np.clip(self.x, 0, CFG.world_w))
        self.y = float(np.clip(self.y, 0, CFG.world_h))
        self.age += 1

    def try_kill(self, organisms: List[Organism]):
        """Kill organism within kill radius."""
        killed = 0
        for o in organisms:
            if o.alive and math.hypot(o.x-self.x, o.y-self.y) <= CFG.enemy_kill_radius:
                o.alive = False
                killed += 1
        return killed


# ─────────────────────────────────────────────────────────────────────────────
# 7. WORLD / ENVIRONMENT
# ─────────────────────────────────────────────────────────────────────────────
class World:
    def __init__(self):
        self.organisms : List[Organism] = []
        self.enemies   : List[Enemy]    = []
        self.food      : List[Tuple[float, float]] = []
        self.bad_zones : List[Tuple[float, float]] = []
        self.step_count = 0
        self.stats = {
            "population": [], "food_count": [], "births": [],
            "deaths_starve": [], "deaths_killed": [],
            "avg_speed": [], "avg_vision": [], "avg_generation": [],
            "total_reward": [],
        }
        self._births = 0
        self._deaths_starve = 0
        self._deaths_killed = 0

        self._init_world()

    def _rand_pos(self):
        return (random.uniform(0, CFG.world_w), random.uniform(0, CFG.world_h))

    def _init_world(self):
        # Bad zones
        for _ in range(CFG.n_bad_zones):
            self.bad_zones.append(self._rand_pos())
        # Food
        for _ in range(CFG.init_food):
            self.food.append(self._rand_pos())
        # Organisms
        for _ in range(CFG.init_organisms):
            x, y = self._rand_pos()
            self.organisms.append(Organism(x, y))
        # Enemies
        for _ in range(CFG.init_enemies):
            x, y = self._rand_pos()
            self.enemies.append(Enemy(x, y))

    # ── Reward function ──────────────────────────────────────────────────────
    def _reward(self, org: Organism, ate: bool, in_bad_zone: bool,
                 near_enemy: bool) -> float:
        r = 0.0
        r += 2.0  if ate         else 0.0
        r -= 1.5  if in_bad_zone else 0.0
        r -= 1.0  if near_enemy  else 0.0
        r += 0.05                          # survival bonus each step
        r -= 0.1 * (org.genome.speed - 1)  # speed is expensive; penalise slightly
        return float(r)

    # ── Reproduction ─────────────────────────────────────────────────────────
    def _try_reproduce(self, org: Organism):
        if (org.energy >= CFG.reproduce_threshold
                and len(self.organisms) < CFG.max_organisms):
            # Find a mate within vision range
            mates = [o for o in self.organisms
                     if o.alive and o.id != org.id
                     and math.hypot(o.x-org.x, o.y-org.y) <= org.genome.vision]
            if mates:
                mate   = random.choice(mates)
                genome = org.genome.crossover(mate.genome)
            else:
                genome = org.genome.mutate()

            child_energy = CFG.reproduce_cost * 0.6
            org.energy  -= CFG.reproduce_cost
            child = Organism(
                x          = org.x + random.uniform(-2, 2),
                y          = org.y + random.uniform(-2, 2),
                genome     = genome,
                energy     = child_energy,
                generation = max(org.generation, getattr(mate if mates else org, "generation", 1)) + 1,
            )
            self.organisms.append(child)
            self._births += 1

    # ── Food spawning ─────────────────────────────────────────────────────────
    def _spawn_food(self):
        if (random.random() < CFG.food_spawn_rate
                and len(self.food) < CFG.max_food):
            self.food.append(self._rand_pos())

    # ── Main step ─────────────────────────────────────────────────────────────
    def step(self, policy: PolicyPool) -> dict:
        self.step_count += 1
        self._births         = 0
        dead_starve_this_step = 0
        dead_killed_this_step = 0
        total_reward = 0.0

        alive = [o for o in self.organisms if o.alive]

        # 1. Observe → Act (batched on GPU for efficiency)
        obs_batch = []
        for org in alive:
            obs = org.observe(self.food, self.enemies, self.bad_zones)
            obs_batch.append(obs)

        if obs_batch:
            obs_t = torch.tensor(np.array(obs_batch), dtype=torch.float32, device=DEVICE)
            with torch.no_grad():
                logits_t, vals_t = policy.net(obs_t)
            dists   = torch.distributions.Categorical(logits=logits_t)
            acts_t  = dists.sample()
            logps_t = dists.log_prob(acts_t)
            actions = acts_t.cpu().numpy()
            logps   = logps_t.cpu().numpy()
            vals    = vals_t.squeeze(-1).cpu().numpy()

            for i, org in enumerate(alive):
                org._obs  = obs_batch[i]
                org._act  = int(actions[i])
                org._logp = float(logps[i])
                org._val  = float(vals[i])

        # 2. Apply actions
        food_set = list(self.food)
        for org in alive:
            org.step(org._act)
            if not org.alive:
                dead_starve_this_step += 1

        # 3. Eat food
        food_to_remove = set()
        for org in alive:
            if not org.alive:
                continue
            for fi, f in enumerate(food_set):
                if fi not in food_to_remove:
                    if math.hypot(f[0]-org.x, f[1]-org.y) <= 1.5:
                        org.energy = min(org.energy + CFG.food_energy, CFG.max_energy)
                        food_to_remove.add(fi)
                        break
        self.food = [f for i, f in enumerate(food_set) if i not in food_to_remove]

        # 4. Compute rewards & store in buffer
        for org in alive:
            if org._obs is None:
                continue
            in_bad = any(
                math.hypot(z[0]-org.x, z[1]-org.y) <= CFG.bad_zone_radius
                for z in self.bad_zones
            )
            near_enemy = any(
                math.hypot(e.x-org.x, e.y-org.y) <= org.genome.vision
                for e in self.enemies
            )
            ate   = org.energy > org._val  # energy went up → ate
            rew   = self._reward(org, ate, in_bad, near_enemy)
            org.fitness += rew
            total_reward += rew
            done  = not org.alive
            policy.store(org._obs, org._act, org._logp, org._val, rew, done)

        # 5. Enemies act
        for enemy in self.enemies:
            enemy.step(alive)
            killed = enemy.try_kill(alive)
            dead_killed_this_step += killed

        # 6. Reproduce
        for org in alive:
            if org.alive:
                self._try_reproduce(org)

        # 7. Food spawn
        self._spawn_food()

        # 8. Prune dead
        self.organisms = [o for o in self.organisms if o.alive]
        self._deaths_starve += dead_starve_this_step
        self._deaths_killed += dead_killed_this_step

        # 9. Record stats
        alive_now = [o for o in self.organisms if o.alive]
        self.stats["population"].append(len(alive_now))
        self.stats["food_count"].append(len(self.food))
        self.stats["births"].append(self._births)
        self.stats["deaths_starve"].append(dead_starve_this_step)
        self.stats["deaths_killed"].append(dead_killed_this_step)
        self.stats["total_reward"].append(total_reward)
        self.stats["avg_speed"].append(
            float(np.mean([o.genome.speed for o in alive_now])) if alive_now else 0)
        self.stats["avg_vision"].append(
            float(np.mean([o.genome.vision for o in alive_now])) if alive_now else 0)
        self.stats["avg_generation"].append(
            float(np.mean([o.generation for o in alive_now])) if alive_now else 0)

        return {
            "pop": len(alive_now),
            "food": len(self.food),
            "births": self._births,
            "killed": dead_killed_this_step,
            "reward": total_reward,
        }


# ─────────────────────────────────────────────────────────────────────────────
# 8. RENDERER
# ─────────────────────────────────────────────────────────────────────────────
class Renderer:
    def __init__(self):
        self.fig, self.axes = plt.subplots(2, 3, figsize=(18, 10))
        self.fig.patch.set_facecolor("#0d1117")
        plt.subplots_adjust(wspace=0.35, hspace=0.4)
        if CFG.save_plot:
            os.makedirs(CFG.plot_dir, exist_ok=True)
        self._frame = 0

    def _ax_style(self, ax, title="", xlabel="", ylabel=""):
        ax.set_facecolor("#161b22")
        ax.tick_params(colors="#8b949e", labelsize=8)
        ax.spines[:].set_color("#30363d")
        ax.set_title(title, color="#e6edf3", fontsize=9, pad=6, fontweight="bold")
        ax.set_xlabel(xlabel, color="#8b949e", fontsize=8)
        ax.set_ylabel(ylabel, color="#8b949e", fontsize=8)

    def render(self, world: World, step: int):
        for ax in self.axes.flat:
            ax.cla()

        # ── Panel 0: World Map ───────────────────────────────────────────────
        ax = self.axes[0, 0]
        self._ax_style(ax, f"Living World  [step {step}]", "X", "Y")
        ax.set_xlim(0, CFG.world_w)
        ax.set_ylim(0, CFG.world_h)

        # Bad zones
        for z in world.bad_zones:
            circle = plt.Circle(z, CFG.bad_zone_radius,
                                color="#ff3b30", alpha=0.18, zorder=1)
            ax.add_patch(circle)
            circle2 = plt.Circle(z, CFG.bad_zone_radius,
                                 fill=False, edgecolor="#ff3b30",
                                 linewidth=1.2, alpha=0.6, zorder=2)
            ax.add_patch(circle2)

        # Food
        if world.food:
            fx, fy = zip(*world.food)
            ax.scatter(fx, fy, s=6, c="#30d158", alpha=0.75, zorder=3,
                       marker=".", linewidths=0)

        # Organisms — colour by energy
        if world.organisms:
            energies = np.array([o.energy for o in world.organisms])
            norm     = Normalize(vmin=0, vmax=CFG.max_energy)
            speeds   = np.array([o.genome.speed for o in world.organisms])
            sizes    = 10 + 20 * (speeds - CFG.gene_speed_range[0]) / (
                CFG.gene_speed_range[1] - CFG.gene_speed_range[0])
            ox = [o.x for o in world.organisms]
            oy = [o.y for o in world.organisms]
            sc = ax.scatter(ox, oy, s=sizes, c=energies, cmap="YlOrRd",
                            norm=norm, alpha=0.9, zorder=4, linewidths=0.3,
                            edgecolors="#ffffff20")
            # Generation overlay: dot for gen > 3
            high_gen = [o for o in world.organisms if o.generation > 3]
            if high_gen:
                ax.scatter([o.x for o in high_gen], [o.y for o in high_gen],
                           s=4, c="#bf5af2", zorder=5, marker="*",
                           linewidths=0)

        # Enemies
        for e in world.enemies:
            ax.scatter(e.x, e.y, s=80, c="#ff453a", marker="^",
                       zorder=6, edgecolors="#fff", linewidths=0.5)

        # Legend
        legend_els = [
            mpatches.Patch(color="#30d158",  label=f"Food ({len(world.food)})"),
            mpatches.Patch(color="#ff3b30",  label="Bad Zone"),
            mpatches.Patch(color="#ff6b35",  label=f"Organisms ({len(world.organisms)})"),
            mpatches.Patch(color="#ff453a",  label=f"Enemies ({len(world.enemies)})"),
            mpatches.Patch(color="#bf5af2",  label="Gen > 3"),
        ]
        ax.legend(handles=legend_els, loc="upper right",
                  fontsize=6, facecolor="#21262d",
                  labelcolor="#e6edf3", edgecolor="#30363d")

        # ── Panel 1: Population & Food ───────────────────────────────────────
        ax = self.axes[0, 1]
        self._ax_style(ax, "Population & Food", "Step", "Count")
        steps = list(range(len(world.stats["population"])))
        ax.plot(steps, world.stats["population"], color="#0a84ff",
                lw=1.4, label="Population")
        ax.plot(steps, world.stats["food_count"],  color="#30d158",
                lw=1.2, alpha=0.8, label="Food")
        ax.legend(fontsize=7, facecolor="#21262d",
                  labelcolor="#e6edf3", edgecolor="#30363d")

        # ── Panel 2: Deaths ──────────────────────────────────────────────────
        ax = self.axes[0, 2]
        self._ax_style(ax, "Deaths per Step", "Step", "Count")
        ax.plot(steps, world.stats["deaths_starve"], color="#ff9f0a",
                lw=1.2, label="Starvation")
        ax.plot(steps, world.stats["deaths_killed"], color="#ff453a",
                lw=1.2, label="Killed")
        ax.legend(fontsize=7, facecolor="#21262d",
                  labelcolor="#e6edf3", edgecolor="#30363d")

        # ── Panel 3: Gene Evolution ──────────────────────────────────────────
        ax = self.axes[1, 0]
        self._ax_style(ax, "Gene Evolution", "Step", "Value")
        ax.plot(steps, world.stats["avg_speed"],  color="#bf5af2",
                lw=1.4, label="Avg Speed")
        ax.plot(steps, world.stats["avg_vision"], color="#64d2ff",
                lw=1.2, label="Avg Vision")
        ax.legend(fontsize=7, facecolor="#21262d",
                  labelcolor="#e6edf3", edgecolor="#30363d")

        # ── Panel 4: Generation & Reward ────────────────────────────────────
        ax = self.axes[1, 1]
        self._ax_style(ax, "Avg Generation", "Step", "Generation")
        ax.plot(steps, world.stats["avg_generation"], color="#ffd60a",
                lw=1.4)

        # ── Panel 5: Cumulative Reward ───────────────────────────────────────
        ax = self.axes[1, 2]
        self._ax_style(ax, "Total Reward / Step", "Step", "Reward")
        rew_smooth = np.convolve(world.stats["total_reward"],
                                 np.ones(20)/20, mode="same")
        ax.plot(steps, world.stats["total_reward"], color="#8b949e",
                lw=0.7, alpha=0.5, label="Raw")
        ax.plot(steps, rew_smooth, color="#30d158",
                lw=1.5, label="Smoothed (20)")
        ax.legend(fontsize=7, facecolor="#21262d",
                  labelcolor="#e6edf3", edgecolor="#30363d")

        # ── Finalize ─────────────────────────────────────────────────────────
        self.fig.suptitle(
            f"Organism Living System — Step {step} | Pop {len(world.organisms)} | "
            f"Food {len(world.food)} | Enemies {len(world.enemies)}",
            color="#e6edf3", fontsize=11, fontweight="bold", y=1.01
        )
        plt.tight_layout()

        if CFG.save_plot:
            fname = os.path.join(CFG.plot_dir, f"frame_{self._frame:05d}.png")
            self.fig.savefig(fname, dpi=110, bbox_inches="tight",
                             facecolor=self.fig.get_facecolor())
            self._frame += 1
        try:
            plt.pause(0.001)
        except Exception:
            pass
        return self.fig


# ─────────────────────────────────────────────────────────────────────────────
# 9. MAIN SIMULATION LOOP
# ─────────────────────────────────────────────────────────────────────────────
def run_simulation():
    print("\n" + "═"*65)
    print("  ORGANISM LIVING SYSTEM — Starting Simulation")
    print("═"*65)
    print(f"  World: {CFG.world_w}×{CFG.world_h}   |   "
          f"Init orgs: {CFG.init_organisms}   |   "
          f"Init enemies: {CFG.init_enemies}")
    print(f"  RL: PPO with {CFG.hidden_dim}-dim network, GAE-λ={CFG.gae_lambda}")
    print(f"  Max steps: {CFG.max_steps}   |   Render every: {CFG.render_every}")
    print("═"*65 + "\n")

    world    = World()
    policy   = PolicyPool()
    renderer = Renderer()

    ppo_metrics = {}
    t0 = time.time()

    for step in range(1, CFG.max_steps + 1):
        # ── Environment step
        info = world.step(policy)

        # ── PPO update when buffer is large enough
        if len(policy.buf_obs) >= CFG.rollout_len:
            ppo_metrics = policy.update()

        # ── Logging
        if step % 50 == 0:
            elapsed = time.time() - t0
            print(
                f"Step {step:5d} | Pop {info['pop']:4d} | Food {info['food']:4d} | "
                f"Births {info['births']:3d} | Killed {info['killed']:3d} | "
                f"Rew {info['reward']:+6.1f} | "
                f"PLoss {ppo_metrics.get('policy_loss', 0):.3f} | "
                f"VLoss {ppo_metrics.get('value_loss', 0):.3f} | "
                f"{elapsed:.0f}s"
            )

        # ── Render
        if step % CFG.render_every == 0:
            renderer.render(world, step)
            print(f"  [Render] Frame saved to {CFG.plot_dir}/")

        # ── Extinction check
        if not world.organisms:
            print(f"\n⚠  All organisms extinct at step {step}. Simulation ended.")
            break

        # ── Re-seed enemies if all dead (shouldn't happen but safety net)
        if not world.enemies:
            for _ in range(CFG.init_enemies // 2):
                x, y = random.uniform(0, CFG.world_w), random.uniform(0, CFG.world_h)
                world.enemies.append(Enemy(x, y))
            print(f"  [World] Enemy re-seeded at step {step}")

    # ── Final render & summary
    renderer.render(world, world.step_count)
    print("\n" + "═"*65)
    print("  SIMULATION COMPLETE")
    print(f"  Final population : {len(world.organisms)}")
    if world.organisms:
        gens  = [o.generation for o in world.organisms]
        spds  = [o.genome.speed for o in world.organisms]
        viss  = [o.genome.vision for o in world.organisms]
        print(f"  Max generation   : {max(gens)}")
        print(f"  Avg generation   : {np.mean(gens):.2f}")
        print(f"  Avg speed        : {np.mean(spds):.3f}")
        print(f"  Avg vision       : {np.mean(viss):.3f}")
        print(f"  Total births     : {sum(world.stats['births'])}")
        print(f"  Total deaths     : {sum(world.stats['deaths_starve']) + sum(world.stats['deaths_killed'])}")
    print("═"*65 + "\n")

    # ── Save final stats plot
    final_path = os.path.join(CFG.plot_dir, "final_summary.png")
    renderer.fig.savefig(final_path, dpi=130, bbox_inches="tight",
                         facecolor=renderer.fig.get_facecolor())
    print(f"[Done] Final summary saved → {final_path}")

    return world, policy, renderer


# ─────────────────────────────────────────────────────────────────────────────
# 10. ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    world, policy, renderer = run_simulation()

[Device] Running on: cuda
         GPU: Tesla T4
         VRAM: 15.6 GB

═════════════════════════════════════════════════════════════════
  ORGANISM LIVING SYSTEM — Starting Simulation
═════════════════════════════════════════════════════════════════
  World: 120×80   |   Init orgs: 60   |   Init enemies: 6
  RL: PPO with 128-dim network, GAE-λ=0.95
  Max steps: 8000   |   Render every: 100
═════════════════════════════════════════════════════════════════

Step    50 | Pop  197 | Food   38 | Births   1 | Killed   3 | Rew +162.5 | PLoss 0.034 | VLoss 77.401 | 3s
Step   100 | Pop  170 | Food   18 | Births   1 | Killed   0 | Rew  +70.8 | PLoss -0.030 | VLoss 29.339 | 6s
  [Render] Frame saved to sim_frames/
Step   150 | Pop   28 | Food   33 | Births   0 | Killed   2 | Rew   +9.2 | PLoss -0.033 | VLoss 28.209 | 9s

⚠  All organisms extinct at step 165. Simulation ended.

═════════════════════════════════════════════════════════════════
  SIMULATION COMPLETE
  Final population : 0
════════

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║        ORGANISM LIVING SYSTEM v5 — Most Advanced RL: MAPPO + LSTM + PBT     ║
║                                                                              ║
║  RL ALGORITHM:  MAPPO  (Multi-Agent PPO)                                     ║
║    + GRU recurrent memory  (organisms remember past 8 steps)                 ║
║    + Centralised critic    (sees all agents → stable training)               ║
║    + Population-Based Training  (4 hyperparameter configs compete)           ║
║    + Intrinsic curiosity reward  (explore unseen map regions)                ║
║                                                                              ║
║  CIVILIZATION FEATURES (new in v5):                                          ║
║   • WARRIOR RESPAWN    — villages auto-train new warriors when count < min   ║
║   • TRADE ROUTES       — orgs carry food between villages, bonus reward      ║
║   • SEASONS            — food spawn rate oscillates (spring→winter cycle)    ║
║   • DISEASE            — low-immunity orgs in crowded villages get sick      ║
║   • TERRITORY MARKERS  — villages claim territory; enemies invade borders    ║
║   • ELDER SYSTEM       — long-lived orgs become elders, boost nearby births  ║
║   • WAR CHIEFS         — most-kills warrior gets speed/radius bonus          ║
║   • MIGRATION          — overpopulated villages push orgs to found new ones  ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
import numpy as np, matplotlib, random, math, time, os, copy
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from dataclasses import dataclass, field
from typing import List, Optional, Tuple, Dict
from enum import Enum
from collections import deque

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {DEVICE}", end="")
if DEVICE.type == "cuda":
    print(f"  |  {torch.cuda.get_device_name(0)}"
          f"  |  {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB VRAM")
else:
    print()

# ─────────────────────────────────────────────────────────────────────────────
# 1.  CONFIG
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class SimConfig:
    # World
    world_w: int   = 220
    world_h: int   = 140
    n_bad_zones: int       = 3
    bad_zone_radius: float = 8.0
    bad_zone_damage: float = 0.5

    # Seasons (steps per full year cycle)
    season_period:     int   = 800
    season_amp:        float = 0.35   # food spawn oscillation amplitude

    # Villages
    init_villages: int       = 2
    max_villages:  int       = 10
    village_radius: float    = 15.0
    village_food_radius: float= 2.5
    clan_form_density: int   = 20
    clan_form_radius:  float = 22.0
    clan_form_cooldown:int   = 300
    village_breach_radius: float     = 28.0
    village_breach_kill_prob: float  = 0.25
    min_warriors_per_village: int    = 3    # auto-respawn below this
    warrior_respawn_interval: int    = 150  # steps between auto-respawns
    territory_radius: float          = 35.0 # claimed territory around village
    overpop_threshold: int           = 45   # push orgs to migrate

    # Populations
    init_organisms: int  = 100
    init_warriors:  int  = 22
    init_enemies:   int  = 5
    max_organisms:  int  = 300
    max_warriors:   int  = 90
    max_enemies:    int  = 40

    # Food
    init_food: int         = 320
    max_food:  int         = 500
    food_spawn_rate: float = 0.60
    food_eat_radius: float = 2.4

    # Organism physics
    org_friction: float = 0.76
    org_accel:    float = 0.92

    # Organism energy
    init_energy: float         = 105.0
    max_energy:  float         = 190.0
    base_metabolism: float     = 0.028
    move_cost_factor: float    = 0.010
    food_energy: float         = 72.0
    reproduce_threshold: float = 82.0
    reproduce_cost:      float = 26.0
    gestation_steps: int       = 12
    starve_energy: float       = 0.0
    elder_age: int             = 400    # steps to become elder
    elder_birth_bonus: float   = 0.3   # extra birth prob near elders
    sick_duration: int         = 60
    sick_damage: float         = 0.08

    # Genes
    gene_vision_range:    Tuple = (5.0, 24.0)
    gene_meta_range:      Tuple = (0.02, 0.22)
    gene_fertility_range: Tuple = (0.3,  1.0)
    gene_va_range:        Tuple = (0.0,  1.0)
    gene_speed_range_M:   Tuple = (1.5, 4.2)
    gene_aggro_range_M:   Tuple = (0.3, 1.0)
    gene_speed_range_F:   Tuple = (0.8, 2.9)
    gene_aggro_range_F:   Tuple = (0.0, 0.6)
    mutation_rate:   float = 0.11
    mutation_strength: float = 0.12

    # Warrior
    warrior_max_speed:    float = 4.0
    warrior_accel:        float = 1.2
    warrior_friction:     float = 0.79
    warrior_vision:       float = 22.0
    warrior_energy_init:  float = 140.0
    warrior_energy_max:   float = 240.0
    warrior_metabolism:   float = 0.045
    warrior_move_cost:    float = 0.009
    warrior_kill_radius:  float = 4.0
    warrior_protect_r:    float = 18.0
    warrior_food_energy:  float = 40.0
    warrior_rep_threshold:float = 170.0
    warrior_rep_cost:     float = 48.0
    warrior_attack_bonus: float = 30.0
    war_chief_bonus:      float = 0.25  # speed multiplier for war chief

    # Enemy
    enemy_max_speed_init:  float = 1.9
    enemy_accel:           float = 0.82
    enemy_friction:        float = 0.73
    enemy_kill_radius_init:float = 1.9
    enemy_vision_init:     float = 13.0
    enemy_max_speed_cap:   float = 4.5
    enemy_kill_radius_cap: float = 4.0
    enemy_growth_interval: int   = 450
    enemy_growth_rate:     float = 0.055
    enemy_spawn_interval:  int   = 700
    enemy_pack_radius:     float = 28.0

    # ── MAPPO + GRU ────────────────────────────────────────────────────────
    act_dim:          int   = 9
    org_obs_dim:      int   = 42    # local observation
    warrior_obs_dim:  int   = 30
    global_state_dim: int   = 24    # centralised critic global state
    hidden_dim:       int   = 256
    gru_hidden:       int   = 128   # GRU memory size
    lr:               float = 1.5e-4
    gamma:            float = 0.99
    gae_lambda:       float = 0.95
    clip_eps:         float = 0.2
    entropy_coef:     float = 0.018
    value_coef:       float = 0.5
    ppo_epochs:       int   = 4
    batch_size:       int   = 1024
    rollout_len:      int   = 300
    max_grad_norm:    float = 0.5

    # PBT population
    pbt_population:   int   = 4
    pbt_interval:     int   = 1000  # steps between PBT exploit/explore

    # Intrinsic curiosity
    curiosity_coef:   float = 0.02
    curiosity_grid_w: int   = 20
    curiosity_grid_h: int   = 13

    # Simulation
    max_steps:    int  = 8000
    render_every: int  = 100
    save_plot:    bool = True
    plot_dir:     str  = "sim_frames_v5"

CFG = SimConfig()

_ACT_DIRS = np.array([
    [ 0,-1],[ 1,-1],[ 1, 0],[ 1, 1],
    [ 0, 1],[-1, 1],[-1, 0],[-1,-1],[ 0, 0],
], dtype=np.float32)
for _i in range(8):
    _n = np.linalg.norm(_ACT_DIRS[_i])
    if _n > 0: _ACT_DIRS[_i] /= _n

class Sex(Enum):
    MALE = 0; FEMALE = 1

# ─────────────────────────────────────────────────────────────────────────────
# 2.  GENOME
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class Genome:
    speed: float=2.0; vision: float=11.0; metabolism: float=0.10
    aggression: float=0.4; fertility: float=0.6
    village_affinity: float=0.5; dominance: float=0.5
    immune_strength: float=0.5; trade_drive: float=0.5

    @classmethod
    def random(cls, sex: Sex) -> "Genome":
        sr = CFG.gene_speed_range_M if sex==Sex.MALE else CFG.gene_speed_range_F
        ar = CFG.gene_aggro_range_M if sex==Sex.MALE else CFG.gene_aggro_range_F
        return cls(speed=random.uniform(*sr),
                   vision=random.uniform(*CFG.gene_vision_range),
                   metabolism=random.uniform(*CFG.gene_meta_range),
                   aggression=random.uniform(*ar),
                   fertility=random.uniform(*CFG.gene_fertility_range),
                   village_affinity=random.uniform(*CFG.gene_va_range),
                   dominance=random.uniform(0.2,0.8),
                   immune_strength=random.uniform(0.1,0.9),
                   trade_drive=random.uniform(0.0,1.0))

    def mutate(self, sex: Sex) -> "Genome":
        sr = CFG.gene_speed_range_M if sex==Sex.MALE else CFG.gene_speed_range_F
        ar = CFG.gene_aggro_range_M if sex==Sex.MALE else CFG.gene_aggro_range_F
        def m(v,lo,hi):
            if random.random()<CFG.mutation_rate: v+=random.gauss(0,CFG.mutation_strength)*(hi-lo)
            return float(np.clip(v,lo,hi))
        return Genome(speed=m(self.speed,*sr),vision=m(self.vision,*CFG.gene_vision_range),
                      metabolism=m(self.metabolism,*CFG.gene_meta_range),
                      aggression=m(self.aggression,*ar),
                      fertility=m(self.fertility,*CFG.gene_fertility_range),
                      village_affinity=m(self.village_affinity,*CFG.gene_va_range),
                      dominance=m(self.dominance,0.1,0.9),
                      immune_strength=m(self.immune_strength,0.0,1.0),
                      trade_drive=m(self.trade_drive,0.0,1.0))

    def crossover(self, other: "Genome", csex: Sex) -> "Genome":
        dm=other.dominance; df=self.dominance; total=dm+df+1e-8
        pf=dm/total; pick=lambda a,b: b if random.random()<pf else a
        return Genome(speed=pick(self.speed,other.speed),vision=pick(self.vision,other.vision),
                      metabolism=pick(self.metabolism,other.metabolism),
                      aggression=pick(self.aggression,other.aggression),
                      fertility=pick(self.fertility,other.fertility),
                      village_affinity=pick(self.village_affinity,other.village_affinity),
                      dominance=(self.dominance+other.dominance)/2,
                      immune_strength=pick(self.immune_strength,other.immune_strength),
                      trade_drive=pick(self.trade_drive,other.trade_drive)).mutate(csex)

    def to_vec(self) -> np.ndarray:
        def norm(v,lo,hi): return (v-lo)/(hi-lo+1e-8)
        return np.array([norm(self.speed,1.0,4.5),norm(self.vision,*CFG.gene_vision_range),
                         norm(self.metabolism,*CFG.gene_meta_range),norm(self.aggression,0.,1.),
                         norm(self.fertility,*CFG.gene_fertility_range),
                         self.village_affinity,self.dominance,self.immune_strength,self.trade_drive],
                        dtype=np.float32)

# ─────────────────────────────────────────────────────────────────────────────
# 3.  MAPPO ACTOR-CRITIC with GRU memory
# ─────────────────────────────────────────────────────────────────────────────
class GRUActorCritic(nn.Module):
    """
    Actor: local obs → GRU → policy head  (each agent has its own hidden state)
    Critic: global state → value head     (centralised, sees everything)
    This is the MAPPO architecture: decentralised execution, centralised training.
    """
    def __init__(self, obs_dim: int, act_dim: int, global_dim: int,
                 hidden: int = 256, gru_h: int = 128):
        super().__init__()
        # Actor encoder
        self.actor_enc = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.LayerNorm(hidden), nn.Tanh(),
            nn.Linear(hidden, gru_h),   nn.Tanh(),
        )
        # GRU for temporal memory
        self.gru = nn.GRUCell(gru_h, gru_h)
        # Actor head
        self.actor_head = nn.Sequential(
            nn.Linear(gru_h, hidden//2), nn.Tanh(),
            nn.Linear(hidden//2, act_dim)
        )
        # Centralised critic (sees global state)
        self.critic = nn.Sequential(
            nn.Linear(global_dim, hidden), nn.LayerNorm(hidden), nn.Tanh(),
            nn.Linear(hidden, hidden//2), nn.Tanh(),
            nn.Linear(hidden//2, 1)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.actor_head[-1].weight, gain=0.01)
        nn.init.orthogonal_(self.critic[-1].weight,     gain=1.0)

    def actor_forward(self, obs: torch.Tensor, hx: torch.Tensor):
        """Returns logits + new hidden state."""
        enc = self.actor_enc(obs)
        hx  = self.gru(enc, hx)
        return self.actor_head(hx), hx

    def critic_forward(self, global_state: torch.Tensor):
        return self.critic(global_state)

    def forward(self, obs, hx, global_state):
        logits, hx_new = self.actor_forward(obs, hx)
        value = self.critic_forward(global_state)
        return logits, value, hx_new

# ─────────────────────────────────────────────────────────────────────────────
# 4.  PBT MAPPO POLICY
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class PBTConfig:
    """Hyperparameters that PBT mutates."""
    lr:           float = 1.5e-4
    entropy_coef: float = 0.018
    gamma:        float = 0.99
    clip_eps:     float = 0.2
    score:        float = 0.0   # running reward score

    def mutate(self) -> "PBTConfig":
        c = copy.copy(self)
        c.lr           = float(np.clip(c.lr           * random.uniform(0.8,1.25), 1e-5, 5e-3))
        c.entropy_coef = float(np.clip(c.entropy_coef * random.uniform(0.8,1.25), 0.001, 0.1))
        c.gamma        = float(np.clip(c.gamma        + random.uniform(-0.01,0.01), 0.90, 0.999))
        c.clip_eps     = float(np.clip(c.clip_eps     * random.uniform(0.85,1.15), 0.05, 0.5))
        c.score        = 0.0
        return c

class MAPPOPolicy:
    """
    MAPPO policy with:
    - GRU hidden states per agent
    - Centralised critic
    - PBT hyperparameter population
    - Intrinsic curiosity (visit-count bonus)
    """
    def __init__(self, obs_dim: int, global_dim: int, name: str = "policy"):
        self.name = name
        # PBT population of hyperconfigs
        self.pbt_configs = [PBTConfig() for _ in range(CFG.pbt_population)]
        self.active_pbt  = 0   # which config is currently active

        self.net = GRUActorCritic(obs_dim, CFG.act_dim, global_dim,
                                  CFG.hidden_dim, CFG.gru_hidden).to(DEVICE)
        self._make_opt()

        # Per-agent GRU hidden states {agent_id: tensor}
        self.hidden: Dict[int, torch.Tensor] = {}

        # Rollout buffers
        self._reset()

        # Intrinsic curiosity: visit count grid
        self.visit_grid = np.zeros((CFG.curiosity_grid_h, CFG.curiosity_grid_w),
                                   dtype=np.float32)
        self.steps = 0

    def _make_opt(self):
        cfg = self.pbt_configs[self.active_pbt]
        self.opt = optim.Adam(self.net.parameters(), lr=cfg.lr, eps=1e-5)

    def _reset(self):
        self.buf = {k:[] for k in ["obs","acts","logps","vals","rews","dones","hx","gs"]}

    def get_hidden(self, agent_id: int) -> torch.Tensor:
        if agent_id not in self.hidden:
            self.hidden[agent_id] = torch.zeros(1, CFG.gru_hidden, device=DEVICE)
        return self.hidden[agent_id]

    def clear_hidden(self, agent_id: int):
        if agent_id in self.hidden:
            del self.hidden[agent_id]

    def intrinsic_reward(self, x: float, y: float) -> float:
        gx = int(x / CFG.world_w * CFG.curiosity_grid_w)
        gy = int(y / CFG.world_h * CFG.curiosity_grid_h)
        gx = np.clip(gx, 0, CFG.curiosity_grid_w-1)
        gy = np.clip(gy, 0, CFG.curiosity_grid_h-1)
        count = self.visit_grid[gy, gx]
        self.visit_grid[gy, gx] += 1.0
        # Diminishing bonus: more novel = higher reward
        return CFG.curiosity_coef / (1.0 + math.sqrt(count))

    def store(self, obs, act, logp, val, rew, done, hx, gs):
        self.buf["obs"].append(obs);   self.buf["acts"].append(act)
        self.buf["logps"].append(logp);self.buf["vals"].append(val)
        self.buf["rews"].append(rew);  self.buf["dones"].append(float(done))
        self.buf["hx"].append(hx.cpu().numpy().flatten())
        self.buf["gs"].append(gs)

    @torch.no_grad()
    def batch_act(self, obs_list, agent_ids, global_states):
        """Batched inference with per-agent GRU hidden states."""
        if not obs_list: return [], [], []
        obs_t = torch.tensor(np.array(obs_list),  dtype=torch.float32, device=DEVICE)
        gs_t  = torch.tensor(np.array(global_states), dtype=torch.float32, device=DEVICE)

        # Stack hidden states in same order as obs_list
        hx_list = [self.get_hidden(aid) for aid in agent_ids]
        hx_t    = torch.cat(hx_list, dim=0)   # (N, gru_h)

        # Forward
        enc  = self.net.actor_enc(obs_t)
        hx_new = self.net.gru(enc, hx_t)
        logits = self.net.actor_head(hx_new)
        vals   = self.net.critic_forward(gs_t)

        dist  = torch.distributions.Categorical(logits=logits)
        acts  = dist.sample()
        logps = dist.log_prob(acts)

        # Update hidden states
        for i, aid in enumerate(agent_ids):
            self.hidden[aid] = hx_new[i:i+1].detach()

        return (acts.cpu().numpy(), logps.cpu().numpy(),
                vals.squeeze(-1).cpu().numpy(), hx_new.detach())

    def update(self, global_score: float = 0.0) -> dict:
        n = len(self.buf["obs"])
        if n < CFG.batch_size: return {}

        cfg = self.pbt_configs[self.active_pbt]
        cfg.score = cfg.score * 0.95 + global_score * 0.05

        obs   = torch.tensor(np.array(self.buf["obs"]),  dtype=torch.float32, device=DEVICE)
        acts  = torch.tensor(self.buf["acts"],            dtype=torch.long,    device=DEVICE)
        logps = torch.tensor(self.buf["logps"],           dtype=torch.float32, device=DEVICE)
        vals  = torch.tensor(self.buf["vals"],            dtype=torch.float32, device=DEVICE)
        rews  = torch.tensor(self.buf["rews"],            dtype=torch.float32, device=DEVICE)
        dones = torch.tensor(self.buf["dones"],           dtype=torch.float32, device=DEVICE)
        hxs   = torch.tensor(np.array(self.buf["hx"]),   dtype=torch.float32, device=DEVICE)
        hxs   = hxs.view(n, 1, CFG.gru_hidden)
        gs    = torch.tensor(np.array(self.buf["gs"]),   dtype=torch.float32, device=DEVICE)

        # GAE advantage computation
        adv = torch.zeros_like(rews); gae = 0.0
        for t in reversed(range(n)):
            nv = 0.0 if t==n-1 or dones[t] else vals[t+1].item()
            delta = rews[t] + cfg.gamma*nv*(1-dones[t]) - vals[t]
            gae   = delta + cfg.gamma*CFG.gae_lambda*(1-dones[t])*gae
            adv[t]= gae
        ret = adv + vals
        adv = (adv-adv.mean())/(adv.std()+1e-8)

        m = {"pl":0,"vl":0,"ent":0,"n":0}
        idx = torch.randperm(n)
        for _ in range(CFG.ppo_epochs):
            for s in range(0, n, CFG.batch_size):
                mb = idx[s:s+CFG.batch_size]
                hx_mb = hxs[mb].squeeze(1)  # (B, gru_h)

                # Re-run actor with stored hidden states
                enc_mb  = self.net.actor_enc(obs[mb])
                hx_new  = self.net.gru(enc_mb, hx_mb)
                logit2  = self.net.actor_head(hx_new)
                v2      = self.net.critic_forward(gs[mb])

                d2  = torch.distributions.Categorical(logits=logit2)
                nlp = d2.log_prob(acts[mb]); ent = d2.entropy().mean()
                r   = (nlp-logps[mb]).exp(); a = adv[mb]
                pl  = -torch.min(r*a, r.clamp(1-cfg.clip_eps,1+cfg.clip_eps)*a).mean()
                vl  = F.mse_loss(v2.squeeze(-1), ret[mb])
                loss= pl + CFG.value_coef*vl - cfg.entropy_coef*ent

                self.opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(self.net.parameters(), CFG.max_grad_norm)
                self.opt.step()
                m["pl"]+=pl.item(); m["vl"]+=vl.item(); m["ent"]+=ent.item(); m["n"]+=1

        self._reset(); self.steps += n
        nu = max(m["n"],1)
        return {k:v/nu for k,v in m.items() if k!="n"}

    def pbt_step(self):
        """Exploit: copy best config. Explore: mutate."""
        scores = [c.score for c in self.pbt_configs]
        best_i = int(np.argmax(scores))
        worst_i= int(np.argmin(scores))
        if best_i != worst_i and scores[best_i] > scores[worst_i] + 0.5:
            # Copy best weights + mutate hyperparams for worst
            self.pbt_configs[worst_i] = self.pbt_configs[best_i].mutate()
            print(f"  [PBT] Replaced config {worst_i} with mutated copy of {best_i} "
                  f"(score {scores[best_i]:.2f}→mut)")
        # Always run with best config
        self.active_pbt = best_i
        self._make_opt()

# ─────────────────────────────────────────────────────────────────────────────
# 5.  VILLAGE
# ─────────────────────────────────────────────────────────────────────────────
_VID = 0
class Village:
    def __init__(self, x, y, clan_id=None):
        global _VID; _VID+=1
        self.id=_VID; self.x=float(x); self.y=float(y)
        self.clan_id=clan_id or _VID
        self.is_defended=True; self.founded_step=0
        self.last_warrior_spawn=0
        self.trade_stock=0   # food accumulated for trade
        self.plague_active=False; self.plague_timer=0

    def contains(self, x, y): return math.hypot(x-self.x,y-self.y)<=CFG.village_radius
    def in_territory(self, x, y): return math.hypot(x-self.x,y-self.y)<=CFG.territory_radius
    def dist(self, x, y): return math.hypot(x-self.x,y-self.y)

# ─────────────────────────────────────────────────────────────────────────────
# 6.  ORGANISM
# ─────────────────────────────────────────────────────────────────────────────
_OID = 0
class Organism:
    def __init__(self, x, y, genome=None, energy=None, generation=1, sex=None, clan_id=None):
        global _OID; _OID+=1
        self.id=_OID; self.x=float(x); self.y=float(y)
        self.vx=0.; self.vy=0.
        self.sex=sex or random.choice([Sex.MALE,Sex.FEMALE])
        self.genome=genome or Genome.random(self.sex)
        self.energy=float(energy or CFG.init_energy)
        self.alive=True; self.age=0; self.generation=generation; self.fitness=0.
        self.home_village: Optional[int]=None; self.clan_id=clan_id
        self.pregnant=False; self.gestation=0; self.father_genome=None
        self.mate_timer=0
        self.is_elder=False
        self.sick=False; self.sick_timer=0
        self.carrying_food=False   # trade role
        self.trade_destination: Optional[int]=None
        self._obs=None; self._act=8; self._logp=0.; self._val=0.
        self._prev_energy=self.energy

    # ── 42-dim observation ────────────────────────────────────────────────
    def observe(self, food, enemies, warriors, villages, bad_zones,
                elders, season_phase: float) -> np.ndarray:
        W,H=CFG.world_w,CFG.world_h
        obs=np.zeros(CFG.org_obs_dim,dtype=np.float32)
        obs[0]=self.x/W; obs[1]=self.y/H
        obs[2]=self.energy/CFG.max_energy
        obs[3]=min(self.age/700.,1.)
        obs[4]=float(self.sex==Sex.FEMALE)
        obs[5:14]=self.genome.to_vec()   # 9 genes

        def near(items,gx,gy):
            if not items: return 0.,0.,1.,0.
            d2=[(math.hypot(gx(i)-self.x,gy(i)-self.y),i) for i in items]
            d,it=min(d2,key=lambda t:t[0]); ix,iy=gx(it),gy(it)
            return (np.clip((ix-self.x)/W,-1,1),np.clip((iy-self.y)/H,-1,1),
                    min(d/self.genome.vision,1.),float(d<=self.genome.vision))

        obs[14],obs[15],obs[16],obs[17]=near(food,    lambda f:f[0],lambda f:f[1])
        obs[18],obs[19],obs[20],obs[21]=near(enemies, lambda e:e.x, lambda e:e.y)
        obs[22],obs[23],obs[24],obs[25]=near(warriors,lambda w:w.x, lambda w:w.y)

        hv=[v for v in villages if v.id==self.home_village]
        if hv:
            v=hv[0]; d=v.dist(self.x,self.y)
            obs[26]=np.clip((v.x-self.x)/W,-1,1); obs[27]=np.clip((v.y-self.y)/H,-1,1)
            obs[28]=min(d/(W+H),1.); obs[29]=float(d<=CFG.village_radius)
            obs[30]=float(v.is_defended); obs[31]=float(v.plague_active)

        if bad_zones:
            dsts=[math.hypot(z[0]-self.x,z[1]-self.y) for z in bad_zones]
            obs[32]=min(min(dsts)/W,1.); obs[33]=float(min(dsts)<=CFG.bad_zone_radius)

        obs[34]=np.clip(self.vx/4.,-1,1); obs[35]=np.clip(self.vy/4.,-1,1)
        lf=sum(1 for f in food if math.hypot(f[0]-self.x,f[1]-self.y)<=self.genome.vision)
        obs[36]=min(lf/14.,1.)
        obs[37]=np.clip((self.energy-self._prev_energy)/CFG.max_energy,-1,1)
        obs[38]=float(self.pregnant); obs[39]=float(self.sick)
        obs[40]=float(self.carrying_food)
        obs[41]=math.sin(season_phase*2*math.pi)  # season awareness
        return obs

    def apply_action(self, action):
        d=_ACT_DIRS[action]; spd=self.genome.speed
        if self.pregnant: spd*=0.72
        if self.sick:     spd*=0.80
        self.vx=self.vx*CFG.org_friction+d[0]*CFG.org_accel*spd
        self.vy=self.vy*CFG.org_friction+d[1]*CFG.org_accel*spd
        v=math.hypot(self.vx,self.vy)
        if v>spd: self.vx*=spd/v; self.vy*=spd/v
        self._prev_energy=self.energy
        self.x=float(np.clip(self.x+self.vx,0,CFG.world_w))
        self.y=float(np.clip(self.y+self.vy,0,CFG.world_h))
        if self.x<=0 or self.x>=CFG.world_w: self.vx*=-0.5
        if self.y<=0 or self.y>=CFG.world_h: self.vy*=-0.5
        v_sq=self.vx**2+self.vy**2
        cost=CFG.base_metabolism*(1.+self.genome.metabolism)+CFG.move_cost_factor*v_sq
        if self.pregnant: cost*=1.18
        if self.sick:     cost*=1.12; self.energy-=CFG.sick_damage; self.sick_timer-=1
        if self.sick_timer<=0: self.sick=False
        self.energy-=cost; self.age+=1
        if self.mate_timer>0: self.mate_timer-=1
        if self.age>=CFG.elder_age: self.is_elder=True
        if self.energy<=CFG.starve_energy: self.alive=False

# ─────────────────────────────────────────────────────────────────────────────
# 7.  WARRIOR
# ─────────────────────────────────────────────────────────────────────────────
_WID = 0
class Warrior:
    def __init__(self, x, y, energy=None, home_village=None, clan_id=None):
        global _WID; _WID+=1
        self.id=_WID; self.x=float(x); self.y=float(y)
        self.vx=0.; self.vy=0.
        self.energy=float(energy or CFG.warrior_energy_init)
        self.alive=True; self.age=0; self.kills=0
        self.home_village=home_village; self.clan_id=clan_id
        self.is_war_chief=False
        self._obs=None; self._act=8; self._logp=0.; self._val=0.

    def effective_speed(self):
        return CFG.warrior_max_speed*(1.+CFG.war_chief_bonus) if self.is_war_chief else CFG.warrior_max_speed

    def effective_kill_radius(self):
        return CFG.warrior_kill_radius*(1.+CFG.war_chief_bonus*0.5) if self.is_war_chief else CFG.warrior_kill_radius

    # ── 30-dim observation ────────────────────────────────────────────────
    def observe(self, enemies, organisms, villages, bad_zones, season_phase: float) -> np.ndarray:
        W,H=CFG.world_w,CFG.world_h
        obs=np.zeros(CFG.warrior_obs_dim,dtype=np.float32)
        obs[0]=self.x/W; obs[1]=self.y/H
        obs[2]=self.energy/CFG.warrior_energy_max; obs[3]=min(self.age/600.,1.)

        def ns(items,gx,gy,k=0):
            if not items: return 0.,0.,1.,0.
            d2=[(math.hypot(gx(i)-self.x,gy(i)-self.y),i) for i in items]
            d2.sort(key=lambda t:t[0])
            if k>=len(d2): return 0.,0.,1.,0.
            d,it=d2[k]; ix,iy=gx(it),gy(it)
            return (np.clip((ix-self.x)/W,-1,1),np.clip((iy-self.y)/H,-1,1),
                    min(d/CFG.warrior_vision,1.),float(d<=CFG.warrior_vision))

        obs[4],obs[5],obs[6],obs[7]   =ns(enemies,  lambda e:e.x,lambda e:e.y,0)
        obs[8],obs[9],obs[10],obs[11] =ns(organisms,lambda o:o.x,lambda o:o.y,0)
        obs[12],obs[13],obs[14],obs[15]=ns(enemies,  lambda e:e.x,lambda e:e.y,1)

        mv=[v for v in villages if v.id==self.home_village]
        if mv:
            v=mv[0]
            obs[16]=np.clip((v.x-self.x)/W,-1,1); obs[17]=np.clip((v.y-self.y)/H,-1,1)
            obs[18]=float(v.is_defended); obs[19]=float(v.plague_active)

        obs[20]=np.clip(self.vx/self.effective_speed(),-1,1)
        obs[21]=np.clip(self.vy/self.effective_speed(),-1,1)
        obs[22]=min(sum(1 for e in enemies
                       if math.hypot(e.x-self.x,e.y-self.y)<=CFG.warrior_vision)/8.,1.)
        obs[23]=min(sum(1 for o in organisms
                       if o.alive and math.hypot(o.x-self.x,o.y-self.y)<=CFG.warrior_protect_r)/14.,1.)
        obs[24]=min(self.kills/40.,1.); obs[25]=self.energy/CFG.warrior_energy_max
        obs[26]=float(self.is_war_chief)
        obs[27]=float(any(not v.is_defended and v.id==self.home_village for v in villages))
        obs[28]=math.sin(season_phase*2*math.pi)
        obs[29]=float(any(v.plague_active and v.id==self.home_village for v in villages))
        return obs

    def apply_action(self, action):
        spd=self.effective_speed(); d=_ACT_DIRS[action]
        self.vx=self.vx*CFG.warrior_friction+d[0]*CFG.warrior_accel*spd
        self.vy=self.vy*CFG.warrior_friction+d[1]*CFG.warrior_accel*spd
        v=math.hypot(self.vx,self.vy)
        if v>spd: self.vx*=spd/v; self.vy*=spd/v
        self.x=float(np.clip(self.x+self.vx,0,CFG.world_w))
        self.y=float(np.clip(self.y+self.vy,0,CFG.world_h))
        if self.x<=0 or self.x>=CFG.world_w: self.vx*=-0.5
        if self.y<=0 or self.y>=CFG.world_h: self.vy*=-0.5
        self.energy-=CFG.warrior_metabolism+CFG.warrior_move_cost*(self.vx**2+self.vy**2)
        self.age+=1
        if self.energy<=0: self.alive=False

    def override_charge(self, enemies):
        ae=[e for e in enemies if e.alive]
        if not ae: return
        d,tgt=min([(math.hypot(e.x-self.x,e.y-self.y),e) for e in ae],key=lambda t:t[0])
        if d<=CFG.warrior_vision:
            dx=tgt.x-self.x; dy=tgt.y-self.y; dd=math.hypot(dx,dy)+1e-8
            spd=self.effective_speed()
            self.vx=(dx/dd)*spd; self.vy=(dy/dd)*spd
            self.x=float(np.clip(self.x+self.vx,0,CFG.world_w))
            self.y=float(np.clip(self.y+self.vy,0,CFG.world_h))

    def try_kill_enemies(self, enemies):
        killed=0
        for e in enemies:
            if e.alive and math.hypot(e.x-self.x,e.y-self.y)<=self.effective_kill_radius():
                e.alive=False; self.kills+=1
                self.energy=min(self.energy+CFG.warrior_attack_bonus,CFG.warrior_energy_max)
                killed+=1
        return killed

# ─────────────────────────────────────────────────────────────────────────────
# 8.  ENEMY
# ─────────────────────────────────────────────────────────────────────────────
_EID=0
class Enemy:
    current_speed=CFG.enemy_max_speed_init; current_kr=CFG.enemy_kill_radius_init
    current_vision=CFG.enemy_vision_init; generation=1

    @classmethod
    def grow(cls):
        cls.current_speed =min(cls.current_speed*(1+CFG.enemy_growth_rate),CFG.enemy_max_speed_cap)
        cls.current_kr    =min(cls.current_kr*(1+CFG.enemy_growth_rate),CFG.enemy_kill_radius_cap)
        cls.current_vision=min(cls.current_vision*(1+CFG.enemy_growth_rate*0.5),24.)
        cls.generation+=1

    @classmethod
    def reset(cls):
        cls.current_speed=CFG.enemy_max_speed_init; cls.current_kr=CFG.enemy_kill_radius_init
        cls.current_vision=CFG.enemy_vision_init; cls.generation=1

    def __init__(self,x,y):
        global _EID; _EID+=1
        self.id=_EID; self.x=float(x); self.y=float(y)
        self.vx=0.; self.vy=0.; self.alive=True; self.age=0

    @property
    def speed(self): return Enemy.current_speed
    @property
    def kill_radius(self): return Enemy.current_kr
    @property
    def vision(self): return Enemy.current_vision

    def step(self, organisms, warriors, all_enemies, villages):
        ao=[o for o in organisms if o.alive]; aw=[w for w in warriors if w.alive]
        fx,fy=0.,0.
        for w in aw:
            d=math.hypot(w.x-self.x,w.y-self.y)
            if d<14.: rx=(self.x-w.x)/(d+1e-8); ry=(self.y-w.y)/(d+1e-8); fx+=rx*(14.-d); fy+=ry*(14.-d)
        flee_len=math.hypot(fx,fy)
        cx,cy=0.,0.
        cands=[(math.hypot(o.x-self.x,o.y-self.y),o) for o in ao
               if math.hypot(o.x-self.x,o.y-self.y)<=self.vision]
        if cands:
            cands.sort(key=lambda t:t[1].energy); tgt=cands[0][1]
            dx=tgt.x-self.x; dy=tgt.y-self.y; d=math.hypot(dx,dy)+1e-8; cx=dx/d; cy=dy/d
        elif ao:
            tgt=random.choice(ao); dx=tgt.x-self.x; dy=tgt.y-self.y
            d=math.hypot(dx,dy)+1e-8; cx=dx/d*0.35; cy=dy/d*0.35
        px,py=0.,0.
        pack=[e for e in all_enemies if e is not self and e.alive
              and math.hypot(e.x-self.x,e.y-self.y)<CFG.enemy_pack_radius]
        if pack:
            pcx=sum(e.x for e in pack)/len(pack); pcy=sum(e.y for e in pack)/len(pack)
            dd=math.hypot(pcx-self.x,pcy-self.y)+1e-8
            px=(pcx-self.x)/dd*0.15; py=(pcy-self.y)/dd*0.15
        if flee_len>0:
            fx/=flee_len; fy/=flee_len; mx=0.65*fx+0.25*cx+0.1*px; my=0.65*fy+0.25*cy+0.1*py
        else: mx=0.85*cx+0.15*px; my=0.85*cy+0.15*py
        ml=math.hypot(mx,my)+1e-8; ax=(mx/ml)*self.speed; ay=(my/ml)*self.speed
        self.vx=self.vx*CFG.enemy_friction+ax*CFG.enemy_accel
        self.vy=self.vy*CFG.enemy_friction+ay*CFG.enemy_accel
        v=math.hypot(self.vx,self.vy)
        if v>self.speed: self.vx*=self.speed/v; self.vy*=self.speed/v
        self.x=float(np.clip(self.x+self.vx,0,CFG.world_w))
        self.y=float(np.clip(self.y+self.vy,0,CFG.world_h))
        if self.x<=0 or self.x>=CFG.world_w: self.vx*=-0.6
        if self.y<=0 or self.y>=CFG.world_h: self.vy*=-0.6
        self.age+=1
        for v in villages:
            if v.contains(self.x,self.y) and v.is_defended:
                dx=self.x-v.x; dy=self.y-v.y; dd=math.hypot(dx,dy)+1e-8; push=CFG.village_radius+0.5
                self.x=float(np.clip(v.x+(dx/dd)*push,0,CFG.world_w))
                self.y=float(np.clip(v.y+(dy/dd)*push,0,CFG.world_h))
                self.vx*=-0.3; self.vy*=-0.3

    def try_kill(self, organisms, villages):
        killed=0; breach=0
        for o in organisms:
            if not o.alive: continue
            d=math.hypot(o.x-self.x,o.y-self.y)
            if d>self.kill_radius: continue
            in_v=False; v_def=False
            for v in villages:
                if v.contains(o.x,o.y): in_v=True; v_def=v.is_defended; break
            if in_v and v_def: continue
            elif in_v and not v_def:
                if random.random()<CFG.village_breach_kill_prob: o.alive=False; killed+=1; breach+=1
            else: o.alive=False; killed+=1
        return killed,breach

# ─────────────────────────────────────────────────────────────────────────────
# 9.  GLOBAL STATE  (for centralised critic)
# ─────────────────────────────────────────────────────────────────────────────
def build_global_state(world) -> np.ndarray:
    """24-dim global state for centralised critic — aggregated population stats."""
    W,H=CFG.world_w,CFG.world_h
    ao=world.organisms; aw=world.warriors; ae=world.enemies
    gs=np.zeros(CFG.global_state_dim,dtype=np.float32)
    gs[0] =min(len(ao)/CFG.max_organisms,1.)
    gs[1] =min(len(aw)/CFG.max_warriors,1.)
    gs[2] =min(len(ae)/CFG.max_enemies,1.)
    gs[3] =min(len(world.food)/CFG.max_food,1.)
    gs[4] =min(len(world.villages)/CFG.max_villages,1.)
    gs[5] =Enemy.current_speed/CFG.enemy_max_speed_cap
    gs[6] =Enemy.current_kr/CFG.enemy_kill_radius_cap
    gs[7] =math.sin(world.season_phase*2*math.pi)  # season
    gs[8] =math.cos(world.season_phase*2*math.pi)
    # Population centre of mass
    if ao:
        gs[9] =np.mean([o.x for o in ao])/W
        gs[10]=np.mean([o.y for o in ao])/H
        gs[11]=np.mean([o.energy for o in ao])/CFG.max_energy
        gs[12]=np.mean([o.genome.speed for o in ao])/4.5
        gs[13]=float(sum(1 for o in ao if o.sex==Sex.MALE))/max(len(ao),1)
        gs[14]=float(sum(1 for o in ao if o.pregnant))/max(len(ao),1)
        gs[15]=float(sum(1 for o in ao if o.is_elder))/max(len(ao),1)
    if aw:
        gs[16]=np.mean([w.x for w in aw])/W
        gs[17]=np.mean([w.y for w in aw])/H
        gs[18]=np.mean([w.energy for w in aw])/CFG.warrior_energy_max
        gs[19]=float(sum(1 for w in aw if w.is_war_chief))/max(len(aw),1)
    if ae:
        gs[20]=np.mean([e.x for e in ae])/W
        gs[21]=np.mean([e.y for e in ae])/H
    gs[22]=min(sum(1 for v in world.villages if v.plague_active)/max(len(world.villages),1),1.)
    gs[23]=float(world.step_count)/CFG.max_steps
    return gs

# ─────────────────────────────────────────────────────────────────────────────
# 10.  WORLD
# ─────────────────────────────────────────────────────────────────────────────
class World:
    def __init__(self):
        Enemy.reset()
        self.villages:  List[Village]  = []
        self.organisms: List[Organism] = []
        self.warriors:  List[Warrior]  = []
        self.enemies:   List[Enemy]    = []
        self.food:      List[Tuple]    = []
        self.bad_zones: List[Tuple]    = []
        self.step_count=0; self._last_village_step=-CFG.clan_form_cooldown
        self.season_phase=0.0    # 0→1 per season_period steps
        self.stats={k:[] for k in [
            "pop","pop_m","pop_f","warriors","enemies","food",
            "in_village","n_villages","pregnancies","elders","sick",
            "births_org","births_war","deaths_starve","deaths_killed",
            "deaths_breach","deaths_war","enemy_kills","trade_trips",
            "avg_spd_m","avg_spd_f","avg_vision","avg_gen","avg_immune",
            "enemy_speed","enemy_kr","reward_org","reward_war","season",
        ]}
        self._init()

    def _rp(self,margin=8):
        return (random.uniform(margin,CFG.world_w-margin),
                random.uniform(margin,CFG.world_h-margin))

    def _init(self):
        # Seed initial villages spread across world
        positions=[(CFG.world_w*0.3,CFG.world_h*0.5),(CFG.world_w*0.7,CFG.world_h*0.5)]
        for i in range(min(CFG.init_villages,2)):
            v=Village(*positions[i],clan_id=i+1); v.founded_step=0
            self.villages.append(v)
        att=0
        while len(self.bad_zones)<CFG.n_bad_zones and att<300:
            x,y=self._rp()
            if all(math.hypot(x-v.x,y-v.y)>CFG.village_radius+CFG.bad_zone_radius+10
                   for v in self.villages):
                self.bad_zones.append((x,y))
            att+=1
        for _ in range(CFG.init_food): self.food.append(self._rp())
        for vi,v in enumerate(self.villages):
            per=CFG.init_organisms//len(self.villages)
            for k in range(per):
                ang=random.uniform(0,2*math.pi); r=random.uniform(0,CFG.village_radius*0.85)
                x=float(np.clip(v.x+r*math.cos(ang),0,CFG.world_w))
                y=float(np.clip(v.y+r*math.sin(ang),0,CFG.world_h))
                sex=Sex.MALE if k%2==0 else Sex.FEMALE
                org=Organism(x,y,sex=sex,clan_id=v.clan_id); org.home_village=v.id
                self.organisms.append(org)
            for k in range(CFG.init_warriors//len(self.villages)):
                ang=random.uniform(0,2*math.pi); r=random.uniform(0,CFG.village_radius)
                x=float(np.clip(v.x+r*math.cos(ang),0,CFG.world_w))
                y=float(np.clip(v.y+r*math.sin(ang),0,CFG.world_h))
                self.warriors.append(Warrior(x,y,home_village=v.id,clan_id=v.clan_id))
        for _ in range(CFG.init_enemies):
            for _ in range(100):
                edge=random.choice(["T","B","L","R"])
                if   edge=="T": x,y=random.uniform(0,CFG.world_w),random.uniform(0,8)
                elif edge=="B": x,y=random.uniform(0,CFG.world_w),random.uniform(CFG.world_h-8,CFG.world_h)
                elif edge=="L": x,y=random.uniform(0,8),random.uniform(0,CFG.world_h)
                else:           x,y=random.uniform(CFG.world_w-8,CFG.world_w),random.uniform(0,CFG.world_h)
                if all(math.hypot(x-v.x,y-v.y)>CFG.village_radius+12 for v in self.villages):
                    self.enemies.append(Enemy(x,y)); break

    # ── Season ────────────────────────────────────────────────────────────
    def _update_season(self):
        self.season_phase=(self.step_count%CFG.season_period)/CFG.season_period
        season_mult=1.0+CFG.season_amp*math.sin(self.season_phase*2*math.pi)
        return season_mult

    # ── Village defence ───────────────────────────────────────────────────
    def _update_defence(self,aw):
        for v in self.villages:
            v.is_defended=any(math.hypot(w.x-v.x,w.y-v.y)<=CFG.village_breach_radius for w in aw)

    # ── Auto warrior respawn ──────────────────────────────────────────────
    def _auto_respawn_warriors(self, aw):
        for v in self.villages:
            local_wars=[w for w in aw if w.home_village==v.id]
            if (len(local_wars)<CFG.min_warriors_per_village
                    and self.step_count-v.last_warrior_spawn>=CFG.warrior_respawn_interval
                    and len(self.warriors)<CFG.max_warriors):
                ang=random.uniform(0,2*math.pi); r=random.uniform(0,CFG.village_radius*0.6)
                x=float(np.clip(v.x+r*math.cos(ang),0,CFG.world_w))
                y=float(np.clip(v.y+r*math.sin(ang),0,CFG.world_h))
                self.warriors.append(Warrior(x,y,
                    energy=CFG.warrior_energy_init*0.8,
                    home_village=v.id,clan_id=v.clan_id))
                v.last_warrior_spawn=self.step_count

    # ── War chief election ────────────────────────────────────────────────
    def _elect_war_chief(self, aw):
        for v in self.villages:
            vw=[w for w in aw if w.home_village==v.id]
            if not vw: continue
            best=max(vw,key=lambda w:w.kills)
            for w in vw: w.is_war_chief=False
            if best.kills>5: best.is_war_chief=True

    # ── Disease ───────────────────────────────────────────────────────────
    def _spread_disease(self, ao):
        for v in self.villages:
            local=[o for o in ao if o.alive and v.contains(o.x,o.y)]
            density=len(local)/max(1,(math.pi*CFG.village_radius**2))
            if density>0.12 and random.random()<0.002:
                v.plague_active=True; v.plague_timer=200
            if v.plague_active:
                v.plague_timer-=1
                if v.plague_timer<=0: v.plague_active=False
                for o in local:
                    if not o.sick and random.random()<0.05*(1.-o.genome.immune_strength):
                        o.sick=True; o.sick_timer=CFG.sick_duration

    # ── Trade routes ──────────────────────────────────────────────────────
    def _process_trade(self, ao) -> int:
        trips=0
        for o in ao:
            if not o.alive or not o.carrying_food: continue
            if o.trade_destination is None: o.carrying_food=False; continue
            dv=[v for v in self.villages if v.id==o.trade_destination]
            if not dv: o.carrying_food=False; o.trade_destination=None; continue
            v=dv[0]
            if v.dist(o.x,o.y)<=CFG.village_radius:
                v.trade_stock+=1; o.carrying_food=False
                o.trade_destination=None; o.energy=min(o.energy+15.,CFG.max_energy)
                trips+=1
        # Assign new trade carriers
        for o in ao:
            if not o.alive or o.carrying_food or o.pregnant: continue
            hv=[v for v in self.villages if v.id==o.home_village]
            if not hv: continue
            v=hv[0]
            if (v.trade_stock<5 and random.random()<0.002*o.genome.trade_drive
                    and len(self.villages)>1):
                others=[vv for vv in self.villages if vv.id!=v.id]
                if others:
                    o.carrying_food=True
                    o.trade_destination=random.choice(others).id
        return trips

    # ── Dynamic village formation ─────────────────────────────────────────
    def _try_form_village(self, ao):
        if len(self.villages)>=CFG.max_villages: return
        if self.step_count-self._last_village_step<CFG.clan_form_cooldown: return
        for _ in range(80):
            cx=random.uniform(25,CFG.world_w-25); cy=random.uniform(25,CFG.world_h-25)
            if any(math.hypot(cx-v.x,cy-v.y)<CFG.village_radius*2.8 for v in self.villages): continue
            if any(math.hypot(cx-z[0],cy-z[1])<CFG.bad_zone_radius+CFG.village_radius
                   for z in self.bad_zones): continue
            nearby=[o for o in ao if math.hypot(o.x-cx,o.y-cy)<=CFG.clan_form_radius]
            if len(nearby)>=CFG.clan_form_density:
                rcx=sum(o.x for o in nearby)/len(nearby)
                rcy=sum(o.y for o in nearby)/len(nearby)
                cc={}
                for o in nearby: c=o.clan_id or 0; cc[c]=cc.get(c,0)+1
                dom=max(cc,key=lambda k:cc[k])
                nv=Village(rcx,rcy,clan_id=dom); nv.founded_step=self.step_count
                self.villages.append(nv)
                for o in nearby:
                    if o.home_village is None or random.random()<0.5:
                        o.home_village=nv.id; o.clan_id=dom
                self._last_village_step=self.step_count
                print(f"  [Village+] ({rcx:.0f},{rcy:.0f}) clan={dom} "
                      f"n={len(nearby)} total={len(self.villages)}")
                return

    # ── Overpopulation migration ──────────────────────────────────────────
    def _handle_migration(self, ao):
        for v in self.villages:
            local=[o for o in ao if o.alive and v.contains(o.x,o.y)]
            if len(local)>CFG.overpop_threshold:
                migrants=random.sample(local,len(local)//4)
                candidates=[vv for vv in self.villages if vv.id!=v.id]
                if not candidates: continue
                target=random.choice(candidates)
                for m in migrants:
                    m.home_village=target.id; m.clan_id=target.clan_id

    # ── Food spawn with seasons ───────────────────────────────────────────
    def _spawn_food(self, season_mult: float):
        ratio=len(self.food)/CFG.max_food
        rate=CFG.food_spawn_rate*season_mult*max(0.,1.-ratio**1.4)
        if random.random()<rate and len(self.food)<CFG.max_food:
            if random.random()<0.45 and self.villages:
                v=random.choice(self.villages)
                ang=random.uniform(0,2*math.pi); r=random.uniform(0,CFG.village_radius*1.4)
                self.food.append((float(np.clip(v.x+r*math.cos(ang),0,CFG.world_w)),
                                  float(np.clip(v.y+r*math.sin(ang),0,CFG.world_h))))
            else: self.food.append(self._rp())

    # ── Threshold ─────────────────────────────────────────────────────────
    def _org_threshold(self):
        fr=len(self.food)/max(CFG.max_food,1); return CFG.reproduce_threshold+(1.-fr)*22.

    # ── Rewards ───────────────────────────────────────────────────────────
    def _org_rew(self, o, ate, in_bad, near_enemy, in_village, near_warrior, v_def,
                 curiosity, near_elder):
        r = 3.2 if ate         else 0.
        r -= (2.5*(1.-o.genome.immune_strength)) if in_bad else 0.
        r -= 1.6 if (near_enemy and not in_village) else 0.
        r -= 0.7 if (near_enemy and in_village and not v_def) else 0.
        r += (0.9 if v_def else 0.35) if in_village else 0.
        r += 0.35 if near_warrior else 0.
        r += 0.5  if near_elder   else 0.
        r += curiosity                   # intrinsic exploration bonus
        r += 0.08                        # survival
        r -= 0.12*max(o.genome.speed-2.3,0)
        r -= 0.4 if o.sick else 0.
        return float(r)

    def _war_rew(self, w, near_orgs, in_bad, v_undef, kills):
        r  = 10.0*kills; r += 0.5*near_orgs
        r -= 2.5 if in_bad else 0.; r += 2.0 if v_undef else 0.; r += 0.12
        return float(r)

    # ── Sexual reproduction ───────────────────────────────────────────────
    def _try_mate(self, ao, threshold):
        births=0
        males  =[o for o in ao if o.sex==Sex.MALE  and o.energy>=threshold and o.mate_timer==0]
        females=[o for o in ao if o.sex==Sex.FEMALE and o.energy>=threshold
                 and not o.pregnant and o.mate_timer==0]
        # Elder bonus: increase fertility threshold temporarily
        elders=[o for o in ao if o.is_elder]
        for f in females:
            if len(self.organisms)>=CFG.max_organisms: break
            near_elder=any(math.hypot(e.x-f.x,e.y-f.y)<=CFG.village_radius for e in elders)
            boost=CFG.elder_birth_bonus if near_elder else 0.
            if random.random()>f.genome.fertility+boost: continue
            cands=[(math.hypot(m.x-f.x,m.y-f.y),m) for m in males
                   if math.hypot(m.x-f.x,m.y-f.y)<=f.genome.vision*1.3]
            if not cands: continue
            _,father=min(cands,key=lambda t:t[0])
            f.pregnant=True; f.gestation=CFG.gestation_steps
            f.father_genome=father.genome
            f.energy-=CFG.reproduce_cost*0.58; father.energy-=CFG.reproduce_cost*0.42
            f.mate_timer=55; father.mate_timer=32
            if father in males: males.remove(father)
        for f in [o for o in ao if o.sex==Sex.FEMALE and o.pregnant]:
            f.gestation-=1
            if f.gestation<=0:
                f.pregnant=False
                if f.father_genome is None: continue
                csex=random.choice([Sex.MALE,Sex.FEMALE])
                cg=f.genome.crossover(f.father_genome,csex)
                child=Organism(
                    x=float(np.clip(f.x+random.uniform(-5,5),0,CFG.world_w)),
                    y=float(np.clip(f.y+random.uniform(-5,5),0,CFG.world_h)),
                    genome=cg,sex=csex,energy=CFG.reproduce_cost*0.42,
                    generation=f.generation+1,clan_id=f.clan_id)
                child.home_village=f.home_village; f.father_genome=None
                self.organisms.append(child); births+=1
        return births

    def _reproduce_war(self, war):
        if war.energy<CFG.warrior_rep_threshold or len(self.warriors)>=CFG.max_warriors: return 0
        war.energy-=CFG.warrior_rep_cost
        self.warriors.append(Warrior(
            x=float(np.clip(war.x+random.uniform(-5,5),0,CFG.world_w)),
            y=float(np.clip(war.y+random.uniform(-5,5),0,CFG.world_h)),
            energy=CFG.warrior_rep_cost*0.5,
            home_village=war.home_village,clan_id=war.clan_id)); return 1

    def _enemy_dynamics(self):
        if self.step_count%CFG.enemy_growth_interval==0 and self.step_count>0:
            Enemy.grow()
            print(f"  [Enemy↑] Spd={Enemy.current_speed:.2f} KR={Enemy.current_kr:.2f} Gen={Enemy.generation}")
        if (self.step_count%CFG.enemy_spawn_interval==0 and self.step_count>0
                and len(self.enemies)<CFG.max_enemies):
            for _ in range(100):
                edge=random.choice(["T","B","L","R"])
                if   edge=="T": x,y=random.uniform(0,CFG.world_w),0.
                elif edge=="B": x,y=random.uniform(0,CFG.world_w),float(CFG.world_h)
                elif edge=="L": x,y=0.,random.uniform(0,CFG.world_h)
                else:           x,y=float(CFG.world_w),random.uniform(0,CFG.world_h)
                if all(math.hypot(x-v.x,y-v.y)>CFG.village_radius+6 for v in self.villages):
                    self.enemies.append(Enemy(x,y))
                    print(f"  [Enemy+] ({x:.0f},{y:.0f}) total={len(self.enemies)}"); break

    # ── MAIN STEP ─────────────────────────────────────────────────────────
    def step(self, op: MAPPOPolicy, wp: MAPPOPolicy) -> dict:
        self.step_count+=1
        season_mult=self._update_season()
        ao=[o for o in self.organisms if o.alive]
        aw=[w for w in self.warriors  if w.alive]
        ae=[e for e in self.enemies   if e.alive]
        births_o=0;births_w=0;d_starve=0;d_killed=0;d_breach=0;d_war=0;e_kills=0;trade=0
        rew_o=0.;rew_w=0.

        self._update_defence(aw)
        self._auto_respawn_warriors(aw)
        self._elect_war_chief(aw)

        gs = build_global_state(self)

        # ── Observe + MAPPO act (organisms) ───────────────────────────────
        elders=[o for o in ao if o.is_elder]
        obs_o=[o.observe(self.food,ae,aw,self.villages,self.bad_zones,
                         elders,self.season_phase) for o in ao]
        ids_o=[o.id for o in ao]
        gs_list=[gs]*len(ao)
        acts_o,logps_o,vals_o,hxs_o=op.batch_act(obs_o,ids_o,gs_list)
        for i,o in enumerate(ao):
            o._obs=obs_o[i]; o._act=int(acts_o[i])
            o._logp=float(logps_o[i]); o._val=float(vals_o[i])

        # ── Observe + MAPPO act (warriors) ────────────────────────────────
        obs_w=[w.observe(ae,ao,self.villages,self.bad_zones,self.season_phase) for w in aw]
        ids_w=[w.id for w in aw]
        gs_wlist=[gs]*len(aw)
        acts_w,logps_w,vals_w,hxs_w=wp.batch_act(obs_w,ids_w,gs_wlist)
        for i,w in enumerate(aw):
            w._obs=obs_w[i]; w._act=int(acts_w[i])
            w._logp=float(logps_w[i]); w._val=float(vals_w[i])

        # ── Move organisms (flee heuristic + home return) ──────────────────
        for o in ao:
            fled=False
            for e in ae:
                d=math.hypot(e.x-o.x,e.y-o.y)
                if d<o.genome.vision*0.55:
                    dx=o.x-e.x;dy=o.y-e.y;dd=math.hypot(dx,dy)+1e-8
                    fx=dx/dd;fy=dy/dd;best=8;bdot=-99
                    for ai in range(8):
                        dot=_ACT_DIRS[ai][0]*fx+_ACT_DIRS[ai][1]*fy
                        if dot>bdot: bdot=dot;best=ai
                    o._act=best;fled=True;break
            in_v=any(v.contains(o.x,o.y) for v in self.villages)
            # Trade: head to destination
            if o.carrying_food and o.trade_destination and not fled:
                dv=[v for v in self.villages if v.id==o.trade_destination]
                if dv:
                    v=dv[0];dx=v.x-o.x;dy=v.y-o.y;dd=math.hypot(dx,dy)+1e-8
                    best=8;bdot=-99
                    for ai in range(8):
                        dot=_ACT_DIRS[ai][0]*(dx/dd)+_ACT_DIRS[ai][1]*(dy/dd)
                        if dot>bdot:bdot=dot;best=ai
                    o._act=best
            elif in_v and o.energy>CFG.reproduce_threshold*0.72 and not fled:
                of=[(math.hypot(f[0]-o.x,f[1]-o.y),f) for f in self.food
                    if not any(v.contains(f[0],f[1]) for v in self.villages)]
                if of:
                    _,tf=min(of,key=lambda t:t[0])
                    dx=tf[0]-o.x;dy=tf[1]-o.y;dd=math.hypot(dx,dy)+1e-8
                    best=8;bdot=-99
                    for ai in range(8):
                        dot=_ACT_DIRS[ai][0]*(dx/dd)+_ACT_DIRS[ai][1]*(dy/dd)
                        if dot>bdot:bdot=dot;best=ai
                    o._act=best
            hv=[v for v in self.villages if v.id==o.home_village]
            if hv and o.energy<CFG.init_energy*0.32 and not fled and not o.carrying_food:
                v=hv[0];dx=v.x-o.x;dy=v.y-o.y;dd=math.hypot(dx,dy)+1e-8
                best=8;bdot=-99
                for ai in range(8):
                    dot=_ACT_DIRS[ai][0]*(dx/dd)+_ACT_DIRS[ai][1]*(dy/dd)
                    if dot>bdot:bdot=dot;best=ai
                o._act=best
            o.apply_action(o._act)
            if not o.alive: d_starve+=1

        # ── Move warriors ─────────────────────────────────────────────────
        for w in aw:
            w.apply_action(w._act); w.override_charge(ae)
            if not w.alive: d_war+=1

        # ── Move enemies ──────────────────────────────────────────────────
        for e in ae: e.step(ao,aw,ae,self.villages)

        # ── Eat food ──────────────────────────────────────────────────────
        food_set=list(self.food);taken=set()
        for o in ao:
            if not o.alive or o.carrying_food: continue
            for fi,f in enumerate(food_set):
                if fi in taken: continue
                er=(CFG.village_food_radius if any(v.contains(o.x,o.y) for v in self.villages)
                    else CFG.food_eat_radius)
                if math.hypot(f[0]-o.x,f[1]-o.y)<=er:
                    o.energy=min(o.energy+CFG.food_energy,CFG.max_energy); taken.add(fi); break
        for w in aw:
            if not w.alive: continue
            for fi,f in enumerate(food_set):
                if fi in taken: continue
                if math.hypot(f[0]-w.x,f[1]-w.y)<=CFG.food_eat_radius:
                    w.energy=min(w.energy+CFG.warrior_food_energy,CFG.warrior_energy_max)
                    taken.add(fi); break
        self.food=[f for i,f in enumerate(food_set) if i not in taken]

        # ── Warriors kill enemies ─────────────────────────────────────────
        for w in aw:
            if w.alive: e_kills+=w.try_kill_enemies(ae)

        # ── Bad zone damage ───────────────────────────────────────────────
        for o in ao:
            if o.alive and any(math.hypot(z[0]-o.x,z[1]-o.y)<=CFG.bad_zone_radius
                               for z in self.bad_zones):
                dmg=CFG.bad_zone_damage*(1.-o.genome.immune_strength*0.5)
                o.energy-=dmg
                if o.energy<=0: o.alive=False; d_starve+=1

        # ── Enemies kill organisms ────────────────────────────────────────
        for e in ae:
            if e.alive: k,br=e.try_kill(ao,self.villages); d_killed+=k; d_breach+=br

        # ── Disease ───────────────────────────────────────────────────────
        self._spread_disease(ao)

        # ── Trade ─────────────────────────────────────────────────────────
        trade=self._process_trade(ao)

        # ── MAPPO rewards (organism) ──────────────────────────────────────
        threshold=self._org_threshold()
        for o in ao:
            if o._obs is None: continue
            hid=op.get_hidden(o.id)
            in_bad=any(math.hypot(z[0]-o.x,z[1]-o.y)<=CFG.bad_zone_radius for z in self.bad_zones)
            n_enemy=any(math.hypot(e.x-o.x,e.y-o.y)<=o.genome.vision for e in ae if e.alive)
            in_vill=any(v.contains(o.x,o.y) for v in self.villages)
            n_war=any(math.hypot(w.x-o.x,w.y-o.y)<=CFG.warrior_protect_r for w in aw if w.alive)
            v_def=next((v.is_defended for v in self.villages if v.contains(o.x,o.y)),True)
            n_elder=any(math.hypot(e.x-o.x,e.y-o.y)<=CFG.village_radius
                        for e in elders if e.id!=o.id)
            ate=o.energy>o._prev_energy
            cur=op.intrinsic_reward(o.x,o.y)
            rew=self._org_rew(o,ate,in_bad,n_enemy,in_vill,n_war,v_def,cur,n_elder)
            o.fitness+=rew; rew_o+=rew
            op.store(o._obs,o._act,o._logp,o._val,rew,not o.alive,hid,gs)

        # ── MAPPO rewards (warrior) ───────────────────────────────────────
        for w in aw:
            if w._obs is None: continue
            hid=wp.get_hidden(w.id)
            in_bad=any(math.hypot(z[0]-w.x,z[1]-w.y)<=CFG.bad_zone_radius for z in self.bad_zones)
            n_orgs=sum(1 for o in ao if o.alive and math.hypot(o.x-w.x,o.y-w.y)<=CFG.warrior_protect_r)
            v_undef=any(not v.is_defended and v.id==w.home_village for v in self.villages)
            rew=self._war_rew(w,n_orgs,in_bad,v_undef,0)
            rew_w+=rew
            wp.store(w._obs,w._act,w._logp,w._val,rew,not w.alive,hid,gs)

        # ── Reproduction ──────────────────────────────────────────────────
        births_o=self._try_mate(ao,threshold)
        for w in aw:
            if w.alive: births_w+=self._reproduce_war(w)

        # ── Clean up dead hidden states ───────────────────────────────────
        for o in ao:
            if not o.alive: op.clear_hidden(o.id)
        for w in aw:
            if not w.alive: wp.clear_hidden(w.id)

        # ── Civilisation systems ──────────────────────────────────────────
        self._spawn_food(season_mult)
        self._enemy_dynamics()
        self._try_form_village(self.organisms)
        self._handle_migration(self.organisms)

        # ── Prune dead ────────────────────────────────────────────────────
        self.organisms=[o for o in self.organisms if o.alive]
        self.warriors =[w for w in self.warriors  if w.alive]
        self.enemies  =[e for e in self.enemies   if e.alive]

        # ── Stats ─────────────────────────────────────────────────────────
        ao2=self.organisms;aw2=self.warriors;ae2=self.enemies
        males=[o for o in ao2 if o.sex==Sex.MALE];females=[o for o in ao2 if o.sex==Sex.FEMALE]
        in_v=sum(1 for o in ao2 if any(v.contains(o.x,o.y) for v in self.villages))
        preg=sum(1 for o in ao2 if o.pregnant)
        eld=sum(1 for o in ao2 if o.is_elder)
        sick=sum(1 for o in ao2 if o.sick)
        self.stats["pop"].append(len(ao2));self.stats["pop_m"].append(len(males))
        self.stats["pop_f"].append(len(females));self.stats["warriors"].append(len(aw2))
        self.stats["enemies"].append(len(ae2));self.stats["food"].append(len(self.food))
        self.stats["in_village"].append(in_v);self.stats["n_villages"].append(len(self.villages))
        self.stats["pregnancies"].append(preg);self.stats["elders"].append(eld)
        self.stats["sick"].append(sick)
        self.stats["births_org"].append(births_o);self.stats["births_war"].append(births_w)
        self.stats["deaths_starve"].append(d_starve);self.stats["deaths_killed"].append(d_killed)
        self.stats["deaths_breach"].append(d_breach);self.stats["deaths_war"].append(d_war)
        self.stats["enemy_kills"].append(e_kills);self.stats["trade_trips"].append(trade)
        self.stats["avg_spd_m"].append(np.mean([o.genome.speed for o in males]) if males else 0)
        self.stats["avg_spd_f"].append(np.mean([o.genome.speed for o in females]) if females else 0)
        self.stats["avg_vision"].append(np.mean([o.genome.vision for o in ao2]) if ao2 else 0)
        self.stats["avg_gen"].append(np.mean([o.generation for o in ao2]) if ao2 else 0)
        self.stats["avg_immune"].append(np.mean([o.genome.immune_strength for o in ao2]) if ao2 else 0)
        self.stats["enemy_speed"].append(Enemy.current_speed)
        self.stats["enemy_kr"].append(Enemy.current_kr)
        self.stats["reward_org"].append(rew_o);self.stats["reward_war"].append(rew_w)
        self.stats["season"].append(season_mult)
        return {"pop":len(ao2),"male":len(males),"female":len(females),
                "war":len(aw2),"enm":len(ae2),"food":len(self.food),
                "in_v":in_v,"n_v":len(self.villages),"preg":preg,"eld":eld,
                "births":births_o,"births_w":births_w,
                "killed":d_killed,"breach":d_breach,"d_war":d_war,
                "ekills":e_kills,"trade":trade,"rew_o":rew_o,"rew_w":rew_w}

# ─────────────────────────────────────────────────────────────────────────────
# 11.  RENDERER
# ─────────────────────────────────────────────────────────────────────────────
class Renderer:
    def __init__(self):
        self.fig=plt.figure(figsize=(30,18),facecolor="#0d1117")
        gs=self.fig.add_gridspec(3,5,wspace=0.40,hspace=0.52)
        self.axes=[self.fig.add_subplot(gs[r,c]) for r in range(3) for c in range(5)]
        os.makedirs(CFG.plot_dir,exist_ok=True) if CFG.save_plot else None
        self._frame=0

    def _sty(self,ax,title="",xl="",yl=""):
        ax.set_facecolor("#161b22");ax.tick_params(colors="#8b949e",labelsize=6.5)
        ax.spines[:].set_color("#30363d")
        ax.set_title(title,color="#e6edf3",fontsize=7.5,pad=4,fontweight="bold")
        ax.set_xlabel(xl,color="#8b949e",fontsize=6.5);ax.set_ylabel(yl,color="#8b949e",fontsize=6.5)

    def render(self,world:World,step:int):
        for ax in self.axes: ax.cla()
        ao=world.organisms;aw=world.warriors
        ae=[e for e in world.enemies if e.alive]
        st=world.stats;xs=list(range(len(st["pop"])))
        sm=lambda s,n:(np.convolve(s,np.ones(n)/n,mode="same").tolist() if len(s)>=n else s)
        males=[o for o in ao if o.sex==Sex.MALE];females=[o for o in ao if o.sex==Sex.FEMALE]
        elders=[o for o in ao if o.is_elder];sick=[o for o in ao if o.sick]
        chiefs=[w for w in aw if w.is_war_chief];preg=[o for o in ao if o.pregnant]

        # ── [0] World Map ─────────────────────────────────────────────────
        ax=self.axes[0];self._sty(ax,f"Living World [step {step}]","X","Y")
        ax.set_xlim(0,CFG.world_w);ax.set_ylim(0,CFG.world_h)
        for v in world.villages:
            col="#30d158" if v.is_defended else "#ff9f0a"
            ax.add_patch(plt.Circle((v.x,v.y),CFG.territory_radius,color=col,alpha=0.04,zorder=0))
            ax.add_patch(plt.Circle((v.x,v.y),CFG.village_radius,color=col,alpha=0.14,zorder=1))
            ax.add_patch(plt.Circle((v.x,v.y),CFG.village_radius,fill=False,edgecolor=col,lw=1.8,alpha=0.8,zorder=2))
            label=f"V{v.id}"+(f"\n⚠" if not v.is_defended else "")+(f"\n☣" if v.plague_active else "")
            ax.text(v.x,v.y,label,color=col,fontsize=5,ha="center",va="center",zorder=3,fontweight="bold")
        for z in world.bad_zones:
            ax.add_patch(plt.Circle(z,CFG.bad_zone_radius,color="#ff3b30",alpha=0.15,zorder=1))
            ax.add_patch(plt.Circle(z,CFG.bad_zone_radius,fill=False,edgecolor="#ff3b30",lw=1,alpha=0.5,zorder=2))
        if world.food:
            fx,fy=zip(*world.food); ax.scatter(fx,fy,s=3.5,c="#30d158",alpha=0.4,zorder=3,marker=".",linewidths=0)
        if males:
            me=np.array([o.energy/CFG.max_energy for o in males]); ms=5+13*me
            ax.scatter([o.x for o in males],[o.y for o in males],s=ms,c="#0a84ff",alpha=0.8,zorder=4,linewidths=0.1)
        if females:
            fe=np.array([o.energy/CFG.max_energy for o in females]); fs=6+14*fe
            ax.scatter([o.x for o in females],[o.y for o in females],s=fs,c="#ff375f",alpha=0.8,zorder=4,linewidths=0.1)
        if preg:   ax.scatter([o.x for o in preg],[o.y for o in preg],s=11,c="#ffd60a",zorder=5,marker="*",linewidths=0)
        if elders: ax.scatter([o.x for o in elders],[o.y for o in elders],s=14,c="#bf5af2",zorder=5,marker="D",linewidths=0)
        if sick:   ax.scatter([o.x for o in sick],[o.y for o in sick],s=10,c="#ff9f0a",zorder=5,marker="x",linewidths=0.8)
        if aw:
            ax.scatter([w.x for w in aw],[w.y for w in aw],s=42,c="#30d158",marker="D",zorder=6,edgecolors="#fff",lw=0.5)
        if chiefs: ax.scatter([w.x for w in chiefs],[w.y for w in chiefs],s=70,c="#ffd60a",marker="*",zorder=7)
        for e in ae:
            sz=min(65+45*(Enemy.generation-1),200)
            ax.scatter(e.x,e.y,s=sz,c="#ff453a",marker="^",zorder=8,edgecolors="#fff",lw=0.5)
        leg=[mpatches.Patch(color="#0a84ff",label=f"♂({len(males)})"),
             mpatches.Patch(color="#ff375f",label=f"♀({len(females)})"),
             mpatches.Patch(color="#ffd60a",label=f"Preg({len(preg)})"),
             mpatches.Patch(color="#bf5af2",label=f"Elder({len(elders)})"),
             mpatches.Patch(color="#ff9f0a",label=f"Sick({len(sick)})"),
             mpatches.Patch(color="#30d158",label=f"War({len(aw)})"),
             mpatches.Patch(color="#ffd60a",label=f"Chief({len(chiefs)})"),
             mpatches.Patch(color="#ff453a",label=f"Enemy({len(ae)})")]
        ax.legend(handles=leg,loc="upper right",fontsize=4.5,
                  facecolor="#21262d",labelcolor="#e6edf3",edgecolor="#30363d",ncol=2)

        # ── [1] Population ────────────────────────────────────────────────
        ax=self.axes[1];self._sty(ax,"Population","Step","Count")
        ax.plot(xs,st["pop"],color="#ffffff",lw=1.3,label="Total")
        ax.plot(xs,st["pop_m"],color="#0a84ff",lw=1.0,label="♂")
        ax.plot(xs,st["pop_f"],color="#ff375f",lw=1.0,label="♀")
        ax.plot(xs,st["warriors"],color="#30d158",lw=1.0,label="War")
        ax.plot(xs,st["enemies"],color="#ff453a",lw=0.9,alpha=0.7,label="Enm")
        ax.legend(fontsize=5,facecolor="#21262d",labelcolor="#e6edf3",edgecolor="#30363d",ncol=2)

        # ── [2] Reproduction & Health ─────────────────────────────────────
        ax=self.axes[2];self._sty(ax,"Reproduction & Health","Step","Count")
        ax.plot(xs,sm(st["pregnancies"],10),color="#ffd60a",lw=1.2,label="Pregnant")
        ax.plot(xs,sm(st["births_org"],10), color="#30d158",lw=1.2,label="Births")
        ax.plot(xs,sm(st["sick"],10),       color="#ff9f0a",lw=1.0,label="Sick")
        ax.plot(xs,sm(st["elders"],5),      color="#bf5af2",lw=0.9,label="Elders")
        ax.legend(fontsize=5,facecolor="#21262d",labelcolor="#e6edf3",edgecolor="#30363d")

        # ── [3] Village & Trade ───────────────────────────────────────────
        ax=self.axes[3];self._sty(ax,"Villages & Trade","Step","Count")
        ax.plot(xs,st["n_villages"],color="#30d158",lw=1.4,label="Villages")
        ax.plot(xs,sm(st["in_village"],5),color="#64d2ff",lw=1.0,label="In Village")
        ax2=ax.twinx();ax2.tick_params(colors="#8b949e",labelsize=6.5);ax2.spines[:].set_color("#30363d")
        ax2.plot(xs,sm(st["trade_trips"],10),color="#ff9f0a",lw=1.0,linestyle="--",label="Trades")
        ax.set_ylabel("Count",color="#30d158",fontsize=6.5);ax2.set_ylabel("Trades",color="#ff9f0a",fontsize=6.5)
        h1,l1=ax.get_legend_handles_labels();h2,l2=ax2.get_legend_handles_labels()
        ax.legend(h1+h2,l1+l2,fontsize=5,facecolor="#21262d",labelcolor="#e6edf3",edgecolor="#30363d")

        # ── [4] Seasons ───────────────────────────────────────────────────
        ax=self.axes[4];self._sty(ax,"Season & Food","Step","Food")
        ax.plot(xs,st["food"],color="#30d158",lw=1.2,label="Food")
        ax2=ax.twinx();ax2.tick_params(colors="#8b949e",labelsize=6.5);ax2.spines[:].set_color("#30363d")
        ax2.plot(xs,st["season"],color="#ffd60a",lw=1.0,linestyle="--",label="Season mult")
        ax.set_ylabel("Food",color="#30d158",fontsize=6.5);ax2.set_ylabel("Season",color="#ffd60a",fontsize=6.5)
        h1,l1=ax.get_legend_handles_labels();h2,l2=ax2.get_legend_handles_labels()
        ax.legend(h1+h2,l1+l2,fontsize=5,facecolor="#21262d",labelcolor="#e6edf3",edgecolor="#30363d")

        # ── [5] Deaths ────────────────────────────────────────────────────
        ax=self.axes[5];self._sty(ax,"Deaths","Step","Count")
        ax.plot(xs,sm(st["deaths_starve"],10),color="#ff9f0a",lw=1.0,label="Starve")
        ax.plot(xs,sm(st["deaths_killed"],10),color="#ff453a",lw=1.0,label="Killed")
        ax.plot(xs,sm(st["deaths_breach"],10),color="#ff375f",lw=0.9,linestyle="--",label="Breach")
        ax.plot(xs,sm(st["deaths_war"],10),   color="#0a84ff",lw=0.9,label="Warrior ☠")
        ax.plot(xs,sm(st["enemy_kills"],10),  color="#30d158",lw=1.1,label="EnemyKills")
        ax.legend(fontsize=5,facecolor="#21262d",labelcolor="#e6edf3",edgecolor="#30363d")

        # ── [6] Enemy Evolution ───────────────────────────────────────────
        ax=self.axes[6];self._sty(ax,"Enemy Evolution","Step","Speed")
        ax2=ax.twinx();ax2.tick_params(colors="#8b949e",labelsize=6.5);ax2.spines[:].set_color("#30363d")
        ax.plot(xs,st["enemy_speed"],color="#ff453a",lw=1.3,label="Speed")
        ax2.plot(xs,st["enemy_kr"],color="#ff9f0a",lw=1.1,linestyle="--",label="Kill R")
        ax.set_ylabel("Speed",color="#ff453a",fontsize=6.5);ax2.set_ylabel("Kill R",color="#ff9f0a",fontsize=6.5)
        h1,l1=ax.get_legend_handles_labels();h2,l2=ax2.get_legend_handles_labels()
        ax.legend(h1+h2,l1+l2,fontsize=5,facecolor="#21262d",labelcolor="#e6edf3",edgecolor="#30363d")

        # ── [7] Gene: Speed dimorphism ────────────────────────────────────
        ax=self.axes[7];self._sty(ax,"Speed Gene ♂ vs ♀","Step","Speed")
        ax.plot(xs,st["avg_spd_m"],color="#0a84ff",lw=1.2,label="♂ Avg Speed")
        ax.plot(xs,st["avg_spd_f"],color="#ff375f",lw=1.1,label="♀ Avg Speed")
        ax.legend(fontsize=5,facecolor="#21262d",labelcolor="#e6edf3",edgecolor="#30363d")

        # ── [8] Gene: Vision & Immune ─────────────────────────────────────
        ax=self.axes[8];self._sty(ax,"Vision & Immunity Gene","Step","Value")
        ax.plot(xs,st["avg_vision"],color="#64d2ff",lw=1.2,label="Avg Vision")
        ax.plot(xs,st["avg_immune"],color="#30d158",lw=1.1,label="Avg Immunity")
        ax.legend(fontsize=5,facecolor="#21262d",labelcolor="#e6edf3",edgecolor="#30363d")

        # ── [9] Generation ────────────────────────────────────────────────
        ax=self.axes[9];self._sty(ax,"Avg Generation","Step","Gen")
        ax.plot(xs,st["avg_gen"],color="#ffd60a",lw=1.4)
        if xs: ax.fill_between(xs,st["avg_gen"],alpha=0.12,color="#ffd60a")

        # ── [10] Births ───────────────────────────────────────────────────
        ax=self.axes[10];self._sty(ax,"Births (smoothed)","Step","Count")
        ax.plot(xs,sm(st["births_org"],25),color="#ff9f0a",lw=1.2,label="Org")
        ax.plot(xs,sm(st["births_war"],25),color="#30d158",lw=1.0,label="War")
        ax.legend(fontsize=5,facecolor="#21262d",labelcolor="#e6edf3",edgecolor="#30363d")

        # ── [11] Reward ───────────────────────────────────────────────────
        ax=self.axes[11];self._sty(ax,"MAPPO Reward / Step","Step","Reward")
        ax.plot(xs,sm(st["reward_org"],25),color="#ff9f0a",lw=1.2,label="Org")
        ax.plot(xs,sm(st["reward_war"],25),color="#30d158",lw=1.1,label="Warrior")
        ax.axhline(0,color="#30363d",lw=0.8,linestyle="--")
        ax.legend(fontsize=5,facecolor="#21262d",labelcolor="#e6edf3",edgecolor="#30363d")

        # ── [12] Curiosity heatmap ────────────────────────────────────────
        ax=self.axes[12];self._sty(ax,"Curiosity Visit Map","X","Y")
        vm=op_ref[0].visit_grid if op_ref else None
        if vm is not None:
            ax.imshow(vm,aspect="auto",cmap="hot",origin="lower",
                      extent=[0,CFG.world_w,0,CFG.world_h],alpha=0.9)
        ax.set_xlim(0,CFG.world_w);ax.set_ylim(0,CFG.world_h)

        # ── [13] PBT config scores ────────────────────────────────────────
        ax=self.axes[13];self._sty(ax,"PBT Config Scores","Config","Score")
        if op_ref:
            scores=[c.score for c in op_ref[0].pbt_configs]
            bars=ax.bar(range(len(scores)),scores,color=["#0a84ff","#30d158","#ff9f0a","#ff453a"])
            ax.set_xticks(range(len(scores)))
            ax.set_xticklabels([f"C{i}" for i in range(len(scores))],
                               color="#8b949e",fontsize=6)
            active=op_ref[0].active_pbt
            if 0<=active<len(bars): bars[active].set_edgecolor("#ffffff"); bars[active].set_linewidth(1.5)

        # ── [14] Civilisation overview ────────────────────────────────────
        ax=self.axes[14];self._sty(ax,"Civilisation Stats","","")
        ax.axis("off")
        wc=sum(w.kills for w in aw)
        tc=sum(world.stats["trade_trips"]) if world.stats["trade_trips"] else 0
        pl_v=sum(1 for v in world.villages if v.plague_active)
        lines=[
            f"Step: {step}",
            f"Total orgs born: {sum(st['births_org'])}",
            f"Total enemy kills: {sum(st['enemy_kills'])}",
            f"Total breach deaths: {sum(st['deaths_breach'])}",
            f"Total trades: {tc}",
            f"Plague villages: {pl_v}",
            f"War chief kills: {wc}",
            f"Enemy gen: {Enemy.generation}",
            f"Villages founded: {len(world.villages)}",
            f"PBT active: C{op_ref[0].active_pbt if op_ref else 0}",
            f"Curiosity total: {op_ref[0].visit_grid.sum():.0f}" if op_ref else "",
        ]
        for i,l in enumerate(lines):
            ax.text(0.05,0.95-i*0.085,l,transform=ax.transAxes,
                    color="#e6edf3",fontsize=7.5,va="top")

        self.fig.suptitle(
            f"Organism Living System v5  ·  MAPPO+GRU+PBT  ·  Step {step}  |  "
            f"Orgs {len(ao)} (♂{len(males)} ♀{len(females)})  |  "
            f"Warriors {len(aw)}  |  Enemies {len(ae)} (Gen {Enemy.generation})  |  "
            f"Villages {len(world.villages)}  |  Food {len(world.food)}  |  "
            f"Season {world.season_phase:.2f}",
            color="#e6edf3",fontsize=9.5,fontweight="bold",y=1.002)
        plt.tight_layout()
        if CFG.save_plot:
            self.fig.savefig(os.path.join(CFG.plot_dir,f"frame_{self._frame:05d}.png"),
                             dpi=100,bbox_inches="tight",facecolor=self.fig.get_facecolor())
            self._frame+=1
        try: plt.pause(0.001)
        except: pass

op_ref = []  # global ref for renderer access to policy

# ─────────────────────────────────────────────────────────────────────────────
# 12.  MAIN
# ─────────────────────────────────────────────────────────────────────────────
def run():
    print("\n"+"═"*74)
    print("  ORGANISM LIVING SYSTEM v5  —  MAPPO + GRU Memory + PBT")
    print("  Civilisation: Seasons · Trade · Disease · Elders · War Chiefs")
    print("═"*74)
    print(f"  World {CFG.world_w}×{CFG.world_h}  |  RL: MAPPO+GRU{CFG.gru_hidden}+PBT×{CFG.pbt_population}")
    print(f"  Obs: org={CFG.org_obs_dim}  war={CFG.warrior_obs_dim}  global={CFG.global_state_dim}")
    print(f"  Warrior auto-respawn: min {CFG.min_warriors_per_village}/village every {CFG.warrior_respawn_interval}s")
    print("═"*74+"\n")

    world=World()
    op=MAPPOPolicy(CFG.org_obs_dim,    CFG.global_state_dim, name="org")
    wp=MAPPOPolicy(CFG.warrior_obs_dim,CFG.global_state_dim, name="warrior")
    op_ref.append(op)   # make available to renderer
    rend=Renderer(); t0=time.time(); mo={}; mw={}; respawns=0

    for step in range(1,CFG.max_steps+1):
        info=world.step(op,wp)

        if len(op.buf["obs"])>=CFG.rollout_len:
            mo=op.update(global_score=info["rew_o"])
        if len(wp.buf["obs"])>=CFG.rollout_len:
            mw=wp.update(global_score=info["rew_w"])

        # PBT step
        if step%CFG.pbt_interval==0:
            op.pbt_step(); wp.pbt_step()

        if step%50==0:
            el=time.time()-t0
            print(f"Step {step:5d} | "
                  f"Org {info['pop']:4d}(♂{info['male']:3d}♀{info['female']:3d}"
                  f"V{info['in_v']:3d}E{info['eld']:2d}) | "
                  f"War {info['war']:3d} | Enm {info['enm']:2d}(G{Enemy.generation}) | "
                  f"V{info['n_v']} | Fd {info['food']:4d} | "
                  f"B{info['births']:2d} K{info['killed']:2d} "
                  f"Br{info['breach']:2d} EK{info['ekills']:2d} Tr{info['trade']:2d} | "
                  f"OPL {mo.get('pl',0):+.3f} WPL {mw.get('pl',0):+.3f} | "
                  f"PBT C{op.active_pbt} | {el:.0f}s")

        if step%CFG.render_every==0: rend.render(world,step)

        if not world.organisms:
            respawns+=1
            print(f"\n⚠  Extinction at step {step}. Respawn #{respawns}")
            if respawns<=8:
                for v in world.villages:
                    for k in range(18):
                        ang=random.uniform(0,2*math.pi); r=random.uniform(0,CFG.village_radius*0.7)
                        sex=Sex.MALE if k%2==0 else Sex.FEMALE
                        org=Organism(float(np.clip(v.x+r*math.cos(ang),0,CFG.world_w)),
                                     float(np.clip(v.y+r*math.sin(ang),0,CFG.world_h)),
                                     sex=sex,energy=CFG.init_energy*0.75,clan_id=v.clan_id)
                        org.home_village=v.id; world.organisms.append(org)
            else:
                print("   Too many extinctions. Ending."); break

        if not world.enemies and step<CFG.max_steps-300:
            world.enemies.append(Enemy(*world._rp()))

    rend.render(world,world.step_count)
    print("\n"+"═"*74)
    ao=world.organisms;aw=world.warriors
    males=[o for o in ao if o.sex==Sex.MALE];females=[o for o in ao if o.sex==Sex.FEMALE]
    print(f"  Final: Orgs {len(ao)} (♂{len(males)} ♀{len(females)}) | Warriors {len(aw)}")
    print(f"  Villages {len(world.villages)} | Enemies {len(world.enemies)} (Gen {Enemy.generation})")
    if ao:
        gens=[o.generation for o in ao]
        print(f"  Gen avg {np.mean(gens):.2f} max {max(gens)}")
        if males: print(f"  ♂ speed {np.mean([o.genome.speed for o in males]):.3f}")
        if females: print(f"  ♀ speed {np.mean([o.genome.speed for o in females]):.3f}")
    print(f"  Total births: {sum(world.stats['births_org'])}")
    print(f"  Total enemy kills: {sum(world.stats['enemy_kills'])}")
    print(f"  Total trades: {sum(world.stats['trade_trips'])}")
    print(f"  PBT best config: C{op.active_pbt} score {op.pbt_configs[op.active_pbt].score:.2f}")
    print("═"*74+"\n")
    fp=os.path.join(CFG.plot_dir,"final_summary_v5.png")
    rend.fig.savefig(fp,dpi=125,bbox_inches="tight",facecolor=rend.fig.get_facecolor())
    print(f"[Done] → {fp}")
    return world,op,wp,rend

if __name__=="__main__":
    # CFG.max_steps=3000; CFG.render_every=200  # faster test run
    world,op,wp,rend=run()

[Device] cuda  |  Tesla T4  |  15.6 GB VRAM

══════════════════════════════════════════════════════════════════════════
  ORGANISM LIVING SYSTEM v5  —  MAPPO + GRU Memory + PBT
  Civilisation: Seasons · Trade · Disease · Elders · War Chiefs
══════════════════════════════════════════════════════════════════════════
  World 220×140  |  RL: MAPPO+GRU128+PBT×4
  Obs: org=42  war=30  global=24
  Warrior auto-respawn: min 3/village every 150s
══════════════════════════════════════════════════════════════════════════

Step    50 | Org  150(♂ 76♀ 74V 51E 0) | War  75 | Enm  1(G1) | V2 | Fd  115 | B 0 K 0 Br 0 EK 0 Tr 0 | OPL +0.000 WPL -0.018 | PBT C0 | 11s
Step   100 | Org  170(♂ 86♀ 84V 58E 0) | War  90 | Enm  1(G1) | V2 | Fd   18 | B 0 K 0 Br 0 EK 0 Tr 0 | OPL +0.000 WPL -0.069 | PBT C0 | 18s


/tmp/ipykernel_2047/597266517.py:1511: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Step   150 | Org  192(♂ 97♀ 95V 71E 0) | War  90 | Enm  1(G1) | V2 | Fd   14 | B 0 K 0 Br 0 EK 0 Tr 0 | OPL +0.000 WPL +0.000 | PBT C0 | 26s
Step   200 | Org  204(♂101♀103V 67E 0) | War  90 | Enm  0(G1) | V2 | Fd    9 | B 0 K 0 Br 0 EK 1 Tr 0 | OPL -0.006 WPL +0.000 | PBT C0 | 33s


/tmp/ipykernel_2047/597266517.py:1511: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Step   250 | Org  217(♂106♀111V 72E 0) | War  90 | Enm  0(G1) | V2 | Fd   10 | B 1 K 0 Br 0 EK 1 Tr 0 | OPL +0.000 WPL +0.000 | PBT C0 | 44s
  [Village+] (118,70) clan=1 n=23 total=3
Step   300 | Org  221(♂106♀115V 65E 0) | War  90 | Enm  1(G1) | V3 | Fd    7 | B 0 K 0 Br 0 EK 0 Tr 0 | OPL +0.000 WPL +0.000 | PBT C0 | 50s


...

Step  7850 | Org  219(♂123♀ 96V 55E162) | War  30 | Enm  0(G18) | V10 | Fd    8 | B 0 K 0 Br 0 EK 0 Tr 0 | OPL +0.000 WPL +0.000 | PBT C0 | 1954s
Step  7900 | Org  220(♂122♀ 98V 66E161) | War  28 | Enm  0(G18) | V10 | Fd   10 | B 0 K 0 Br 0 EK 0 Tr 0 | OPL +0.000 WPL +0.000 | PBT C0 | 1959s


/tmp/ipykernel_2047/597266517.py:1511: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Step  7950 | Org  229(♂127♀102V 71E166) | War  27 | Enm  0(G18) | V10 | Fd   13 | B 1 K 0 Br 0 EK 0 Tr 1 | OPL +0.000 WPL +0.030 | PBT C0 | 1985s
  [PBT] Replaced config 1 with mutated copy of 0 (score 203.56→mut)
  [PBT] Replaced config 1 with mutated copy of 0 (score 148.58→mut)
Step  8000 | Org  227(♂124♀103V 75E170) | War  30 | Enm  0(G18) | V10 | Fd   13 | B 0 K 0 Br 0 EK 0 Tr 0 | OPL +0.000 WPL +0.000 | PBT C0 | 1991s


/tmp/ipykernel_2047/597266517.py:1511: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipykernel_2047/597266517.py:1511: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()



══════════════════════════════════════════════════════════════════════════
  Final: Orgs 227 (♂124 ♀103) | Warriors 30
  Villages 10 | Enemies 0 (Gen 18)
  Gen avg 4.77 max 10
  ♂ speed 2.729
  ♀ speed 2.661
  Total births: 2873
  Total enemy kills: 1330
  Total trades: 360
  PBT best config: C0 score 203.56
══════════════════════════════════════════════════════════════════════════

[Done] → sim_frames_v5/final_summary_v5.png


# RL Architecture & Concepts: Organism Civilization Simulation

This repository implements a **Multi-Agent Reinforcement Learning (MARL)** framework where organisms and warriors evolve behaviors in a shared environment. The core engine is built on **MAPPO (Multi-Agent Proximal Policy Optimization)** enhanced with **GRU-based memory** and **Intrinsic Curiosity**.

---

## 🚀 1. RL Concepts Used

### 1.1 Multi-Agent PPO (MAPPO)
MAPPO is an extension of PPO designed for multi-agent environments. It utilizes **Centralized Training with Decentralized Execution (CTDE)**.

* **Policies:** Shared or partially shared across agents.
* **Stability:** Training is stabilized using a global critic (centralized) while agents act independently based on local observations (decentralized).

#### 🧮 Mathematical Formulation
**Policy Objective (PPO Clip Loss):**
$$L_{CLIP}(\theta) = \mathbb{E}_t \left[ \min(r_t(\theta) A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) A_t) \right]$$

**Where:**
* $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$
* $A_t$ = Advantage
* $\epsilon$ = Clip range (**0.2** in this config)

**Final Loss Function:**
$$L = L_{CLIP} + c_v L_{value} - c_e L_{entropy}$$

---

### 1.2 Centralized Critic
Unlike standard RL where the critic only sees what the agent sees, our **Centralized Critic** uses the global state to reduce variance during training.
* **Actor Input:** Local observation (`obs_dim`)
* **Critic Input:** Global environment state (`global_state_dim`)

### 1.3 Generalized Advantage Estimation (GAE)
Used to reduce variance in advantage estimation by balancing bias and variance.
* **TD Error:** $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$
* **Advantage:** $A_t = \delta_t + \gamma\lambda\delta_{t+1} + \gamma^2\lambda^2\delta_{t+2} + \dots$

### 1.4 Recurrent Policy (GRU)
To handle **Partially Observable Markov Decision Processes (POMDP)**, agents use a Gated Recurrent Unit (GRU) to maintain temporal memory.
* **Update:** $h_t = \text{GRU}(x_t, h_{t-1})$
* **Why?** Agents cannot see the entire world at once and must remember past locations of food or threats.

### 1.5 Intrinsic Curiosity
Agents are rewarded for exploring less-visited states to prevent stagnation.
* **Reward:** $r_{intrinsic} = \frac{\beta}{1 + \sqrt{N(s)}}$
* **Components:** $N(s)$ is the visit count; $\beta$ is the curiosity coefficient.

---

## 🏗️ 2. Architecture

### 🔷 High-Level Flow
1.  **Environment:** Shared world state.
2.  **Observation:** Local spatial slice for each agent.
3.  **Actor Network:** Outputs action logits via GRU memory.
4.  **Reward:** Combined Extrinsic (environment) + Intrinsic (curiosity).
5.  **Critic Network:** Evaluates the global state.
6.  **PPO Update:** Policy refinement based on GAE.

### 🧠 Neural Network Design
**Actor-Critic Network:**
* **Input:** Observation $\to$ Linear $\to$ LayerNorm + Tanh $\to$ Linear.
* **Memory:** GRU Cell (stores hidden state).
* **Heads:** Actor Head (Action Logits) | Critic Head (State Value $V(s)$).

---

## 📊 3. Training Pipeline

1.  **Collect Rollouts:** Store observations, actions, rewards, and hidden states in a buffer.
2.  **Compute Returns:** Calculate Advantages using GAE and target values.
3.  **Optimization:** * Run multiple PPO epochs per batch.
    * Apply gradient clipping to prevent massive policy shifts.
    * Inject Entropy Bonus to maintain exploration.

---

## ⚙️ 4. Key Hyperparameters

| Parameter | Value | Description |
| :--- | :--- | :--- |
| **Learning Rate** | `1.5e-4` | Step size for weight updates |
| **Gamma ($\gamma$)** | `0.99` | Discount factor for future rewards |
| **GAE Lambda ($\lambda$)** | `0.95` | Variance reduction for Advantage |
| **Clip Epsilon ($\epsilon$)** | `0.2` | PPO clipping threshold |
| **Entropy Coef** | `0.020` | Encourages diverse actions |
| **Curiosity Coef** | `0.025` | Weight of the exploration reward |
| **Batch Size** | `1024` | Experience sampled per update |
| **PPO Epochs** | `4` | Number of passes over the same data |

---

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   ORGANISM LIVING SYSTEM v7  —  INTERNAL CIVILIZATION & RELIGIOUS CONFLICT  ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  CORE CHANGES FROM v6:                                                       ║
║   • NO external enemies — conflict is entirely between civilizations         ║
║   • VILLAGE WARS — villages declare war over resources / religion / territory║
║   • RAID SYSTEM — warriors raid enemy villages, steal resources, kill orgs   ║
║   • RELIGION — 2 faiths spawn from random orgs, spread by proximity/charm   ║
║       - Faith grows/declines by conversion, births, and death of followers   ║
║       - Religious conflict: interfaith tension → war declarations             ║
║       - Zealots: high-faith orgs get speed/fertility bonus, attack heretics  ║
║   • RESOURCE WARS — villages fight when one stockpile drops below threshold  ║
║   • DIPLOMACY — alliances, truces, trade agreements between villages         ║
║   • VILLAGE CONCENTRATION — orgs naturally cluster → found villages          ║
║   • REAL-WORLD PHENOMENA:                                                    ║
║       - Population pressure → expansion → territorial conflict               ║
║       - Resource scarcity → raiding → arms race (warrior count)              ║
║       - Religious schism → civil war inside a village                        ║
║       - Golden Age: high-resource, no-war villages boom in population        ║
║       - Dark Age: post-war villages collapse, survivors flee as refugees     ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
import numpy as np, matplotlib, random, math, time, os, copy
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from dataclasses import dataclass, field
from typing import List, Optional, Tuple, Dict, Set
from enum import Enum
from collections import deque, defaultdict

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {DEVICE}", end="")
if DEVICE.type == "cuda":
    print(f"  |  {torch.cuda.get_device_name(0)}"
          f"  |  {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB VRAM")
else: print()

# ─────────────────────────────────────────────────────────────────────────────
#  ENUMS
# ─────────────────────────────────────────────────────────────────────────────
class Sex(Enum):         MALE=0;   FEMALE=1
class Profession(Enum): FARMER=0; TRADER=1; WARRIOR=2; EXPLORER=3; HEALER=4
class Governance(Enum): DEMOCRACY=0; AUTOCRACY=1; THEOCRACY=2; ANARCHY=3
class TerrainType(Enum):PLAINS=0; FOREST=1; DESERT=2; MOUNTAIN=3; RIVER=4
class WarState(Enum):   PEACE=0;  AT_WAR=1; TRUCE=2;  ALLIANCE=3
class WarCause(Enum):   RESOURCES=0; RELIGION=1; TERRITORY=2; RETALIATION=3

# ─────────────────────────────────────────────────────────────────────────────
#  CONFIG
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class SimConfig:
    # World
    world_w: int   = 260
    world_h: int   = 170
    terrain_tile:  int   = 20
    n_bad_zones: int       = 2
    bad_zone_radius: float = 9.0
    bad_zone_damage: float = 0.35
    fog_radius: float      = 22.0

    # Seasons
    season_period: int   = 900
    season_amp:    float = 0.38

    # ── VILLAGES ──────────────────────────────────────────────────────────
    init_villages: int       = 2
    max_villages:  int       = 14
    village_radius: float    = 16.0
    village_food_radius: float = 3.0
    clan_form_density: int   = 15      # orgs needed to found a new village
    clan_form_radius:  float = 20.0
    clan_form_cooldown:int   = 200
    territory_radius: float         = 45.0
    overpop_threshold: int          = 55
    min_warriors_per_village: int   = 4
    warrior_respawn_interval: int   = 100

    # ── RELIGION ──────────────────────────────────────────────────────────
    n_religions: int            = 3
    religion_spawn_step: int    = 300    # earliest step a religion can spawn
    religion_spread_radius: float= 12.0  # radius for conversion attempts
    religion_spread_prob: float  = 0.008 # base prob per step to convert nearby
    religion_charm_weight: float = 0.4   # charisma gene weight in conversion
    zealot_threshold: float      = 0.85  # faith level to become zealot
    zealot_speed_bonus: float    = 0.20
    zealot_fertility_bonus: float= 0.15
    zealot_attack_bonus: float   = 0.20  # zealots attack different-faith orgs
    schism_threshold: float      = 0.45  # fraction minority faith → civil war
    apostasy_prob: float         = 0.001 # chance to lose faith per step

    # ── DISASTERS ─────────────────────────────────────────────────────────
    disaster_base_prob:   float = 0.0006  # per step chance to spawn
    disaster_duration:    int   = 150     # steps a disaster lasts
    drought_food_kill_r:  float = 40.0
    flood_damage:         float = 0.18
    flood_radius:         float = 35.0
    fire_radius:          float = 28.0
    fire_food_kill_frac:  float = 0.6
    fire_damage:          float = 0.10
    earthquake_radius:    float = 45.0
    earthquake_damage:    float = 0.25
    famine_food_mult:     float = 0.20   # food spawn multiplier during famine

    # ── WAR / RAID SYSTEM ─────────────────────────────────────────────────
    war_declare_resource_thresh: float = 0.15  # LOWERED: raid if target has > this × own
    war_declare_territory_dist:  float = 70.0  # RAISED: more villages in range
    war_declare_religion_thresh: float = 0.35  # LOWERED: religion sparks war easier
    war_declare_prob:            float = 0.012 # 4× HIGHER base probability
    war_dominance_prob:          float = 0.004 # NEW: random dominance war, no reason needed
    war_zealot_prob:             float = 0.008 # NEW: zealots in village push for holy war
    pacifist_war_damp:           float = 0.65  # RAISED from 0.4 — pacifist still can war
    truce_duration:              int   = 300   # SHORTER truces
    alliance_duration:           int   = 600
    raid_warrior_speed_bonus:    float = 0.25  # warriors faster when raiding
    raid_kill_prob:              float = 0.25  # prob to kill org during raid
    raid_resource_steal:         float = 0.30  # fraction of resources stolen
    war_attrition_damage:        float = 0.05  # passive energy drain on warriors at war
    refugee_spawn_on_village_fall: int = 8     # orgs that flee destroyed village

    # ── DIPLOMACY ─────────────────────────────────────────────────────────
    trade_agreement_prob: float  = 0.002  # allied villages trade automatically
    trade_agreement_amount: float= 5.0    # food transferred per step (allied)
    open_trade_prob:      float  = 0.006  # any peaceful village pair can trade
    open_trade_amount:    float  = 3.0    # smaller transfer per open trade trip
    trade_route_dist:     float  = 130.0  # max distance for any trade

    # ── POPULATIONS ───────────────────────────────────────────────────────
    init_organisms: int  = 120
    init_warriors:  int  = 20
    max_organisms:  int  = 350
    max_warriors:   int  = 110

    # ── FOOD / RESOURCES ──────────────────────────────────────────────────
    init_food: int         = 380
    max_food:  int         = 600
    food_spawn_rate: float = 0.70
    food_eat_radius: float = 2.6
    # Multi-resource (food, wood, stone, gold)
    resource_names: Tuple  = ("food","wood","stone","gold")
    resource_max:   Tuple  = (80., 50., 40., 20.)

    # ── ORGANISM ──────────────────────────────────────────────────────────
    org_friction: float = 0.76
    org_accel:    float = 0.94
    init_energy: float         = 108.0
    max_energy:  float         = 200.0
    base_metabolism: float     = 0.025
    move_cost_factor: float    = 0.009
    food_energy: float         = 75.0
    reproduce_threshold: float = 78.0
    reproduce_cost:      float = 24.0
    gestation_steps: int       = 12
    starve_energy: float       = 0.0
    elder_age: int             = 420
    sick_duration: int         = 60
    sick_damage: float         = 0.06

    # Genes
    gene_vision_range:    Tuple = (5.0, 26.0)
    gene_meta_range:      Tuple = (0.02, 0.22)
    gene_fertility_range: Tuple = (0.3,  1.0)
    gene_va_range:        Tuple = (0.0,  1.0)
    gene_speed_range_M:   Tuple = (1.5, 4.5)
    gene_aggro_range_M:   Tuple = (0.2, 1.0)
    gene_speed_range_F:   Tuple = (0.8, 3.0)
    gene_aggro_range_F:   Tuple = (0.0, 0.6)
    gene_charisma_range:  Tuple = (0.0, 1.0)  # NEW: drives religious spreading
    mutation_rate:   float = 0.10
    mutation_strength: float = 0.11

    # Warrior
    warrior_max_speed:    float = 4.0
    warrior_accel:        float = 1.2
    warrior_friction:     float = 0.79
    warrior_vision:       float = 25.0
    warrior_energy_init:  float = 145.0
    warrior_energy_max:   float = 250.0
    warrior_metabolism:   float = 0.040
    warrior_move_cost:    float = 0.008
    warrior_kill_radius:  float = 4.5
    warrior_protect_r:    float = 18.0
    warrior_food_energy:  float = 45.0
    warrior_rep_threshold:float = 170.0
    warrior_rep_cost:     float = 48.0
    warrior_attack_bonus: float = 28.0   # energy gain per kill in war
    war_chief_bonus:      float = 0.28

    # ── RL (MAPPO + GRU) ──────────────────────────────────────────────────
    act_dim:          int   = 9
    org_obs_dim:      int   = 52
    warrior_obs_dim:  int   = 36
    global_state_dim: int   = 30
    hidden_dim:       int   = 256
    gru_hidden:       int   = 128
    lr:               float = 1.5e-4
    gamma:            float = 0.99
    gae_lambda:       float = 0.95
    clip_eps:         float = 0.2
    entropy_coef:     float = 0.020
    value_coef:       float = 0.5
    ppo_epochs:       int   = 4
    batch_size:       int   = 1024
    rollout_len:      int   = 300
    max_grad_norm:    float = 0.5
    curiosity_coef:   float = 0.025
    curiosity_grid_w: int   = 26
    curiosity_grid_h: int   = 17

    # Simulation
    max_steps:    int  = 8000
    render_every: int  = 100
    save_plot:    bool = True
    plot_dir:     str  = "sim_frames_v7"

CFG = SimConfig()

_ACT_DIRS = np.array([
    [ 0,-1],[ 1,-1],[ 1, 0],[ 1, 1],
    [ 0, 1],[-1, 1],[-1, 0],[-1,-1],[ 0, 0],
], dtype=np.float32)
for _i in range(8):
    _n = np.linalg.norm(_ACT_DIRS[_i])
    if _n > 0: _ACT_DIRS[_i] /= _n

TERRAIN_COLORS = {
    TerrainType.PLAINS:"#2d4a1e", TerrainType.FOREST:"#1a3a12",
    TerrainType.DESERT:"#5a4a1a", TerrainType.MOUNTAIN:"#3a3a3a",
    TerrainType.RIVER:"#1a2a4a",
}
TERRAIN_FOOD_MULT  = {TerrainType.PLAINS:1.0, TerrainType.FOREST:1.6,
                      TerrainType.DESERT:0.25,TerrainType.MOUNTAIN:0.4,TerrainType.RIVER:1.4}
TERRAIN_SPEED_MULT = {TerrainType.PLAINS:1.0, TerrainType.FOREST:0.80,
                      TerrainType.DESERT:0.65,TerrainType.MOUNTAIN:0.45,TerrainType.RIVER:0.88}

# ─────────────────────────────────────────────────────────────────────────────
#  TERRAIN MAP
# ─────────────────────────────────────────────────────────────────────────────
class TerrainMap:
    def __init__(self):
        W = CFG.terrain_tile
        H = max(int(W * CFG.world_h / CFG.world_w), 1)
        self.W = W; self.H = H
        self.grid = np.full((H, W), TerrainType.PLAINS)
        self._generate()

    def _generate(self):
        for ty in range(self.H):
            for tx in range(self.W):
                x = tx / self.W; y = ty / self.H
                if abs(math.sin(x*math.pi*3+0.5)*math.cos(y*math.pi*2)-0.05) < 0.08:
                    self.grid[ty,tx] = TerrainType.RIVER
                elif (x < 0.12 or x > 0.88) and (y < 0.18 or y > 0.82):
                    self.grid[ty,tx] = TerrainType.MOUNTAIN
                elif 0.28 < x < 0.72 and 0.04 < y < 0.22:
                    self.grid[ty,tx] = TerrainType.DESERT
                elif x < 0.18 or x > 0.82 or y < 0.12 or y > 0.88:
                    self.grid[ty,tx] = TerrainType.FOREST

    def get(self, x, y) -> TerrainType:
        tx = int(np.clip(x/CFG.world_w*self.W, 0, self.W-1))
        ty = int(np.clip(y/CFG.world_h*self.H, 0, self.H-1))
        return self.grid[ty, tx]

    def food_mult(self, x, y): return TERRAIN_FOOD_MULT[self.get(x,y)]
    def speed_mult(self, x, y): return TERRAIN_SPEED_MULT[self.get(x,y)]

TERRAIN = TerrainMap()

# ─────────────────────────────────────────────────────────────────────────────
#  RELIGION
# ─────────────────────────────────────────────────────────────────────────────
RELIGION_NAMES = [
    ("The Radiant Path",   "#FFD700", "☀"),
    ("The Iron Covenant",  "#8B0000", "⚔"),
    ("The Green Circle",   "#228B22", "🌿"),
    ("The Deep Current",   "#1E90FF", "〰"),
]

class Religion:
    _next_id = 0
    def __init__(self, name: str, color: str, symbol: str, founder_id: int,
                 founder_x: float, founder_y: float, step: int):
        Religion._next_id += 1
        self.id          = Religion._next_id
        self.name        = name
        self.color       = color
        self.symbol      = symbol
        self.founder_id  = founder_id
        self.origin_x    = founder_x
        self.origin_y    = founder_y
        self.founded_step= step
        self.follower_count = 1
        self.total_ever  = 1
        self.holy_sites: List[Tuple[float,float]] = [(founder_x, founder_y)]
        self.is_active   = True
        # Religious tenets — mix can be contradictory (peaceful + extremist = real world)
        self.tenet_pacifist    = random.random() < 0.4   # lower war desire
        self.tenet_expansionist= random.random() < 0.5   # spread fast
        self.tenet_trade       = random.random() < 0.5   # more traders
        self.tenet_isolation   = random.random() < 0.3   # stay in village
        self.tenet_extremist   = random.random() < 0.35  # NEW: zealots push holy wars
        # Historical stats
        self.wars_caused = 0
        self.conversions = 0
        self.schisms     = 0

    def tension_with(self, other: "Religion") -> float:
        """
        0 = compatible, 1 = maximum tension.
        Incompatible tenets increase tension.
        Pacifist + extremist = still can war (real world: peaceful faith, violent arm).
        """
        t = 0.0
        if self.tenet_expansionist and other.tenet_expansionist: t += 0.4
        if self.tenet_expansionist and other.tenet_isolation:     t += 0.3
        if not self.tenet_pacifist  and not other.tenet_pacifist: t += 0.3
        if self.tenet_extremist:   t += 0.35  # extremist religion is aggressive regardless
        if other.tenet_extremist:  t += 0.25  # target extremism also provokes
        return min(t, 1.0)

# ─────────────────────────────────────────────────────────────────────────────
#  DIPLOMACY LEDGER  (relationships between village pairs)
# ─────────────────────────────────────────────────────────────────────────────
class DiplomacyLedger:
    def __init__(self):
        self._state: Dict[Tuple[int,int], WarState] = {}
        self._timer: Dict[Tuple[int,int], int] = {}
        self._cause: Dict[Tuple[int,int], WarCause] = {}
        self.war_history: List[Dict] = []
        # Trade route tracking: (vid_a, vid_b) → total food transferred
        self.trade_routes: Dict[Tuple[int,int], float] = {}
        self.active_trade_this_step: Set[Tuple[int,int]] = set()

    def _key(self, a: int, b: int) -> Tuple[int,int]:
        return (min(a,b), max(a,b))

    def state(self, a: int, b: int) -> WarState:
        return self._state.get(self._key(a,b), WarState.PEACE)

    def declare_war(self, a: int, b: int, cause: WarCause, step: int):
        k = self._key(a,b)
        if self._state.get(k) == WarState.AT_WAR: return
        self._state[k] = WarState.AT_WAR
        self._cause[k] = cause
        self.war_history.append({"step":step,"a":a,"b":b,"cause":cause.name})
        print(f"  [WAR] Village {a} ↔ {b} | Cause: {cause.name} | Step {step}")

    def make_truce(self, a: int, b: int):
        k = self._key(a,b)
        self._state[k] = WarState.TRUCE
        self._timer[k] = CFG.truce_duration

    def make_alliance(self, a: int, b: int):
        k = self._key(a,b)
        self._state[k] = WarState.ALLIANCE
        self._timer[k] = CFG.alliance_duration

    def tick(self):
        expired = []
        for k, state in self._state.items():
            if state in (WarState.TRUCE, WarState.ALLIANCE):
                self._timer[k] = self._timer.get(k, 0) - 1
                if self._timer[k] <= 0:
                    self._state[k] = WarState.PEACE
                    expired.append(k)
        for k in expired:
            self._timer.pop(k, None)

    def at_war_with(self, vid: int) -> List[int]:
        enemies = []
        for (a,b), st in self._state.items():
            if st == WarState.AT_WAR:
                if a == vid: enemies.append(b)
                if b == vid: enemies.append(a)
        return enemies

    def allied_with(self, vid: int) -> List[int]:
        allies = []
        for (a,b), st in self._state.items():
            if st == WarState.ALLIANCE:
                if a == vid: allies.append(b)
                if b == vid: allies.append(a)
        return allies

    def total_wars(self) -> int:
        return sum(1 for s in self._state.values() if s == WarState.AT_WAR)

# ─────────────────────────────────────────────────────────────────────────────
#  GENOME
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class Genome:
    speed: float=2.0; vision: float=11.0; metabolism: float=0.10
    aggression: float=0.4; fertility: float=0.6
    village_affinity: float=0.5; dominance: float=0.5
    immune_strength: float=0.5; trade_drive: float=0.5
    cooperation: float=0.5; charisma: float=0.5

    @classmethod
    def random(cls, sex: Sex) -> "Genome":
        sr = CFG.gene_speed_range_M if sex==Sex.MALE else CFG.gene_speed_range_F
        ar = CFG.gene_aggro_range_M if sex==Sex.MALE else CFG.gene_aggro_range_F
        return cls(speed=random.uniform(*sr),
                   vision=random.uniform(*CFG.gene_vision_range),
                   metabolism=random.uniform(*CFG.gene_meta_range),
                   aggression=random.uniform(*ar),
                   fertility=random.uniform(*CFG.gene_fertility_range),
                   village_affinity=random.uniform(*CFG.gene_va_range),
                   dominance=random.uniform(0.2,0.8),
                   immune_strength=random.uniform(0.1,0.9),
                   trade_drive=random.uniform(0.0,1.0),
                   cooperation=random.uniform(0.0,1.0),
                   charisma=random.uniform(*CFG.gene_charisma_range))

    def mutate(self, sex: Sex) -> "Genome":
        sr = CFG.gene_speed_range_M if sex==Sex.MALE else CFG.gene_speed_range_F
        ar = CFG.gene_aggro_range_M if sex==Sex.MALE else CFG.gene_aggro_range_F
        def m(v,lo,hi):
            if random.random()<CFG.mutation_rate: v+=random.gauss(0,CFG.mutation_strength)*(hi-lo)
            return float(np.clip(v,lo,hi))
        return Genome(speed=m(self.speed,*sr), vision=m(self.vision,*CFG.gene_vision_range),
                      metabolism=m(self.metabolism,*CFG.gene_meta_range),
                      aggression=m(self.aggression,*ar),
                      fertility=m(self.fertility,*CFG.gene_fertility_range),
                      village_affinity=m(self.village_affinity,*CFG.gene_va_range),
                      dominance=m(self.dominance,0.1,0.9),
                      immune_strength=m(self.immune_strength,0.0,1.0),
                      trade_drive=m(self.trade_drive,0.0,1.0),
                      cooperation=m(self.cooperation,0.0,1.0),
                      charisma=m(self.charisma,*CFG.gene_charisma_range))

    def crossover(self, other: "Genome", csex: Sex) -> "Genome":
        dm=other.dominance; df=self.dominance; total=dm+df+1e-8; pf=dm/total
        pick=lambda a,b: b if random.random()<pf else a
        return Genome(speed=pick(self.speed,other.speed),
                      vision=pick(self.vision,other.vision),
                      metabolism=pick(self.metabolism,other.metabolism),
                      aggression=pick(self.aggression,other.aggression),
                      fertility=pick(self.fertility,other.fertility),
                      village_affinity=pick(self.village_affinity,other.village_affinity),
                      dominance=(self.dominance+other.dominance)/2,
                      immune_strength=pick(self.immune_strength,other.immune_strength),
                      trade_drive=pick(self.trade_drive,other.trade_drive),
                      cooperation=pick(self.cooperation,other.cooperation),
                      charisma=pick(self.charisma,other.charisma)).mutate(csex)

    def to_vec(self) -> np.ndarray:
        def norm(v,lo,hi): return (v-lo)/(hi-lo+1e-8)
        return np.array([norm(self.speed,1.0,4.5),norm(self.vision,*CFG.gene_vision_range),
                         norm(self.metabolism,*CFG.gene_meta_range),norm(self.aggression,0.,1.),
                         norm(self.fertility,*CFG.gene_fertility_range),
                         self.village_affinity,self.dominance,self.immune_strength,
                         self.trade_drive,self.cooperation,self.charisma],
                        dtype=np.float32)  # 11 genes

# ─────────────────────────────────────────────────────────────────────────────
#  ACTOR-CRITIC  (MAPPO + GRU)
# ─────────────────────────────────────────────────────────────────────────────
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim, global_dim, hidden=256, gru_h=128):
        super().__init__()
        self.enc    = nn.Sequential(nn.Linear(obs_dim,hidden),nn.LayerNorm(hidden),nn.Tanh(),
                                    nn.Linear(hidden,gru_h),nn.Tanh())
        self.gru    = nn.GRUCell(gru_h, gru_h)
        self.actor  = nn.Sequential(nn.Linear(gru_h,hidden//2),nn.Tanh(),
                                    nn.Linear(hidden//2,act_dim))
        self.critic = nn.Sequential(nn.Linear(global_dim,hidden),nn.LayerNorm(hidden),nn.Tanh(),
                                    nn.Linear(hidden,hidden//2),nn.Tanh(),nn.Linear(hidden//2,1))
        for mod in self.modules():
            if isinstance(mod,nn.Linear):
                nn.init.orthogonal_(mod.weight,gain=np.sqrt(2)); nn.init.zeros_(mod.bias)
        nn.init.orthogonal_(self.actor[-1].weight,gain=0.01)
        nn.init.orthogonal_(self.critic[-1].weight,gain=1.0)

    def act_step(self, obs_t, hx_t):
        enc = self.enc(obs_t); hx = self.gru(enc,hx_t)
        return self.actor(hx), hx

    def val_step(self, gs_t): return self.critic(gs_t)

# ─────────────────────────────────────────────────────────────────────────────
#  MAPPO POLICY
# ─────────────────────────────────────────────────────────────────────────────
class MAPPOPolicy:
    def __init__(self, obs_dim, global_dim, name="policy"):
        self.name = name
        self.net  = ActorCritic(obs_dim, CFG.act_dim, global_dim,
                                CFG.hidden_dim, CFG.gru_hidden).to(DEVICE)
        self.opt  = optim.Adam(self.net.parameters(), lr=CFG.lr, eps=1e-5)
        self.hidden: Dict[int,torch.Tensor] = {}
        self.visit_grid = np.zeros((CFG.curiosity_grid_h,CFG.curiosity_grid_w),dtype=np.float32)
        self._reset(); self.steps=0

    def _reset(self):
        self.buf={k:[] for k in ["obs","acts","logps","vals","rews","dones","hx","gs"]}

    def _hx(self, aid):
        if aid not in self.hidden:
            self.hidden[aid]=torch.zeros(1,CFG.gru_hidden,device=DEVICE)
        return self.hidden[aid]

    def clear(self, aid):
        self.hidden.pop(aid,None)

    def intrinsic(self, x, y):
        gx=int(np.clip(x/CFG.world_w*CFG.curiosity_grid_w,0,CFG.curiosity_grid_w-1))
        gy=int(np.clip(y/CFG.world_h*CFG.curiosity_grid_h,0,CFG.curiosity_grid_h-1))
        c=self.visit_grid[gy,gx]; self.visit_grid[gy,gx]+=1.
        return CFG.curiosity_coef/(1.+math.sqrt(c))

    def store(self, obs,act,logp,val,rew,done,hx,gs):
        self.buf["obs"].append(obs); self.buf["acts"].append(act)
        self.buf["logps"].append(logp); self.buf["vals"].append(val)
        self.buf["rews"].append(rew); self.buf["dones"].append(float(done))
        self.buf["hx"].append(hx.cpu().numpy().flatten()); self.buf["gs"].append(gs)

    @torch.no_grad()
    def batch_act(self, obs_list, agent_ids, global_states):
        if not obs_list: return [],[],[],None
        obs_t = torch.tensor(np.array(obs_list),dtype=torch.float32,device=DEVICE)
        gs_t  = torch.tensor(np.array(global_states),dtype=torch.float32,device=DEVICE)
        hx_t  = torch.cat([self._hx(aid) for aid in agent_ids],dim=0)
        logits,hx_new = self.net.act_step(obs_t,hx_t)
        vals          = self.net.val_step(gs_t)
        dist  = torch.distributions.Categorical(logits=logits)
        acts  = dist.sample(); logps=dist.log_prob(acts)
        for i,aid in enumerate(agent_ids):
            self.hidden[aid]=hx_new[i:i+1].detach()
        return acts.cpu().numpy(),logps.cpu().numpy(),vals.squeeze(-1).cpu().numpy(),hx_new.detach()

    def update(self, score=0.0) -> dict:
        n=len(self.buf["obs"])
        if n<CFG.batch_size: return {}
        obs  =torch.tensor(np.array(self.buf["obs"]),dtype=torch.float32,device=DEVICE)
        acts =torch.tensor(self.buf["acts"],dtype=torch.long,device=DEVICE)
        logps=torch.tensor(self.buf["logps"],dtype=torch.float32,device=DEVICE)
        vals =torch.tensor(self.buf["vals"],dtype=torch.float32,device=DEVICE)
        rews =torch.tensor(self.buf["rews"],dtype=torch.float32,device=DEVICE)
        dones=torch.tensor(self.buf["dones"],dtype=torch.float32,device=DEVICE)
        hxs  =torch.tensor(np.array(self.buf["hx"]),dtype=torch.float32,device=DEVICE).view(n,1,CFG.gru_hidden)
        gs   =torch.tensor(np.array(self.buf["gs"]),dtype=torch.float32,device=DEVICE)
        adv=torch.zeros_like(rews); gae=0.0
        for t in reversed(range(n)):
            nv=0. if t==n-1 or dones[t] else vals[t+1].item()
            delta=rews[t]+CFG.gamma*nv*(1-dones[t])-vals[t]
            gae=delta+CFG.gamma*CFG.gae_lambda*(1-dones[t])*gae; adv[t]=gae
        ret=adv+vals; adv=(adv-adv.mean())/(adv.std()+1e-8)
        m={"pl":0,"vl":0,"ent":0,"n":0}; idx=torch.randperm(n)
        for _ in range(CFG.ppo_epochs):
            for s in range(0,n,CFG.batch_size):
                mb=idx[s:s+CFG.batch_size]
                hx_mb=hxs[mb].squeeze(1)
                enc_mb=self.net.enc(obs[mb]); hx2=self.net.gru(enc_mb,hx_mb)
                l2=self.net.actor(hx2); v2=self.net.val_step(gs[mb])
                d2=torch.distributions.Categorical(logits=l2)
                nlp=d2.log_prob(acts[mb]); ent=d2.entropy().mean()
                r=(nlp-logps[mb]).exp(); a=adv[mb]
                pl=-torch.min(r*a,r.clamp(1-CFG.clip_eps,1+CFG.clip_eps)*a).mean()
                vl=F.mse_loss(v2.squeeze(-1),ret[mb])
                loss=pl+CFG.value_coef*vl-CFG.entropy_coef*ent
                self.opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(self.net.parameters(),CFG.max_grad_norm)
                self.opt.step()
                m["pl"]+=pl.item(); m["vl"]+=vl.item(); m["ent"]+=ent.item(); m["n"]+=1
        self._reset(); self.steps+=n; nu=max(m["n"],1)
        return {k:v/nu for k,v in m.items() if k!="n"}

# ─────────────────────────────────────────────────────────────────────────────
#  VILLAGE
# ─────────────────────────────────────────────────────────────────────────────
_VID = 0
class Village:
    def __init__(self, x, y, clan_id=None):
        global _VID; _VID+=1
        self.id=_VID; self.x=float(x); self.y=float(y)
        self.clan_id=clan_id or _VID
        self.founded_step=0
        self.is_defended=True
        self.last_warrior_spawn=0
        # Resources: food, wood, stone, gold
        self.resources=np.array([20.,10.,5.,2.],dtype=np.float32)
        # Culture & Governance
        self.governance=random.choice(list(Governance))
        self.culture=np.random.rand(8).astype(np.float32)
        self.leader_id: Optional[int]=None
        self.policy_trade=random.random()
        self.policy_aggression=random.random()
        # Religion
        self.dominant_religion: Optional[int]=None  # religion id
        self.religion_counts: Dict[int,int]={}
        # War state
        self.war_exhaustion=0.0   # 0-1, rises during war, reduces policy_aggression
        self.morale=1.0           # 0-1, drops after raids
        self.raided_step=0
        # Prosperity metrics
        self.golden_age=False
        self.dark_age=False
        self.population_history: deque = deque(maxlen=50)
        # Refugees stored here when village falls
        self.fallen=False
        self.color=None  # assigned by renderer

    def contains(self, x, y): return math.hypot(x-self.x,y-self.y)<=CFG.village_radius
    def in_territory(self, x, y): return math.hypot(x-self.x,y-self.y)<=CFG.territory_radius
    def dist(self, x, y): return math.hypot(x-self.x,y-self.y)

    def total_wealth(self) -> float:
        weights=np.array([1.,1.5,2.,4.])
        return float(np.dot(self.resources,weights))

    def update_governance(self, local_orgs, local_wars):
        if not local_orgs: return
        if self.governance==Governance.DEMOCRACY:
            self.policy_trade=float(np.mean([o.genome.trade_drive for o in local_orgs]))
            self.policy_aggression=float(np.mean([o.genome.aggression for o in local_orgs]))
        elif self.governance==Governance.AUTOCRACY:
            if self.leader_id:
                ldrs=[o for o in local_orgs if o.id==self.leader_id]
                if ldrs:
                    self.policy_trade=ldrs[0].genome.trade_drive
                    self.policy_aggression=ldrs[0].genome.aggression
        elif self.governance==Governance.THEOCRACY:
            elders=[o for o in local_orgs if o.is_elder]
            if elders: self.policy_trade=float(np.mean([e.genome.trade_drive for e in elders]))
        # War exhaustion reduces aggression
        self.policy_aggression=max(0.,self.policy_aggression-self.war_exhaustion*0.3)
        # Elect leader
        if local_orgs:
            self.leader_id=max(local_orgs,key=lambda o:o.fitness).id
        # Golden/dark age
        pop=len(local_orgs)
        self.population_history.append(pop)
        avg_pop=float(np.mean(self.population_history)) if self.population_history else pop
        self.golden_age=(pop>avg_pop*1.3 and self.total_wealth()>30 and not local_wars)
        self.dark_age=(pop<avg_pop*0.6 or self.war_exhaustion>0.7)

    def update_religion_counts(self, local_orgs):
        counts: Dict[int,int]=defaultdict(int)
        for o in local_orgs:
            if o.religion_id is not None: counts[o.religion_id]+=1
        self.religion_counts=dict(counts)
        if counts:
            self.dominant_religion=max(counts,key=counts.get)
        else:
            self.dominant_religion=None

    def religion_tension(self, religions: Dict[int,"Religion"]) -> float:
        """Tension between two religions in this village (0-1)."""
        rids=[rid for rid,cnt in self.religion_counts.items() if cnt>0]
        if len(rids)<2: return 0.0
        max_t=0.0
        for i in range(len(rids)):
            for j in range(i+1,len(rids)):
                r1=religions.get(rids[i]); r2=religions.get(rids[j])
                if r1 and r2: max_t=max(max_t,r1.tension_with(r2))
        # Scale by how evenly split the village is
        total=sum(self.religion_counts.values())+1e-8
        minority_frac=sum(sorted(self.religion_counts.values())[:-1])/total
        return max_t*min(minority_frac*2,1.0)

# ─────────────────────────────────────────────────────────────────────────────
#  ORGANISM
# ─────────────────────────────────────────────────────────────────────────────
_OID = 0
class Organism:
    def __init__(self, x, y, genome=None, energy=None, generation=1,
                 sex=None, clan_id=None):
        global _OID; _OID+=1
        self.id=_OID; self.x=float(x); self.y=float(y)
        self.vx=0.; self.vy=0.
        self.sex=sex or random.choice([Sex.MALE,Sex.FEMALE])
        self.genome=genome or Genome.random(self.sex)
        self.energy=float(energy or CFG.init_energy)
        self.alive=True; self.age=0; self.generation=generation; self.fitness=0.
        self.home_village: Optional[int]=None; self.clan_id=clan_id
        self.profession=self._assign_profession()
        self.pregnant=False; self.gestation=0; self.father_genome=None
        self.mate_timer=0; self.is_elder=False
        self.sick=False; self.sick_timer=0
        self.carrying_food=False; self.trade_destination: Optional[int]=None
        # Religion
        self.religion_id: Optional[int]=None   # which religion they follow
        self.faith_level: float=0.0             # 0=atheist, 1=zealot
        self.is_zealot: bool=False
        # War
        self.is_raiding: bool=False             # currently on raid
        self.raid_target_village: Optional[int]=None
        # State
        self._obs=None; self._act=8; self._logp=0.; self._val=0.
        self._prev_energy=self.energy

    def _assign_profession(self) -> Profession:
        g=self.genome
        if g.aggression>0.65:      return Profession.WARRIOR
        if g.trade_drive>0.65:     return Profession.TRADER
        if g.charisma>0.65:        return Profession.EXPLORER  # repurposed as missionary
        if g.immune_strength>0.65: return Profession.HEALER
        return Profession.FARMER

    # 52-dim observation
    def observe(self, food, warriors, villages, bad_zones,
                season_phase, terrain, religions, diplomacy) -> np.ndarray:
        W,H=CFG.world_w,CFG.world_h
        obs=np.zeros(CFG.org_obs_dim,dtype=np.float32)
        obs[0]=self.x/W; obs[1]=self.y/H
        obs[2]=self.energy/CFG.max_energy; obs[3]=min(self.age/700.,1.)
        obs[4]=float(self.sex==Sex.FEMALE)
        obs[5:16]=self.genome.to_vec()   # 11 genes

        # Nearest food
        vis_food=[f for f in food if math.hypot(f[0]-self.x,f[1]-self.y)<=CFG.fog_radius]
        if vis_food:
            d2=[(math.hypot(f[0]-self.x,f[1]-self.y),f) for f in vis_food]
            d,f=min(d2,key=lambda t:t[0])
            obs[16]=np.clip((f[0]-self.x)/W,-1,1); obs[17]=np.clip((f[1]-self.y)/H,-1,1)
            obs[18]=min(d/self.genome.vision,1.); obs[19]=float(d<=self.genome.vision)

        # Home village
        hv=[v for v in villages if v.id==self.home_village]
        if hv:
            v=hv[0]; d=v.dist(self.x,self.y)
            obs[20]=np.clip((v.x-self.x)/W,-1,1); obs[21]=np.clip((v.y-self.y)/H,-1,1)
            obs[22]=min(d/(W+H),1.); obs[23]=float(d<=CFG.village_radius)
            obs[24]=float(v.is_defended); obs[25]=float(v.golden_age); obs[26]=float(v.dark_age)
            obs[27]=v.policy_aggression; obs[28]=v.policy_trade
            obs[29]=min(v.total_wealth()/100.,1.)
            # Are we at war?
            enemy_vids=diplomacy.at_war_with(v.id)
            obs[30]=float(len(enemy_vids)>0)
            # Nearest enemy village direction
            if enemy_vids:
                ev=[vv for vv in villages if vv.id in enemy_vids]
                if ev:
                    ne=min(ev,key=lambda vv:vv.dist(self.x,self.y))
                    obs[31]=np.clip((ne.x-self.x)/W,-1,1)
                    obs[32]=np.clip((ne.y-self.y)/H,-1,1)

        # Religion state
        obs[33]=float(self.religion_id is not None)
        obs[34]=self.faith_level; obs[35]=float(self.is_zealot)

        # Nearest warrior
        if warriors:
            dw=[(math.hypot(w.x-self.x,w.y-self.y),w) for w in warriors]
            d,w=min(dw,key=lambda t:t[0])
            obs[36]=np.clip((w.x-self.x)/W,-1,1); obs[37]=np.clip((w.y-self.y)/H,-1,1)
            obs[38]=min(d/self.genome.vision,1.)
            obs[39]=float(w.clan_id==self.clan_id)  # friendly or enemy warrior

        if bad_zones:
            dsts=[math.hypot(z[0]-self.x,z[1]-self.y) for z in bad_zones]
            obs[40]=min(min(dsts)/W,1.); obs[41]=float(min(dsts)<=CFG.bad_zone_radius)

        obs[42]=np.clip(self.vx/4.,-1,1); obs[43]=np.clip(self.vy/4.,-1,1)
        obs[44]=min(len(vis_food)/14.,1.)
        obs[45]=np.clip((self.energy-self._prev_energy)/CFG.max_energy,-1,1)
        obs[46]=float(self.pregnant); obs[47]=float(self.sick)
        obs[48]=float(self.carrying_food); obs[49]=float(self.is_raiding)
        obs[50]=math.sin(season_phase*2*math.pi); obs[51]=math.cos(season_phase*2*math.pi)
        return obs

    def apply_action(self, action, terrain):
        d=_ACT_DIRS[action]; spd=self.genome.speed
        if self.pregnant: spd*=0.72
        if self.sick:     spd*=0.78
        if self.is_zealot:spd*=(1.+CFG.zealot_speed_bonus)
        spd*=terrain.speed_mult(self.x,self.y)
        self.vx=self.vx*CFG.org_friction+d[0]*CFG.org_accel*spd
        self.vy=self.vy*CFG.org_friction+d[1]*CFG.org_accel*spd
        v=math.hypot(self.vx,self.vy)
        if v>spd: self.vx*=spd/v; self.vy*=spd/v
        self._prev_energy=self.energy
        self.x=float(np.clip(self.x+self.vx,0,CFG.world_w))
        self.y=float(np.clip(self.y+self.vy,0,CFG.world_h))
        if self.x<=0 or self.x>=CFG.world_w: self.vx*=-0.5
        if self.y<=0 or self.y>=CFG.world_h: self.vy*=-0.5
        cost=CFG.base_metabolism*(1.+self.genome.metabolism)+CFG.move_cost_factor*(self.vx**2+self.vy**2)
        if self.pregnant: cost*=1.18
        if self.sick:     cost*=1.10; self.energy-=CFG.sick_damage; self.sick_timer-=1
        if self.sick_timer<=0: self.sick=False
        self.energy-=cost; self.age+=1
        if self.mate_timer>0: self.mate_timer-=1
        if self.age>=CFG.elder_age: self.is_elder=True
        if self.energy<=CFG.starve_energy: self.alive=False

# ─────────────────────────────────────────────────────────────────────────────
#  WARRIOR
# ─────────────────────────────────────────────────────────────────────────────
_WID = 0
class Warrior:
    def __init__(self, x, y, energy=None, home_village=None, clan_id=None):
        global _WID; _WID+=1
        self.id=_WID; self.x=float(x); self.y=float(y)
        self.vx=0.; self.vy=0.
        self.energy=float(energy or CFG.warrior_energy_init)
        self.alive=True; self.age=0; self.kills=0
        self.home_village=home_village; self.clan_id=clan_id
        self.is_war_chief=False
        self.is_raiding=False
        self.raid_target: Optional[int]=None
        self._obs=None; self._act=8; self._logp=0.; self._val=0.

    def effective_speed(self):
        s=CFG.warrior_max_speed*(1.+CFG.war_chief_bonus if self.is_war_chief else 1.)
        if self.is_raiding: s*=(1.+CFG.raid_warrior_speed_bonus)
        return s

    def observe(self, organisms, villages, bad_zones, season_phase, terrain, diplomacy) -> np.ndarray:
        W,H=CFG.world_w,CFG.world_h
        obs=np.zeros(CFG.warrior_obs_dim,dtype=np.float32)
        obs[0]=self.x/W; obs[1]=self.y/H
        obs[2]=self.energy/CFG.warrior_energy_max; obs[3]=min(self.age/600.,1.)

        def ns(items,gx,gy,k=0):
            if not items: return 0.,0.,1.,0.
            d2=[(math.hypot(gx(i)-self.x,gy(i)-self.y),i) for i in items]
            d2.sort(key=lambda t:t[0])
            if k>=len(d2): return 0.,0.,1.,0.
            d,it=d2[k]; ix,iy=gx(it),gy(it)
            return (np.clip((ix-self.x)/W,-1,1),np.clip((iy-self.y)/H,-1,1),
                    min(d/CFG.warrior_vision,1.),float(d<=CFG.warrior_vision))

        # Nearest friendly org
        friendly=[o for o in organisms if o.alive and o.clan_id==self.clan_id]
        enemy_orgs=[o for o in organisms if o.alive and o.clan_id!=self.clan_id]
        obs[4],obs[5],obs[6],obs[7]   =ns(friendly,  lambda o:o.x,lambda o:o.y,0)
        obs[8],obs[9],obs[10],obs[11] =ns(enemy_orgs,lambda o:o.x,lambda o:o.y,0)

        # Enemy village
        enemy_vids=diplomacy.at_war_with(self.home_village) if self.home_village else []
        enemy_vs=[v for v in villages if v.id in enemy_vids]
        if enemy_vs:
            ne=min(enemy_vs,key=lambda v:v.dist(self.x,self.y))
            obs[12]=np.clip((ne.x-self.x)/W,-1,1); obs[13]=np.clip((ne.y-self.y)/H,-1,1)
            obs[14]=min(ne.dist(self.x,self.y)/(W+H),1.); obs[15]=1.

        # Home village
        mv=[v for v in villages if v.id==self.home_village]
        if mv:
            v=mv[0]
            obs[16]=np.clip((v.x-self.x)/W,-1,1); obs[17]=np.clip((v.y-self.y)/H,-1,1)
            obs[18]=float(v.is_defended); obs[19]=v.war_exhaustion
            obs[20]=v.morale; obs[21]=min(v.total_wealth()/100.,1.)

        obs[22]=np.clip(self.vx/self.effective_speed(),-1,1)
        obs[23]=np.clip(self.vy/self.effective_speed(),-1,1)
        obs[24]=min(self.kills/30.,1.); obs[25]=self.energy/CFG.warrior_energy_max
        obs[26]=float(self.is_war_chief); obs[27]=float(self.is_raiding)
        obs[28]=min(sum(1 for o in friendly
                        if math.hypot(o.x-self.x,o.y-self.y)<=CFG.warrior_protect_r)/12.,1.)
        obs[29]=float(len(enemy_vids)>0)
        obs[30]=math.sin(season_phase*2*math.pi); obs[31]=math.cos(season_phase*2*math.pi)
        t=terrain.get(self.x,self.y)
        obs[32]=float(t==TerrainType.MOUNTAIN); obs[33]=float(t==TerrainType.FOREST)
        obs[34]=float(t==TerrainType.RIVER); obs[35]=float(t==TerrainType.DESERT)
        return obs

    def apply_action(self, action, terrain):
        spd=self.effective_speed()*terrain.speed_mult(self.x,self.y)
        d=_ACT_DIRS[action]
        self.vx=self.vx*CFG.warrior_friction+d[0]*CFG.warrior_accel*spd
        self.vy=self.vy*CFG.warrior_friction+d[1]*CFG.warrior_accel*spd
        v=math.hypot(self.vx,self.vy)
        if v>spd: self.vx*=spd/v; self.vy*=spd/v
        self.x=float(np.clip(self.x+self.vx,0,CFG.world_w))
        self.y=float(np.clip(self.y+self.vy,0,CFG.world_h))
        if self.x<=0 or self.x>=CFG.world_w: self.vx*=-0.5
        if self.y<=0 or self.y>=CFG.world_h: self.vy*=-0.5
        cost=CFG.warrior_metabolism+CFG.warrior_move_cost*(self.vx**2+self.vy**2)
        if self.is_raiding: cost*=1.15
        self.energy-=cost; self.age+=1
        if self.energy<=0: self.alive=False

    def override_move(self, tx, ty):
        """Force warrior toward target position (raid/defend)."""
        dx=tx-self.x; dy=ty-self.y; dd=math.hypot(dx,dy)+1e-8
        spd=self.effective_speed()
        self.vx=(dx/dd)*spd; self.vy=(dy/dd)*spd
        self.x=float(np.clip(self.x+self.vx,0,CFG.world_w))
        self.y=float(np.clip(self.y+self.vy,0,CFG.world_h))

    def try_kill_combat(self, targets: List) -> int:
        killed=0
        for t in targets:
            if hasattr(t,'alive') and t.alive and math.hypot(t.x-self.x,t.y-self.y)<=CFG.warrior_kill_radius:
                t.alive=False; self.kills+=1
                self.energy=min(self.energy+CFG.warrior_attack_bonus,CFG.warrior_energy_max)
                killed+=1
        return killed

# ─────────────────────────────────────────────────────────────────────────────
#  GLOBAL STATE
# ─────────────────────────────────────────────────────────────────────────────
def build_global_state(world) -> np.ndarray:
    W,H=CFG.world_w,CFG.world_h
    ao=world.organisms; aw=world.warriors
    gs=np.zeros(CFG.global_state_dim,dtype=np.float32)
    gs[0]=min(len(ao)/CFG.max_organisms,1.);  gs[1]=min(len(aw)/CFG.max_warriors,1.)
    gs[2]=min(len(world.food)/CFG.max_food,1.); gs[3]=min(len(world.villages)/CFG.max_villages,1.)
    gs[4]=math.sin(world.season_phase*2*math.pi); gs[5]=math.cos(world.season_phase*2*math.pi)
    gs[6]=float(world.diplomacy.total_wars())/max(len(world.villages),1)
    if ao:
        gs[7]=np.mean([o.x for o in ao])/W;  gs[8]=np.mean([o.y for o in ao])/H
        gs[9]=np.mean([o.energy for o in ao])/CFG.max_energy
        gs[10]=float(sum(1 for o in ao if o.sex==Sex.MALE))/max(len(ao),1)
        gs[11]=float(sum(1 for o in ao if o.pregnant))/max(len(ao),1)
        gs[12]=float(sum(1 for o in ao if o.is_elder))/max(len(ao),1)
        gs[13]=float(sum(1 for o in ao if o.is_zealot))/max(len(ao),1)
        gs[14]=float(sum(1 for o in ao if o.religion_id is not None))/max(len(ao),1)
    # Religion stats
    for ri, (rid, rel) in enumerate(world.religions.items()):
        if ri>=2: break
        gs[15+ri*3]=min(rel.follower_count/max(len(ao),1),1.)
        gs[16+ri*3]=float(rel.tenet_expansionist)
        gs[17+ri*3]=float(rel.tenet_pacifist)
    gs[21]=min(sum(v.total_wealth() for v in world.villages)/500.,1.)
    gs[22]=min(sum(w.kills for w in aw)/200.,1.)
    # War count
    gs[23]=float(world.diplomacy.total_wars())/5.
    # Alliance count
    gs[24]=float(sum(1 for s in world.diplomacy._state.values() if s==WarState.ALLIANCE))/5.
    gs[25]=float(world.step_count)/CFG.max_steps
    if world.villages:
        gs[26]=float(sum(1 for v in world.villages if v.golden_age))/len(world.villages)
        gs[27]=float(sum(1 for v in world.villages if v.dark_age))/len(world.villages)
    gs[28]=float(len(world.religions))/CFG.n_religions
    gs[29]=float(sum(1 for o in ao if o.sick))/max(len(ao),1)
    return gs

# ─────────────────────────────────────────────────────────────────────────────
#  WORLD
# ─────────────────────────────────────────────────────────────────────────────
# ─────────────────────────────────────────────────────────────────────────────
#  DISASTER MANAGER
# ─────────────────────────────────────────────────────────────────────────────
class DisasterType(Enum):
    DROUGHT=0; FLOOD=1; FIRE=2; EARTHQUAKE=3; FAMINE=4

DISASTER_COLORS = {
    DisasterType.DROUGHT:    "#8B4513",
    DisasterType.FLOOD:      "#1E90FF",
    DisasterType.FIRE:       "#FF4500",
    DisasterType.EARTHQUAKE: "#8B8B00",
    DisasterType.FAMINE:     "#4B0082",
}

class DisasterManager:
    def __init__(self):
        self.active: List[Dict] = []
        self.history: List[Dict] = []
        self._famine_active = False

    @property
    def famine_mult(self) -> float:
        """Food spawn multiplier; <1 during famine."""
        return CFG.famine_food_mult if self._famine_active else 1.0

    def tick(self, world, season_mult: float, step: int):
        # Spawn probability rises in winter and when villages are dense
        prob = CFG.disaster_base_prob
        if season_mult < 0.85: prob *= 2.2   # winter = more disasters
        if len(world.villages) > 5: prob *= 1.3
        if random.random() < prob:
            dtype = random.choice(list(DisasterType))
            # Pick an epicentre — bias toward populated areas
            if world.organisms and random.random() < 0.6:
                org = random.choice(world.organisms)
                cx, cy = org.x + random.gauss(0, 20), org.y + random.gauss(0, 20)
            else:
                cx = random.uniform(15, CFG.world_w - 15)
                cy = random.uniform(15, CFG.world_h - 15)
            cx = float(np.clip(cx, 0, CFG.world_w))
            cy = float(np.clip(cy, 0, CFG.world_h))
            intensity = random.uniform(0.6, 1.5)
            entry = {"type": dtype, "x": cx, "y": cy,
                     "intensity": intensity, "timer": CFG.disaster_duration,
                     "step": step}
            self.active.append(entry)
            self.history.append(entry.copy())
            if dtype == DisasterType.FAMINE:
                self._famine_active = True
            vname = ", ".join(f"V{v.id}" for v in world.villages
                              if math.hypot(v.x-cx, v.y-cy) < 60)
            print(f"  [Disaster] {dtype.name} at ({cx:.0f},{cy:.0f})"
                  f" intensity={intensity:.2f} near=[{vname}] step={step}")

        # Apply active disasters
        for d in self.active:
            d["timer"] -= 1
            dtype = d["type"]; cx = d["x"]; cy = d["y"]; inten = d["intensity"]

            if dtype == DisasterType.DROUGHT:
                # Periodically kill food in radius
                if d["timer"] % 5 == 0:
                    world.food = [f for f in world.food
                                  if math.hypot(f[0]-cx, f[1]-cy) > CFG.drought_food_kill_r*inten]

            elif dtype == DisasterType.FLOOD:
                for o in world.organisms:
                    if o.alive and math.hypot(o.x-cx, o.y-cy) < CFG.flood_radius*inten:
                        o.energy -= CFG.flood_damage * inten
                        if o.energy <= 0: o.alive = False
                for w in world.warriors:
                    if w.alive and math.hypot(w.x-cx, w.y-cy) < CFG.flood_radius*inten:
                        w.energy -= CFG.flood_damage * 0.5 * inten

            elif dtype == DisasterType.FIRE:
                if d["timer"] % 3 == 0:
                    world.food = [f for f in world.food
                                  if math.hypot(f[0]-cx, f[1]-cy) > CFG.fire_radius*inten
                                  or random.random() > CFG.fire_food_kill_frac]
                for o in world.organisms:
                    if o.alive and math.hypot(o.x-cx, o.y-cy) < CFG.fire_radius*inten:
                        o.energy -= CFG.fire_damage * inten
                        if o.energy <= 0: o.alive = False

            elif dtype == DisasterType.EARTHQUAKE:
                # One-time burst at first tick, then lingering low damage
                dmg = CFG.earthquake_damage if d["timer"] == CFG.disaster_duration - 1 else 0.015
                for o in world.organisms:
                    if o.alive and math.hypot(o.x-cx, o.y-cy) < CFG.earthquake_radius*inten:
                        o.energy -= dmg * inten
                        if o.energy <= 0: o.alive = False
                # Damage village resources
                if d["timer"] == CFG.disaster_duration - 1:
                    for v in world.villages:
                        if math.hypot(v.x-cx, v.y-cy) < CFG.earthquake_radius*inten:
                            v.resources *= 0.6
                            v.morale = max(0., v.morale - 0.3)
                            print(f"  [Earthquake] V{v.id} resources damaged!")

            elif dtype == DisasterType.FAMINE:
                # Drain energy from all organisms slowly
                for o in world.organisms:
                    if o.alive:
                        o.energy -= 0.03 * inten
                        if o.energy <= 0: o.alive = False

        # Clear expired
        expired = [d for d in self.active if d["timer"] <= 0]
        for d in expired:
            if d["type"] == DisasterType.FAMINE:
                self._famine_active = False
        self.active = [d for d in self.active if d["timer"] > 0]

    def n_active(self) -> int:
        return len(self.active)

    def active_names(self) -> str:
        if not self.active: return "None"
        return ", ".join(d["type"].name for d in self.active)


class World:
    def __init__(self):
        self.terrain   = TERRAIN
        self.villages: List[Village]  = []
        self.organisms: List[Organism]= []
        self.warriors:  List[Warrior] = []
        self.food:      List[Tuple]   = []
        self.bad_zones: List[Tuple]   = []
        self.religions: Dict[int,Religion] = {}
        self.diplomacy = DiplomacyLedger()
        self.disaster_mgr = DisasterManager()
        self.step_count=0; self._last_village_step=-CFG.clan_form_cooldown
        self.season_phase=0.0
        self.stats={k:[] for k in [
            "pop","pop_m","pop_f","warriors","food",
            "n_villages","in_village","pregnancies","elders","sick","zealots",
            "births","births_war","deaths_starve","deaths_combat",
            "n_wars","n_alliances","total_raids","resources_stolen",
            "religion_followers","religion2_followers","religion3_followers",
            "avg_spd_m","avg_spd_f","avg_gen","avg_faith",
            "golden_ages","dark_ages","season","n_religions_active",
            "war_declarations","trade_trips","schisms",
            "n_disasters","disaster_history",
        ]}
        self._total_raids=0; self._total_stolen=0.0; self._war_decls=0
        self._trade_trips=0; self._schisms=0
        self._init()

    def _rp(self,margin=12):
        return (random.uniform(margin,CFG.world_w-margin),random.uniform(margin,CFG.world_h-margin))

    def _init(self):
        # Two starting villages on opposite sides
        positions=[(CFG.world_w*0.25,CFG.world_h*0.5),(CFG.world_w*0.75,CFG.world_h*0.5)]
        for i in range(min(CFG.init_villages,2)):
            v=Village(*positions[i],clan_id=i+1); v.founded_step=0
            self.villages.append(v)
        att=0
        while len(self.bad_zones)<CFG.n_bad_zones and att<200:
            x,y=self._rp()
            if all(v.dist(x,y)>CFG.village_radius+CFG.bad_zone_radius+12 for v in self.villages):
                self.bad_zones.append((x,y))
            att+=1
        for _ in range(CFG.init_food): self.food.append(self._rp())
        for vi,v in enumerate(self.villages):
            per=CFG.init_organisms//len(self.villages)
            for k in range(per):
                ang=random.uniform(0,2*math.pi); r=random.uniform(0,CFG.village_radius*0.85)
                x=float(np.clip(v.x+r*math.cos(ang),0,CFG.world_w))
                y=float(np.clip(v.y+r*math.sin(ang),0,CFG.world_h))
                sex=Sex.MALE if k%2==0 else Sex.FEMALE
                org=Organism(x,y,sex=sex,clan_id=v.clan_id); org.home_village=v.id
                self.organisms.append(org)
            for k in range(CFG.init_warriors//len(self.villages)):
                ang=random.uniform(0,2*math.pi); r=random.uniform(0,CFG.village_radius)
                x=float(np.clip(v.x+r*math.cos(ang),0,CFG.world_w))
                y=float(np.clip(v.y+r*math.sin(ang),0,CFG.world_h))
                self.warriors.append(Warrior(x,y,home_village=v.id,clan_id=v.clan_id))

    # ── SEASON ────────────────────────────────────────────────────────────
    def _update_season(self):
        self.season_phase=(self.step_count%CFG.season_period)/CFG.season_period
        return 1.+CFG.season_amp*math.sin(self.season_phase*2*math.pi)

    # ── VILLAGE DEFENCE ───────────────────────────────────────────────────
    def _update_defence(self,aw):
        breach_r = CFG.village_radius * 2.0
        for v in self.villages:
            v.is_defended=any(math.hypot(w.x-v.x,w.y-v.y)<=breach_r
                              for w in aw if w.home_village==v.id)


    def _auto_respawn_warriors(self,aw):
        for v in self.villages:
            local=[w for w in aw if w.home_village==v.id]
            if (len(local)<CFG.min_warriors_per_village
                    and self.step_count-v.last_warrior_spawn>=CFG.warrior_respawn_interval
                    and len(self.warriors)<CFG.max_warriors):
                ang=random.uniform(0,2*math.pi); r=random.uniform(0,CFG.village_radius*0.6)
                x=float(np.clip(v.x+r*math.cos(ang),0,CFG.world_w))
                y=float(np.clip(v.y+r*math.sin(ang),0,CFG.world_h))
                self.warriors.append(Warrior(x,y,energy=CFG.warrior_energy_init*0.8,
                                             home_village=v.id,clan_id=v.clan_id))
                v.last_warrior_spawn=self.step_count

    def _elect_war_chiefs(self,aw):
        for v in self.villages:
            vw=[w for w in aw if w.home_village==v.id]
            if not vw: continue
            best=max(vw,key=lambda w:w.kills)
            for w in vw: w.is_war_chief=False
            if best.kills>4: best.is_war_chief=True

    # ── RELIGION ──────────────────────────────────────────────────────────
    def _try_spawn_religion(self):
        if self.step_count<CFG.religion_spawn_step: return
        if len(self.religions)>=CFG.n_religions: return
        # Chance per step to spawn a new religion from a charismatic org
        ao=[o for o in self.organisms if o.alive and o.religion_id is None
            and o.genome.charisma>0.5]
        if not ao: return
        if random.random()>0.0005*len(ao): return
        founder=max(random.sample(ao,min(5,len(ao))),key=lambda o:o.genome.charisma)
        ridx=len(self.religions)
        if ridx>=len(RELIGION_NAMES): return
        rn=RELIGION_NAMES[ridx]
        rel=Religion(rn[0],rn[1],rn[2],founder.id,founder.x,founder.y,self.step_count)
        self.religions[rel.id]=rel
        founder.religion_id=rel.id; founder.faith_level=1.0
        hv=[v for v in self.villages if v.id==founder.home_village]
        if hv: rel.holy_sites.append((hv[0].x,hv[0].y))
        print(f"  [Religion] '{rel.name}' ({rel.symbol}) founded at step {self.step_count} "
              f"by org {founder.id} charisma={founder.genome.charisma:.2f}")

    def _spread_religion(self,ao):
        for rel in self.religions.values():
            if not rel.is_active: continue
            # Followers try to convert nearby unbelievers / different-faith
            followers=[o for o in ao if o.alive and o.religion_id==rel.id]
            rel.follower_count=len(followers)
            for f in followers:
                if random.random()>CFG.religion_spread_prob*(1.+f.genome.charisma): continue
                nearby=[o for o in ao if o.alive and o.id!=f.id
                        and math.hypot(o.x-f.x,o.y-f.y)<=CFG.religion_spread_radius]
                for t in nearby:
                    if t.religion_id==rel.id: continue
                    # Conversion probability based on charisma + tenets
                    conv_prob=0.08*f.genome.charisma*(1.+CFG.religion_charm_weight)
                    if rel.tenet_expansionist: conv_prob*=1.5
                    if t.religion_id is not None: conv_prob*=0.3  # harder to convert believers
                    if random.random()<conv_prob:
                        old_rid=t.religion_id
                        t.religion_id=rel.id
                        t.faith_level=min(t.faith_level+0.3,1.0)
                        rel.conversions+=1
                        if old_rid is not None and old_rid in self.religions:
                            self.religions[old_rid].follower_count=max(0,self.religions[old_rid].follower_count-1)

    def _update_faith(self,ao):
        for o in ao:
            if not o.alive: continue
            # Natural faith decay
            if o.religion_id is not None and random.random()<CFG.apostasy_prob:
                o.faith_level=max(0.,o.faith_level-0.05)
                if o.faith_level<=0.0: o.religion_id=None
            # Zealot status
            was_zealot=o.is_zealot
            o.is_zealot=(o.faith_level>=CFG.zealot_threshold and o.religion_id is not None)
            # Zealot attacks: different-faith organism nearby
            if o.is_zealot:
                nearby=[t for t in ao if t.alive and t.id!=o.id
                        and t.religion_id!=o.religion_id and t.religion_id is not None
                        and math.hypot(t.x-o.x,t.y-o.y)<=4.0]
                for t in nearby[:1]:
                    t.energy-=0.5*CFG.zealot_attack_bonus
                    if t.energy<=0: t.alive=False

    def _check_religious_schism(self,ao):
        """If minority religion reaches schism threshold in a village → civil war (auto-war declaration)."""
        for v in self.villages:
            local=[o for o in ao if o.alive and v.contains(o.x,o.y)]
            if not local: continue
            tension=v.religion_tension(self.religions)
            if tension>CFG.schism_threshold and random.random()<0.01*tension:
                # The minority religion group leaves and joins a different village
                total=sum(v.religion_counts.values())+1e-8
                for rid,cnt in v.religion_counts.items():
                    frac=cnt/total
                    if 0.15<frac<0.45:  # minority
                        victims=[o for o in local if o.religion_id==rid]
                        # They flee to another village with same religion or found a new one
                        other_vs=[vv for vv in self.villages if vv.id!=v.id
                                  and vv.dominant_religion==rid]
                        if other_vs:
                            dest=random.choice(other_vs)
                            for vm in victims:
                                vm.home_village=dest.id; vm.clan_id=dest.clan_id
                        self._schisms+=1
                        print(f"  [Schism] Religion {rid} minority flees V{v.id} → step {self.step_count}")
                        break

    # ── WAR / DIPLOMACY ───────────────────────────────────────────────────
    def _evaluate_war_decisions(self,ao,aw):
        if len(self.villages)<2: return
        for v in self.villages:
            if v.war_exhaustion>0.75: continue   # genuinely exhausted
            local_w=[w for w in aw if w.home_village==v.id]
            if len(local_w)<1: continue           # need at least 1 warrior
            enemies=self.diplomacy.at_war_with(v.id)
            if len(enemies)>=3: continue          # don't fight 3+ fronts

            local_orgs=[o for o in ao if o.alive and v.contains(o.x,o.y)]
            zealot_count=sum(1 for o in local_orgs if o.is_zealot)
            v_rel=v.dominant_religion
            v_religion=self.religions.get(v_rel) if v_rel else None

            for target in self.villages:
                if target.id==v.id: continue
                if self.diplomacy.state(v.id,target.id)!=WarState.PEACE: continue
                war_desire=0.0; cause=None

                # ── 1. RESOURCE WAR ──────────────────────────────────────
                # Trigger if target is meaningfully richer OR attacker is poor
                wealth_ratio = target.total_wealth() / max(v.total_wealth(), 1.)
                if wealth_ratio > 1.4:   # target at least 40% richer
                    war_desire += 0.35 * min(wealth_ratio - 1.0, 1.5)
                    cause = WarCause.RESOURCES
                # Desperate poor attack anyone
                if v.total_wealth() < 8. and target.total_wealth() > 5.:
                    war_desire += 0.4; cause = WarCause.RESOURCES

                # ── 2. TERRITORY WAR ─────────────────────────────────────
                dist = v.dist(target.x, target.y)
                if dist < CFG.war_declare_territory_dist:
                    proximity_pressure = 1. - dist/CFG.war_declare_territory_dist
                    war_desire += 0.3 * proximity_pressure
                    cause = cause or WarCause.TERRITORY
                # Overpopulated village expands aggressively
                local_pop = len(local_orgs)
                if local_pop > CFG.overpop_threshold * 0.8:
                    war_desire += 0.2 * (local_pop / CFG.overpop_threshold)
                    cause = cause or WarCause.TERRITORY

                # ── 3. RELIGIOUS WAR ─────────────────────────────────────
                t_rel = target.dominant_religion
                if v_rel and t_rel and v_rel != t_rel:
                    r2 = self.religions.get(t_rel)
                    if v_religion and r2:
                        tension = v_religion.tension_with(r2)
                        war_desire += tension * CFG.war_declare_religion_thresh
                        if tension > 0.4: cause = WarCause.RELIGION
                # Zealots in village actively push for holy war
                if zealot_count > 0 and t_rel and t_rel != v_rel:
                    war_desire += 0.06 * zealot_count
                    cause = WarCause.RELIGION
                # Extremist religion goes to war even against peaceful targets
                if v_religion and v_religion.tenet_extremist:
                    war_desire += CFG.war_zealot_prob * 30  # flat boost

                # ── 4. RETALIATION ───────────────────────────────────────
                if v.raided_step > 0 and self.step_count - v.raided_step < 300:
                    war_desire += 0.55; cause = WarCause.RETALIATION

                # ── 5. DOMINANCE WAR — no reason needed ──────────────────
                # Any village with enough warriors may just want to dominate
                if len(local_w) >= 3 and random.random() < CFG.war_dominance_prob:
                    war_desire += 0.25; cause = cause or WarCause.TERRITORY

                # ── MODIFIERS ────────────────────────────────────────────
                # Aggressive governance amplifies; democracy is more cautious
                gov_mult = {"AUTOCRACY":1.4,"ANARCHY":1.6,"DEMOCRACY":0.85,"THEOCRACY":1.1}
                war_desire *= gov_mult.get(v.governance.name, 1.0)
                # Policy aggression (from genes of population)
                war_desire *= max(0.2, v.policy_aggression)
                # War exhaustion reduces desire
                war_desire *= (1. - v.war_exhaustion * 0.6)
                # Pacifist DAMPENS but does NOT prevent (real world: pacifist faith ≠ pacifist state)
                if v_religion and v_religion.tenet_pacifist:
                    war_desire *= CFG.pacifist_war_damp

                # ── DECLARE ───────────────────────────────────────────────
                if war_desire > 0.18 and random.random() < CFG.war_declare_prob * war_desire:
                    self.diplomacy.declare_war(v.id, target.id,
                                               cause or WarCause.RESOURCES,
                                               self.step_count)
                    self._war_decls += 1
                    if cause == WarCause.RELIGION and v_rel:
                        self.religions[v_rel].wars_caused += 1

            # ── ALLIANCE: share religion + nearby + at peace ─────────────
            for target in self.villages:
                if target.id == v.id: continue
                if self.diplomacy.state(v.id, target.id) != WarState.PEACE: continue
                if (v_rel and v_rel == target.dominant_religion
                        and v.dist(target.x, target.y) < CFG.territory_radius * 2.0
                        and random.random() < 0.003):
                    self.diplomacy.make_alliance(v.id, target.id)

            # ── AUTO TRUCE when exhausted ─────────────────────────────────
            for eid in list(self.diplomacy.at_war_with(v.id)):
                if v.war_exhaustion > 0.75:
                    self.diplomacy.make_truce(v.id, eid)
                    v.war_exhaustion = 0.35

    def _conduct_raids(self,ao,aw):
        """Warriors at war move toward enemy village and raid it."""
        total_stolen=0.0; total_raids=0
        for v in self.villages:
            enemy_vids=self.diplomacy.at_war_with(v.id)
            if not enemy_vids: continue
            v.war_exhaustion=min(1.,v.war_exhaustion+0.001)
            local_w=[w for w in aw if w.home_village==v.id and w.alive]
            for w in local_w:
                w.is_raiding=True
                # Pick nearest enemy village
                ev=[vv for vv in self.villages if vv.id in enemy_vids]
                if not ev: break
                target_v=min(ev,key=lambda vv:vv.dist(w.x,w.y))
                w.override_move(target_v.x+random.uniform(-8,8),
                                target_v.y+random.uniform(-8,8))
                w.energy-=CFG.war_attrition_damage
                # Kill enemy orgs near enemy village
                for o in ao:
                    if not o.alive: continue
                    if o.clan_id==w.clan_id: continue
                    if target_v.contains(o.x,o.y) and math.hypot(o.x-w.x,o.y-w.y)<=CFG.warrior_kill_radius:
                        if random.random()<CFG.raid_kill_prob:
                            o.alive=False; w.kills+=1
                            w.energy=min(w.energy+CFG.warrior_attack_bonus,CFG.warrior_energy_max)
                # Steal resources when warrior inside enemy village
                if target_v.dist(w.x,w.y)<=CFG.village_radius:
                    steal=target_v.resources*CFG.raid_resource_steal/max(1,len(local_w))
                    target_v.resources=np.maximum(0.,target_v.resources-steal)
                    # Find own village and add resources
                    mv=[vv for vv in self.villages if vv.id==v.id]
                    if mv: mv[0].resources+=steal
                    target_v.morale=max(0.,target_v.morale-0.05)
                    target_v.raided_step=self.step_count
                    target_v.war_exhaustion=min(1.,target_v.war_exhaustion+0.003)
                    total_stolen+=float(steal.sum()); total_raids+=1

        # Defenders fight back
        for v in self.villages:
            enemy_vids=self.diplomacy.at_war_with(v.id)
            if not enemy_vids: continue
            defenders=[w for w in aw if w.home_village==v.id and w.alive]
            attackers=[w for w in aw if w.home_village in enemy_vids and w.alive
                       and v.dist(w.x,w.y)<=CFG.village_radius+5]
            for d in defenders:
                kills=d.try_kill_combat(attackers)
            for a in attackers:
                kills=a.try_kill_combat(defenders)
        self._total_raids+=total_raids; self._total_stolen+=total_stolen

    # ── TRADE (allied + open) ─────────────────────────────────────────────
    def _allied_trade(self):
        trips=0
        self.diplomacy.active_trade_this_step=set()

        # Allied trade: automatic, larger transfer
        for v in self.villages:
            for allied_id in self.diplomacy.allied_with(v.id):
                av=[vv for vv in self.villages if vv.id==allied_id]
                if not av: continue
                a=av[0]
                if v.resources[0]>a.resources[0]+8.:
                    transfer=min(CFG.trade_agreement_amount,v.resources[0]-a.resources[0])
                    v.resources[0]-=transfer; a.resources[0]+=transfer
                    k=self.diplomacy._key(v.id,a.id)
                    self.diplomacy.trade_routes[k]=self.diplomacy.trade_routes.get(k,0.)+transfer
                    self.diplomacy.active_trade_this_step.add(k)
                    trips+=1

        # Open trade: any peaceful pair within range, if there's imbalance
        for i,v in enumerate(self.villages):
            for j,a in enumerate(self.villages):
                if j<=i: continue
                if self.diplomacy.state(v.id,a.id)==WarState.AT_WAR: continue
                if v.dist(a.x,a.y)>CFG.trade_route_dist: continue
                # Trade the resource each village has more of
                for ri in range(4):
                    diff=v.resources[ri]-a.resources[ri]
                    if abs(diff)<5.: continue
                    if random.random()>CFG.open_trade_prob: continue
                    transfer=min(CFG.open_trade_amount, abs(diff)*0.3)
                    if diff>0:
                        v.resources[ri]-=transfer; a.resources[ri]+=transfer
                    else:
                        a.resources[ri]-=transfer; v.resources[ri]+=transfer
                    k=self.diplomacy._key(v.id,a.id)
                    self.diplomacy.trade_routes[k]=self.diplomacy.trade_routes.get(k,0.)+transfer
                    self.diplomacy.active_trade_this_step.add(k)
                    trips+=1; break  # one resource per pair per step

        self._trade_trips+=trips; return trips

    # ── VILLAGE FORMATION ─────────────────────────────────────────────────
    def _try_form_village(self,ao):
        if len(self.villages)>=CFG.max_villages: return
        if self.step_count-self._last_village_step<CFG.clan_form_cooldown: return
        for _ in range(80):
            cx=random.uniform(20,CFG.world_w-20); cy=random.uniform(20,CFG.world_h-20)
            if any(v.dist(cx,cy)<CFG.village_radius*3.2 for v in self.villages): continue
            if any(math.hypot(cx-z[0],cy-z[1])<CFG.bad_zone_radius+CFG.village_radius
                   for z in self.bad_zones): continue
            if self.terrain.get(cx,cy) in [TerrainType.MOUNTAIN,TerrainType.DESERT]: continue
            nearby=[o for o in ao if math.hypot(o.x-cx,o.y-cy)<=CFG.clan_form_radius]
            if len(nearby)>=CFG.clan_form_density:
                rcx=sum(o.x for o in nearby)/len(nearby)
                rcy=sum(o.y for o in nearby)/len(nearby)
                cc: Dict[int,int]=defaultdict(int)
                for o in nearby: cc[o.clan_id or 0]+=1
                dom=max(cc,key=lambda k:cc[k])
                nv=Village(rcx,rcy,clan_id=dom); nv.founded_step=self.step_count
                self.villages.append(nv)
                for o in nearby:
                    if o.home_village is None or random.random()<0.5:
                        o.home_village=nv.id; o.clan_id=dom
                self._last_village_step=self.step_count
                print(f"  [Village+] ({rcx:.0f},{rcy:.0f}) clan={dom} gov={nv.governance.name}"
                      f" n={len(nearby)} total={len(self.villages)}")
                return

    # ── MIGRATION ─────────────────────────────────────────────────────────
    def _handle_migration(self,ao):
        for v in self.villages:
            local=[o for o in ao if o.alive and v.contains(o.x,o.y)]
            if len(local)>CFG.overpop_threshold and len(self.villages)>1:
                migrants=random.sample(local,len(local)//4)
                candidates=[vv for vv in self.villages if vv.id!=v.id
                             and self.diplomacy.state(v.id,vv.id)!=WarState.AT_WAR]
                if not candidates: continue
                target=min(candidates,key=lambda vv:sum(
                    1 for o in ao if o.alive and vv.contains(o.x,o.y)))
                for m in migrants: m.home_village=target.id; m.clan_id=target.clan_id

    # ── FOOD ──────────────────────────────────────────────────────────────
    def _spawn_food(self,season_mult):
        ratio=len(self.food)/CFG.max_food
        famine_m = self.disaster_mgr.famine_mult
        for _ in range(3):
            if len(self.food)>=CFG.max_food: break
            x,y=self._rp()
            rate=CFG.food_spawn_rate*season_mult*famine_m*max(0.,1.-ratio**1.3)*self.terrain.food_mult(x,y)
            if random.random()<rate/3.: self.food.append((x,y))

    # ── DISEASE ───────────────────────────────────────────────────────────
    def _spread_disease(self,ao):
        for v in self.villages:
            local=[o for o in ao if o.alive and v.contains(o.x,o.y)]
            density=len(local)/max(1,math.pi*CFG.village_radius**2)
            if density>0.12 and random.random()<0.0015:
                for o in local:
                    if not o.sick and random.random()<0.04*(1.-o.genome.immune_strength):
                        o.sick=True; o.sick_timer=CFG.sick_duration

    # ── REPRODUCTION ──────────────────────────────────────────────────────
    def _org_threshold(self):
        fr=len(self.food)/max(CFG.max_food,1)
        return CFG.reproduce_threshold+(1.-fr)*20.

    def _try_mate(self,ao,threshold):
        births=0
        males  =[o for o in ao if o.sex==Sex.MALE  and o.energy>=threshold and o.mate_timer==0]
        females=[o for o in ao if o.sex==Sex.FEMALE and o.energy>=threshold
                 and not o.pregnant and o.mate_timer==0]
        elders =[o for o in ao if o.is_elder]
        for f in females:
            if len(self.organisms)>=CFG.max_organisms: break
            boost=(0.25 if any(math.hypot(e.x-f.x,e.y-f.y)<=CFG.village_radius for e in elders) else 0.)
            if f.is_zealot: boost+=CFG.zealot_fertility_bonus
            if random.random()>f.genome.fertility+boost: continue
            cands=[(math.hypot(m.x-f.x,m.y-f.y),m) for m in males
                   if math.hypot(m.x-f.x,m.y-f.y)<=f.genome.vision*1.3]
            if not cands: continue
            _,father=min(cands,key=lambda t:t[0])
            f.pregnant=True; f.gestation=CFG.gestation_steps; f.father_genome=father.genome
            f.energy-=CFG.reproduce_cost*0.58; father.energy-=CFG.reproduce_cost*0.42
            f.mate_timer=50; father.mate_timer=30
            if father in males: males.remove(father)
        for f in [o for o in ao if o.sex==Sex.FEMALE and o.pregnant]:
            f.gestation-=1
            if f.gestation<=0:
                f.pregnant=False
                if f.father_genome is None: continue
                csex=random.choice([Sex.MALE,Sex.FEMALE])
                cg=f.genome.crossover(f.father_genome,csex)
                child=Organism(
                    x=float(np.clip(f.x+random.uniform(-5,5),0,CFG.world_w)),
                    y=float(np.clip(f.y+random.uniform(-5,5),0,CFG.world_h)),
                    genome=cg,sex=csex,energy=CFG.reproduce_cost*0.4,
                    generation=f.generation+1,clan_id=f.clan_id)
                child.home_village=f.home_village
                # Inherit religion from mother
                child.religion_id=f.religion_id
                child.faith_level=f.faith_level*0.5
                f.father_genome=None; self.organisms.append(child); births+=1
        return births

    def _reproduce_war(self,aw):
        births=0
        for w in aw:
            if w.alive and w.energy>=CFG.warrior_rep_threshold and len(self.warriors)<CFG.max_warriors:
                w.energy-=CFG.warrior_rep_cost
                self.warriors.append(Warrior(
                    x=float(np.clip(w.x+random.uniform(-5,5),0,CFG.world_w)),
                    y=float(np.clip(w.y+random.uniform(-5,5),0,CFG.world_h)),
                    energy=CFG.warrior_rep_cost*0.5,
                    home_village=w.home_village,clan_id=w.clan_id))
                births+=1
        return births

    # ── REWARD FUNCTIONS ──────────────────────────────────────────────────
    def _org_rew(self, o, ate, in_bad, in_village, v_golden, v_dark, curiosity, v_at_war):
        r  = 3.5 if ate else 0.
        r -= (2.5*(1.-o.genome.immune_strength)) if in_bad else 0.
        r += (1.2 if v_golden else (0.5 if in_village else 0.))
        r -= 0.4 if v_dark else 0.
        r -= 0.6 if (v_at_war and in_village) else 0.
        r += curiosity; r += 0.08
        r -= 0.12*max(o.genome.speed-2.3,0)
        r -= 0.5 if o.sick else 0.
        if o.is_zealot: r += 0.2
        return float(r)

    def _war_rew(self, w, near_friendly, v_at_war, kills):
        r = 12.0*kills + 0.4*near_friendly
        r += 2.0 if (v_at_war and w.is_raiding) else 0.
        r += 0.12; return float(r)

    # ── MAIN STEP ─────────────────────────────────────────────────────────
    def step(self, op: MAPPOPolicy, wp: MAPPOPolicy) -> dict:
        self.step_count+=1
        season_mult=self._update_season()
        ao=[o for o in self.organisms if o.alive]
        aw=[w for w in self.warriors  if w.alive]
        births_o=0; births_w=0; d_starve=0; d_combat=0
        rew_o=0.; rew_w=0.

        # Village updates
        for v in self.villages:
            local=[o for o in ao if v.contains(o.x,o.y)]
            local_w=[w for w in aw if w.home_village==v.id]
            v_wars=self.diplomacy.at_war_with(v.id)
            v.update_governance(local,v_wars)
            v.update_religion_counts(local)
            # Resource generation: scales with POPULATION density + terrain
            # More people = more production; forest/river terrain = more wood/food
            pop_density = len(local) / max(1, CFG.overpop_threshold)  # 0-1+
            terrain_m   = self.terrain.food_mult(v.x, v.y)
            # Food accumulates from farmers in village
            farmers_here = sum(1 for o in local if o.profession==Profession.FARMER)
            v.resources[0] = min(v.resources[0] + 0.05*farmers_here*terrain_m*season_mult,
                                 CFG.resource_max[0])
            # Wood: dense forests, more workers
            v.resources[1] = min(v.resources[1] + 0.02*pop_density*terrain_m*season_mult,
                                 CFG.resource_max[1])
            # Stone: mountain bonus, slow
            stone_m = 2.0 if self.terrain.get(v.x,v.y)==TerrainType.MOUNTAIN else 0.8
            v.resources[2] = min(v.resources[2] + 0.008*pop_density*stone_m, CFG.resource_max[2])
            # Gold: rare, depends on population size (trade & taxation)
            v.resources[3] = min(v.resources[3] + 0.003*pop_density, CFG.resource_max[3])
            # Morale recovery
            v.morale=min(1.,v.morale+0.001)
            if not v_wars: v.war_exhaustion=max(0.,v.war_exhaustion-0.002)

        self._update_defence(aw)
        self._auto_respawn_warriors(aw)
        self._elect_war_chiefs(aw)

        # Religion
        self._try_spawn_religion()
        self._spread_religion(ao)
        self._update_faith(ao)
        self._check_religious_schism(ao)

        # Diplomacy
        self.diplomacy.tick()
        self._evaluate_war_decisions(ao,aw)
        self._conduct_raids(ao,aw)
        self._allied_trade()

        gs=build_global_state(self)

        # ── Org MAPPO act ──────────────────────────────────────────────────
        obs_o=[o.observe(self.food,aw,self.villages,self.bad_zones,
                         self.season_phase,self.terrain,self.religions,self.diplomacy)
               for o in ao]
        ids_o=[o.id for o in ao]; gs_list=[gs]*len(ao)
        if obs_o:
            acts_o,logps_o,vals_o,hxs_o=op.batch_act(obs_o,ids_o,gs_list)
            for i,o in enumerate(ao):
                o._obs=obs_o[i]; o._act=int(acts_o[i])
                o._logp=float(logps_o[i]); o._val=float(vals_o[i])

        # ── Warrior MAPPO act ──────────────────────────────────────────────
        obs_w=[w.observe(ao,self.villages,self.bad_zones,self.season_phase,
                         self.terrain,self.diplomacy) for w in aw]
        ids_w=[w.id for w in aw]; gs_wlist=[gs]*len(aw)
        if obs_w:
            acts_w,logps_w,vals_w,hxs_w=wp.batch_act(obs_w,ids_w,gs_wlist)
            for i,w in enumerate(aw):
                w._obs=obs_w[i]; w._act=int(acts_w[i])
                w._logp=float(logps_w[i]); w._val=float(vals_w[i])

        # ── Move organisms ─────────────────────────────────────────────────
        for o in ao:
            in_v=any(v.contains(o.x,o.y) for v in self.villages)
            # Foraging: leave village for food
            if in_v and o.energy>CFG.reproduce_threshold*0.72:
                vis_food=[f for f in self.food
                          if math.hypot(f[0]-o.x,f[1]-o.y)<=CFG.fog_radius
                          and not any(v.contains(f[0],f[1]) for v in self.villages)]
                if vis_food:
                    _,tf=min([(math.hypot(f[0]-o.x,f[1]-o.y),f) for f in vis_food],
                              key=lambda t:t[0])
                    dx=tf[0]-o.x; dy=tf[1]-o.y; dd=math.hypot(dx,dy)+1e-8
                    best=8; bdot=-99
                    for ai in range(8):
                        dot=_ACT_DIRS[ai][0]*(dx/dd)+_ACT_DIRS[ai][1]*(dy/dd)
                        if dot>bdot: bdot=dot; best=ai
                    o._act=best
            # Return home when low energy or in dark age
            hv=[v for v in self.villages if v.id==o.home_village]
            if hv and (o.energy<CFG.init_energy*0.30 or hv[0].dark_age):
                v=hv[0]; dx=v.x-o.x; dy=v.y-o.y; dd=math.hypot(dx,dy)+1e-8
                best=8; bdot=-99
                for ai in range(8):
                    dot=_ACT_DIRS[ai][0]*(dx/dd)+_ACT_DIRS[ai][1]*(dy/dd)
                    if dot>bdot: bdot=dot; best=ai
                o._act=best
            # Zealot missionary: head toward unbelievers
            if o.is_zealot and random.random()<0.3:
                unbelievers=[t for t in ao if t.alive and t.id!=o.id
                             and t.religion_id!=o.religion_id
                             and math.hypot(t.x-o.x,t.y-o.y)<=o.genome.vision]
                if unbelievers:
                    tgt=random.choice(unbelievers)
                    dx=tgt.x-o.x; dy=tgt.y-o.y; dd=math.hypot(dx,dy)+1e-8
                    best=8; bdot=-99
                    for ai in range(8):
                        dot=_ACT_DIRS[ai][0]*(dx/dd)+_ACT_DIRS[ai][1]*(dy/dd)
                        if dot>bdot: bdot=dot; best=ai
                    o._act=best
            o.apply_action(o._act,self.terrain)
            if not o.alive: d_starve+=1

        # ── Move warriors ──────────────────────────────────────────────────
        for w in aw:
            if not w.is_raiding: w.apply_action(w._act,self.terrain)
            if not w.alive: d_combat+=1

        # ── Eat food ──────────────────────────────────────────────────────
        food_set=list(self.food); taken=set()
        for o in ao:
            if not o.alive: continue
            for fi,f in enumerate(food_set):
                if fi in taken: continue
                er=(CFG.village_food_radius if any(v.contains(o.x,o.y) for v in self.villages)
                    else CFG.food_eat_radius)
                if math.hypot(f[0]-o.x,f[1]-o.y)<=er:
                    o.energy=min(o.energy+CFG.food_energy,CFG.max_energy); taken.add(fi); break
        for w in aw:
            if not w.alive: continue
            for fi,f in enumerate(food_set):
                if fi in taken: continue
                if math.hypot(f[0]-w.x,f[1]-w.y)<=CFG.food_eat_radius:
                    w.energy=min(w.energy+CFG.warrior_food_energy,CFG.warrior_energy_max)
                    taken.add(fi); break
        self.food=[f for i,f in enumerate(food_set) if i not in taken]

        # ── Bad zone damage ────────────────────────────────────────────────
        for o in ao:
            if o.alive and any(math.hypot(z[0]-o.x,z[1]-o.y)<=CFG.bad_zone_radius
                               for z in self.bad_zones):
                o.energy-=CFG.bad_zone_damage*(1.-o.genome.immune_strength*0.5)
                if o.energy<=0: o.alive=False; d_starve+=1

        # ── Disease ────────────────────────────────────────────────────────
        self._spread_disease(ao)

        # ── Natural disasters ──────────────────────────────────────────────
        self.disaster_mgr.tick(self, season_mult, self.step_count)

        # ── Org rewards ────────────────────────────────────────────────────
        threshold=self._org_threshold()
        for oi,o in enumerate(ao):
            if o._obs is None: continue
            hid=op._hx(o.id)
            in_bad=any(math.hypot(z[0]-o.x,z[1]-o.y)<=CFG.bad_zone_radius for z in self.bad_zones)
            in_v=any(v.contains(o.x,o.y) for v in self.villages)
            hv=[v for v in self.villages if v.id==o.home_village]
            v_golden=hv[0].golden_age if hv else False
            v_dark  =hv[0].dark_age   if hv else False
            v_at_war=len(self.diplomacy.at_war_with(o.home_village or 0))>0 if hv else False
            cur=op.intrinsic(o.x,o.y)
            rew=self._org_rew(o,o.energy>o._prev_energy,in_bad,in_v,v_golden,v_dark,cur,v_at_war)
            o.fitness+=rew; rew_o+=rew
            op.store(o._obs,o._act,o._logp,o._val,rew,not o.alive,hid,gs)

        # ── Warrior rewards ────────────────────────────────────────────────
        for wi,w in enumerate(aw):
            if w._obs is None: continue
            hid=wp._hx(w.id)
            v_at_war=len(self.diplomacy.at_war_with(w.home_village or 0))>0
            near_friendly=sum(1 for o in ao if o.alive and o.clan_id==w.clan_id
                              and math.hypot(o.x-w.x,o.y-w.y)<=CFG.warrior_protect_r)
            rew=self._war_rew(w,near_friendly,v_at_war,0)
            rew_w+=rew; wp.store(w._obs,w._act,w._logp,w._val,rew,not w.alive,hid,gs)

        # ── Reproduction ──────────────────────────────────────────────────
        births_o=self._try_mate(ao,threshold); births_w=self._reproduce_war(aw)

        # ── Clear dead ────────────────────────────────────────────────────
        for o in ao:
            if not o.alive: op.clear(o.id)
        for w in aw:
            if not w.alive: wp.clear(w.id)

        # ── Civilisation systems ───────────────────────────────────────────
        self._spawn_food(season_mult)
        self._try_form_village(self.organisms)
        self._handle_migration(self.organisms)

        # ── Prune dead ─────────────────────────────────────────────────────
        self.organisms=[o for o in self.organisms if o.alive]
        self.warriors =[w for w in self.warriors  if w.alive]

        # ── Stats ──────────────────────────────────────────────────────────
        ao2=self.organisms; aw2=self.warriors
        males=[o for o in ao2 if o.sex==Sex.MALE]; females=[o for o in ao2 if o.sex==Sex.FEMALE]
        in_v=sum(1 for o in ao2 if any(v.contains(o.x,o.y) for v in self.villages))
        preg=sum(1 for o in ao2 if o.pregnant); eld=sum(1 for o in ao2 if o.is_elder)
        sick=sum(1 for o in ao2 if o.sick); zealots=sum(1 for o in ao2 if o.is_zealot)
        r1f=r2f=r3f=0
        rlist=list(self.religions.values())
        if len(rlist)>0: r1f=rlist[0].follower_count
        if len(rlist)>1: r2f=rlist[1].follower_count
        if len(rlist)>2: r3f=rlist[2].follower_count
        self.stats["pop"].append(len(ao2)); self.stats["pop_m"].append(len(males))
        self.stats["pop_f"].append(len(females)); self.stats["warriors"].append(len(aw2))
        self.stats["food"].append(len(self.food)); self.stats["n_villages"].append(len(self.villages))
        self.stats["in_village"].append(in_v); self.stats["pregnancies"].append(preg)
        self.stats["elders"].append(eld); self.stats["sick"].append(sick); self.stats["zealots"].append(zealots)
        self.stats["births"].append(births_o); self.stats["births_war"].append(births_w)
        self.stats["deaths_starve"].append(d_starve); self.stats["deaths_combat"].append(d_combat)
        self.stats["n_wars"].append(self.diplomacy.total_wars())
        self.stats["n_alliances"].append(sum(1 for s in self.diplomacy._state.values() if s==WarState.ALLIANCE))
        self.stats["total_raids"].append(self._total_raids); self.stats["resources_stolen"].append(self._total_stolen)
        self.stats["religion_followers"].append(r1f); self.stats["religion2_followers"].append(r2f)
        self.stats["religion3_followers"].append(r3f)
        believers=[o for o in ao2 if o.religion_id is not None]
        self.stats["avg_spd_m"].append(np.mean([o.genome.speed for o in males]) if males else 0)
        self.stats["avg_spd_f"].append(np.mean([o.genome.speed for o in females]) if females else 0)
        self.stats["avg_gen"].append(np.mean([o.generation for o in ao2]) if ao2 else 0)
        self.stats["avg_faith"].append(np.mean([o.faith_level for o in believers]) if believers else 0.0)
        self.stats["golden_ages"].append(sum(1 for v in self.villages if v.golden_age))
        self.stats["dark_ages"].append(sum(1 for v in self.villages if v.dark_age))
        self.stats["season"].append(season_mult)
        self.stats["n_religions_active"].append(len(self.religions))
        self.stats["war_declarations"].append(self._war_decls)
        self.stats["trade_trips"].append(self._trade_trips)
        self.stats["schisms"].append(self._schisms)
        self.stats["n_disasters"].append(self.disaster_mgr.n_active())
        self.stats["disaster_history"].append(len(self.disaster_mgr.history))
        return {"pop":len(ao2),"male":len(males),"female":len(females),
                "war":len(aw2),"food":len(self.food),"n_v":len(self.villages),
                "in_v":in_v,"preg":preg,"eld":eld,"zealots":zealots,
                "births":births_o,"d_starve":d_starve,"d_combat":d_combat,
                "n_wars":self.diplomacy.total_wars(),"r1f":r1f,"r2f":r2f,"r3f":r3f,
                "rew_o":rew_o,"rew_w":rew_w}

# ─────────────────────────────────────────────────────────────────────────────
#  RENDERER  (16-panel civilization dashboard)
# ─────────────────────────────────────────────────────────────────────────────
op_ref=[]
VILLAGE_COLORS=["#30d158","#0a84ff","#ff9f0a","#bf5af2","#ff375f","#64d2ff",
                "#ffd60a","#ff6b35","#30d158","#ff453a","#8efaae","#c084fc",
                "#fbbf24","#34d399"]

class Renderer:
    def __init__(self):
        self.fig=plt.figure(figsize=(34,22),facecolor="#0a0c10",layout="constrained")
        gs_layout=self.fig.add_gridspec(4,4,wspace=0.40,hspace=0.52)
        self.axes=[self.fig.add_subplot(gs_layout[r,c]) for r in range(4) for c in range(4)]
        os.makedirs(CFG.plot_dir,exist_ok=True) if CFG.save_plot else None
        self._frame=0
        self._village_color_map: Dict[int,str]={}

    def _sty(self,ax,title="",xl="",yl=""):
        ax.set_facecolor("#13161e"); ax.tick_params(colors="#7a8394",labelsize=6.5)
        ax.spines[:].set_color("#252a35")
        ax.set_title(title,color="#e8edf5",fontsize=7.5,pad=4,fontweight="bold")
        ax.set_xlabel(xl,color="#7a8394",fontsize=6.5); ax.set_ylabel(yl,color="#7a8394",fontsize=6.5)

    def _get_vcol(self, vid):
        if vid not in self._village_color_map:
            self._village_color_map[vid]=VILLAGE_COLORS[len(self._village_color_map)%len(VILLAGE_COLORS)]
        return self._village_color_map[vid]

    def render(self, world: World, step: int):
        for ax in self.axes: ax.cla()
        ao=world.organisms; aw=world.warriors
        st=world.stats; xs=list(range(len(st["pop"])))
        sm=lambda s,n: (np.convolve(s,np.ones(n)/n,mode="same").tolist() if len(s)>=n else s)
        males=[o for o in ao if o.sex==Sex.MALE]; females=[o for o in ao if o.sex==Sex.FEMALE]
        preg=[o for o in ao if o.pregnant]; elders=[o for o in ao if o.is_elder]
        sick=[o for o in ao if o.sick]; zealots=[o for o in ao if o.is_zealot]
        chiefs=[w for w in aw if w.is_war_chief]; raiders=[w for w in aw if w.is_raiding]
        rlist=list(world.religions.values())

        # ── [0] World Map ─────────────────────────────────────────────────
        ax=self.axes[0]; self._sty(ax,f"Civilization World [step {step}]","X","Y")
        ax.set_xlim(0,CFG.world_w); ax.set_ylim(0,CFG.world_h)
        # Terrain
        tm=world.terrain
        for ty in range(tm.H):
            for tx in range(tm.W):
                t=tm.grid[ty,tx]; col=TERRAIN_COLORS[t]
                ax.add_patch(plt.Rectangle((tx/tm.W*CFG.world_w,ty/tm.H*CFG.world_h),
                             CFG.world_w/tm.W,CFG.world_h/tm.H,color=col,alpha=0.45,zorder=0))
        # Disaster overlays
        for d in world.disaster_mgr.active:
            dc=DISASTER_COLORS.get(d["type"],"#ff0000")
            dr={"DROUGHT":CFG.drought_food_kill_r,"FLOOD":CFG.flood_radius,
                "FIRE":CFG.fire_radius,"EARTHQUAKE":CFG.earthquake_radius,
                "FAMINE":60.0}.get(d["type"].name,30.0)*d["intensity"]
            ax.add_patch(plt.Circle((d["x"],d["y"]),dr,color=dc,alpha=0.22,zorder=1))
            ax.add_patch(plt.Circle((d["x"],d["y"]),dr,fill=False,edgecolor=dc,lw=1.5,alpha=0.7,zorder=2))
            ax.text(d["x"],d["y"],d["type"].name[:3],color="#ffffff",fontsize=5,
                    ha="center",va="center",zorder=3,fontweight="bold")
        # Territory
        for v in world.villages:
            col=self._get_vcol(v.id)
            ax.add_patch(plt.Circle((v.x,v.y),CFG.territory_radius,color=col,alpha=0.06,zorder=1))
        # War lines + Trade route lines
        for (a,b),st_r in world.diplomacy._state.items():
            va=[v for v in world.villages if v.id==a]; vb=[v for v in world.villages if v.id==b]
            if not va or not vb: continue
            if st_r==WarState.AT_WAR:
                ax.plot([va[0].x,vb[0].x],[va[0].y,vb[0].y],'r-',lw=1.5,alpha=0.6,zorder=3)
                ax.text((va[0].x+vb[0].x)/2,(va[0].y+vb[0].y)/2,"⚔",
                        color="#ff453a",fontsize=7,ha="center",va="center",zorder=4)
            elif st_r==WarState.ALLIANCE:
                ax.plot([va[0].x,vb[0].x],[va[0].y,vb[0].y],color="#ffd60a",lw=1.2,
                        alpha=0.5,linestyle="--",zorder=3)
        # Active trade routes this step
        for (a,b) in world.diplomacy.active_trade_this_step:
            va=[v for v in world.villages if v.id==a]; vb=[v for v in world.villages if v.id==b]
            if not va or not vb: continue
            vol=world.diplomacy.trade_routes.get((a,b),1.)
            lw=min(0.8+vol/60.,3.0)
            ax.plot([va[0].x,vb[0].x],[va[0].y,vb[0].y],
                    color="#00ffcc",lw=lw,alpha=0.55,linestyle="-",zorder=3)
            mx=(va[0].x+vb[0].x)/2; my=(va[0].y+vb[0].y)/2
            ax.text(mx,my,"$",color="#00ffcc",fontsize=5.5,ha="center",va="center",zorder=4)
        # Villages
        for v in world.villages:
            col=self._get_vcol(v.id)
            gold=v.golden_age; dark=v.dark_age
            ring_col="#ffd60a" if gold else ("#ff3b30" if dark else col)
            ax.add_patch(plt.Circle((v.x,v.y),CFG.village_radius,color=col,alpha=0.18,zorder=2))
            ax.add_patch(plt.Circle((v.x,v.y),CFG.village_radius,fill=False,edgecolor=ring_col,lw=2.2,alpha=0.9,zorder=3))
            # Religion symbol
            rel_sym=""
            if v.dominant_religion and v.dominant_religion in world.religions:
                rel_sym=world.religions[v.dominant_religion].symbol
            gov_abbr=v.governance.name[:1]
            lbl=f"V{v.id}{rel_sym}\n{gov_abbr}{'🌟' if gold else ('💀' if dark else '')}"
            ax.text(v.x,v.y,lbl,color=col,fontsize=5,ha="center",va="center",zorder=4,fontweight="bold")
        # Bad zones
        for z in world.bad_zones:
            ax.add_patch(plt.Circle(z,CFG.bad_zone_radius,color="#ff3b30",alpha=0.14,zorder=2))
            ax.add_patch(plt.Circle(z,CFG.bad_zone_radius,fill=False,edgecolor="#ff3b30",lw=1,alpha=0.5,zorder=3))
        # Food
        if world.food:
            fx,fy=zip(*world.food); ax.scatter(fx,fy,s=2.5,c="#30d158",alpha=0.35,zorder=4,marker=".",linewidths=0)
        # Organisms colored by clan/village
        for o in ao:
            col=self._get_vcol(o.home_village or 0) if o.home_village else "#555"
            sz=4+8*(o.energy/CFG.max_energy)
            marker="*" if o.is_zealot else ("^" if o.sex==Sex.MALE else "o")
            ax.scatter(o.x,o.y,s=sz,c=col,alpha=0.75,zorder=5,marker=marker,linewidths=0)
        # Warriors
        if aw:
            for w in aw:
                col=self._get_vcol(w.home_village or 0) if w.home_village else "#888"
                ec="#ff0000" if w.is_raiding else "#ffffff"
                ax.scatter(w.x,w.y,s=35,c=col,marker="D",zorder=6,edgecolors=ec,lw=0.8,alpha=0.9)
        leg=[mpatches.Patch(color="#30d158",label=f"Orgs({len(ao)})"),
             mpatches.Patch(color="#0a84ff",label=f"Warriors({len(aw)})"),
             mpatches.Patch(color="#ffd60a",label=f"Zealots({len(zealots)})"),
             mpatches.Patch(color="#ff453a",label="⚔ At War"),
             mpatches.Patch(color="#ffd60a",label="-- Alliance"),
             mpatches.Patch(color="#00ffcc",label="$ Trade route"),
             mpatches.Patch(color="#ff453a",label=f"Raiders({len(raiders)})")]
        ax.legend(handles=leg,loc="upper right",fontsize=4.5,
                  facecolor="#1a1d25",labelcolor="#e8edf5",edgecolor="#252a35",ncol=2)

        # ── [1] Population ─────────────────────────────────────────────────
        ax=self.axes[1]; self._sty(ax,"Population","Step","Count")
        ax.plot(xs,st["pop"],color="#ffffff",lw=1.3,label="Total")
        ax.plot(xs,st["pop_m"],color="#0a84ff",lw=1.0,label="♂")
        ax.plot(xs,st["pop_f"],color="#ff375f",lw=1.0,label="♀")
        ax.plot(xs,st["warriors"],color="#ffd60a",lw=1.0,label="Warriors")
        ax.legend(fontsize=5,facecolor="#1a1d25",labelcolor="#e8edf5",edgecolor="#252a35")

        # ── [2] Religion Dynamics ─────────────────────────────────────────
        ax=self.axes[2]; self._sty(ax,"Religion Dynamics","Step","Followers")
        rc=["#FFD700","#8B0000","#228B22","#1E90FF"]
        rkeys=["religion_followers","religion2_followers","religion3_followers"]
        for ri,rel in enumerate(rlist[:3]):
            if rkeys[ri] in st and st[rkeys[ri]]:
                ax.plot(xs,st[rkeys[ri]],color=rc[ri],lw=1.4,label=f"{rel.symbol} {rel.name[:12]}")
        ax.plot(xs,sm(st["zealots"],10),color="#ff6b35",lw=1.0,linestyle="--",label="Zealots")
        ax.plot(xs,sm(st["avg_faith"],10),color="#bf5af2",lw=0.9,linestyle=":",label="Avg Faith")
        ax2=ax.twinx(); ax2.tick_params(colors="#7a8394",labelsize=6.5); ax2.spines[:].set_color("#252a35")
        ax2.plot(xs,sm(st["n_disasters"],5),color="#ff4500",lw=1.0,linestyle="-.",label="Disasters")
        ax2.set_ylabel("Active Disasters",color="#ff4500",fontsize=6)
        h1,l1=ax.get_legend_handles_labels(); h2,l2=ax2.get_legend_handles_labels()
        ax.legend(h1+h2,l1+l2,fontsize=4.2,facecolor="#1a1d25",labelcolor="#e8edf5",edgecolor="#252a35",ncol=2)

        # ── [3] War & Diplomacy ────────────────────────────────────────────
        ax=self.axes[3]; self._sty(ax,"War & Diplomacy","Step","Count")
        ax.plot(xs,sm(st["n_wars"],5),         color="#ff453a",lw=1.4,label="Active Wars")
        ax.plot(xs,sm(st["n_alliances"],5),    color="#ffd60a",lw=1.2,label="Alliances")
        ax2=ax.twinx(); ax2.tick_params(colors="#7a8394",labelsize=6.5); ax2.spines[:].set_color("#252a35")
        ax2.plot(xs,sm(st["total_raids"],10),  color="#ff6b35",lw=1.0,linestyle="--",label="Raids")
        ax.set_ylabel("Diplo Count",color="#ff453a",fontsize=6.5)
        ax2.set_ylabel("Raids (cumul)",color="#ff6b35",fontsize=6.5)
        h1,l1=ax.get_legend_handles_labels(); h2,l2=ax2.get_legend_handles_labels()
        ax.legend(h1+h2,l1+l2,fontsize=5,facecolor="#1a1d25",labelcolor="#e8edf5",edgecolor="#252a35")

        # ── [4] Village Map (resource view) ───────────────────────────────
        ax=self.axes[4]; self._sty(ax,"Village Wealth & State","Village","Wealth")
        if world.villages:
            vids=[v.id for v in world.villages]
            wealth=[v.total_wealth() for v in world.villages]
            cols=[("#ffd60a" if v.golden_age else ("#ff3b30" if v.dark_age else
                   ("#ff6b35" if world.diplomacy.at_war_with(v.id) else
                    self._get_vcol(v.id)))) for v in world.villages]
            bars=ax.bar(range(len(vids)),wealth,color=cols,edgecolor="#252a35",lw=0.5)
            ax.set_xticks(range(len(vids)))
            ax.set_xticklabels([f"V{v.id}" for v in world.villages],color="#7a8394",fontsize=6)
            # Religion symbol on bar
            for bi,v in enumerate(world.villages):
                if v.dominant_religion and v.dominant_religion in world.religions:
                    sym=world.religions[v.dominant_religion].symbol
                    ax.text(bi,wealth[bi]+0.5,sym,ha="center",va="bottom",fontsize=8,color="#e8edf5")

        # ── [5] Deaths ─────────────────────────────────────────────────────
        ax=self.axes[5]; self._sty(ax,"Deaths & Health","Step","Count")
        ax.plot(xs,sm(st["deaths_starve"],10), color="#ff9f0a",lw=1.1,label="Starve")
        ax.plot(xs,sm(st["deaths_combat"],10), color="#ff453a",lw=1.1,label="Combat")
        ax.plot(xs,sm(st["sick"],10),          color="#8B4513",lw=1.0,label="Sick")
        ax.legend(fontsize=5,facecolor="#1a1d25",labelcolor="#e8edf5",edgecolor="#252a35")

        # ── [6] Villages over time ─────────────────────────────────────────
        ax=self.axes[6]; self._sty(ax,"Villages & Golden/Dark Ages","Step","Count")
        ax.plot(xs,st["n_villages"],   color="#30d158",lw=1.4,label="Villages")
        ax.plot(xs,sm(st["in_village"],5),color="#64d2ff",lw=1.0,label="In Village")
        ax.plot(xs,sm(st["golden_ages"],5),color="#ffd60a",lw=1.1,label="Golden Age")
        ax.plot(xs,sm(st["dark_ages"],5),color="#8B0000",lw=1.1,label="Dark Age")
        ax.legend(fontsize=5,facecolor="#1a1d25",labelcolor="#e8edf5",edgecolor="#252a35")

        # ── [7] Reproduction & Society ────────────────────────────────────
        ax=self.axes[7]; self._sty(ax,"Society","Step","Count")
        ax.plot(xs,sm(st["pregnancies"],10),color="#ffd60a",lw=1.2,label="Pregnant")
        ax.plot(xs,sm(st["births"],10),     color="#30d158",lw=1.2,label="Births")
        ax.plot(xs,sm(st["elders"],5),      color="#bf5af2",lw=0.9,label="Elders")
        ax.plot(xs,sm(st["trade_trips"],10),color="#64d2ff",lw=1.0,label="Trades")
        ax.legend(fontsize=5,facecolor="#1a1d25",labelcolor="#e8edf5",edgecolor="#252a35")

        # ── [8] Speed Gene ─────────────────────────────────────────────────
        ax=self.axes[8]; self._sty(ax,"Speed Gene ♂ vs ♀","Step","Speed")
        ax.plot(xs,st["avg_spd_m"],color="#0a84ff",lw=1.2,label="♂")
        ax.plot(xs,st["avg_spd_f"],color="#ff375f",lw=1.1,label="♀")
        ax.legend(fontsize=5,facecolor="#1a1d25",labelcolor="#e8edf5",edgecolor="#252a35")

        # ── [9] Generation ─────────────────────────────────────────────────
        ax=self.axes[9]; self._sty(ax,"Avg Generation","Step","Gen")
        ax.plot(xs,st["avg_gen"],color="#ffd60a",lw=1.4)
        if xs: ax.fill_between(xs,st["avg_gen"],alpha=0.13,color="#ffd60a")

        # ── [10] Food & Season ─────────────────────────────────────────────
        ax=self.axes[10]; self._sty(ax,"Food & Season","Step","Food")
        ax.plot(xs,st["food"],color="#30d158",lw=1.2,label="Food")
        ax2=ax.twinx(); ax2.tick_params(colors="#7a8394",labelsize=6.5); ax2.spines[:].set_color("#252a35")
        ax2.plot(xs,st["season"],color="#ffd60a",lw=1.0,linestyle="--",label="Season")
        ax.set_ylabel("Food",color="#30d158",fontsize=6.5); ax2.set_ylabel("Season",color="#ffd60a",fontsize=6.5)
        h1,l1=ax.get_legend_handles_labels(); h2,l2=ax2.get_legend_handles_labels()
        ax.legend(h1+h2,l1+l2,fontsize=5,facecolor="#1a1d25",labelcolor="#e8edf5",edgecolor="#252a35")

        # ── [11] MAPPO Reward ─────────────────────────────────────────────
        ax=self.axes[11]; self._sty(ax,"MAPPO Reward / Step","Step","Reward")
        ax.plot(xs,sm(st["reward_org"] if "reward_org" in st else [0]*len(xs),25),
                color="#ff9f0a",lw=1.2,label="Org")
        ax.plot(xs,sm(st["reward_war"] if "reward_war" in st else [0]*len(xs),25),
                color="#30d158",lw=1.1,label="Warrior")
        ax.axhline(0,color="#252a35",lw=0.8,linestyle="--")
        ax.legend(fontsize=5,facecolor="#1a1d25",labelcolor="#e8edf5",edgecolor="#252a35")

        # Patch missing reward stats
        if "reward_org" not in world.stats:
            world.stats["reward_org"]=[rew_o:=0]*len(xs)
            world.stats["reward_war"]=[0]*len(xs)

        # ── [12] Curiosity Map ────────────────────────────────────────────
        ax=self.axes[12]; self._sty(ax,"Exploration Heatmap","X","Y")
        if op_ref:
            vm=op_ref[0].visit_grid
            ax.imshow(vm,aspect="auto",cmap="inferno",origin="lower",
                      extent=[0,CFG.world_w,0,CFG.world_h],alpha=0.92)
        ax.set_xlim(0,CFG.world_w); ax.set_ylim(0,CFG.world_h)
        for v in world.villages:
            ax.plot(v.x,v.y,"w.",markersize=5,zorder=5)

        # ── [13] War History Timeline ─────────────────────────────────────
        ax=self.axes[13]; self._sty(ax,"War Declarations Timeline","Step","Village pair")
        for wi,wh in enumerate(world.diplomacy.war_history[-20:]):
            col={"RESOURCES":"#ff9f0a","RELIGION":"#8B0000",
                 "TERRITORY":"#1E90FF","RETALIATION":"#ff453a"}.get(wh["cause"],"#fff")
            ax.scatter(wh["step"],f"V{wh['a']}↔V{wh['b']}",c=col,s=60,marker="|",zorder=3)
            ax.text(wh["step"]+20,f"V{wh['a']}↔V{wh['b']}",wh["cause"][:3],
                    color=col,fontsize=5,va="center")
        self._sty(ax,"War History","Step","")
        # Legend for causes
        cause_leg=[mpatches.Patch(color="#ff9f0a",label="Resources"),
                   mpatches.Patch(color="#8B0000",label="Religion"),
                   mpatches.Patch(color="#1E90FF",label="Territory"),
                   mpatches.Patch(color="#ff453a",label="Retaliation")]
        ax.legend(handles=cause_leg,fontsize=4.5,facecolor="#1a1d25",
                  labelcolor="#e8edf5",edgecolor="#252a35")

        # ── [14] Religion Detail ──────────────────────────────────────────
        ax=self.axes[14]; self._sty(ax,"Religion Detail","","")
        ax.axis("off")
        lines=["── RELIGIONS ──"]
        for ri,rel in enumerate(rlist):
            lines.append(f"{rel.symbol} {rel.name}")
            lines.append(f"  Followers: {rel.follower_count} | Founded: {rel.founded_step}")
            tenets=[]
            if rel.tenet_pacifist:     tenets.append("Pacifist")
            if rel.tenet_expansionist: tenets.append("Expans.")
            if rel.tenet_trade:        tenets.append("Trade")
            if rel.tenet_isolation:    tenets.append("Isolat.")
            lines.append(f"  Tenets: {', '.join(tenets) or 'None'}")
            lines.append(f"  Wars caused: {rel.wars_caused} | Conv: {rel.conversions}")
            lines.append(f"  Schisms: {world.stats['schisms'][-1] if world.stats['schisms'] else 0}")
            lines.append("")
        lines.append("── WAR STATS ──")
        lines.append(f"Total declarations: {world._war_decls}")
        lines.append(f"Total raids: {world._total_raids}")
        lines.append(f"Resources stolen: {world._total_stolen:.0f}")
        lines.append(f"Allied trades: {world._trade_trips}")
        lines.append("── DISASTERS ──")
        lines.append(f"Active: {world.disaster_mgr.active_names()}")
        lines.append(f"Total events: {len(world.disaster_mgr.history)}")
        for i,l in enumerate(lines[:20]):
            col="#e8edf5"
            if "─" in l: col="#7a8394"
            if rlist and len(rlist)>0 and l.startswith(rlist[0].symbol): col=rc[0]
            if rlist and len(rlist)>1 and l.startswith(rlist[1].symbol): col=rc[1]
            if rlist and len(rlist)>2 and l.startswith(rlist[2].symbol): col=rc[2]
            if "DISASTER" in l or "Active:" in l or "Total events" in l: col="#ff4500"
            ax.text(0.04,0.97-i*0.049,l,transform=ax.transAxes,
                    color=col,fontsize=6.8,va="top",family="monospace")

        # ── [15] Civilization Status ──────────────────────────────────────
        ax=self.axes[15]; self._sty(ax,"Civilization Status","","")
        ax.axis("off")
        govs={g.name:sum(1 for v in world.villages if v.governance==g) for g in Governance}
        dominant_r=(rlist[0].name if rlist else "None")
        lines2=[
            f"Step: {step}",
            f"Pop: {len(ao)} (♂{len(males)} ♀{len(females)})",
            f"Births: {sum(st['births'])} | Combat: {sum(st['deaths_combat'])}",
            f"Avg gen: {st['avg_gen'][-1]:.2f}" if st["avg_gen"] else "",
            f"Villages: {len(world.villages)} | Golden:{sum(1 for v in world.villages if v.golden_age)}",
            f"Dark age: {sum(1 for v in world.villages if v.dark_age)}",
            f"Active wars: {world.diplomacy.total_wars()}",
            f"Alliances: {sum(1 for s in world.diplomacy._state.values() if s==WarState.ALLIANCE)}",
            f"War decls: {world._war_decls} | Raids: {world._total_raids}",
            f"Stolen: {world._total_stolen:.0f} | Schisms: {world._schisms}",
            f"Zealots: {len(zealots)}",
            f"Governance: D{govs.get('DEMOCRACY',0)} A{govs.get('AUTOCRACY',0)}"
            f" T{govs.get('THEOCRACY',0)} An{govs.get('ANARCHY',0)}",
            f"Dominant faith: {dominant_r[:18]}",
            f"── Disasters ──",
            f"Active: {world.disaster_mgr.n_active()}  ({world.disaster_mgr.active_names()[:22]})",
            f"Total events: {len(world.disaster_mgr.history)}",
            f"Famine: {'YES' if world.disaster_mgr._famine_active else 'no'}",
        ]
        for i,l in enumerate(lines2):
            col="#ff4500" if ("Disaster" in l or "Active:" in l or "Famine" in l
                              or "Total events" in l) else "#e8edf5"
            if "──" in l: col="#7a8394"
            ax.text(0.04,0.97-i*0.057,l,transform=ax.transAxes,
                    color=col,fontsize=7.0,va="top",family="monospace")

        # Add reward stats if missing
        if "reward_org" not in world.stats:
            world.stats["reward_org"]=[0]*len(xs)
            world.stats["reward_war"]=[0]*len(xs)

        self.fig.suptitle(
            f"Civilization v7  ·  No External Enemies  ·  Step {step}  |  "
            f"Orgs {len(ao)} (♂{len(males)} ♀{len(females)})  |  "
            f"Warriors {len(aw)}  |  Villages {len(world.villages)}  |  "
            f"Wars {world.diplomacy.total_wars()}  |  "
            f"Religions {len(world.religions)}  |  Zealots {len(zealots)}  |  "
            f"Disasters {world.disaster_mgr.n_active()}  |  "
            f"Food {len(world.food)}  |  Season {world.season_phase:.2f}",
            color="#e8edf5",fontsize=9,fontweight="bold",y=1.002)
        # layout managed by constrained_layout set at figure creation
        if CFG.save_plot:
            self.fig.savefig(os.path.join(CFG.plot_dir,f"frame_{self._frame:05d}.png"),
                             dpi=92,bbox_inches="tight",facecolor=self.fig.get_facecolor())
            self._frame+=1
        try: plt.pause(0.001)
        except: pass

# ─────────────────────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────────────────────
def run():
    print("\n"+"═"*76)
    print("  CIVILIZATION LIVING SYSTEM v7")
    print("  No External Enemies · Village Wars · Religion · Resource Conflict")
    print("  Raid System · Diplomacy · Golden Age / Dark Age · Religious Schism")
    print("═"*76)
    print(f"  World {CFG.world_w}×{CFG.world_h}  |  RL: MAPPO+GRU{CFG.gru_hidden}")
    print(f"  Religions: {CFG.n_religions} max (DROUGHT·FLOOD·FIRE·EARTHQUAKE·FAMINE disasters active)")
    print(f"  War triggers: resource ratio>{CFG.war_declare_resource_thresh:.1f}"
          f"  territory<{CFG.war_declare_territory_dist:.0f}u"
          f"  religion tension>{CFG.war_declare_religion_thresh:.1f}")
    print("═"*76+"\n")

    world=World()
    op=MAPPOPolicy(CFG.org_obs_dim,    CFG.global_state_dim,name="org")
    wp=MAPPOPolicy(CFG.warrior_obs_dim,CFG.global_state_dim,name="warrior")
    op_ref.append(op)
    rend=Renderer(); t0=time.time(); mo={}; mw={}; respawns=0

    # Track rewards for stats
    world.stats["reward_org"]=[]; world.stats["reward_war"]=[]

    for step in range(1,CFG.max_steps+1):
        info=world.step(op,wp)
        world.stats["reward_org"].append(info["rew_o"])
        world.stats["reward_war"].append(info["rew_w"])

        if len(op.buf["obs"])>=CFG.rollout_len: mo=op.update(score=info["rew_o"])
        if len(wp.buf["obs"])>=CFG.rollout_len: mw=wp.update(score=info["rew_w"])

        if step%50==0:
            el=time.time()-t0
            rlist=list(world.religions.values())
            r1n=rlist[0].name[:8] if rlist else "—"
            r2n=rlist[1].name[:8] if len(rlist)>1 else "—"
            r3n=rlist[2].name[:8] if len(rlist)>2 else "—"
            print(f"Step {step:5d} | Org {info['pop']:4d}(♂{info['male']:3d}♀{info['female']:3d}"
                  f"E{info['eld']:2d}Z{info['zealots']:2d}) | War {info['war']:3d} | "
                  f"V{info['n_v']} Wars:{info['n_wars']} | "
                  f"F{info['food']:4d} | Dis:{world.disaster_mgr.n_active()} | "
                  f"R1:{r1n}({info['r1f']}) R2:{r2n}({info['r2f']}) R3:{r3n}({info['r3f']}) | "
                  f"B{info['births']:2d} Dc{info['d_combat']:2d} | "
                  f"OPL {mo.get('pl',0):+.3f} | {el:.0f}s")

        if step%CFG.render_every==0: rend.render(world,step)

        if not world.organisms:
            respawns+=1
            print(f"\n⚠  Extinction at step {step}. Respawn #{respawns}")
            if respawns<=8:
                for v in world.villages:
                    for k in range(18):
                        ang=random.uniform(0,2*math.pi); r=random.uniform(0,CFG.village_radius*0.7)
                        sex=Sex.MALE if k%2==0 else Sex.FEMALE
                        org=Organism(float(np.clip(v.x+r*math.cos(ang),0,CFG.world_w)),
                                     float(np.clip(v.y+r*math.sin(ang),0,CFG.world_h)),
                                     sex=sex,energy=CFG.init_energy*0.75,clan_id=v.clan_id)
                        org.home_village=v.id; world.organisms.append(org)
            else: print("   Too many extinctions."); break

    rend.render(world,world.step_count)
    print("\n"+"═"*76)
    ao=world.organisms; aw=world.warriors
    rlist=list(world.religions.values())
    males=[o for o in ao if o.sex==Sex.MALE]; females=[o for o in ao if o.sex==Sex.FEMALE]
    print(f"  Final: Orgs {len(ao)}(♂{len(males)}♀{len(females)}) | War {len(aw)}")
    print(f"  Villages: {len(world.villages)} | Active wars: {world.diplomacy.total_wars()}")
    if ao:
        gens=[o.generation for o in ao]
        print(f"  Gen avg {np.mean(gens):.2f} max {max(gens)}")
    for rel in rlist:
        tenets=[k for k,v in [("Pacifist",rel.tenet_pacifist),("Expansionist",rel.tenet_expansionist),
                               ("Trade",rel.tenet_trade),("Isolation",rel.tenet_isolation)] if v]
        print(f"  {rel.symbol} {rel.name}: {rel.follower_count} followers | "
              f"wars caused: {rel.wars_caused} | conversions: {rel.conversions} | "
              f"tenets: {', '.join(tenets)}")
    print(f"  Total war declarations: {world._war_decls}")
    print(f"  Total raids: {world._total_raids} | stolen: {world._total_stolen:.0f}")
    print(f"  Total schisms: {world._schisms}")
    govs={g.name:sum(1 for v in world.villages if v.governance==g) for g in Governance}
    print(f"  Governance: {govs}")
    print("═"*76+"\n")
    fp=os.path.join(CFG.plot_dir,"final_summary_v7.png")
    rend.fig.savefig(fp,dpi=118,bbox_inches="tight",facecolor=rend.fig.get_facecolor())
    print(f"[Done] → {fp}")
    return world,op,wp,rend

if __name__=="__main__":
    # CFG.max_steps=2000; CFG.render_every=200  # quick test
    world,op,wp,rend=run()

[Device] cuda  |  Tesla T4  |  15.6 GB VRAM

════════════════════════════════════════════════════════════════════════════
  CIVILIZATION LIVING SYSTEM v7
  No External Enemies · Village Wars · Religion · Resource Conflict
  Raid System · Diplomacy · Golden Age / Dark Age · Religious Schism
════════════════════════════════════════════════════════════════════════════
  World 260×170  |  RL: MAPPO+GRU128
  Religions: 3 max (DROUGHT·FLOOD·FIRE·EARTHQUAKE·FAMINE disasters active)
  War triggers: resource ratio>0.1  territory<70u  religion tension>0.3
════════════════════════════════════════════════════════════════════════════

Step    50 | Org  183(♂ 98♀ 85E 0Z 0) | War  57 | V2 Wars:0 | F 208 | Dis:0 | R1:—(0) R2:—(0) R3:—(0) | B 0 Dc 0 | OPL +0.000 | 22s
Step   100 | Org  226(♂121♀105E 0Z 0) | War  95 | V2 Wars:0 | F  94 | Dis:0 | R1:—(0) R2:—(0) R3:—(0) | B 0 Dc 0 | OPL +0.036 | 42s
  [Village+] (108,89) clan=2 gov=AUTOCRACY n=18 total=3
Step   150 | Org  240(♂124♀116E 0Z 0) | War 110 | 

/tmp/ipykernel_1866/3698779702.py:2155: UserWarning: Glyph 127775 (\N{GLOWING STAR}) missing from font(s) DejaVu Sans.
  self.fig.savefig(os.path.join(CFG.plot_dir,f"frame_{self._frame:05d}.png"),
/tmp/ipykernel_1866/3698779702.py:2158: UserWarning: Glyph 127775 (\N{GLOWING STAR}) missing from font(s) DejaVu Sans.
  try: plt.pause(0.001)


Step   250 | Org  240(♂125♀115E 0Z 0) | War 110 | V3 Wars:0 | F  31 | Dis:0 | R1:—(0) R2:—(0) R3:—(0) | B 0 Dc 0 | OPL +0.052 | 88s
  [Religion] 'The Radiant Path' (☀) founded at step 300 by org 329 charisma=0.93
Step   300 | Org  237(♂121♀116E 0Z 1) | War 110 | V3 Wars:0 | F  20 | Dis:0 | R1:The Radi(1) R2:—(0) R3:—(0) | B 1 Dc 0 | OPL -0.005 | 97s
  [Religion] 'The Iron Covenant' (⚔) founded at step 312 by org 19 charisma=1.00
  [Religion] 'The Green Circle' (🌿) founded at step 318 by org 94 charisma=0.78
Step   350 | Org  244(♂125♀119E 0Z 3) | War 110 | V3 Wars:0 | F  18 | Dis:0 | R1:The Radi(13) R2:The Iron(1) R3:The Gree(2) | B 0 Dc 0 | OPL +0.002 | 115s
Step   400 | Org  237(♂124♀113E 0Z 2) | War 110 | V3 Wars:0 | F  13 | Dis:0 | R1:The Radi(15) R2:The Iron(1) R3:The Gree(2) | B 0 Dc 0 | OPL +0.022 | 125s


/tmp/ipykernel_1866/3698779702.py:2155: UserWarning: Glyph 127807 (\N{HERB}) missing from font(s) DejaVu Sans.
  self.fig.savefig(os.path.join(CFG.plot_dir,f"frame_{self._frame:05d}.png"),
/tmp/ipykernel_1866/3698779702.py:2155: UserWarning: Glyph 127807 (\N{HERB}) missing from font(s) DejaVu Sans Mono.
  self.fig.savefig(os.path.join(CFG.plot_dir,f"frame_{self._frame:05d}.png"),
/tmp/ipykernel_1866/3698779702.py:2158: UserWarning: Glyph 127807 (\N{HERB}) missing from font(s) DejaVu Sans.
  try: plt.pause(0.001)
/tmp/ipykernel_1866/3698779702.py:2158: UserWarning: Glyph 127807 (\N{HERB}) missing from font(s) DejaVu Sans Mono.
  try: plt.pause(0.001)


  [Village+] (243,39) clan=1 gov=ANARCHY n=16 total=4
Step   450 | Org  237(♂121♀116E134Z 2) | War 110 | V4 Wars:0 | F  21 | Dis:0 | R1:The Radi(16) R2:The Iron(1) R3:The Gree(4) | B 0 Dc 0 | OPL -0.003 | 143s
Step   500 | Org  239(♂122♀117E137Z 2) | War 110 | V4 Wars:0 | F  26 | Dis:0 | R1:The Radi(12) R2:The Iron(2) R3:The Gree(5) | B 0 Dc 0 | OPL +0.016 | 153s


...

/tmp/ipykernel_1866/3698779702.py:2155: UserWarning: Glyph 128128 (\N{SKULL}) missing from font(s) DejaVu Sans.
  self.fig.savefig(os.path.join(CFG.plot_dir,f"frame_{self._frame:05d}.png"),
/tmp/ipykernel_1866/3698779702.py:2155: UserWarning: Glyph 127775 (\N{GLOWING STAR}) missing from font(s) DejaVu Sans.
  self.fig.savefig(os.path.join(CFG.plot_dir,f"frame_{self._frame:05d}.png"),
/tmp/ipykernel_1866/3698779702.py:2155: UserWarning: Glyph 127807 (\N{HERB}) missing from font(s) DejaVu Sans.
  self.fig.savefig(os.path.join(CFG.plot_dir,f"frame_{self._frame:05d}.png"),
/tmp/ipykernel_1866/3698779702.py:2155: UserWarning: Glyph 127807 (\N{HERB}) missing from font(s) DejaVu Sans Mono.
  self.fig.savefig(os.path.join(CFG.plot_dir,f"frame_{self._frame:05d}.png"),
/tmp/ipykernel_1866/3698779702.py:2158: UserWarning: Glyph 128128 (\N{SKULL}) missing from font(s) DejaVu Sans.
  try: plt.pause(0.001)
/tmp/ipykernel_1866/3698779702.py:2158: UserWarning: Glyph 127775 (\N{GLOWING STAR}) missing f


════════════════════════════════════════════════════════════════════════════
  Final: Orgs 144(♂76♀68) | War 110
  Villages: 10 | Active wars: 6
  Gen avg 3.81 max 7
  ☀ The Radiant Path: 0 followers | wars caused: 0 | conversions: 193 | tenets: Trade, Isolation
  ⚔ The Iron Covenant: 130 followers | wars caused: 0 | conversions: 752 | tenets: Pacifist, Isolation
  🌿 The Green Circle: 0 followers | wars caused: 0 | conversions: 413 | tenets: 
  Total war declarations: 6
  Total raids: 160366 | stolen: 195065
  Total schisms: 1
  Governance: {'DEMOCRACY': 3, 'AUTOCRACY': 3, 'THEOCRACY': 1, 'ANARCHY': 3}
════════════════════════════════════════════════════════════════════════════



/tmp/ipykernel_1866/3698779702.py:2247: UserWarning: Glyph 128128 (\N{SKULL}) missing from font(s) DejaVu Sans.
  rend.fig.savefig(fp,dpi=118,bbox_inches="tight",facecolor=rend.fig.get_facecolor())
/tmp/ipykernel_1866/3698779702.py:2247: UserWarning: Glyph 127775 (\N{GLOWING STAR}) missing from font(s) DejaVu Sans.
  rend.fig.savefig(fp,dpi=118,bbox_inches="tight",facecolor=rend.fig.get_facecolor())
/tmp/ipykernel_1866/3698779702.py:2247: UserWarning: Glyph 127807 (\N{HERB}) missing from font(s) DejaVu Sans.
  rend.fig.savefig(fp,dpi=118,bbox_inches="tight",facecolor=rend.fig.get_facecolor())
/tmp/ipykernel_1866/3698779702.py:2247: UserWarning: Glyph 127807 (\N{HERB}) missing from font(s) DejaVu Sans Mono.
  rend.fig.savefig(fp,dpi=118,bbox_inches="tight",facecolor=rend.fig.get_facecolor())


[Done] → sim_frames_v7/final_summary_v7.png


/usr/local/lib/python3.12/dist-packages/IPython/core/events.py:89: UserWarning: Glyph 128128 (\N{SKULL}) missing from font(s) DejaVu Sans.
  func(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/IPython/core/events.py:89: UserWarning: Glyph 127775 (\N{GLOWING STAR}) missing from font(s) DejaVu Sans.
  func(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/IPython/core/events.py:89: UserWarning: Glyph 127807 (\N{HERB}) missing from font(s) DejaVu Sans.
  func(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/IPython/core/events.py:89: UserWarning: Glyph 127807 (\N{HERB}) missing from font(s) DejaVu Sans Mono.
  func(*args, **kwargs)


# RL Architecture & Concepts: Organism Living System

This version  represents a significant leap from previous versions by introducing **Hierarchical RL**, **World Models**, and **Transformer-based memory**, moving beyond standard MAPPO with basic GRU architectures.

---

## 🚀 1. RL Concepts Used

### 1.1 Hierarchical Reinforcement Learning (HRL)
V6 breaks down complex behaviors into a multi-layered hierarchy.
* **Definition:** A framework that uses a **High-Level Policy (Manager)** to set abstract goals and a **Low-Level Policy (Worker)** to execute primitive actions to reach those goals.
* **Mathematical Explanation:**
    * The high-level policy $\pi_{high}(g|s)$ samples a goal $g$.
    * The low-level policy $\pi_{low}(a|s, g)$ is conditioned on that goal.
    * **Reward:** $R = r_{extrinsic} + r_{intrinsic}(s, g)$, where the low-level is rewarded specifically for goal-reaching behaviors.



### 1.2 World Model (Dreamer/Hallucination)
Inspired by the Dreamer architecture, agents "hallucinate" future states to learn without constant environmental interaction.
* **Definition:** A model that learns to predict the next state $s_{t+1}$ and reward $r_t$ given current state $s_t$ and action $a_t$.
* **Mathematical Explanation:**
    * **Transition Model:** $P(s_{t+1} | s_t, a_t)$
    * **Imagination Training:** Agents update their policies by acting within this learned model (hallucination) to simulate consequences before they happen in the real world.

### 1.3 Multi-Agent PPO (MAPPO) with CTDE
V6 utilizes MAPPO with an upgraded **Centralized Training with Decentralized Execution** pipeline.
* **Definition:** The **Critic** has access to global information (all agent positions, resource maps), while the **Actor** only sees the "Fog of War" local observation.
* **Objective Function:**
  $$L_{CLIP}(\theta) = \mathbb{E}_t [ \min(r_t(\theta) A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+ \epsilon) A_t) ]$$.



### 1.4 Transformer Memory (8-Step Attention)
To handle long-range dependencies better than standard GRUs, v6 introduces spatial-temporal attention.
* **Definition:** Uses an attention mechanism to weigh the importance of past observations over a fixed window.
* **Mathematical Explanation:**
  $$Attention(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
  * In V6, $Q, K, V$ are derived from the **8 most recent time-steps**, allowing the agent to "attend" to specific past events (like the last known location of a predator).

---

## 🏗️ 2. Architecture

### 🔷 High-Level Flow
1.  **Environment:** 2D World with Fog of War and multi-resource nodes.
2.  **Transformer Window:** Last 8 observations are processed via Self-Attention.
3.  **HRL Controller:** High-level sets a "Goal Embedding" (e.g., "Find Food" or "Defend Village").
4.  **Actor-Critic Core:**
    * **Actor:** Combines Attention Context + Goal Embedding + GRU state $\to$ Action Logits.
    * **Critic:** Processes Global State $\to$ Value $V(s)$.
5.  **Ensemble Output:** MAPPO output is merged with **NEAT** (NeuroEvolution of Augmenting Topologies) output for final action selection.

### 🧠 Neural Network Design: "Integrated Civ-Brain"
* **Encoder:** Linear layers for local spatial and status data.
* **Memory Layer 1 (Transformer):** 8-step temporal attention window.
* **Memory Layer 2 (GRU):** Recurrent state for long-term persistence beyond the attention window.
* **Action Heads:** * **Primary:** (Move, Eat, Attack, Trade, Communicate).
    * **Communication:** 8-dim learned message vectors for social coordination.

---

## 📊 3. Training Pipeline

* **Self-Play Archive:** Agents train against current and historical versions of themselves to ensure robust evolution.
* **PBT (Population Based Training):** Hyperparameters (like curiosity weights) are evolved in real-time based on population fitness.
* **Auto-Curriculum:** Environment difficulty (resource scarcity, disaster frequency) scales dynamically based on the current population's average health.

---

## ⚙️ 4. Key Hyperparameters (v6)

| Parameter | Value | Description |
| :--- | :--- | :--- |
| **Learning Rate** | `1.2e-4` | Optimized for Transformer stability |
| **Gamma ($\gamma$)** | `0.99` | Reward discount factor |
| **GAE Lambda ($\lambda$)** | `0.95` | Advantage smoothing |
| **Entropy Coef** | `0.018` | Lowered from v5 to encourage policy specialization |
| **Curiosity Coef** | `0.025` | Intrinsic exploration reward scaling |
| **Attention Window** | `8` | Number of past steps the Transformer "sees" |
| **Batch Size** | `1024` | Transitions sampled per gradient update |

---

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   ORGANISM LIVING SYSTEM   —  CIVILIZATION-GRADE OPEN-ENDED EVOLUTION     ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  INTELLIGENCE                                                                ║
║   • Hierarchical RL  (HRL)    — goal-conditioned high/low policy             ║
║   • World Model  (Dreamer)    — agents hallucinate futures before acting     ║
║   • Transformer Memory        — 8-step attention window per agent            ║
║  EVOLUTION                                                                   ║
║   • Neural Evolution (NEAT)   — evolve network topology + weights            ║
║   • Cultural Inheritance      — village behavior transmitted to offspring    ║
║   • Speciation                — clades diverge by gene distance              ║
║  SOCIETY                                                                     ║
║   • Communication             — learned 8-dim message vectors                ║
║   • Professions               — farmer / trader / warrior / explorer         ║
║   • Governance                — democracy, autocracy, theocracy policies     ║
║   • Multi-Resource Economy    — food / wood / metal / medicine               ║
║  ENVIRONMENT                                                                 ║
║   • Terrain                   — forest, desert, mountain, river              ║
║   • Natural Disasters         — drought, flood, fire, plague                 ║
║   • Fog of War                — partial observability per agent              ║
║  CONFLICT                                                                    ║
║   • Formation Combat          — warrior squads, flanking                     ║
║   • Territory Borders         — contested zones, resource wars               ║
║  META-LEARNING                                                               ║
║   • Auto-Curriculum           — difficulty auto-scales to pop health         ║
║   • Self-Play Archive         — agents compete vs. best past policies        ║
║   • MAPPO + GRU + PBT         — upgraded from v5                            ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
# ─────────────────────────────────────────────────────────────────────────────
#  IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
import numpy as np, matplotlib, random, math, time, os, copy, heapq
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.patches import FancyArrowPatch
from dataclasses import dataclass, field
from typing import List, Optional, Tuple, Dict, Any
from enum import Enum, auto
from collections import deque, defaultdict

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {DEVICE}", end="")
if DEVICE.type == "cuda":
    print(f"  |  {torch.cuda.get_device_name(0)}"
          f"  |  {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB VRAM")
else:
    print()

# ─────────────────────────────────────────────────────────────────────────────
#  ENUMS
# ─────────────────────────────────────────────────────────────────────────────
class Sex(Enum):         MALE=0;   FEMALE=1
class Profession(Enum): FARMER=0; TRADER=1; WARRIOR=2; EXPLORER=3; HEALER=4
class Governance(Enum): DEMOCRACY=0; AUTOCRACY=1; THEOCRACY=2; ANARCHY=3
class TerrainType(Enum):PLAINS=0; FOREST=1; DESERT=2; MOUNTAIN=3; RIVER=4
class GoalType(Enum):   FORAGE=0; RETURN=1; FLEE=2; TRADE=3; EXPLORE=4; DEFEND=5
class DisasterType(Enum): DROUGHT=0; FLOOD=1; FIRE=2; PLAGUE=3

# ─────────────────────────────────────────────────────────────────────────────
#  CONFIG
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class SimConfig:
    # World
    world_w: int   = 240
    world_h: int   = 160
    terrain_tile:  int   = 16       # tiles per axis for terrain grid
    n_bad_zones: int       = 3
    bad_zone_radius: float = 8.0
    bad_zone_damage: float = 0.4
    fog_radius: float      = 18.0   # visibility radius per organism

    # Seasons
    season_period: int   = 1000
    season_amp:    float = 0.40

    # Disasters
    disaster_prob:  float = 0.0008  # per step base probability
    disaster_duration: int= 120

    # Villages / Civilization
    init_villages: int       = 2
    max_villages:  int       = 12
    village_radius: float    = 16.0
    village_food_radius: float = 2.8
    clan_form_density: int   = 18
    clan_form_radius:  float = 24.0
    clan_form_cooldown:int   = 250
    village_breach_radius: float    = 30.0
    village_breach_kill_prob: float = 0.20
    min_warriors_per_village: int   = 3
    warrior_respawn_interval: int   = 120
    territory_radius: float         = 40.0
    overpop_threshold: int          = 50

    # Resources
    resource_types: int = 4    # food, wood, metal, medicine
    trade_reward:   float= 2.0

    # Communication
    msg_dim: int = 8

    # Populations
    init_organisms: int  = 110
    init_warriors:  int  = 24
    init_enemies:   int  = 6
    max_organisms:  int  = 320
    max_warriors:   int  = 100
    max_enemies:    int  = 45

    # Food / Resources
    init_food: int         = 350
    max_food:  int         = 550
    food_spawn_rate: float = 0.65
    food_eat_radius: float = 2.5

    # Organism physics
    org_friction: float = 0.76
    org_accel:    float = 0.94

    # Energy
    init_energy: float         = 108.0
    max_energy:  float         = 200.0
    base_metabolism: float     = 0.026
    move_cost_factor: float    = 0.009
    food_energy: float         = 75.0
    reproduce_threshold: float = 80.0
    reproduce_cost:      float = 24.0
    gestation_steps: int       = 12
    starve_energy: float       = 0.0
    elder_age: int             = 450
    elder_birth_bonus: float   = 0.28
    sick_duration: int         = 70
    sick_damage: float         = 0.07

    # Genes
    gene_vision_range:    Tuple = (5.0, 26.0)
    gene_meta_range:      Tuple = (0.02, 0.22)
    gene_fertility_range: Tuple = (0.3, 1.0)
    gene_va_range:        Tuple = (0.0, 1.0)
    gene_speed_range_M:   Tuple = (1.5, 4.5)
    gene_aggro_range_M:   Tuple = (0.2, 1.0)
    gene_speed_range_F:   Tuple = (0.8, 3.0)
    gene_aggro_range_F:   Tuple = (0.0, 0.6)
    mutation_rate:   float = 0.10
    mutation_strength: float = 0.11
    speciation_threshold: float = 1.8  # gene distance for new species

    # Warrior
    warrior_max_speed:    float = 4.2
    warrior_accel:        float = 1.2
    warrior_friction:     float = 0.79
    warrior_vision:       float = 24.0
    warrior_energy_init:  float = 145.0
    warrior_energy_max:   float = 250.0
    warrior_metabolism:   float = 0.042
    warrior_move_cost:    float = 0.008
    warrior_kill_radius:  float = 4.2
    warrior_protect_r:    float = 20.0
    warrior_food_energy:  float = 42.0
    warrior_rep_threshold:float = 175.0
    warrior_rep_cost:     float = 50.0
    warrior_attack_bonus: float = 32.0
    war_chief_bonus:      float = 0.28

    # Enemy
    enemy_max_speed_init:  float = 1.9
    enemy_accel:           float = 0.85
    enemy_friction:        float = 0.73
    enemy_kill_radius_init:float = 1.9
    enemy_vision_init:     float = 14.0
    enemy_max_speed_cap:   float = 5.0
    enemy_kill_radius_cap: float = 4.5
    enemy_growth_interval: int   = 500
    enemy_growth_rate:     float = 0.055
    enemy_spawn_interval:  int   = 800
    enemy_pack_radius:     float = 30.0

    # ── HRL ─────────────────────────────────────────────────────────────
    n_goals:         int   = 6     # number of high-level goals
    goal_horizon:    int   = 20    # steps per high-level decision
    goal_embed_dim:  int   = 16    # goal embedding size

    # ── World Model (Dreamer) ───────────────────────────────────────────
    wm_latent_dim:   int   = 32    # world model latent state
    wm_rollout:      int   = 3     # dream steps before acting
    wm_train_freq:   int   = 500   # train world model every N steps
    wm_lr:           float = 3e-4

    # ── Transformer Memory ───────────────────────────────────────────────
    mem_len:         int   = 8     # memory window
    mem_heads:       int   = 4     # attention heads
    mem_dim:         int   = 64    # memory embedding dim

    # ── NEAT Evolution ────────────────────────────────────────────────────
    neat_pop_size:   int   = 8     # network variants in evolution pool
    neat_mutate_freq:int   = 600   # mutate network structure every N steps
    neat_elites:     int   = 2     # top k networks survive

    # ── MAPPO + GRU + PBT ────────────────────────────────────────────────
    act_dim:          int   = 9
    org_obs_dim:      int   = 56    # expanded observation
    warrior_obs_dim:  int   = 38
    global_state_dim: int   = 32
    hidden_dim:       int   = 256
    gru_hidden:       int   = 128
    lr:               float = 1.2e-4
    gamma:            float = 0.99
    gae_lambda:       float = 0.95
    clip_eps:         float = 0.2
    entropy_coef:     float = 0.018
    value_coef:       float = 0.5
    ppo_epochs:       int   = 4
    batch_size:       int   = 1024
    rollout_len:      int   = 320
    max_grad_norm:    float = 0.5
    pbt_population:   int   = 4
    pbt_interval:     int   = 1200
    curiosity_coef:   float = 0.025
    curiosity_grid_w: int   = 24
    curiosity_grid_h: int   = 16

    # ── Auto-Curriculum ───────────────────────────────────────────────────
    curriculum_target_survival: float = 0.35  # target fraction surviving
    curriculum_check_interval:  int   = 300

    # ── Self-Play Archive ─────────────────────────────────────────────────
    selfplay_archive_size: int  = 5
    selfplay_swap_interval:int  = 2000

    # Simulation
    max_steps:    int  = 8000
    render_every: int  = 100
    save_plot:    bool = True
    plot_dir:     str  = "sim_frames_v6"

CFG = SimConfig()

_ACT_DIRS = np.array([
    [ 0,-1],[ 1,-1],[ 1, 0],[ 1, 1],
    [ 0, 1],[-1, 1],[-1, 0],[-1,-1],[ 0, 0],
], dtype=np.float32)
for _i in range(8):
    _n = np.linalg.norm(_ACT_DIRS[_i])
    if _n > 0: _ACT_DIRS[_i] /= _n

TERRAIN_COLORS = {
    TerrainType.PLAINS:   "#2d4a1e",
    TerrainType.FOREST:   "#1a3a12",
    TerrainType.DESERT:   "#5a4a1a",
    TerrainType.MOUNTAIN: "#3a3a3a",
    TerrainType.RIVER:    "#1a2a4a",
}
TERRAIN_FOOD_MULT = {
    TerrainType.PLAINS:   1.0,
    TerrainType.FOREST:   1.5,
    TerrainType.DESERT:   0.3,
    TerrainType.MOUNTAIN: 0.5,
    TerrainType.RIVER:    1.3,
}
TERRAIN_SPEED_MULT = {
    TerrainType.PLAINS:   1.0,
    TerrainType.FOREST:   0.8,
    TerrainType.DESERT:   0.7,
    TerrainType.MOUNTAIN: 0.5,
    TerrainType.RIVER:    0.9,
}

# ─────────────────────────────────────────────────────────────────────────────
#  TERRAIN MAP
# ─────────────────────────────────────────────────────────────────────────────
class TerrainMap:
    def __init__(self):
        W, H = CFG.terrain_tile, int(CFG.terrain_tile * CFG.world_h / CFG.world_w)
        self.W = W; self.H = max(H, 1)
        self.grid = np.full((self.H, W), TerrainType.PLAINS)
        self._generate()

    def _generate(self):
        # Simple noise-based terrain generation
        for ty in range(self.H):
            for tx in range(self.W):
                x = tx / self.W; y = ty / self.H
                # Rivers: diagonal bands
                if abs(math.sin(x * math.pi * 3 + 0.5) * math.cos(y * math.pi * 2) - 0.05) < 0.08:
                    self.grid[ty, tx] = TerrainType.RIVER
                # Mountains: corners
                elif (x < 0.15 or x > 0.85) and (y < 0.2 or y > 0.8):
                    self.grid[ty, tx] = TerrainType.MOUNTAIN
                # Desert: centre-top band
                elif 0.3 < x < 0.7 and 0.05 < y < 0.25:
                    self.grid[ty, tx] = TerrainType.DESERT
                # Forest: edges
                elif x < 0.2 or x > 0.8 or y < 0.15 or y > 0.85:
                    self.grid[ty, tx] = TerrainType.FOREST

    def get(self, x: float, y: float) -> TerrainType:
        tx = int(np.clip(x / CFG.world_w * self.W, 0, self.W - 1))
        ty = int(np.clip(y / CFG.world_h * self.H, 0, self.H - 1))
        return self.grid[ty, tx]

    def food_mult(self, x, y): return TERRAIN_FOOD_MULT[self.get(x, y)]
    def speed_mult(self, x, y): return TERRAIN_SPEED_MULT[self.get(x, y)]

TERRAIN = TerrainMap()

# ─────────────────────────────────────────────────────────────────────────────
#  GENOME  (expanded with cultural + speciation genes)
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class Genome:
    speed: float=2.0;  vision: float=11.0; metabolism: float=0.10
    aggression: float=0.4; fertility: float=0.6
    village_affinity: float=0.5; dominance: float=0.5
    immune_strength: float=0.5; trade_drive: float=0.5
    cooperation: float=0.5      # tendency to help nearby kin
    curiosity_drive: float=0.5  # intrinsic exploration tendency
    cultural_memory: float=0.5  # how much offspring inherit culture
    species_id: int = 0         # set during speciation

    @classmethod
    def random(cls, sex: Sex) -> "Genome":
        sr = CFG.gene_speed_range_M if sex==Sex.MALE else CFG.gene_speed_range_F
        ar = CFG.gene_aggro_range_M if sex==Sex.MALE else CFG.gene_aggro_range_F
        return cls(speed=random.uniform(*sr),
                   vision=random.uniform(*CFG.gene_vision_range),
                   metabolism=random.uniform(*CFG.gene_meta_range),
                   aggression=random.uniform(*ar),
                   fertility=random.uniform(*CFG.gene_fertility_range),
                   village_affinity=random.uniform(*CFG.gene_va_range),
                   dominance=random.uniform(0.2, 0.8),
                   immune_strength=random.uniform(0.1, 0.9),
                   trade_drive=random.uniform(0.0, 1.0),
                   cooperation=random.uniform(0.0, 1.0),
                   curiosity_drive=random.uniform(0.0, 1.0),
                   cultural_memory=random.uniform(0.2, 0.8))

    def mutate(self, sex: Sex) -> "Genome":
        sr = CFG.gene_speed_range_M if sex==Sex.MALE else CFG.gene_speed_range_F
        ar = CFG.gene_aggro_range_M if sex==Sex.MALE else CFG.gene_aggro_range_F
        def m(v, lo, hi):
            if random.random() < CFG.mutation_rate:
                v += random.gauss(0, CFG.mutation_strength) * (hi - lo)
            return float(np.clip(v, lo, hi))
        return Genome(speed=m(self.speed,*sr),
                      vision=m(self.vision,*CFG.gene_vision_range),
                      metabolism=m(self.metabolism,*CFG.gene_meta_range),
                      aggression=m(self.aggression,*ar),
                      fertility=m(self.fertility,*CFG.gene_fertility_range),
                      village_affinity=m(self.village_affinity,*CFG.gene_va_range),
                      dominance=m(self.dominance,0.1,0.9),
                      immune_strength=m(self.immune_strength,0.0,1.0),
                      trade_drive=m(self.trade_drive,0.0,1.0),
                      cooperation=m(self.cooperation,0.0,1.0),
                      curiosity_drive=m(self.curiosity_drive,0.0,1.0),
                      cultural_memory=m(self.cultural_memory,0.1,0.9),
                      species_id=self.species_id)

    def crossover(self, other: "Genome", csex: Sex) -> "Genome":
        dm = other.dominance; df = self.dominance
        total = dm + df + 1e-8; pf = dm / total
        pick = lambda a, b: b if random.random() < pf else a
        return Genome(speed=pick(self.speed,other.speed),
                      vision=pick(self.vision,other.vision),
                      metabolism=pick(self.metabolism,other.metabolism),
                      aggression=pick(self.aggression,other.aggression),
                      fertility=pick(self.fertility,other.fertility),
                      village_affinity=pick(self.village_affinity,other.village_affinity),
                      dominance=(self.dominance+other.dominance)/2,
                      immune_strength=pick(self.immune_strength,other.immune_strength),
                      trade_drive=pick(self.trade_drive,other.trade_drive),
                      cooperation=pick(self.cooperation,other.cooperation),
                      curiosity_drive=pick(self.curiosity_drive,other.curiosity_drive),
                      cultural_memory=pick(self.cultural_memory,other.cultural_memory),
                      species_id=self.species_id).mutate(csex)

    def distance(self, other: "Genome") -> float:
        """Genetic distance for speciation."""
        return math.sqrt(sum((a-b)**2 for a,b in [
            (self.speed/4.5,        other.speed/4.5),
            (self.vision/26.,       other.vision/26.),
            (self.aggression,       other.aggression),
            (self.cooperation,      other.cooperation),
            (self.trade_drive,      other.trade_drive),
            (self.curiosity_drive,  other.curiosity_drive),
        ]))

    def to_vec(self) -> np.ndarray:
        def norm(v, lo, hi): return (v-lo)/(hi-lo+1e-8)
        return np.array([
            norm(self.speed,1.0,4.5), norm(self.vision,*CFG.gene_vision_range),
            norm(self.metabolism,*CFG.gene_meta_range), norm(self.aggression,0.,1.),
            norm(self.fertility,*CFG.gene_fertility_range),
            self.village_affinity, self.dominance, self.immune_strength,
            self.trade_drive, self.cooperation, self.curiosity_drive, self.cultural_memory,
        ], dtype=np.float32)  # 12 genes

# ─────────────────────────────────────────────────────────────────────────────
#  TRANSFORMER MEMORY MODULE
# ─────────────────────────────────────────────────────────────────────────────
class TransformerMemory(nn.Module):
    """
    Replaces GRU with a small causal transformer memory window.
    Each agent maintains a rolling buffer of past mem_len embeddings.
    Multi-head self-attention reads across the window to produce context.
    """
    def __init__(self, input_dim: int, mem_dim: int = 64,
                 heads: int = 4, mem_len: int = 8):
        super().__init__()
        self.mem_len = mem_len
        self.mem_dim = mem_dim
        self.proj = nn.Linear(input_dim, mem_dim)
        self.attn = nn.MultiheadAttention(mem_dim, heads, batch_first=True)
        self.norm = nn.LayerNorm(mem_dim)
        self.out  = nn.Linear(mem_dim, mem_dim)
        # Positional encoding
        pe = torch.zeros(mem_len, mem_dim)
        pos = torch.arange(mem_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, mem_dim, 2).float() * (-math.log(10000.0)/mem_dim))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div[:mem_dim//2])
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor, mem_buf: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        x:       (B, input_dim)  current observation
        mem_buf: (B, mem_len, mem_dim)  past memory
        Returns: context (B, mem_dim), new_mem_buf
        """
        emb = self.proj(x).unsqueeze(1)  # (B, 1, mem_dim)
        new_buf = torch.cat([mem_buf[:, 1:, :], emb], dim=1)  # roll window
        buf_pe  = new_buf + self.pe.unsqueeze(0)
        ctx, _  = self.attn(buf_pe, buf_pe, buf_pe)
        ctx     = self.norm(ctx[:, -1, :])  # use most recent token
        ctx     = self.out(ctx)
        return ctx, new_buf

# ─────────────────────────────────────────────────────────────────────────────
#  WORLD MODEL  (Dreamer-lite: predict next latent + reward)
# ─────────────────────────────────────────────────────────────────────────────
class WorldModel(nn.Module):
    """
    Learns to predict: next_obs, reward given (obs, action).
    Agents use this to mentally simulate CFG.wm_rollout steps
    before committing to an action.
    """
    def __init__(self, obs_dim: int, act_dim: int, latent: int = 32):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(obs_dim, 128), nn.Tanh(),
            nn.Linear(128, latent)
        )
        self.trans = nn.Sequential(
            nn.Linear(latent + act_dim, 128), nn.Tanh(),
            nn.Linear(128, latent)
        )
        self.dec_obs    = nn.Linear(latent, obs_dim)
        self.dec_reward = nn.Linear(latent, 1)
        self._buf: List = []

    def forward(self, obs, act_onehot):
        z    = self.enc(obs)
        z2   = self.trans(torch.cat([z, act_onehot], -1))
        pred_obs = self.dec_obs(z2)
        pred_rew = self.dec_reward(z2)
        return pred_obs, pred_rew, z2

    def encode(self, obs: torch.Tensor) -> torch.Tensor:
        return self.enc(obs)

    def dream_reward(self, obs_np: np.ndarray, act: int, steps: int = 3) -> float:
        """Quick dreaming: simulate steps forward, sum predicted rewards."""
        if len(self._buf) < 64: return 0.0
        obs_t = torch.tensor(obs_np, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        total = 0.0
        for _ in range(steps):
            ah = torch.zeros(1, CFG.act_dim, device=DEVICE)
            ah[0, act] = 1.0
            with torch.no_grad():
                obs_t, rew, _ = self(obs_t, ah)
            total += rew.item()
        return float(total)

    def store(self, obs, act, next_obs, rew):
        self._buf.append((obs, act, next_obs, rew))
        if len(self._buf) > 20000: self._buf.pop(0)

    def train_step(self, opt) -> float:
        if len(self._buf) < 128: return 0.0
        batch = random.sample(self._buf, min(256, len(self._buf)))
        obs   = torch.tensor([b[0] for b in batch], dtype=torch.float32, device=DEVICE)
        acts  = torch.tensor([b[1] for b in batch], dtype=torch.long,    device=DEVICE)
        nobs  = torch.tensor([b[2] for b in batch], dtype=torch.float32, device=DEVICE)
        rews  = torch.tensor([b[3] for b in batch], dtype=torch.float32, device=DEVICE).unsqueeze(-1)
        ah    = F.one_hot(acts, num_classes=CFG.act_dim).float()
        p_obs, p_rew, _ = self(obs, ah)
        loss = F.mse_loss(p_obs, nobs) + F.mse_loss(p_rew, rews)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(self.parameters(), 1.0)
        opt.step()
        return loss.item()

# ─────────────────────────────────────────────────────────────────────────────
#  HRL  — High-Level Goal Policy + Low-Level Goal-Conditioned Actor
# ─────────────────────────────────────────────────────────────────────────────
class HighLevelPolicy(nn.Module):
    """Outputs a goal index every goal_horizon steps."""
    def __init__(self, obs_dim: int, n_goals: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 128), nn.Tanh(),
            nn.Linear(128, 64), nn.Tanh(),
            nn.Linear(64, n_goals)
        )
        self.goal_emb = nn.Embedding(n_goals, CFG.goal_embed_dim)

    def forward(self, obs: torch.Tensor):
        return self.net(obs)

    def embed(self, goal_idx: int) -> torch.Tensor:
        return self.goal_emb(torch.tensor(goal_idx, device=DEVICE))


class HRLWrapper:
    """Wraps a goal-conditioned low-level policy with a high-level scheduler."""
    def __init__(self, obs_dim: int):
        self.hl = HighLevelPolicy(obs_dim, CFG.n_goals).to(DEVICE)
        self.hl_opt = optim.Adam(self.hl.parameters(), lr=5e-4)
        self.goal_embed_dim = CFG.goal_embed_dim
        # Per-agent current goal
        self.agent_goals: Dict[int, int] = {}
        self.agent_goal_steps: Dict[int, int] = {}

    def get_goal(self, agent_id: int, obs_np: np.ndarray) -> Tuple[int, np.ndarray]:
        steps = self.agent_goal_steps.get(agent_id, 0)
        if steps <= 0 or agent_id not in self.agent_goals:
            obs_t = torch.tensor(obs_np, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            with torch.no_grad():
                logits = self.hl(obs_t)
            goal = torch.distributions.Categorical(logits=logits).sample().item()
            self.agent_goals[agent_id] = goal
            self.agent_goal_steps[agent_id] = CFG.goal_horizon
        else:
            goal = self.agent_goals[agent_id]
        self.agent_goal_steps[agent_id] -= 1
        with torch.no_grad():
            embed = self.hl.embed(goal).cpu().numpy()
        return goal, embed

    def clear(self, agent_id: int):
        self.agent_goals.pop(agent_id, None)
        self.agent_goal_steps.pop(agent_id, None)

# ─────────────────────────────────────────────────────────────────────────────
#  NEAT-STYLE NEURAL EVOLUTION POOL
# ─────────────────────────────────────────────────────────────────────────────
class NEATPool:
    """
    Maintains a small population of network variants.
    Every neat_mutate_freq steps, prunes worst, mutates best.
    """
    def __init__(self, obs_dim: int):
        self.obs_dim = obs_dim
        self.act_dim = CFG.act_dim
        self.population: List[nn.Module] = []
        self.scores: List[float] = []
        for _ in range(CFG.neat_pop_size):
            net = self._make_net()
            self.population.append(net)
            self.scores.append(0.0)
        self.active_idx = 0

    def _make_net(self) -> nn.Module:
        hidden = random.choice([64, 128, 192, 256])
        layers = random.choice([2, 3])
        mods = []
        inp = self.obs_dim + CFG.goal_embed_dim
        for _ in range(layers):
            mods += [nn.Linear(inp, hidden), nn.Tanh()]
            inp = hidden
        mods.append(nn.Linear(hidden, self.act_dim))
        net = nn.Sequential(*mods).to(DEVICE)
        for m in net.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)
        return net

    def get_active(self) -> nn.Module:
        return self.population[self.active_idx]

    def update_score(self, reward: float):
        self.scores[self.active_idx] = self.scores[self.active_idx]*0.95 + reward*0.05

    def evolve(self):
        """Prune worst, mutate best."""
        order = sorted(range(len(self.scores)), key=lambda i: self.scores[i])
        # Replace bottom half with mutated copies of top
        for i in order[:len(order)//2]:
            src_idx = order[-1]
            src = self.population[src_idx]
            child = copy.deepcopy(src)
            with torch.no_grad():
                for p in child.parameters():
                    p.add_(torch.randn_like(p) * 0.05)
            self.population[i] = child
            self.scores[i] = 0.0
        # Best network stays active
        self.active_idx = order[-1]
        print(f"  [NEAT] Evolved. Best score={self.scores[self.active_idx]:.3f}")

# ─────────────────────────────────────────────────────────────────────────────
#  COMMUNICATION  (learned message passing)
# ─────────────────────────────────────────────────────────────────────────────
class CommModule(nn.Module):
    """
    Agents broadcast msg_dim message vectors.
    Nearby agents aggregate received messages.
    """
    def __init__(self, obs_dim: int, msg_dim: int = 8):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.Tanh(),
            nn.Linear(64, msg_dim), nn.Tanh()
        )
        self.decoder = nn.Sequential(
            nn.Linear(msg_dim, 32), nn.Tanh(),
            nn.Linear(32, msg_dim)
        )

    def encode(self, obs: torch.Tensor) -> torch.Tensor:
        return self.encoder(obs)

    def aggregate(self, messages: List[torch.Tensor]) -> torch.Tensor:
        if not messages:
            return torch.zeros(CFG.msg_dim, device=DEVICE)
        flat = [m.squeeze(0) if m.dim() == 2 else m for m in messages]
        stacked = torch.stack(flat, dim=0)   # (K, msg_dim)
        return self.decoder(stacked.mean(0)) # (msg_dim,)

# ─────────────────────────────────────────────────────────────────────────────
#  MAIN ACTOR-CRITIC  (MAPPO + Transformer Memory + Goal-conditioned)
# ─────────────────────────────────────────────────────────────────────────────
class CivActorCritic(nn.Module):
    """
    Full architecture:
    obs → transformer_memory → [mem_ctx, goal_embed, msg] → GRU → actor/critic
    """
    def __init__(self, obs_dim: int, act_dim: int, global_dim: int):
        super().__init__()
        self.tmem = TransformerMemory(obs_dim, CFG.mem_dim, CFG.mem_heads, CFG.mem_len)
        gru_in = CFG.mem_dim + CFG.goal_embed_dim + CFG.msg_dim
        self.gru = nn.GRUCell(gru_in, CFG.gru_hidden)
        self.actor = nn.Sequential(
            nn.Linear(CFG.gru_hidden, CFG.hidden_dim//2), nn.Tanh(),
            nn.Linear(CFG.hidden_dim//2, act_dim)
        )
        self.critic = nn.Sequential(
            nn.Linear(global_dim, CFG.hidden_dim), nn.LayerNorm(CFG.hidden_dim), nn.Tanh(),
            nn.Linear(CFG.hidden_dim, CFG.hidden_dim//2), nn.Tanh(),
            nn.Linear(CFG.hidden_dim//2, 1)
        )
        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.zeros_(m.bias)
        nn.init.orthogonal_(self.actor[-1].weight, gain=0.01)
        nn.init.orthogonal_(self.critic[-1].weight, gain=1.0)

    def actor_step(self, obs_t, mem_buf, hx, goal_embed, msg_embed):
        ctx, new_buf = self.tmem(obs_t, mem_buf)
        inp  = torch.cat([ctx, goal_embed, msg_embed], dim=-1)
        hx   = self.gru(inp, hx)
        return self.actor(hx), hx, new_buf

    def critic_step(self, gs_t):
        return self.critic(gs_t)

# ─────────────────────────────────────────────────────────────────────────────
#  PBT CONFIG
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class PBTConfig:
    lr: float=1.2e-4; entropy_coef: float=0.018
    gamma: float=0.99; clip_eps: float=0.2; score: float=0.0

    def mutate(self) -> "PBTConfig":
        c = copy.copy(self)
        c.lr           = float(np.clip(c.lr*random.uniform(0.8,1.25), 1e-5, 5e-3))
        c.entropy_coef = float(np.clip(c.entropy_coef*random.uniform(0.8,1.25), 0.001, 0.1))
        c.gamma        = float(np.clip(c.gamma+random.uniform(-0.01,0.01), 0.90, 0.999))
        c.clip_eps     = float(np.clip(c.clip_eps*random.uniform(0.85,1.15), 0.05, 0.5))
        c.score        = 0.0; return c

# ─────────────────────────────────────────────────────────────────────────────
#  CIV POLICY  (MAPPO + Transformer + HRL + WorldModel + PBT + NEAT)
# ─────────────────────────────────────────────────────────────────────────────
class CivPolicy:
    def __init__(self, obs_dim: int, global_dim: int, name: str = "policy"):
        self.name = name
        self.obs_dim = obs_dim
        self.pbt_configs = [PBTConfig() for _ in range(CFG.pbt_population)]
        self.active_pbt  = 0
        self.net = CivActorCritic(obs_dim, CFG.act_dim, global_dim).to(DEVICE)
        self.comm = CommModule(obs_dim, CFG.msg_dim).to(DEVICE)
        self.world_model = WorldModel(obs_dim, CFG.act_dim, CFG.wm_latent_dim).to(DEVICE)
        self.wm_opt = optim.Adam(self.world_model.parameters(), lr=CFG.wm_lr)
        self.hrl = HRLWrapper(obs_dim)
        self.neat = NEATPool(obs_dim)
        self._make_opt()
        # Per-agent states
        self.hidden:  Dict[int, torch.Tensor] = {}
        self.mem_buf: Dict[int, torch.Tensor] = {}
        self.msgs:    Dict[int, torch.Tensor] = {}  # last broadcast messages
        self._reset()
        self.visit_grid = np.zeros((CFG.curiosity_grid_h, CFG.curiosity_grid_w), dtype=np.float32)
        self.steps = 0
        self.wm_loss_hist: List[float] = []

        # Self-play archive: store copies of past nets
        self.archive: List[nn.Module] = []

        # Auto-curriculum state
        self.survival_history: deque = deque(maxlen=100)
        self.curriculum_difficulty = 1.0  # multiplier on enemy strength

    def _make_opt(self):
        cfg = self.pbt_configs[self.active_pbt]
        params = list(self.net.parameters()) + list(self.comm.parameters())
        self.opt = optim.Adam(params, lr=cfg.lr, eps=1e-5)

    def _reset(self):
        self.buf = {k: [] for k in ["obs","acts","logps","vals","rews","dones","hx","gs","goal_e","msg_e"]}

    def _get_hidden(self, aid: int) -> torch.Tensor:
        if aid not in self.hidden:
            self.hidden[aid] = torch.zeros(1, CFG.gru_hidden, device=DEVICE)
        return self.hidden[aid]

    def _get_membuf(self, aid: int) -> torch.Tensor:
        if aid not in self.mem_buf:
            self.mem_buf[aid] = torch.zeros(1, CFG.mem_len, CFG.mem_dim, device=DEVICE)
        return self.mem_buf[aid]

    def _get_msg(self, aid: int) -> torch.Tensor:
        if aid not in self.msgs:
            self.msgs[aid] = torch.zeros(1, CFG.msg_dim, device=DEVICE)
        return self.msgs[aid]

    def clear_agent(self, aid: int):
        for d in [self.hidden, self.mem_buf, self.msgs]:
            d.pop(aid, None)
        self.hrl.clear(aid)

    def intrinsic_reward(self, x, y) -> float:
        gx = int(np.clip(x/CFG.world_w*CFG.curiosity_grid_w, 0, CFG.curiosity_grid_w-1))
        gy = int(np.clip(y/CFG.world_h*CFG.curiosity_grid_h, 0, CFG.curiosity_grid_h-1))
        cnt = self.visit_grid[gy, gx]; self.visit_grid[gy, gx] += 1.0
        return CFG.curiosity_coef / (1.0 + math.sqrt(cnt))

    def store(self, obs, act, logp, val, rew, done, hx, gs, ge, me):
        self.buf["obs"].append(obs);   self.buf["acts"].append(act)
        self.buf["logps"].append(logp);self.buf["vals"].append(val)
        self.buf["rews"].append(rew);  self.buf["dones"].append(float(done))
        self.buf["hx"].append(hx.cpu().numpy().flatten())
        self.buf["gs"].append(gs); self.buf["goal_e"].append(ge)
        self.buf["msg_e"].append(me)

    @torch.no_grad()
    def batch_act(self, obs_list, agent_ids, global_states,
                  nearby_agent_ids_list):
        if not obs_list: return [], [], [], None, [], []
        obs_t = torch.tensor(np.array(obs_list), dtype=torch.float32, device=DEVICE)
        gs_t  = torch.tensor(np.array(global_states), dtype=torch.float32, device=DEVICE)
        N = len(obs_list)

        # Compute broadcast messages for each agent
        msgs_t = self.comm.encode(obs_t)   # (N, msg_dim)
        for i, aid in enumerate(agent_ids):
            self.msgs[aid] = msgs_t[i:i+1].detach()

        # Aggregate incoming messages per agent — each agg is (msg_dim,)
        agg_msgs = []
        for i, (aid, nearby_ids) in enumerate(zip(agent_ids, nearby_agent_ids_list)):
            incoming = [self.msgs[nid] for nid in nearby_ids if nid in self.msgs and nid != aid]
            agg = self.comm.aggregate(incoming)          # (msg_dim,)
            agg_msgs.append(agg)
        agg_msg_t = torch.stack(agg_msgs, dim=0).to(DEVICE)   # (N, msg_dim) ✓

        # HRL: get goal embeddings
        goal_embeds = []
        goal_indices = []
        for i, (aid, obs_np) in enumerate(zip(agent_ids, obs_list)):
            goal_idx, goal_e = self.hrl.get_goal(aid, obs_np)
            goal_indices.append(goal_idx)
            goal_embeds.append(goal_e)
        goal_t = torch.tensor(np.array(goal_embeds), dtype=torch.float32, device=DEVICE)

        # Stack per-agent hidden + mem_buf
        hx_list  = [self._get_hidden(aid) for aid in agent_ids]
        mb_list  = [self._get_membuf(aid) for aid in agent_ids]
        hx_t     = torch.cat(hx_list,  dim=0)           # (N, gru_h)
        mb_t     = torch.cat(mb_list,  dim=0)           # (N, mem_len, mem_dim)

        # Forward pass
        ctx, new_mb = self.net.tmem(obs_t, mb_t)
        inp  = torch.cat([ctx, goal_t, agg_msg_t], dim=-1)
        hx_new = self.net.gru(inp, hx_t)
        logits = self.net.actor(hx_new)
        vals   = self.net.critic_step(gs_t)

        # NEAT net also votes (blend)
        neat_net = self.neat.get_active()
        neat_inp = torch.cat([obs_t, goal_t], dim=-1)
        neat_logits = neat_net(neat_inp)
        logits = 0.7 * logits + 0.3 * neat_logits  # ensemble

        dist   = torch.distributions.Categorical(logits=logits)
        acts   = dist.sample(); logps = dist.log_prob(acts)

        # Update per-agent states
        for i, aid in enumerate(agent_ids):
            self.hidden[aid]  = hx_new[i:i+1].detach()
            self.mem_buf[aid] = new_mb[i:i+1].detach()

        return (acts.cpu().numpy(), logps.cpu().numpy(),
                vals.squeeze(-1).cpu().numpy(),
                hx_new.detach(),
                [goal_embeds[i] for i in range(N)],
                [agg_msgs[i].detach().cpu().numpy() for i in range(N)])

    def update(self, global_score: float = 0.0, survival_frac: float = 1.0) -> dict:
        n = len(self.buf["obs"])
        if n < CFG.batch_size: return {}
        cfg = self.pbt_configs[self.active_pbt]
        cfg.score = cfg.score * 0.95 + global_score * 0.05
        self.survival_history.append(survival_frac)

        obs   = torch.tensor(np.array(self.buf["obs"]),   dtype=torch.float32, device=DEVICE)
        acts  = torch.tensor(self.buf["acts"],             dtype=torch.long,    device=DEVICE)
        logps = torch.tensor(self.buf["logps"],            dtype=torch.float32, device=DEVICE)
        vals  = torch.tensor(self.buf["vals"],             dtype=torch.float32, device=DEVICE)
        rews  = torch.tensor(self.buf["rews"],             dtype=torch.float32, device=DEVICE)
        dones = torch.tensor(self.buf["dones"],            dtype=torch.float32, device=DEVICE)
        hxs   = torch.tensor(np.array(self.buf["hx"]),    dtype=torch.float32, device=DEVICE)
        hxs   = hxs.view(n, 1, CFG.gru_hidden)
        gs    = torch.tensor(np.array(self.buf["gs"]),    dtype=torch.float32, device=DEVICE)
        ge    = torch.tensor(np.array(self.buf["goal_e"]),dtype=torch.float32, device=DEVICE)
        me    = torch.tensor(np.array(self.buf["msg_e"]), dtype=torch.float32, device=DEVICE)

        adv = torch.zeros_like(rews); gae = 0.0
        for t in reversed(range(n)):
            nv = 0.0 if t==n-1 or dones[t] else vals[t+1].item()
            delta = rews[t] + cfg.gamma*nv*(1-dones[t]) - vals[t]
            gae   = delta + cfg.gamma*CFG.gae_lambda*(1-dones[t])*gae
            adv[t]= gae
        ret = adv + vals
        adv = (adv-adv.mean())/(adv.std()+1e-8)

        m = {"pl":0,"vl":0,"ent":0,"n":0}
        idx = torch.randperm(n)
        for _ in range(CFG.ppo_epochs):
            for s in range(0, n, CFG.batch_size):
                mb   = idx[s:s+CFG.batch_size]
                hx_mb= hxs[mb].squeeze(1)
                # Dummy mem_buf for update (use zeros — approximate)
                zero_mb = torch.zeros(len(mb), CFG.mem_len, CFG.mem_dim, device=DEVICE)
                ctx_mb, _ = self.net.tmem(obs[mb], zero_mb)
                inp_mb    = torch.cat([ctx_mb, ge[mb], me[mb]], dim=-1)
                hx_new_mb = self.net.gru(inp_mb, hx_mb)
                logit2    = self.net.actor(hx_new_mb)
                v2        = self.net.critic_step(gs[mb])
                # NEAT blend
                neat_inp2 = torch.cat([obs[mb], ge[mb]], dim=-1)
                neat_l2   = self.neat.get_active()(neat_inp2)
                logit2    = 0.7*logit2 + 0.3*neat_l2
                d2   = torch.distributions.Categorical(logits=logit2)
                nlp  = d2.log_prob(acts[mb]); ent = d2.entropy().mean()
                r    = (nlp-logps[mb]).exp(); a = adv[mb]
                pl   = -torch.min(r*a, r.clamp(1-cfg.clip_eps,1+cfg.clip_eps)*a).mean()
                vl   = F.mse_loss(v2.squeeze(-1), ret[mb])
                loss = pl + CFG.value_coef*vl - cfg.entropy_coef*ent
                self.opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    list(self.net.parameters())+list(self.comm.parameters()),
                    CFG.max_grad_norm)
                self.opt.step()
                m["pl"]+=pl.item(); m["vl"]+=vl.item(); m["ent"]+=ent.item(); m["n"]+=1

        self.neat.update_score(global_score)
        self._reset(); self.steps += n
        nu = max(m["n"], 1)
        return {k: v/nu for k,v in m.items() if k!="n"}

    def pbt_step(self):
        scores  = [c.score for c in self.pbt_configs]
        best_i  = int(np.argmax(scores)); worst_i = int(np.argmin(scores))
        if best_i!=worst_i and scores[best_i]>scores[worst_i]+0.5:
            self.pbt_configs[worst_i] = self.pbt_configs[best_i].mutate()
            print(f"  [PBT] {self.name}: replaced C{worst_i} with mut(C{best_i}) score={scores[best_i]:.2f}")
        self.active_pbt = best_i; self._make_opt()

    def neat_step(self):
        self.neat.evolve()

    def train_world_model(self):
        loss = self.world_model.train_step(self.wm_opt)
        if loss > 0: self.wm_loss_hist.append(loss)

    def auto_curriculum(self) -> float:
        """Adjust difficulty based on recent survival fraction."""
        if len(self.survival_history) < 20: return self.curriculum_difficulty
        avg = float(np.mean(self.survival_history))
        if avg < CFG.curriculum_target_survival - 0.1:
            # Too hard → reduce enemy aggression
            self.curriculum_difficulty = max(0.5, self.curriculum_difficulty - 0.02)
        elif avg > CFG.curriculum_target_survival + 0.1:
            # Too easy → increase
            self.curriculum_difficulty = min(2.0, self.curriculum_difficulty + 0.02)
        return self.curriculum_difficulty

    def archive_policy(self):
        if len(self.archive) >= CFG.selfplay_archive_size:
            self.archive.pop(0)
        self.archive.append(copy.deepcopy(self.net))
        print(f"  [Archive] {self.name}: saved policy (archive size={len(self.archive)})")

# ─────────────────────────────────────────────────────────────────────────────
#  VILLAGE  (governance, multi-resource, culture)
# ─────────────────────────────────────────────────────────────────────────────
_VID = 0
class Village:
    def __init__(self, x, y, clan_id=None):
        global _VID; _VID += 1
        self.id = _VID; self.x = float(x); self.y = float(y)
        self.clan_id = clan_id or _VID
        self.is_defended = True; self.founded_step = 0
        self.last_warrior_spawn = 0
        # Governance
        self.governance = random.choice(list(Governance))
        self.leader_id: Optional[int] = None
        self.policy_attack = random.random()    # 0=pacifist, 1=aggressive
        self.policy_trade  = random.random()
        self.policy_expand = random.random()
        # Multi-resource stockpile: food, wood, metal, medicine
        self.resources = np.array([10., 5., 2., 1.], dtype=np.float32)
        # Culture vector (transmitted to offspring)
        self.culture = np.random.rand(8).astype(np.float32)
        # Disease
        self.plague_active = False; self.plague_timer = 0
        # Disaster tracking
        self.disaster: Optional[DisasterType] = None
        self.disaster_timer = 0
        self.species_counts: Dict[int, int] = {}

    def contains(self, x, y): return math.hypot(x-self.x,y-self.y)<=CFG.village_radius
    def in_territory(self, x, y): return math.hypot(x-self.x,y-self.y)<=CFG.territory_radius
    def dist(self, x, y): return math.hypot(x-self.x,y-self.y)

    def update_governance(self, organisms, warriors):
        """Village governance affects policy."""
        local_orgs = [o for o in organisms if o.alive and self.contains(o.x,o.y)]
        local_wars = [w for w in warriors if w.alive and w.home_village==self.id]
        if not local_orgs: return
        if self.governance == Governance.DEMOCRACY:
            # Policy is average preference of residents
            self.policy_trade  = float(np.mean([o.genome.trade_drive for o in local_orgs]))
            self.policy_attack = float(np.mean([o.genome.aggression  for o in local_orgs]))
        elif self.governance == Governance.AUTOCRACY:
            # Leader sets policy
            if self.leader_id:
                leaders = [o for o in local_orgs if o.id == self.leader_id]
                if leaders:
                    ldr = leaders[0]
                    self.policy_trade  = ldr.genome.trade_drive
                    self.policy_attack = ldr.genome.aggression
        elif self.governance == Governance.THEOCRACY:
            # Elders dominate
            elders = [o for o in local_orgs if o.is_elder]
            if elders:
                self.policy_trade = float(np.mean([e.genome.trade_drive for e in elders]))
        # Elect leader: highest fitness in village
        if local_orgs:
            new_ldr = max(local_orgs, key=lambda o: o.fitness)
            self.leader_id = new_ldr.id

    def culture_drift(self):
        """Culture evolves slowly over time."""
        noise = np.random.randn(8).astype(np.float32) * 0.005
        self.culture = np.clip(self.culture + noise, 0, 1)

# ─────────────────────────────────────────────────────────────────────────────
#  ORGANISM
# ─────────────────────────────────────────────────────────────────────────────
_OID = 0
class Organism:
    def __init__(self, x, y, genome=None, energy=None, generation=1,
                 sex=None, clan_id=None, culture=None):
        global _OID; _OID += 1
        self.id=_OID; self.x=float(x); self.y=float(y)
        self.vx=0.; self.vy=0.
        self.sex=sex or random.choice([Sex.MALE,Sex.FEMALE])
        self.genome=genome or Genome.random(self.sex)
        self.energy=float(energy or CFG.init_energy)
        self.alive=True; self.age=0; self.generation=generation; self.fitness=0.
        self.home_village: Optional[int]=None; self.clan_id=clan_id
        self.profession=self._assign_profession()
        self.pregnant=False; self.gestation=0; self.father_genome=None
        self.mate_timer=0
        self.is_elder=False
        self.sick=False; self.sick_timer=0
        self.carrying_food=False
        self.trade_destination: Optional[int]=None
        # Cultural memory (inherited + learned)
        self.culture = culture if culture is not None else np.random.rand(8).astype(np.float32)
        # HRL goal state
        self.current_goal = GoalType.FORAGE
        self.goal_timer = 0
        # Fog of war: seen food positions (short-term episodic memory)
        self.known_food: deque = deque(maxlen=20)
        self._obs=None; self._act=8; self._logp=0.; self._val=0.
        self._prev_energy=self.energy

    def _assign_profession(self) -> Profession:
        """Profession from genes."""
        g = self.genome
        if g.aggression > 0.65:     return Profession.WARRIOR
        if g.trade_drive > 0.65:    return Profession.TRADER
        if g.curiosity_drive > 0.65:return Profession.EXPLORER
        if g.immune_strength > 0.65:return Profession.HEALER
        return Profession.FARMER

    def profession_bonus(self) -> Dict[str, float]:
        """Stat bonuses from profession."""
        if self.profession == Profession.FARMER:
            return {"food_mult": 1.3, "meta_mult": 0.85}
        if self.profession == Profession.TRADER:
            return {"trade_reward": 1.5}
        if self.profession == Profession.WARRIOR:
            return {"speed_mult": 1.15, "aggro_mult": 1.2}
        if self.profession == Profession.EXPLORER:
            return {"vision_mult": 1.25, "curiosity": 1.5}
        if self.profession == Profession.HEALER:
            return {"sick_resist": 1.5, "heal_nearby": 0.5}
        return {}

    # 56-dim observation
    def observe(self, food, enemies, warriors, villages, bad_zones,
                elders, season_phase, terrain: TerrainMap,
                agg_msg: np.ndarray) -> np.ndarray:
        W,H=CFG.world_w,CFG.world_h
        obs=np.zeros(CFG.org_obs_dim,dtype=np.float32)
        obs[0]=self.x/W; obs[1]=self.y/H
        obs[2]=self.energy/CFG.max_energy; obs[3]=min(self.age/700.,1.)
        obs[4]=float(self.sex==Sex.FEMALE)
        obs[5:17]=self.genome.to_vec()   # 12 genes

        def near(items,gx,gy):
            if not items: return 0.,0.,1.,0.
            d2=[(math.hypot(gx(i)-self.x,gy(i)-self.y),i) for i in items]
            d,it=min(d2,key=lambda t:t[0]); ix,iy=gx(it),gy(it)
            return (np.clip((ix-self.x)/W,-1,1),np.clip((iy-self.y)/H,-1,1),
                    min(d/self.genome.vision,1.),float(d<=self.genome.vision))

        # Only include food within fog radius
        vis_food=[f for f in food if math.hypot(f[0]-self.x,f[1]-self.y)<=CFG.fog_radius]
        obs[17],obs[18],obs[19],obs[20]=near(vis_food, lambda f:f[0],lambda f:f[1])
        obs[21],obs[22],obs[23],obs[24]=near(enemies,  lambda e:e.x, lambda e:e.y)
        obs[25],obs[26],obs[27],obs[28]=near(warriors, lambda w:w.x, lambda w:w.y)

        hv=[v for v in villages if v.id==self.home_village]
        if hv:
            v=hv[0]; d=v.dist(self.x,self.y)
            obs[29]=np.clip((v.x-self.x)/W,-1,1); obs[30]=np.clip((v.y-self.y)/H,-1,1)
            obs[31]=min(d/(W+H),1.); obs[32]=float(d<=CFG.village_radius)
            obs[33]=float(v.is_defended); obs[34]=float(v.plague_active)
            obs[35]=float(v.policy_attack); obs[36]=float(v.policy_trade)
            # Resource levels (normalised)
            res_norm=v.resources/np.array([50.,30.,15.,10.])
            obs[37:41]=np.clip(res_norm,0,1)

        if bad_zones:
            dsts=[math.hypot(z[0]-self.x,z[1]-self.y) for z in bad_zones]
            obs[41]=min(min(dsts)/W,1.); obs[42]=float(min(dsts)<=CFG.bad_zone_radius)

        obs[43]=np.clip(self.vx/4.,-1,1); obs[44]=np.clip(self.vy/4.,-1,1)
        obs[45]=min(len(vis_food)/14.,1.)
        obs[46]=np.clip((self.energy-self._prev_energy)/CFG.max_energy,-1,1)
        obs[47]=float(self.pregnant); obs[48]=float(self.sick)
        obs[49]=float(self.carrying_food)
        obs[50]=math.sin(season_phase*2*math.pi); obs[51]=math.cos(season_phase*2*math.pi)
        # Terrain info
        t=terrain.get(self.x,self.y)
        obs[52]=float(t==TerrainType.FOREST); obs[53]=float(t==TerrainType.DESERT)
        obs[54]=float(t==TerrainType.MOUNTAIN); obs[55]=float(t==TerrainType.RIVER)
        # Communication message embedded into obs
        # (msg included separately as goal_embed in policy)
        return obs

    def apply_action(self, action, terrain: TerrainMap):
        d=_ACT_DIRS[action]; spd=self.genome.speed
        if self.pregnant: spd*=0.72
        if self.sick:     spd*=0.78
        spd *= terrain.speed_mult(self.x,self.y)
        prof_bonus = self.profession_bonus()
        spd *= prof_bonus.get("speed_mult", 1.0)
        self.vx=self.vx*CFG.org_friction+d[0]*CFG.org_accel*spd
        self.vy=self.vy*CFG.org_friction+d[1]*CFG.org_accel*spd
        v=math.hypot(self.vx,self.vy)
        if v>spd: self.vx*=spd/v; self.vy*=spd/v
        self._prev_energy=self.energy
        self.x=float(np.clip(self.x+self.vx,0,CFG.world_w))
        self.y=float(np.clip(self.y+self.vy,0,CFG.world_h))
        if self.x<=0 or self.x>=CFG.world_w: self.vx*=-0.5
        if self.y<=0 or self.y>=CFG.world_h: self.vy*=-0.5
        meta_mult = prof_bonus.get("meta_mult", 1.0)
        cost=(CFG.base_metabolism*(1.+self.genome.metabolism)*meta_mult
              +CFG.move_cost_factor*(self.vx**2+self.vy**2))
        if self.pregnant: cost*=1.18
        if self.sick:
            cost*=1.12; self.energy-=CFG.sick_damage
            self.sick_timer-=1
            if self.sick_timer<=0: self.sick=False
        self.energy-=cost; self.age+=1
        if self.mate_timer>0: self.mate_timer-=1
        if self.age>=CFG.elder_age: self.is_elder=True
        if self.energy<=CFG.starve_energy: self.alive=False

# ─────────────────────────────────────────────────────────────────────────────
#  WARRIOR
# ─────────────────────────────────────────────────────────────────────────────
_WID = 0
class Warrior:
    def __init__(self, x, y, energy=None, home_village=None, clan_id=None):
        global _WID; _WID+=1
        self.id=_WID; self.x=float(x); self.y=float(y)
        self.vx=0.; self.vy=0.
        self.energy=float(energy or CFG.warrior_energy_init)
        self.alive=True; self.age=0; self.kills=0
        self.home_village=home_village; self.clan_id=clan_id
        self.is_war_chief=False
        self.squad_id: Optional[int]=None  # formation combat
        self._obs=None; self._act=8; self._logp=0.; self._val=0.

    def effective_speed(self):
        spd = CFG.warrior_max_speed*(1.+CFG.war_chief_bonus) if self.is_war_chief else CFG.warrior_max_speed
        return spd

    def effective_kill_radius(self):
        return (CFG.warrior_kill_radius*(1.+CFG.war_chief_bonus*0.5)
                if self.is_war_chief else CFG.warrior_kill_radius)

    def observe(self, enemies, organisms, villages, bad_zones,
                season_phase: float, terrain: TerrainMap) -> np.ndarray:
        W,H=CFG.world_w,CFG.world_h
        obs=np.zeros(CFG.warrior_obs_dim,dtype=np.float32)
        obs[0]=self.x/W; obs[1]=self.y/H
        obs[2]=self.energy/CFG.warrior_energy_max; obs[3]=min(self.age/600.,1.)

        def ns(items,gx,gy,k=0):
            if not items: return 0.,0.,1.,0.
            d2=[(math.hypot(gx(i)-self.x,gy(i)-self.y),i) for i in items]
            d2.sort(key=lambda t:t[0])
            if k>=len(d2): return 0.,0.,1.,0.
            d,it=d2[k]; ix,iy=gx(it),gy(it)
            return (np.clip((ix-self.x)/W,-1,1),np.clip((iy-self.y)/H,-1,1),
                    min(d/CFG.warrior_vision,1.),float(d<=CFG.warrior_vision))

        obs[4],obs[5],obs[6],obs[7]    =ns(enemies,  lambda e:e.x,lambda e:e.y,0)
        obs[8],obs[9],obs[10],obs[11]  =ns(organisms,lambda o:o.x,lambda o:o.y,0)
        obs[12],obs[13],obs[14],obs[15]=ns(enemies,  lambda e:e.x,lambda e:e.y,1)
        obs[16],obs[17],obs[18],obs[19]=ns(enemies,  lambda e:e.x,lambda e:e.y,2)  # 3rd enemy

        mv=[v for v in villages if v.id==self.home_village]
        if mv:
            v=mv[0]
            obs[20]=np.clip((v.x-self.x)/W,-1,1); obs[21]=np.clip((v.y-self.y)/H,-1,1)
            obs[22]=float(v.is_defended); obs[23]=float(v.plague_active)
            obs[24]=float(v.policy_attack); obs[25]=float(v.policy_trade)

        obs[26]=np.clip(self.vx/self.effective_speed(),-1,1)
        obs[27]=np.clip(self.vy/self.effective_speed(),-1,1)
        obs[28]=min(sum(1 for e in enemies
                        if math.hypot(e.x-self.x,e.y-self.y)<=CFG.warrior_vision)/8.,1.)
        obs[29]=min(sum(1 for o in organisms
                        if o.alive and math.hypot(o.x-self.x,o.y-self.y)<=CFG.warrior_protect_r)/14.,1.)
        obs[30]=min(self.kills/40.,1.); obs[31]=self.energy/CFG.warrior_energy_max
        obs[32]=float(self.is_war_chief)
        obs[33]=float(any(not v.is_defended and v.id==self.home_village for v in villages))
        obs[34]=math.sin(season_phase*2*math.pi); obs[35]=math.cos(season_phase*2*math.pi)
        t=terrain.get(self.x,self.y)
        obs[36]=float(t==TerrainType.MOUNTAIN); obs[37]=float(t==TerrainType.FOREST)
        return obs

    def apply_action(self, action, terrain: TerrainMap):
        spd=self.effective_speed()*terrain.speed_mult(self.x,self.y)
        d=_ACT_DIRS[action]
        self.vx=self.vx*CFG.warrior_friction+d[0]*CFG.warrior_accel*spd
        self.vy=self.vy*CFG.warrior_friction+d[1]*CFG.warrior_accel*spd
        v=math.hypot(self.vx,self.vy)
        if v>spd: self.vx*=spd/v; self.vy*=spd/v
        self.x=float(np.clip(self.x+self.vx,0,CFG.world_w))
        self.y=float(np.clip(self.y+self.vy,0,CFG.world_h))
        if self.x<=0 or self.x>=CFG.world_w: self.vx*=-0.5
        if self.y<=0 or self.y>=CFG.world_h: self.vy*=-0.5
        self.energy-=CFG.warrior_metabolism+CFG.warrior_move_cost*(self.vx**2+self.vy**2)
        self.age+=1
        if self.energy<=0: self.alive=False

    def override_charge(self, enemies):
        ae=[e for e in enemies if e.alive]
        if not ae: return
        d,tgt=min([(math.hypot(e.x-self.x,e.y-self.y),e) for e in ae],key=lambda t:t[0])
        if d<=CFG.warrior_vision:
            dx=tgt.x-self.x; dy=tgt.y-self.y; dd=math.hypot(dx,dy)+1e-8
            spd=self.effective_speed()
            self.vx=(dx/dd)*spd; self.vy=(dy/dd)*spd
            self.x=float(np.clip(self.x+self.vx,0,CFG.world_w))
            self.y=float(np.clip(self.y+self.vy,0,CFG.world_h))

    def try_kill_enemies(self, enemies):
        killed=0
        for e in enemies:
            if e.alive and math.hypot(e.x-self.x,e.y-self.y)<=self.effective_kill_radius():
                e.alive=False; self.kills+=1
                self.energy=min(self.energy+CFG.warrior_attack_bonus,CFG.warrior_energy_max)
                killed+=1
        return killed

# ─────────────────────────────────────────────────────────────────────────────
#  ENEMY  (auto-curriculum adjusted)
# ─────────────────────────────────────────────────────────────────────────────
_EID=0
class Enemy:
    current_speed=CFG.enemy_max_speed_init; current_kr=CFG.enemy_kill_radius_init
    current_vision=CFG.enemy_vision_init; generation=1

    @classmethod
    def grow(cls, diff=1.0):
        rate=CFG.enemy_growth_rate*diff
        cls.current_speed =min(cls.current_speed*(1+rate), CFG.enemy_max_speed_cap)
        cls.current_kr    =min(cls.current_kr*(1+rate),    CFG.enemy_kill_radius_cap)
        cls.current_vision=min(cls.current_vision*(1+rate*0.5), 26.)
        cls.generation+=1

    @classmethod
    def reset(cls):
        cls.current_speed=CFG.enemy_max_speed_init; cls.current_kr=CFG.enemy_kill_radius_init
        cls.current_vision=CFG.enemy_vision_init; cls.generation=1

    def __init__(self,x,y):
        global _EID; _EID+=1
        self.id=_EID; self.x=float(x); self.y=float(y)
        self.vx=0.; self.vy=0.; self.alive=True; self.age=0

    @property
    def speed(self): return Enemy.current_speed
    @property
    def kill_radius(self): return Enemy.current_kr
    @property
    def vision(self): return Enemy.current_vision

    def step(self, organisms, warriors, all_enemies, villages, terrain: TerrainMap):
        ao=[o for o in organisms if o.alive]; aw=[w for w in warriors if w.alive]
        fx,fy=0.,0.
        for w in aw:
            d=math.hypot(w.x-self.x,w.y-self.y)
            if d<15.: rx=(self.x-w.x)/(d+1e-8); ry=(self.y-w.y)/(d+1e-8); fx+=rx*(15.-d); fy+=ry*(15.-d)
        flee_len=math.hypot(fx,fy)
        cx,cy=0.,0.
        # Target weakest visible organism
        cands=[(math.hypot(o.x-self.x,o.y-self.y),o) for o in ao
               if math.hypot(o.x-self.x,o.y-self.y)<=self.vision]
        if cands:
            cands.sort(key=lambda t:t[1].energy); tgt=cands[0][1]
            dx=tgt.x-self.x; dy=tgt.y-self.y; d=math.hypot(dx,dy)+1e-8; cx=dx/d; cy=dy/d
        elif ao:
            tgt=random.choice(ao); dx=tgt.x-self.x; dy=tgt.y-self.y
            d=math.hypot(dx,dy)+1e-8; cx=dx/d*0.35; cy=dy/d*0.35
        # Pack cohesion
        px,py=0.,0.
        pack=[e for e in all_enemies if e is not self and e.alive
              and math.hypot(e.x-self.x,e.y-self.y)<CFG.enemy_pack_radius]
        if pack:
            pcx=sum(e.x for e in pack)/len(pack); pcy=sum(e.y for e in pack)/len(pack)
            dd=math.hypot(pcx-self.x,pcy-self.y)+1e-8
            px=(pcx-self.x)/dd*0.18; py=(pcy-self.y)/dd*0.18
        if flee_len>0:
            fx/=flee_len; fy/=flee_len; mx=0.65*fx+0.25*cx+0.1*px; my=0.65*fy+0.25*cy+0.1*py
        else: mx=0.85*cx+0.15*px; my=0.85*cy+0.15*py
        ml=math.hypot(mx,my)+1e-8
        spd=self.speed*terrain.speed_mult(self.x,self.y)
        ax=(mx/ml)*spd; ay=(my/ml)*spd
        self.vx=self.vx*CFG.enemy_friction+ax*CFG.enemy_accel
        self.vy=self.vy*CFG.enemy_friction+ay*CFG.enemy_accel
        v=math.hypot(self.vx,self.vy)
        if v>spd: self.vx*=spd/v; self.vy*=spd/v
        self.x=float(np.clip(self.x+self.vx,0,CFG.world_w))
        self.y=float(np.clip(self.y+self.vy,0,CFG.world_h))
        if self.x<=0 or self.x>=CFG.world_w: self.vx*=-0.6
        if self.y<=0 or self.y>=CFG.world_h: self.vy*=-0.6
        self.age+=1
        for v in villages:
            if v.contains(self.x,self.y) and v.is_defended:
                dx=self.x-v.x; dy=self.y-v.y; dd=math.hypot(dx,dy)+1e-8
                push=CFG.village_radius+0.5
                self.x=float(np.clip(v.x+(dx/dd)*push,0,CFG.world_w))
                self.y=float(np.clip(v.y+(dy/dd)*push,0,CFG.world_h))
                self.vx*=-0.3; self.vy*=-0.3

    def try_kill(self, organisms, villages):
        killed=0; breach=0
        for o in organisms:
            if not o.alive: continue
            d=math.hypot(o.x-self.x,o.y-self.y)
            if d>self.kill_radius: continue
            in_v=False; v_def=False
            for v in villages:
                if v.contains(o.x,o.y): in_v=True; v_def=v.is_defended; break
            if in_v and v_def: continue
            elif in_v and not v_def:
                if random.random()<CFG.village_breach_kill_prob: o.alive=False; killed+=1; breach+=1
            else: o.alive=False; killed+=1
        return killed,breach

# ─────────────────────────────────────────────────────────────────────────────
#  SPECIATION TRACKER
# ─────────────────────────────────────────────────────────────────────────────
class SpeciesTracker:
    def __init__(self):
        self.species: Dict[int, Dict] = {}  # species_id → {centroid, count}
        self._next_id = 1

    def assign(self, genome: Genome) -> int:
        """Assign or create species for genome."""
        if not self.species:
            sid = self._next_id; self._next_id += 1
            self.species[sid] = {"centroid": genome, "count": 1}
            genome.species_id = sid; return sid
        best_d = float("inf"); best_sid = -1
        for sid, info in self.species.items():
            d = genome.distance(info["centroid"])
            if d < best_d: best_d = d; best_sid = sid
        if best_d > CFG.speciation_threshold:
            sid = self._next_id; self._next_id += 1
            self.species[sid] = {"centroid": copy.copy(genome), "count": 1}
            print(f"  [Species] New species {sid} diverged! dist={best_d:.2f}")
        else:
            sid = best_sid; self.species[sid]["count"] += 1
        genome.species_id = sid; return sid

    def prune(self, organisms):
        """Update species counts."""
        counts: Dict[int,int] = defaultdict(int)
        for o in organisms: counts[o.genome.species_id] += 1
        self.species = {sid: {"centroid": info["centroid"], "count": counts.get(sid,0)}
                        for sid, info in self.species.items() if counts.get(sid,0) > 0}

# ─────────────────────────────────────────────────────────────────────────────
#  NATURAL DISASTER MANAGER
# ─────────────────────────────────────────────────────────────────────────────
class DisasterManager:
    def __init__(self):
        self.active: List[Dict] = []

    def tick(self, world, season_mult: float):
        # Spawn new disasters
        base_prob = CFG.disaster_prob
        if season_mult < 0.8: base_prob *= 2.0  # winter more dangerous
        if random.random() < base_prob:
            dtype = random.choice(list(DisasterType))
            cx = random.uniform(20, CFG.world_w-20)
            cy = random.uniform(20, CFG.world_h-20)
            r  = random.uniform(15, 35)
            self.active.append({
                "type": dtype, "x": cx, "y": cy, "radius": r,
                "timer": CFG.disaster_duration,
                "intensity": random.uniform(0.5, 1.5)
            })
            print(f"  [Disaster] {dtype.name} at ({cx:.0f},{cy:.0f}) r={r:.0f}")

        # Apply disasters
        for d in self.active:
            d["timer"] -= 1
            if d["type"] == DisasterType.DROUGHT:
                # Kill food in radius
                world.food = [f for f in world.food
                              if math.hypot(f[0]-d["x"],f[1]-d["y"]) > d["radius"]]
            elif d["type"] == DisasterType.FLOOD:
                # Damage orgs in radius
                for o in world.organisms:
                    if o.alive and math.hypot(o.x-d["x"],o.y-d["y"]) < d["radius"]:
                        o.energy -= 0.15 * d["intensity"]
                        if o.energy<=0: o.alive=False
            elif d["type"] == DisasterType.FIRE:
                # Kill food and damage anyone in radius
                world.food = [f for f in world.food
                              if math.hypot(f[0]-d["x"],f[1]-d["y"]) > d["radius"]*0.5]
                for o in world.organisms:
                    if o.alive and math.hypot(o.x-d["x"],o.y-d["y"]) < d["radius"]*0.7:
                        o.energy -= 0.08 * d["intensity"]
                        if o.energy<=0: o.alive=False
            elif d["type"] == DisasterType.PLAGUE:
                for o in world.organisms:
                    if o.alive and math.hypot(o.x-d["x"],o.y-d["y"]) < d["radius"]:
                        if not o.sick and random.random() < 0.03*(1-o.genome.immune_strength):
                            o.sick=True; o.sick_timer=CFG.sick_duration

        self.active = [d for d in self.active if d["timer"] > 0]

# ─────────────────────────────────────────────────────────────────────────────
#  GLOBAL STATE  (centralised critic input)
# ─────────────────────────────────────────────────────────────────────────────
def build_global_state(world) -> np.ndarray:
    W,H=CFG.world_w,CFG.world_h
    ao=world.organisms; aw=world.warriors; ae=world.enemies
    gs=np.zeros(CFG.global_state_dim,dtype=np.float32)
    gs[0]=min(len(ao)/CFG.max_organisms,1.);    gs[1]=min(len(aw)/CFG.max_warriors,1.)
    gs[2]=min(len(ae)/CFG.max_enemies,1.);      gs[3]=min(len(world.food)/CFG.max_food,1.)
    gs[4]=min(len(world.villages)/CFG.max_villages,1.)
    gs[5]=Enemy.current_speed/CFG.enemy_max_speed_cap
    gs[6]=Enemy.current_kr/CFG.enemy_kill_radius_cap
    gs[7]=math.sin(world.season_phase*2*math.pi); gs[8]=math.cos(world.season_phase*2*math.pi)
    if ao:
        gs[9]=np.mean([o.x for o in ao])/W;     gs[10]=np.mean([o.y for o in ao])/H
        gs[11]=np.mean([o.energy for o in ao])/CFG.max_energy
        gs[12]=np.mean([o.genome.speed for o in ao])/4.5
        gs[13]=float(sum(1 for o in ao if o.sex==Sex.MALE))/max(len(ao),1)
        gs[14]=float(sum(1 for o in ao if o.pregnant))/max(len(ao),1)
        gs[15]=float(sum(1 for o in ao if o.is_elder))/max(len(ao),1)
        gs[16]=float(sum(1 for o in ao if o.sick))/max(len(ao),1)
        # Profession distribution
        for pi, prof in enumerate(Profession):
            gs[17+pi]=sum(1 for o in ao if o.profession==prof)/max(len(ao),1)
    if aw:
        gs[22]=np.mean([w.energy for w in aw])/CFG.warrior_energy_max
        gs[23]=float(sum(1 for w in aw if w.is_war_chief))/max(len(aw),1)
    gs[24]=min(len(world.species_tracker.species)/10.,1.)
    gs[25]=float(len(world.disaster_mgr.active))/5.
    gs[26]=world.op_policy.curriculum_difficulty if world.op_policy else 1.0
    if ae:
        gs[27]=np.mean([e.x for e in ae])/W; gs[28]=np.mean([e.y for e in ae])/H
    gs[29]=min(sum(v.resources[2] for v in world.villages)/100.,1.)   # total metal
    gs[30]=min(sum(v.resources[3] for v in world.villages)/50.,1.)    # total medicine
    gs[31]=float(world.step_count)/CFG.max_steps
    return gs

# ─────────────────────────────────────────────────────────────────────────────
#  WORLD
# ─────────────────────────────────────────────────────────────────────────────
class World:
    def __init__(self):
        Enemy.reset()
        self.terrain   = TERRAIN
        self.villages: List[Village]  = []
        self.organisms: List[Organism]= []
        self.warriors:  List[Warrior] = []
        self.enemies:   List[Enemy]   = []
        self.food:      List[Tuple]   = []
        self.bad_zones: List[Tuple]   = []
        self.step_count=0; self._last_village_step=-CFG.clan_form_cooldown
        self.season_phase=0.0
        self.species_tracker = SpeciesTracker()
        self.disaster_mgr    = DisasterManager()
        self.op_policy = None   # set by run()
        self.stats = {k:[] for k in [
            "pop","pop_m","pop_f","warriors","enemies","food",
            "in_village","n_villages","pregnancies","elders","sick",
            "births_org","births_war","deaths_starve","deaths_killed",
            "deaths_breach","deaths_war","enemy_kills","trade_trips",
            "avg_spd_m","avg_spd_f","avg_vision","avg_gen","avg_immune","avg_cooperation",
            "enemy_speed","enemy_kr","reward_org","reward_war","season",
            "n_species","n_disasters","wm_loss","curriculum_diff",
        ]}
        self._init()

    def _rp(self, margin=10):
        return (random.uniform(margin,CFG.world_w-margin),
                random.uniform(margin,CFG.world_h-margin))

    def _init(self):
        positions=[(CFG.world_w*0.28,CFG.world_h*0.5),(CFG.world_w*0.72,CFG.world_h*0.5)]
        for i in range(min(CFG.init_villages,2)):
            v=Village(*positions[i],clan_id=i+1); v.founded_step=0
            self.villages.append(v)
        att=0
        while len(self.bad_zones)<CFG.n_bad_zones and att<300:
            x,y=self._rp()
            if all(math.hypot(x-v.x,y-v.y)>CFG.village_radius+CFG.bad_zone_radius+10
                   for v in self.villages):
                self.bad_zones.append((x,y))
            att+=1
        for _ in range(CFG.init_food):
            x,y=self._rp()
            self.food.append((x,y))
        for vi,v in enumerate(self.villages):
            per=CFG.init_organisms//len(self.villages)
            for k in range(per):
                ang=random.uniform(0,2*math.pi); r=random.uniform(0,CFG.village_radius*0.85)
                x=float(np.clip(v.x+r*math.cos(ang),0,CFG.world_w))
                y=float(np.clip(v.y+r*math.sin(ang),0,CFG.world_h))
                sex=Sex.MALE if k%2==0 else Sex.FEMALE
                org=Organism(x,y,sex=sex,clan_id=v.clan_id,culture=v.culture.copy())
                org.home_village=v.id
                self.species_tracker.assign(org.genome)
                self.organisms.append(org)
            for k in range(CFG.init_warriors//len(self.villages)):
                ang=random.uniform(0,2*math.pi); r=random.uniform(0,CFG.village_radius)
                x=float(np.clip(v.x+r*math.cos(ang),0,CFG.world_w))
                y=float(np.clip(v.y+r*math.sin(ang),0,CFG.world_h))
                self.warriors.append(Warrior(x,y,home_village=v.id,clan_id=v.clan_id))
        for _ in range(CFG.init_enemies):
            for _ in range(100):
                edge=random.choice(["T","B","L","R"])
                if   edge=="T": x,y=random.uniform(0,CFG.world_w),random.uniform(0,8)
                elif edge=="B": x,y=random.uniform(0,CFG.world_w),random.uniform(CFG.world_h-8,CFG.world_h)
                elif edge=="L": x,y=random.uniform(0,8),random.uniform(0,CFG.world_h)
                else:           x,y=random.uniform(CFG.world_w-8,CFG.world_w),random.uniform(0,CFG.world_h)
                if all(math.hypot(x-v.x,y-v.y)>CFG.village_radius+12 for v in self.villages):
                    self.enemies.append(Enemy(x,y)); break

    def _update_season(self):
        self.season_phase=(self.step_count%CFG.season_period)/CFG.season_period
        return 1.0+CFG.season_amp*math.sin(self.season_phase*2*math.pi)

    def _update_defence(self, aw):
        for v in self.villages:
            v.is_defended=any(math.hypot(w.x-v.x,w.y-v.y)<=CFG.village_breach_radius for w in aw)

    def _auto_respawn_warriors(self, aw):
        for v in self.villages:
            local=[w for w in aw if w.home_village==v.id]
            if (len(local)<CFG.min_warriors_per_village
                    and self.step_count-v.last_warrior_spawn>=CFG.warrior_respawn_interval
                    and len(self.warriors)<CFG.max_warriors):
                ang=random.uniform(0,2*math.pi); r=random.uniform(0,CFG.village_radius*0.6)
                x=float(np.clip(v.x+r*math.cos(ang),0,CFG.world_w))
                y=float(np.clip(v.y+r*math.sin(ang),0,CFG.world_h))
                self.warriors.append(Warrior(x,y,energy=CFG.warrior_energy_init*0.8,
                                             home_village=v.id,clan_id=v.clan_id))
                v.last_warrior_spawn=self.step_count

    def _elect_war_chief(self, aw):
        for v in self.villages:
            vw=[w for w in aw if w.home_village==v.id]
            if not vw: continue
            best=max(vw,key=lambda w:w.kills)
            for w in vw: w.is_war_chief=False
            if best.kills>4: best.is_war_chief=True

    def _formation_combat(self, aw):
        """Assign warriors to squads, coordinate flanking."""
        squads: Dict[int, List[Warrior]] = defaultdict(list)
        for w in aw:
            if w.home_village is not None:
                squads[w.home_village].append(w)
        for vid, squad in squads.items():
            if len(squad) < 2: continue
            # Find centroid of squad
            cx = sum(w.x for w in squad)/len(squad)
            cy = sum(w.y for w in squad)/len(squad)
            # Assign flanking positions (surround nearest enemy)
            ae=[e for e in self.enemies if e.alive]
            if not ae: continue
            nearest_e = min(ae, key=lambda e:math.hypot(e.x-cx,e.y-cy))
            angles = [2*math.pi*i/len(squad) for i in range(len(squad))]
            for w, ang in zip(squad, angles):
                # Target position: surround enemy
                tx = nearest_e.x + math.cos(ang)*CFG.warrior_kill_radius*2
                ty = nearest_e.y + math.sin(ang)*CFG.warrior_kill_radius*2
                dx = tx-w.x; dy = ty-w.y; d=math.hypot(dx,dy)+1e-8
                if d>2:
                    spd=w.effective_speed()*0.8
                    w.vx += (dx/d)*spd*0.3; w.vy += (dy/d)*spd*0.3

    def _spread_disease(self, ao):
        for v in self.villages:
            local=[o for o in ao if o.alive and v.contains(o.x,o.y)]
            density=len(local)/max(1,(math.pi*CFG.village_radius**2))
            if density>0.10 and random.random()<0.0018:
                v.plague_active=True; v.plague_timer=220
            if v.plague_active:
                v.plague_timer-=1
                if v.plague_timer<=0: v.plague_active=False
                for o in local:
                    if not o.sick and random.random()<0.04*(1.-o.genome.immune_strength):
                        o.sick=True; o.sick_timer=CFG.sick_duration
            # Healers reduce sick count
            healers=[o for o in local if o.profession==Profession.HEALER and o.alive]
            for h in healers:
                nearby_sick=[o for o in local if o.sick and o.id!=h.id
                             and math.hypot(o.x-h.x,o.y-h.y)<8.]
                for s in nearby_sick[:2]:
                    s.sick_timer = max(0, s.sick_timer - 15)

    def _process_trade(self, ao) -> int:
        trips=0
        for o in ao:
            if not o.alive or not o.carrying_food: continue
            if o.trade_destination is None: o.carrying_food=False; continue
            dv=[v for v in self.villages if v.id==o.trade_destination]
            if not dv: o.carrying_food=False; o.trade_destination=None; continue
            v=dv[0]
            if v.dist(o.x,o.y)<=CFG.village_radius:
                v.resources[0]+=1.; o.carrying_food=False
                o.trade_destination=None
                bonus=CFG.trade_reward*o.profession_bonus().get("trade_reward",1.0)
                o.energy=min(o.energy+bonus,CFG.max_energy); trips+=1
        for o in ao:
            if not o.alive or o.carrying_food or o.pregnant: continue
            hv=[v for v in self.villages if v.id==o.home_village]
            if not hv: continue
            v=hv[0]
            if (random.random()<0.0025*o.genome.trade_drive*v.policy_trade
                    and len(self.villages)>1):
                others=[vv for vv in self.villages if vv.id!=v.id]
                if others:
                    o.carrying_food=True; o.trade_destination=random.choice(others).id
        return trips

    def _cultural_transmission(self, parent: Organism, child: Organism):
        """Child inherits blend of parent culture and village culture."""
        hv=[v for v in self.villages if v.id==parent.home_village]
        vil_culture = hv[0].culture if hv else np.random.rand(8).astype(np.float32)
        alpha = parent.genome.cultural_memory
        child.culture = (alpha*parent.culture + (1-alpha)*vil_culture
                         + np.random.randn(8).astype(np.float32)*0.02)
        child.culture = np.clip(child.culture, 0, 1)

    def _try_form_village(self, ao):
        if len(self.villages)>=CFG.max_villages: return
        if self.step_count-self._last_village_step<CFG.clan_form_cooldown: return
        for _ in range(80):
            cx=random.uniform(25,CFG.world_w-25); cy=random.uniform(25,CFG.world_h-25)
            if any(math.hypot(cx-v.x,cy-v.y)<CFG.village_radius*3.0 for v in self.villages): continue
            if any(math.hypot(cx-z[0],cy-z[1])<CFG.bad_zone_radius+CFG.village_radius for z in self.bad_zones): continue
            if self.terrain.get(cx,cy) in [TerrainType.MOUNTAIN, TerrainType.DESERT]: continue
            nearby=[o for o in ao if math.hypot(o.x-cx,o.y-cy)<=CFG.clan_form_radius]
            if len(nearby)>=CFG.clan_form_density:
                rcx=sum(o.x for o in nearby)/len(nearby); rcy=sum(o.y for o in nearby)/len(nearby)
                cc={}
                for o in nearby: c=o.clan_id or 0; cc[c]=cc.get(c,0)+1
                dom=max(cc,key=lambda k:cc[k])
                nv=Village(rcx,rcy,clan_id=dom); nv.founded_step=self.step_count
                # Inherit dominant culture
                dom_orgs=[o for o in nearby if o.clan_id==dom]
                if dom_orgs: nv.culture=np.mean([o.culture for o in dom_orgs],axis=0).astype(np.float32)
                self.villages.append(nv)
                for o in nearby:
                    if o.home_village is None or random.random()<0.5:
                        o.home_village=nv.id; o.clan_id=dom
                self._last_village_step=self.step_count
                print(f"  [Village+] ({rcx:.0f},{rcy:.0f}) clan={dom} governance={nv.governance.name} n={len(nearby)} total={len(self.villages)}")
                return

    def _handle_migration(self, ao):
        for v in self.villages:
            local=[o for o in ao if o.alive and v.contains(o.x,o.y)]
            if len(local)>CFG.overpop_threshold:
                migrants=random.sample(local,len(local)//4)
                candidates=[vv for vv in self.villages if vv.id!=v.id]
                if not candidates: continue
                target=random.choice(candidates)
                for m in migrants: m.home_village=target.id; m.clan_id=target.clan_id

    def _spawn_food(self, season_mult):
        ratio=len(self.food)/CFG.max_food
        for _ in range(3):  # try 3 spots per step
            if len(self.food)>=CFG.max_food: break
            x,y=self._rp()
            rate=CFG.food_spawn_rate*season_mult*max(0.,1.-ratio**1.3)*self.terrain.food_mult(x,y)
            if random.random()<rate/3.:
                self.food.append((x,y))

    def _org_threshold(self):
        fr=len(self.food)/max(CFG.max_food,1); return CFG.reproduce_threshold+(1.-fr)*22.

    def _org_rew(self, o, ate, in_bad, near_enemy, in_village, near_warrior, v_def,
                 curiosity, near_elder, near_healer):
        r  = 3.5 if ate else 0.
        r -= (2.5*(1.-o.genome.immune_strength)) if in_bad else 0.
        r -= 1.8 if (near_enemy and not in_village) else 0.
        r -= 0.6 if (near_enemy and in_village and not v_def) else 0.
        r += (1.0 if v_def else 0.4) if in_village else 0.
        r += 0.4 if near_warrior else 0.
        r += 0.6 if near_elder   else 0.
        r += 0.3 if near_healer  else 0.
        r += curiosity
        r += 0.09
        r -= 0.12*max(o.genome.speed-2.3,0)
        r -= 0.5 if o.sick else 0.
        # Profession-specific bonuses
        if o.profession==Profession.EXPLORER: r += 0.3
        if o.profession==Profession.TRADER and o.carrying_food: r += 0.2
        return float(r)

    def _war_rew(self, w, near_orgs, in_bad, v_undef, kills):
        r = 12.0*kills + 0.5*near_orgs
        r -= 2.5 if in_bad else 0.
        r += 2.5 if v_undef else 0.
        r += 0.12; return float(r)

    def _try_mate(self, ao, threshold):
        births=0
        males  =[o for o in ao if o.sex==Sex.MALE  and o.energy>=threshold and o.mate_timer==0]
        females=[o for o in ao if o.sex==Sex.FEMALE and o.energy>=threshold
                 and not o.pregnant and o.mate_timer==0]
        elders =[o for o in ao if o.is_elder]
        for f in females:
            if len(self.organisms)>=CFG.max_organisms: break
            near_elder=any(math.hypot(e.x-f.x,e.y-f.y)<=CFG.village_radius for e in elders)
            boost=CFG.elder_birth_bonus if near_elder else 0.
            if random.random()>f.genome.fertility+boost: continue
            cands=[(math.hypot(m.x-f.x,m.y-f.y),m) for m in males
                   if math.hypot(m.x-f.x,m.y-f.y)<=f.genome.vision*1.3]
            if not cands: continue
            _,father=min(cands,key=lambda t:t[0])
            f.pregnant=True; f.gestation=CFG.gestation_steps
            f.father_genome=father.genome
            f.energy-=CFG.reproduce_cost*0.58; father.energy-=CFG.reproduce_cost*0.42
            f.mate_timer=55; father.mate_timer=32
            if father in males: males.remove(father)
        for f in [o for o in ao if o.sex==Sex.FEMALE and o.pregnant]:
            f.gestation-=1
            if f.gestation<=0:
                f.pregnant=False
                if f.father_genome is None: continue
                csex=random.choice([Sex.MALE,Sex.FEMALE])
                cg=f.genome.crossover(f.father_genome,csex)
                child=Organism(
                    x=float(np.clip(f.x+random.uniform(-5,5),0,CFG.world_w)),
                    y=float(np.clip(f.y+random.uniform(-5,5),0,CFG.world_h)),
                    genome=cg,sex=csex,energy=CFG.reproduce_cost*0.42,
                    generation=f.generation+1,clan_id=f.clan_id)
                child.home_village=f.home_village
                self._cultural_transmission(f, child)
                self.species_tracker.assign(child.genome)
                f.father_genome=None
                self.organisms.append(child); births+=1
        return births

    def _reproduce_war(self, war):
        if war.energy<CFG.warrior_rep_threshold or len(self.warriors)>=CFG.max_warriors: return 0
        war.energy-=CFG.warrior_rep_cost
        self.warriors.append(Warrior(
            x=float(np.clip(war.x+random.uniform(-5,5),0,CFG.world_w)),
            y=float(np.clip(war.y+random.uniform(-5,5),0,CFG.world_h)),
            energy=CFG.warrior_rep_cost*0.5,
            home_village=war.home_village,clan_id=war.clan_id)); return 1

    def _enemy_dynamics(self, curriculum_diff):
        if self.step_count%CFG.enemy_growth_interval==0 and self.step_count>0:
            Enemy.grow(diff=curriculum_diff)
            print(f"  [Enemy↑] Spd={Enemy.current_speed:.2f} KR={Enemy.current_kr:.2f} Gen={Enemy.generation} diff={curriculum_diff:.2f}")
        if (self.step_count%CFG.enemy_spawn_interval==0 and self.step_count>0
                and len(self.enemies)<CFG.max_enemies):
            for _ in range(100):
                edge=random.choice(["T","B","L","R"])
                if   edge=="T": x,y=random.uniform(0,CFG.world_w),0.
                elif edge=="B": x,y=random.uniform(0,CFG.world_w),float(CFG.world_h)
                elif edge=="L": x,y=0.,random.uniform(0,CFG.world_h)
                else:           x,y=float(CFG.world_w),random.uniform(0,CFG.world_h)
                if all(math.hypot(x-v.x,y-v.y)>CFG.village_radius+6 for v in self.villages):
                    self.enemies.append(Enemy(x,y))
                    print(f"  [Enemy+] ({x:.0f},{y:.0f}) total={len(self.enemies)}"); break

    # ── MAIN STEP ─────────────────────────────────────────────────────────
    def step(self, op: CivPolicy, wp: CivPolicy) -> dict:
        self.step_count+=1
        self.op_policy = op
        season_mult=self._update_season()
        ao=[o for o in self.organisms if o.alive]
        aw=[w for w in self.warriors  if w.alive]
        ae=[e for e in self.enemies   if e.alive]
        births_o=0;births_w=0;d_starve=0;d_killed=0;d_breach=0;d_war=0;e_kills=0;trade=0
        rew_o=0.;rew_w=0.

        self._update_defence(aw)
        self._auto_respawn_warriors(aw)
        self._elect_war_chief(aw)
        for v in self.villages:
            v.update_governance(ao, aw)
            v.culture_drift()
        self._formation_combat(aw)

        gs = build_global_state(self)

        # ── Nearby agents for communication ───────────────────────────────
        def get_nearby_ids(agent, all_agents, radius):
            return [a.id for a in all_agents
                    if a.id != agent.id and math.hypot(a.x-agent.x,a.y-agent.y)<=radius]

        # ── Organism MAPPO act ─────────────────────────────────────────────
        elders  = [o for o in ao if o.is_elder]
        healers = [o for o in ao if o.profession==Profession.HEALER]
        obs_o   = [o.observe(self.food,ae,aw,self.villages,self.bad_zones,
                             elders,self.season_phase,self.terrain,
                             np.zeros(CFG.msg_dim,dtype=np.float32)) for o in ao]
        ids_o   = [o.id for o in ao]
        gs_list = [gs]*len(ao)
        nearby_o= [get_nearby_ids(o,ao,20.0) for o in ao]

        if obs_o:
            acts_o,logps_o,vals_o,hxs_o,goal_es,msg_es=op.batch_act(obs_o,ids_o,gs_list,nearby_o)
            for i,o in enumerate(ao):
                o._obs=obs_o[i]; o._act=int(acts_o[i])
                o._logp=float(logps_o[i]); o._val=float(vals_o[i])
        else:
            acts_o=logps_o=vals_o=hxs_o=goal_es=msg_es=[]

        # ── Warrior MAPPO act ──────────────────────────────────────────────
        obs_w   = [w.observe(ae,ao,self.villages,self.bad_zones,self.season_phase,self.terrain) for w in aw]
        ids_w   = [w.id for w in aw]
        gs_wlist= [gs]*len(aw)
        nearby_w= [get_nearby_ids(w,aw,20.) for w in aw]

        if obs_w:
            acts_w,logps_w,vals_w,hxs_w,goal_ew,msg_ew=wp.batch_act(obs_w,ids_w,gs_wlist,nearby_w)
            for i,w in enumerate(aw):
                w._obs=obs_w[i]; w._act=int(acts_w[i])
                w._logp=float(logps_w[i]); w._val=float(vals_w[i])
        else:
            acts_w=logps_w=vals_w=hxs_w=goal_ew=msg_ew=[]

        # ── Move organisms ────────────────────────────────────────────────
        for oi, o in enumerate(ao):
            fled=False
            for e in ae:
                d=math.hypot(e.x-o.x,e.y-o.y)
                if d<o.genome.vision*0.52:
                    dx=o.x-e.x;dy=o.y-e.y;dd=math.hypot(dx,dy)+1e-8
                    fx=dx/dd;fy=dy/dd;best=8;bdot=-99
                    for ai in range(8):
                        dot=_ACT_DIRS[ai][0]*fx+_ACT_DIRS[ai][1]*fy
                        if dot>bdot: bdot=dot;best=ai
                    o._act=best;fled=True;break
            # Trade movement
            if o.carrying_food and o.trade_destination and not fled:
                dv=[v for v in self.villages if v.id==o.trade_destination]
                if dv:
                    v=dv[0];dx=v.x-o.x;dy=v.y-o.y;dd=math.hypot(dx,dy)+1e-8
                    best=8;bdot=-99
                    for ai in range(8):
                        dot=_ACT_DIRS[ai][0]*(dx/dd)+_ACT_DIRS[ai][1]*(dy/dd)
                        if dot>bdot:bdot=dot;best=ai
                    o._act=best
            # Foraging
            in_v=any(v.contains(o.x,o.y) for v in self.villages)
            if in_v and o.energy>CFG.reproduce_threshold*0.72 and not fled:
                vis_food=[f for f in self.food if math.hypot(f[0]-o.x,f[1]-o.y)<=CFG.fog_radius]
                of=[(math.hypot(f[0]-o.x,f[1]-o.y),f) for f in vis_food
                    if not any(v.contains(f[0],f[1]) for v in self.villages)]
                if of:
                    _,tf=min(of,key=lambda t:t[0])
                    dx=tf[0]-o.x;dy=tf[1]-o.y;dd=math.hypot(dx,dy)+1e-8
                    best=8;bdot=-99
                    for ai in range(8):
                        dot=_ACT_DIRS[ai][0]*(dx/dd)+_ACT_DIRS[ai][1]*(dy/dd)
                        if dot>bdot:bdot=dot;best=ai
                    o._act=best
            # Return home
            hv=[v for v in self.villages if v.id==o.home_village]
            if hv and o.energy<CFG.init_energy*0.30 and not fled and not o.carrying_food:
                v=hv[0];dx=v.x-o.x;dy=v.y-o.y;dd=math.hypot(dx,dy)+1e-8
                best=8;bdot=-99
                for ai in range(8):
                    dot=_ACT_DIRS[ai][0]*(dx/dd)+_ACT_DIRS[ai][1]*(dy/dd)
                    if dot>bdot:bdot=dot;best=ai
                o._act=best
            o.apply_action(o._act, self.terrain)
            if not o.alive: d_starve+=1

        # ── Move warriors ─────────────────────────────────────────────────
        for w in aw:
            w.apply_action(w._act, self.terrain); w.override_charge(ae)
            if not w.alive: d_war+=1

        # ── Move enemies ──────────────────────────────────────────────────
        for e in ae: e.step(ao,aw,ae,self.villages,self.terrain)

        # ── Eat food ──────────────────────────────────────────────────────
        food_set=list(self.food); taken=set()
        for o in ao:
            if not o.alive or o.carrying_food: continue
            for fi,f in enumerate(food_set):
                if fi in taken: continue
                er=(CFG.village_food_radius if any(v.contains(o.x,o.y) for v in self.villages)
                    else CFG.food_eat_radius)
                bonus=o.profession_bonus().get("food_mult",1.0)
                if math.hypot(f[0]-o.x,f[1]-o.y)<=er:
                    o.energy=min(o.energy+CFG.food_energy*bonus,CFG.max_energy)
                    taken.add(fi); break
        for w in aw:
            if not w.alive: continue
            for fi,f in enumerate(food_set):
                if fi in taken: continue
                if math.hypot(f[0]-w.x,f[1]-w.y)<=CFG.food_eat_radius:
                    w.energy=min(w.energy+CFG.warrior_food_energy,CFG.warrior_energy_max)
                    taken.add(fi); break
        self.food=[f for i,f in enumerate(food_set) if i not in taken]

        # ── Warriors kill enemies ─────────────────────────────────────────
        for w in aw:
            if w.alive: e_kills+=w.try_kill_enemies(ae)

        # ── Bad zone + disaster damage ─────────────────────────────────────
        for o in ao:
            if o.alive and any(math.hypot(z[0]-o.x,z[1]-o.y)<=CFG.bad_zone_radius for z in self.bad_zones):
                dmg=CFG.bad_zone_damage*(1.-o.genome.immune_strength*0.5)
                o.energy-=dmg
                if o.energy<=0: o.alive=False; d_starve+=1

        # ── Enemies kill organisms ────────────────────────────────────────
        for e in ae:
            if e.alive: k,br=e.try_kill(ao,self.villages); d_killed+=k; d_breach+=br

        # ── Systems ───────────────────────────────────────────────────────
        self._spread_disease(ao)
        trade=self._process_trade(ao)
        self.disaster_mgr.tick(self, season_mult)

        # ── World model store ─────────────────────────────────────────────
        for oi, o in enumerate(ao):
            if o._obs is not None and o.alive:
                op.world_model.store(o._obs, o._act, o.observe(
                    self.food,ae,aw,self.villages,self.bad_zones,
                    elders,self.season_phase,self.terrain,
                    np.zeros(CFG.msg_dim,dtype=np.float32)), 0.0)

        # ── MAPPO rewards (organisms) ─────────────────────────────────────
        threshold=self._org_threshold()
        for oi, o in enumerate(ao):
            if o._obs is None: continue
            hid=op._get_hidden(o.id)
            in_bad=any(math.hypot(z[0]-o.x,z[1]-o.y)<=CFG.bad_zone_radius for z in self.bad_zones)
            n_enemy=any(math.hypot(e.x-o.x,e.y-o.y)<=o.genome.vision for e in ae if e.alive)
            in_vill=any(v.contains(o.x,o.y) for v in self.villages)
            n_war=any(math.hypot(w.x-o.x,w.y-o.y)<=CFG.warrior_protect_r for w in aw if w.alive)
            v_def=next((v.is_defended for v in self.villages if v.contains(o.x,o.y)),True)
            n_elder=any(math.hypot(e.x-o.x,e.y-o.y)<=CFG.village_radius for e in elders if e.id!=o.id)
            n_healer=any(math.hypot(h.x-o.x,h.y-o.y)<=CFG.village_radius for h in healers if h.id!=o.id)
            ate=o.energy>o._prev_energy
            cur=op.intrinsic_reward(o.x,o.y)
            rew=self._org_rew(o,ate,in_bad,n_enemy,in_vill,n_war,v_def,cur,n_elder,n_healer)
            o.fitness+=rew; rew_o+=rew
            ge_np = goal_es[oi] if oi < len(goal_es) else np.zeros(CFG.goal_embed_dim, dtype=np.float32)
            me_np = msg_es[oi]  if oi < len(msg_es)  else np.zeros(CFG.msg_dim, dtype=np.float32)
            if isinstance(me_np, torch.Tensor): me_np=me_np.cpu().numpy()
            op.store(o._obs,o._act,o._logp,o._val,rew,not o.alive,hid,gs,ge_np,me_np)

        # ── MAPPO rewards (warriors) ──────────────────────────────────────
        for wi, w in enumerate(aw):
            if w._obs is None: continue
            hid=wp._get_hidden(w.id)
            in_bad=any(math.hypot(z[0]-w.x,z[1]-w.y)<=CFG.bad_zone_radius for z in self.bad_zones)
            n_orgs=sum(1 for o in ao if o.alive and math.hypot(o.x-w.x,o.y-w.y)<=CFG.warrior_protect_r)
            v_undef=any(not v.is_defended and v.id==w.home_village for v in self.villages)
            rew=self._war_rew(w,n_orgs,in_bad,v_undef,0)
            rew_w+=rew
            ge_np = goal_ew[wi] if wi < len(goal_ew) else np.zeros(CFG.goal_embed_dim,dtype=np.float32)
            me_np = msg_ew[wi]  if wi < len(msg_ew)  else np.zeros(CFG.msg_dim,dtype=np.float32)
            if isinstance(me_np, torch.Tensor): me_np=me_np.cpu().numpy()
            wp.store(w._obs,w._act,w._logp,w._val,rew,not w.alive,hid,gs,ge_np,me_np)

        # ── Reproduction ──────────────────────────────────────────────────
        births_o=self._try_mate(ao,threshold)
        for w in aw:
            if w.alive: births_w+=self._reproduce_war(w)

        # ── Clear dead agent states ───────────────────────────────────────
        for o in ao:
            if not o.alive: op.clear_agent(o.id)
        for w in aw:
            if not w.alive: wp.clear_agent(w.id)

        # ── Systems ───────────────────────────────────────────────────────
        self._spawn_food(season_mult)
        curriculum_diff = op.auto_curriculum()
        self._enemy_dynamics(curriculum_diff)
        self._try_form_village(self.organisms)
        self._handle_migration(self.organisms)
        self.species_tracker.prune(self.organisms)

        # ── Prune dead ────────────────────────────────────────────────────
        self.organisms=[o for o in self.organisms if o.alive]
        self.warriors =[w for w in self.warriors  if w.alive]
        self.enemies  =[e for e in self.enemies   if e.alive]

        # ── Stats ─────────────────────────────────────────────────────────
        ao2=self.organisms; aw2=self.warriors; ae2=self.enemies
        males=[o for o in ao2 if o.sex==Sex.MALE]; females=[o for o in ao2 if o.sex==Sex.FEMALE]
        in_v=sum(1 for o in ao2 if any(v.contains(o.x,o.y) for v in self.villages))
        preg=sum(1 for o in ao2 if o.pregnant); eld=sum(1 for o in ao2 if o.is_elder)
        sick=sum(1 for o in ao2 if o.sick)
        wm_l=float(np.mean(op.wm_loss_hist[-10:])) if op.wm_loss_hist else 0.
        self.stats["pop"].append(len(ao2)); self.stats["pop_m"].append(len(males))
        self.stats["pop_f"].append(len(females)); self.stats["warriors"].append(len(aw2))
        self.stats["enemies"].append(len(ae2)); self.stats["food"].append(len(self.food))
        self.stats["in_village"].append(in_v); self.stats["n_villages"].append(len(self.villages))
        self.stats["pregnancies"].append(preg); self.stats["elders"].append(eld); self.stats["sick"].append(sick)
        self.stats["births_org"].append(births_o); self.stats["births_war"].append(births_w)
        self.stats["deaths_starve"].append(d_starve); self.stats["deaths_killed"].append(d_killed)
        self.stats["deaths_breach"].append(d_breach); self.stats["deaths_war"].append(d_war)
        self.stats["enemy_kills"].append(e_kills); self.stats["trade_trips"].append(trade)
        self.stats["avg_spd_m"].append(np.mean([o.genome.speed for o in males]) if males else 0)
        self.stats["avg_spd_f"].append(np.mean([o.genome.speed for o in females]) if females else 0)
        self.stats["avg_vision"].append(np.mean([o.genome.vision for o in ao2]) if ao2 else 0)
        self.stats["avg_gen"].append(np.mean([o.generation for o in ao2]) if ao2 else 0)
        self.stats["avg_immune"].append(np.mean([o.genome.immune_strength for o in ao2]) if ao2 else 0)
        self.stats["avg_cooperation"].append(np.mean([o.genome.cooperation for o in ao2]) if ao2 else 0)
        self.stats["enemy_speed"].append(Enemy.current_speed); self.stats["enemy_kr"].append(Enemy.current_kr)
        self.stats["reward_org"].append(rew_o); self.stats["reward_war"].append(rew_w)
        self.stats["season"].append(season_mult); self.stats["n_species"].append(len(self.species_tracker.species))
        self.stats["n_disasters"].append(len(self.disaster_mgr.active))
        self.stats["wm_loss"].append(wm_l); self.stats["curriculum_diff"].append(curriculum_diff)
        return {"pop":len(ao2),"male":len(males),"female":len(females),
                "war":len(aw2),"enm":len(ae2),"food":len(self.food),
                "in_v":in_v,"n_v":len(self.villages),"preg":preg,"eld":eld,
                "births":births_o,"births_w":births_w,
                "killed":d_killed,"breach":d_breach,"d_war":d_war,
                "ekills":e_kills,"trade":trade,"rew_o":rew_o,"rew_w":rew_w,
                "n_species":len(self.species_tracker.species),"curriculum":curriculum_diff}

# ─────────────────────────────────────────────────────────────────────────────
#  RENDERER  (15-panel civilization dashboard)
# ─────────────────────────────────────────────────────────────────────────────
op_ref = []

class Renderer:
    def __init__(self):
        self.fig=plt.figure(figsize=(32,20),facecolor="#0a0c10")
        gs_layout=self.fig.add_gridspec(3,5,wspace=0.42,hspace=0.55)
        self.axes=[self.fig.add_subplot(gs_layout[r,c]) for r in range(3) for c in range(5)]
        os.makedirs(CFG.plot_dir,exist_ok=True) if CFG.save_plot else None
        self._frame=0

    def _sty(self,ax,title="",xl="",yl=""):
        ax.set_facecolor("#13161e"); ax.tick_params(colors="#7a8394",labelsize=6.5)
        ax.spines[:].set_color("#252a35")
        ax.set_title(title,color="#e8edf5",fontsize=7.5,pad=4,fontweight="bold")
        ax.set_xlabel(xl,color="#7a8394",fontsize=6.5); ax.set_ylabel(yl,color="#7a8394",fontsize=6.5)

    def render(self, world: World, step: int):
        for ax in self.axes: ax.cla()
        ao=world.organisms; aw=world.warriors
        ae=[e for e in world.enemies if e.alive]
        st=world.stats; xs=list(range(len(st["pop"])))
        sm=lambda s,n:(np.convolve(s,np.ones(n)/n,mode="same").tolist() if len(s)>=n else s)
        males=[o for o in ao if o.sex==Sex.MALE]; females=[o for o in ao if o.sex==Sex.FEMALE]
        elders=[o for o in ao if o.is_elder]; sick=[o for o in ao if o.sick]
        chiefs=[w for w in aw if w.is_war_chief]; preg=[o for o in ao if o.pregnant]
        traders=[o for o in ao if o.carrying_food]
        explorers=[o for o in ao if o.profession==Profession.EXPLORER]

        # ── [0] World Map ─────────────────────────────────────────────────
        ax=self.axes[0]; self._sty(ax,f"Civilization World [step {step}]","X","Y")
        ax.set_xlim(0,CFG.world_w); ax.set_ylim(0,CFG.world_h)

        # Terrain background
        tm=world.terrain
        for ty in range(tm.H):
            for tx in range(tm.W):
                t=tm.grid[ty,tx]
                col=TERRAIN_COLORS[t]
                rx=tx/tm.W*CFG.world_w; ry=ty/tm.H*CFG.world_h
                rw=CFG.world_w/tm.W; rh=CFG.world_h/tm.H
                ax.add_patch(plt.Rectangle((rx,ry),rw,rh,color=col,alpha=0.5,zorder=0))

        # Disaster zones
        for d in world.disaster_mgr.active:
            dcols={DisasterType.DROUGHT:"#8B4513",DisasterType.FLOOD:"#1a3aff",
                   DisasterType.FIRE:"#ff4500",DisasterType.PLAGUE:"#7f007f"}
            dc=dcols.get(d["type"],"#ff0000")
            ax.add_patch(plt.Circle((d["x"],d["y"]),d["radius"],color=dc,alpha=0.22,zorder=1))

        # Villages
        for v in world.villages:
            col="#30d158" if v.is_defended else "#ff9f0a"
            ax.add_patch(plt.Circle((v.x,v.y),CFG.territory_radius,color=col,alpha=0.04,zorder=1))
            ax.add_patch(plt.Circle((v.x,v.y),CFG.village_radius,color=col,alpha=0.15,zorder=2))
            ax.add_patch(plt.Circle((v.x,v.y),CFG.village_radius,fill=False,edgecolor=col,lw=1.8,alpha=0.8,zorder=3))
            txt=f"V{v.id}\n{v.governance.name[:4]}"+(f"\n⚠" if not v.is_defended else "")+(f"\n☣" if v.plague_active else "")
            ax.text(v.x,v.y,txt,color=col,fontsize=4.5,ha="center",va="center",zorder=4,fontweight="bold")

        # Bad zones
        for z in world.bad_zones:
            ax.add_patch(plt.Circle(z,CFG.bad_zone_radius,color="#ff3b30",alpha=0.15,zorder=2))
            ax.add_patch(plt.Circle(z,CFG.bad_zone_radius,fill=False,edgecolor="#ff3b30",lw=1,alpha=0.5,zorder=3))

        # Food
        if world.food:
            fx,fy=zip(*world.food); ax.scatter(fx,fy,s=3,c="#30d158",alpha=0.35,zorder=4,marker=".",linewidths=0)

        # Organisms by sex + profession marker
        PROF_MARKERS={Profession.FARMER:"o",Profession.TRADER:"s",
                      Profession.EXPLORER:"^",Profession.HEALER:"+",Profession.WARRIOR:"D"}
        if males:
            me=np.array([o.energy/CFG.max_energy for o in males]); ms=4+12*me
            ax.scatter([o.x for o in males],[o.y for o in males],s=ms,c="#0a84ff",alpha=0.78,zorder=5,linewidths=0.1)
        if females:
            fe=np.array([o.energy/CFG.max_energy for o in females]); fs=5+13*fe
            ax.scatter([o.x for o in females],[o.y for o in females],s=fs,c="#ff375f",alpha=0.78,zorder=5,linewidths=0.1)
        if preg: ax.scatter([o.x for o in preg],[o.y for o in preg],s=11,c="#ffd60a",zorder=6,marker="*",linewidths=0)
        if elders: ax.scatter([o.x for o in elders],[o.y for o in elders],s=14,c="#bf5af2",zorder=6,marker="D",linewidths=0)
        if sick: ax.scatter([o.x for o in sick],[o.y for o in sick],s=9,c="#ff9f0a",zorder=6,marker="x",linewidths=0.7)
        if traders: ax.scatter([o.x for o in traders],[o.y for o in traders],s=8,c="#ffd60a",zorder=6,marker="s",linewidths=0)
        if explorers: ax.scatter([o.x for o in explorers],[o.y for o in explorers],s=8,c="#64d2ff",zorder=6,marker="^",linewidths=0)

        if aw: ax.scatter([w.x for w in aw],[w.y for w in aw],s=40,c="#30d158",marker="D",zorder=7,edgecolors="#fff",lw=0.4,alpha=0.9)
        if chiefs: ax.scatter([w.x for w in chiefs],[w.y for w in chiefs],s=65,c="#ffd60a",marker="*",zorder=8)
        for e in ae:
            sz=min(60+40*(Enemy.generation-1),200)
            ax.scatter(e.x,e.y,s=sz,c="#ff453a",marker="^",zorder=9,edgecolors="#fff",lw=0.4)

        leg=[mpatches.Patch(color="#0a84ff",label=f"♂({len(males)})"),
             mpatches.Patch(color="#ff375f",label=f"♀({len(females)})"),
             mpatches.Patch(color="#ffd60a",label=f"Preg({len(preg)})"),
             mpatches.Patch(color="#bf5af2",label=f"Elder({len(elders)})"),
             mpatches.Patch(color="#ff9f0a",label=f"Sick({len(sick)})"),
             mpatches.Patch(color="#ffd60a",label=f"Trade({len(traders)})"),
             mpatches.Patch(color="#30d158",label=f"War({len(aw)})"),
             mpatches.Patch(color="#ff453a",label=f"Enm({len(ae)})")]
        ax.legend(handles=leg,loc="upper right",fontsize=4.2,
                  facecolor="#1a1d25",labelcolor="#e8edf5",edgecolor="#252a35",ncol=2)

        # ── [1] Population ─────────────────────────────────────────────────
        ax=self.axes[1]; self._sty(ax,"Population Dynamics","Step","Count")
        ax.plot(xs,st["pop"],color="#ffffff",lw=1.3,label="Total")
        ax.plot(xs,st["pop_m"],color="#0a84ff",lw=1.0,label="♂")
        ax.plot(xs,st["pop_f"],color="#ff375f",lw=1.0,label="♀")
        ax.plot(xs,st["warriors"],color="#30d158",lw=1.0,label="War")
        ax.plot(xs,st["enemies"],color="#ff453a",lw=0.9,alpha=0.7,label="Enm")
        ax.legend(fontsize=5,facecolor="#1a1d25",labelcolor="#e8edf5",edgecolor="#252a35",ncol=2)

        # ── [2] Reproduction & Health ──────────────────────────────────────
        ax=self.axes[2]; self._sty(ax,"Health & Reproduction","Step","Count")
        ax.plot(xs,sm(st["pregnancies"],10),color="#ffd60a",lw=1.2,label="Pregnant")
        ax.plot(xs,sm(st["births_org"],10), color="#30d158",lw=1.2,label="Births")
        ax.plot(xs,sm(st["sick"],10),       color="#ff9f0a",lw=1.0,label="Sick")
        ax.plot(xs,sm(st["elders"],5),      color="#bf5af2",lw=0.9,label="Elders")
        ax.legend(fontsize=5,facecolor="#1a1d25",labelcolor="#e8edf5",edgecolor="#252a35")

        # ── [3] Villages & Trade ───────────────────────────────────────────
        ax=self.axes[3]; self._sty(ax,"Villages & Trade","Step","Count")
        ax.plot(xs,st["n_villages"],color="#30d158",lw=1.4,label="Villages")
        ax.plot(xs,sm(st["in_village"],5),color="#64d2ff",lw=1.0,label="In Village")
        ax2=ax.twinx(); ax2.tick_params(colors="#7a8394",labelsize=6.5); ax2.spines[:].set_color("#252a35")
        ax2.plot(xs,sm(st["trade_trips"],10),color="#ff9f0a",lw=1.0,linestyle="--",label="Trades")
        ax.set_ylabel("Villages",color="#30d158",fontsize=6.5); ax2.set_ylabel("Trades",color="#ff9f0a",fontsize=6.5)
        h1,l1=ax.get_legend_handles_labels(); h2,l2=ax2.get_legend_handles_labels()
        ax.legend(h1+h2,l1+l2,fontsize=5,facecolor="#1a1d25",labelcolor="#e8edf5",edgecolor="#252a35")

        # ── [4] Species & Disasters ────────────────────────────────────────
        ax=self.axes[4]; self._sty(ax,"Species & Disasters","Step","Count")
        ax.plot(xs,st["n_species"],color="#bf5af2",lw=1.4,label="Species")
        ax2=ax.twinx(); ax2.tick_params(colors="#7a8394",labelsize=6.5); ax2.spines[:].set_color("#252a35")
        ax2.plot(xs,sm(st["n_disasters"],5),color="#ff453a",lw=1.0,linestyle="--",label="Disasters")
        ax.set_ylabel("Species",color="#bf5af2",fontsize=6.5); ax2.set_ylabel("Disasters",color="#ff453a",fontsize=6.5)
        h1,l1=ax.get_legend_handles_labels(); h2,l2=ax2.get_legend_handles_labels()
        ax.legend(h1+h2,l1+l2,fontsize=5,facecolor="#1a1d25",labelcolor="#e8edf5",edgecolor="#252a35")

        # ── [5] Deaths ─────────────────────────────────────────────────────
        ax=self.axes[5]; self._sty(ax,"Deaths & Combat","Step","Count")
        ax.plot(xs,sm(st["deaths_starve"],10),color="#ff9f0a",lw=1.0,label="Starve")
        ax.plot(xs,sm(st["deaths_killed"],10),color="#ff453a",lw=1.0,label="Killed")
        ax.plot(xs,sm(st["deaths_breach"],10),color="#ff375f",lw=0.9,linestyle="--",label="Breach")
        ax.plot(xs,sm(st["deaths_war"],10),   color="#0a84ff",lw=0.9,label="War☠")
        ax.plot(xs,sm(st["enemy_kills"],10),  color="#30d158",lw=1.1,label="Enm☠")
        ax.legend(fontsize=5,facecolor="#1a1d25",labelcolor="#e8edf5",edgecolor="#252a35")

        # ── [6] Enemy Evolution ────────────────────────────────────────────
        ax=self.axes[6]; self._sty(ax,"Enemy Evolution","Step","")
        ax2=ax.twinx(); ax2.tick_params(colors="#7a8394",labelsize=6.5); ax2.spines[:].set_color("#252a35")
        ax.plot(xs,st["enemy_speed"],color="#ff453a",lw=1.3,label="Speed")
        ax2.plot(xs,st["enemy_kr"],color="#ff9f0a",lw=1.1,linestyle="--",label="Kill R")
        ax.set_ylabel("Speed",color="#ff453a",fontsize=6.5); ax2.set_ylabel("Kill R",color="#ff9f0a",fontsize=6.5)
        h1,l1=ax.get_legend_handles_labels(); h2,l2=ax2.get_legend_handles_labels()
        ax.legend(h1+h2,l1+l2,fontsize=5,facecolor="#1a1d25",labelcolor="#e8edf5",edgecolor="#252a35")

        # ── [7] Gene: Speed dimorphism ─────────────────────────────────────
        ax=self.axes[7]; self._sty(ax,"Speed Gene ♂ vs ♀","Step","Speed")
        ax.plot(xs,st["avg_spd_m"],color="#0a84ff",lw=1.2,label="♂ Speed")
        ax.plot(xs,st["avg_spd_f"],color="#ff375f",lw=1.1,label="♀ Speed")
        ax.legend(fontsize=5,facecolor="#1a1d25",labelcolor="#e8edf5",edgecolor="#252a35")

        # ── [8] Cooperation & Immunity ─────────────────────────────────────
        ax=self.axes[8]; self._sty(ax,"Social Genes","Step","Value")
        ax.plot(xs,st["avg_cooperation"],color="#30d158",lw=1.2,label="Cooperation")
        ax.plot(xs,st["avg_immune"],     color="#64d2ff",lw=1.1,label="Immunity")
        ax.plot(xs,st["avg_vision"],     color="#ffd60a",lw=1.0,label="Vision")
        ax.legend(fontsize=5,facecolor="#1a1d25",labelcolor="#e8edf5",edgecolor="#252a35")

        # ── [9] Generation ─────────────────────────────────────────────────
        ax=self.axes[9]; self._sty(ax,"Avg Generation","Step","Gen")
        ax.plot(xs,st["avg_gen"],color="#ffd60a",lw=1.4)
        if xs: ax.fill_between(xs,st["avg_gen"],alpha=0.12,color="#ffd60a")

        # ── [10] Reward ────────────────────────────────────────────────────
        ax=self.axes[10]; self._sty(ax,"MAPPO Reward / Step","Step","Reward")
        ax.plot(xs,sm(st["reward_org"],25),color="#ff9f0a",lw=1.2,label="Org")
        ax.plot(xs,sm(st["reward_war"],25),color="#30d158",lw=1.1,label="Warrior")
        ax.axhline(0,color="#252a35",lw=0.8,linestyle="--")
        ax.legend(fontsize=5,facecolor="#1a1d25",labelcolor="#e8edf5",edgecolor="#252a35")

        # ── [11] World Model Loss ──────────────────────────────────────────
        ax=self.axes[11]; self._sty(ax,"World Model (Dreamer) Loss","Step","Loss")
        ax.plot(xs,sm(st["wm_loss"],20),color="#bf5af2",lw=1.2)
        if xs: ax.fill_between(xs,sm(st["wm_loss"],20),alpha=0.10,color="#bf5af2")

        # ── [12] Auto-Curriculum ───────────────────────────────────────────
        ax=self.axes[12]; self._sty(ax,"Auto-Curriculum Difficulty","Step","Difficulty")
        ax.plot(xs,st["curriculum_diff"],color="#ff9f0a",lw=1.3)
        ax.axhline(1.0,color="#252a35",lw=0.8,linestyle="--")
        ax2=ax.twinx(); ax2.tick_params(colors="#7a8394",labelsize=6.5); ax2.spines[:].set_color("#252a35")
        ax2.plot(xs,st["season"],color="#64d2ff",lw=0.9,linestyle=":",label="Season")
        ax2.set_ylabel("Season",color="#64d2ff",fontsize=6.5)

        # ── [13] Curiosity Heatmap ─────────────────────────────────────────
        ax=self.axes[13]; self._sty(ax,"Curiosity Visit Map (Exploration)","X","Y")
        if op_ref:
            vm=op_ref[0].visit_grid
            ax.imshow(vm,aspect="auto",cmap="inferno",origin="lower",
                      extent=[0,CFG.world_w,0,CFG.world_h],alpha=0.92)
        ax.set_xlim(0,CFG.world_w); ax.set_ylim(0,CFG.world_h)
        # Overlay village positions
        for v in world.villages:
            ax.plot(v.x,v.y,"w.",markersize=4,zorder=5)

        # ── [14] Civilization Summary ──────────────────────────────────────
        ax=self.axes[14]; self._sty(ax,"Civilization Status","","")
        ax.axis("off")
        op0=op_ref[0] if op_ref else None
        tc=sum(world.stats["trade_trips"]) if world.stats["trade_trips"] else 0
        wc=sum(w.kills for w in aw)
        govs={g.name:sum(1 for v in world.villages if v.governance==g) for g in Governance}
        spec_counts={sid:info["count"] for sid,info in world.species_tracker.species.items()}
        top_sids=sorted(spec_counts,key=spec_counts.get,reverse=True)[:3]
        lines=[
            f"Step: {step}",
            f"Births: {sum(st['births_org'])} | Kills: {sum(st['enemy_kills'])}",
            f"Trades: {tc} | Breach: {sum(st['deaths_breach'])}",
            f"Species: {len(world.species_tracker.species)}",
            f"  Top: {', '.join(f'S{s}({spec_counts[s]})' for s in top_sids)}",
            f"Villages: {len(world.villages)} | Enemy Gen: {Enemy.generation}",
            f"Governance: D{govs.get('DEMOCRACY',0)} A{govs.get('AUTOCRACY',0)} T{govs.get('THEOCRACY',0)}",
            f"Disasters active: {len(world.disaster_mgr.active)}",
            f"Curriculum: {world.stats['curriculum_diff'][-1]:.2f}" if world.stats['curriculum_diff'] else "",
            f"PBT active: C{op0.active_pbt if op0 else 0}",
            f"WM loss: {world.stats['wm_loss'][-1]:.4f}" if world.stats['wm_loss'] else "",
            f"Archive: {len(op0.archive) if op0 else 0} snapshots",
            f"NEAT best score: {max(op0.neat.scores):.3f}" if op0 else "",
            f"HRL goals: {len(set(op0.hrl.agent_goals.values())) if op0 else 0} active",
        ]
        for i,l in enumerate(lines):
            ax.text(0.04,0.97-i*0.067,l,transform=ax.transAxes,
                    color="#e8edf5",fontsize=7.2,va="top",family="monospace")

        self.fig.suptitle(
            f"Organism Living System v6  ·  MAPPO+GRU+PBT+HRL+WorldModel+NEAT+Comm  ·  Step {step}  |  "
            f"Orgs {len(ao)} (♂{len(males)} ♀{len(females)})  |  "
            f"Warriors {len(aw)}  |  Enemies {len(ae)} (Gen {Enemy.generation})  |  "
            f"Villages {len(world.villages)}  |  Species {len(world.species_tracker.species)}  |  "
            f"Food {len(world.food)}  |  Season {world.season_phase:.2f}",
            color="#e8edf5",fontsize=9,fontweight="bold",y=1.002)
        plt.tight_layout()
        if CFG.save_plot:
            self.fig.savefig(os.path.join(CFG.plot_dir,f"frame_{self._frame:05d}.png"),
                             dpi=95,bbox_inches="tight",facecolor=self.fig.get_facecolor())
            self._frame+=1
        try: plt.pause(0.001)
        except: pass

# ─────────────────────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────────────────────
def run():
    print("\n"+"═"*78)
    print("  ORGANISM LIVING SYSTEM v6  —  FULL CIVILIZATION AI")
    print("  HRL + WorldModel(Dreamer) + TransformerMemory + NEAT + PBT + Comm + Speciation")
    print("  Terrain + Disasters + Governance + Professions + Multi-Resource + Self-Play")
    print("═"*78)
    print(f"  World {CFG.world_w}×{CFG.world_h}  |  {CFG.terrain_tile}×{CFG.terrain_tile} terrain tiles")
    print(f"  RL stack: MAPPO+GRU{CFG.gru_hidden}+Transformer(L={CFG.mem_len},H={CFG.mem_heads})")
    print(f"          + HRL({CFG.n_goals} goals, {CFG.goal_horizon}s) + WorldModel({CFG.wm_latent_dim}d)")
    print(f"          + NEAT(pop={CFG.neat_pop_size}) + PBT×{CFG.pbt_population}")
    print(f"          + Communication({CFG.msg_dim}d) + AutoCurriculum + SelfPlay")
    print("═"*78+"\n")

    world=World()
    op=CivPolicy(CFG.org_obs_dim,    CFG.global_state_dim, name="org")
    wp=CivPolicy(CFG.warrior_obs_dim,CFG.global_state_dim, name="warrior")
    op_ref.append(op)
    rend=Renderer(); t0=time.time(); mo={}; mw={}; respawns=0

    for step in range(1,CFG.max_steps+1):
        info=world.step(op,wp)

        pop_before = sum(world.stats["pop"][-2:-1] or [info["pop"]])
        survival_frac = info["pop"]/(max(pop_before,1))

        if len(op.buf["obs"])>=CFG.rollout_len:
            mo=op.update(global_score=info["rew_o"],survival_frac=survival_frac)
        if len(wp.buf["obs"])>=CFG.rollout_len:
            mw=wp.update(global_score=info["rew_w"],survival_frac=survival_frac)

        # Periodic updates
        if step%CFG.pbt_interval==0:
            op.pbt_step(); wp.pbt_step()
        if step%CFG.neat_mutate_freq==0:
            op.neat_step(); wp.neat_step()
        if step%CFG.wm_train_freq==0:
            op.train_world_model(); wp.train_world_model()
        if step%CFG.selfplay_swap_interval==0:
            op.archive_policy(); wp.archive_policy()
        if step%CFG.curriculum_check_interval==0:
            diff=op.auto_curriculum()

        if step%50==0:
            el=time.time()-t0
            print(f"Step {step:5d} | "
                  f"Org {info['pop']:4d}(♂{info['male']:3d}♀{info['female']:3d}"
                  f"V{info['in_v']:3d}E{info['eld']:2d}) | "
                  f"War {info['war']:3d} | Enm {info['enm']:2d}(G{Enemy.generation}) | "
                  f"V{info['n_v']} Sp{info['n_species']} | Fd {info['food']:4d} | "
                  f"B{info['births']:2d} K{info['killed']:2d} Br{info['breach']:2d} "
                  f"EK{info['ekills']:2d} Tr{info['trade']:2d} | "
                  f"Curr{info['curriculum']:.2f} | "
                  f"OPL {mo.get('pl',0):+.3f} | {el:.0f}s")

        if step%CFG.render_every==0: rend.render(world,step)

        if not world.organisms:
            respawns+=1
            print(f"\n⚠  Extinction at step {step}. Respawn #{respawns}")
            if respawns<=10:
                for v in world.villages:
                    for k in range(20):
                        ang=random.uniform(0,2*math.pi); r=random.uniform(0,CFG.village_radius*0.7)
                        sex=Sex.MALE if k%2==0 else Sex.FEMALE
                        org=Organism(float(np.clip(v.x+r*math.cos(ang),0,CFG.world_w)),
                                     float(np.clip(v.y+r*math.sin(ang),0,CFG.world_h)),
                                     sex=sex,energy=CFG.init_energy*0.75,clan_id=v.clan_id,
                                     culture=v.culture.copy())
                        org.home_village=v.id; world.species_tracker.assign(org.genome)
                        world.organisms.append(org)
            else:
                print("   Too many extinctions."); break

        if not world.enemies and step<CFG.max_steps-300:
            world.enemies.append(Enemy(*world._rp()))

    rend.render(world,world.step_count)
    print("\n"+"═"*78)
    ao=world.organisms; aw=world.warriors
    males=[o for o in ao if o.sex==Sex.MALE]; females=[o for o in ao if o.sex==Sex.FEMALE]
    print(f"  Final: Orgs {len(ao)}(♂{len(males)}♀{len(females)}) | War {len(aw)} | Enm {len(world.enemies)}")
    print(f"  Villages {len(world.villages)} | Species {len(world.species_tracker.species)} | Enemy Gen {Enemy.generation}")
    if ao:
        gens=[o.generation for o in ao]
        print(f"  Gen avg {np.mean(gens):.2f} max {max(gens)}")
        print(f"  ♂ speed {np.mean([o.genome.speed for o in males]):.3f}  ♀ speed {np.mean([o.genome.speed for o in females]):.3f}")
        profs={p.name:sum(1 for o in ao if o.profession==p) for p in Profession}
        print(f"  Professions: {profs}")
        govs={g.name:sum(1 for v in world.villages if v.governance==g) for g in Governance}
        print(f"  Governance: {govs}")
    print(f"  Total births: {sum(world.stats['births_org'])}")
    print(f"  Total enemy kills: {sum(world.stats['enemy_kills'])}")
    print(f"  Total trades: {sum(world.stats['trade_trips'])}")
    print(f"  PBT best: C{op.active_pbt} score {op.pbt_configs[op.active_pbt].score:.2f}")
    print(f"  NEAT best score: {max(op.neat.scores):.3f}")
    print(f"  Archive snapshots: {len(op.archive)}")
    print("═"*78+"\n")
    fp=os.path.join(CFG.plot_dir,"final_summary_v6.png")
    rend.fig.savefig(fp,dpi=120,bbox_inches="tight",facecolor=rend.fig.get_facecolor())
    print(f"[Done] → {fp}")
    return world,op,wp,rend

if __name__=="__main__":
    # CFG.max_steps=2000; CFG.render_every=200  # quick test
    world,op,wp,rend=run()

[Device] cuda  |  Tesla T4  |  15.6 GB VRAM

══════════════════════════════════════════════════════════════════════════════
  ORGANISM LIVING SYSTEM v6  —  FULL CIVILIZATION AI
  HRL + WorldModel(Dreamer) + TransformerMemory + NEAT + PBT + Comm + Speciation
  Terrain + Disasters + Governance + Professions + Multi-Resource + Self-Play
══════════════════════════════════════════════════════════════════════════════
  World 240×160  |  16×16 terrain tiles
  RL stack: MAPPO+GRU128+Transformer(L=8,H=4)
          + HRL(6 goals, 20s) + WorldModel(32d)
          + NEAT(pop=8) + PBT×4
          + Communication(8d) + AutoCurriculum + SelfPlay
══════════════════════════════════════════════════════════════════════════════

  [Village+] (106,90) clan=1 governance=AUTOCRACY n=18 total=3
Step    50 | Org  163(♂ 79♀ 84V 63E 0) | War  67 | Enm  1(G1) | V3 Sp1 | Fd  135 | B 0 K 0 Br 0 EK 0 Tr 0 | Curr1.00 | OPL +0.000 | 20s
Step   100 | Org  178(♂ 85♀ 93V 72E 0) | War  87 | Enm  0(G1) | V3 Sp1 | Fd   54 |

/tmp/ipykernel_3196/3425277089.py:2273: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Step   150 | Org  187(♂ 90♀ 97V 73E 0) | War 100 | Enm  0(G1) | V3 Sp1 | Fd   40 | B 0 K 0 Br 0 EK 1 Tr 0 | Curr1.34 | OPL +0.000 | 53s
Step   200 | Org  196(♂ 92♀104V 74E 0) | War 100 | Enm  1(G1) | V3 Sp1 | Fd   34 | B 0 K 0 Br 0 EK 0 Tr 0 | Curr2.00 | OPL +0.034 | 69s
Step   250 | Org  188(♂ 87♀101V 61E 0) | War 100 | Enm  0(G1) | V3 Sp1 | Fd   31 | B 2 K 0 Br 0 EK 1 Tr 0 | Curr2.00 | OPL +0.000 | 88s
  [Disaster] FLOOD at (55,68) r=27
  [Village+] (215,35) clan=1 governance=ANARCHY n=18 total=4
Step   300 | Org  179(♂ 80♀ 99V 65E 0) | War 100 | Enm  0(G1) | V4 Sp1 | Fd   30 | B 0 K 0 Br 0 EK 1 Tr 0 | Curr2.00 | OPL +0.000 | 104s
Step   350 | Org  171(♂ 82♀ 89V 41E 0) | War 100 | Enm  1(G1) | V4 Sp1 | Fd   27 | B 1 K 0 Br 0 EK 0 Tr 0 | Curr2.00 | OPL +0.069 | 122s
Step   400 | Org  172(♂ 82♀ 90V 42E 0) | War 100 | Enm  1(G1) | V4 Sp1 | Fd   22 | B 0 K 0 Br 0 EK 0 Tr 0 | Curr2.00 | OPL +0.000 | 136s
Step   450 | Org  180(♂ 86♀ 94V 52E102) | War 100 | Enm  1(G1) | V4 Sp1 | Fd   16 | B

/tmp/ipykernel_3196/3425277089.py:496: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  obs   = torch.tensor([b[0] for b in batch], dtype=torch.float32, device=DEVICE)


Step   550 | Org  200(♂ 95♀105V 59E110) | War 100 | Enm  1(G2) | V4 Sp1 | Fd   17 | B 0 K 0 Br 0 EK 0 Tr 0 | Curr2.00 | OPL +0.000 | 191s
  [Village+] (217,137) clan=1 governance=DEMOCRACY n=21 total=5
  [NEAT] Evolved. Best score=215.319
  [NEAT] Evolved. Best score=277.730
Step   600 | Org  204(♂ 94♀110V 77E111) | War 100 | Enm  1(G2) | V5 Sp1 | Fd   15 | B 2 K 0 Br 0 EK 0 Tr 0 | Curr2.00 | OPL +0.049 | 207s
Step   650 | Org  201(♂ 94♀107V 74E114) | War 100 | Enm  1(G2) | V5 Sp1 | Fd   13 | B 0 K 0 Br 0 EK 0 Tr 0 | Curr2.00 | OPL +0.000 | 229s
Step   700 | Org  197(♂ 97♀100V 78E114) | War 100 | Enm  1(G2) | V5 Sp1 | Fd    8 | B 0 K 0 Br 0 EK 0 Tr 0 | Curr2.00 | OPL +0.000 | 245s
Step   750 | Org  196(♂ 98♀ 98V 71E115) | War 100 | Enm  1(G2) | V5 Sp1 | Fd   13 | B 1 K 0 Br 0 EK 0 Tr 0 | Curr2.00 | OPL +0.006 | 266s
  [Enemy+] (135,160) total=2
Step   800 | Org  197(♂100♀ 97V 64E117) | War 100 | Enm  2(G2) | V5 Sp1 | Fd   16 | B 0 K 0 Br 0 EK 0 Tr 2 | Curr2.00 | OPL +0.000 | 282s
Step 